In [1]:

# PHASE00: independent Windows Jupyter cell (no training)
import csv
import hashlib
import io
import json
import os
import platform
import shutil
import sys
import uuid
from datetime import datetime, timezone
from pathlib import Path

PROJECT_ID = "LUAD_TRIADIC_ORDINAL_2026"
ROOT = Path(r"D:\LUAD_TRIADIC_ORDINAL_2026")
PHASE = "PHASE00_PREFLIGHT"

# 1. Runtime and project-safety checks
if sys.version_info < (3, 10):
    raise RuntimeError("FAIL: Python 3.10+ is required.")

if os.name != "nt" or not Path("D:\\").is_dir():
    raise RuntimeError(
        "FAIL: Run in local Windows Jupyter with an available D: drive."
    )

identity_path = (
    ROOT / "00_admin_and_protocol"
    / "project_identity" / "PROJECT_ID.json"
)

if ROOT.exists() and any(ROOT.iterdir()):
    if not identity_path.is_file():
        raise RuntimeError(
            "FAIL: Project folder contains unrecognized content: "
            + str(ROOT)
        )
    prior = json.loads(identity_path.read_text(encoding="utf-8"))
    if prior.get("project_id") != PROJECT_ID:
        raise RuntimeError(
            "FAIL: Existing folder belongs to another project."
        )

# 2. Create a completely separate project
folders = [
    "00_admin_and_protocol/project_identity",
    "00_admin_and_protocol/research_protocol",
    "00_admin_and_protocol/decision_gates",
    "00_admin_and_protocol/run_manifests",
    "01_literature_and_novelty/literature_matrix",
    "01_literature_and_novelty/architecture_overlap",
    "02_raw_immutable/GEO",
    "02_raw_immutable/ArrayExpress",
    "02_raw_immutable/GDC",
    "02_raw_immutable/source_checksums",
    "03_metadata_and_label_evidence/sample_mapping",
    "03_metadata_and_label_evidence/mutation_ontology",
    "03_metadata_and_label_evidence/assay_coverage",
    "03_metadata_and_label_evidence/patient_independence",
    "03_metadata_and_label_evidence/external_eligibility",
    "04_preprocessing/train_only_processing",
    "04_preprocessing/gene_mapping",
    "04_preprocessing/ordinal_features",
    "04_preprocessing/pathway_programs",
    "05_architecture_development/mathematical_specification",
    "05_architecture_development/synthetic_tests",
    "05_architecture_development/ablation_contract",
    "06_baselines_and_development/cohort_splits",
    "06_baselines_and_development/baseline_models",
    "06_baselines_and_development/architecture_pilot",
    "07_external_evaluation/frozen_protocol",
    "07_external_evaluation/predictions",
    "08_biological_analysis/replication",
    "09_figures_and_tables",
    "10_manuscript",
    "11_final_audits",
    "notebooks",
]

for relative in folders:
    (ROOT / relative).mkdir(parents=True, exist_ok=True)

def write_exclusive(path, data):
    """Write a new artifact without overwriting existing work."""
    path.parent.mkdir(parents=True, exist_ok=True)
    blob = data.encode("utf-8") if isinstance(data, str) else data
    with path.open("xb") as f:
        f.write(blob)
        f.flush()
        os.fsync(f.fileno())
    return {
        "path": str(path.relative_to(ROOT)).replace("\\", "/"),
        "sha256": hashlib.sha256(blob).hexdigest(),
        "bytes": len(blob),
    }

def as_json(obj):
    return json.dumps(
        obj, ensure_ascii=False, indent=2, sort_keys=True
    ) + "\n"

def as_csv(rows, keys):
    stream = io.StringIO(newline="")
    writer = csv.DictWriter(
        stream,
        fieldnames=keys,
        lineterminator="\n",
        extrasaction="raise",
    )
    writer.writeheader()
    writer.writerows(rows)
    return stream.getvalue()

if not identity_path.exists():
    write_exclusive(identity_path, as_json({
        "project_id": PROJECT_ID,
        "root": str(ROOT),
        "purpose": "LUAD EGFR/KRAS/double-WT architecture research",
        "historical_projects_read_only": True,
        "created_utc": datetime.now(timezone.utc).isoformat(),
    }))

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run_id = stamp + "_" + uuid.uuid4().hex[:8]

run_dir = (
    ROOT / "00_admin_and_protocol"
    / "run_manifests" / run_id
)
run_dir.mkdir(parents=False, exist_ok=False)

# 3. Source-based dataset candidates
# Published sample counts are NOT final eligible modeling counts.
cohorts = [
    {
        "cohort": "GSE72094",
        "role": "DEVELOPMENT_CANDIDATE",
        "published_n": 442,
        "source_platform": "GPL15048 custom Affymetrix",
        "sample_scope": "LUAD tumor",
        "eligibility": "PENDING_BOTH_GENE_VERIFICATION",
        "rationale": "Large KRAS-associated LUAD cohort",
        "source_url":
            "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE72094",
    },
    {
        "cohort": "GSE31210",
        "role": "DEVELOPMENT_CANDIDATE",
        "published_n": 226,
        "source_platform": "GPL570 Affymetrix U133 Plus 2",
        "sample_scope": "226 primary LUAD among 246 GEO records",
        "eligibility": "PENDING_GSM_MAPPING_AND_ALK_AUDIT",
        "rationale": "EGFR-rich; ALK-positive handling required",
        "source_url":
            "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE31210",
    },
    {
        "cohort": "E-MTAB-923",
        "role": "DEVELOPMENT_CANDIDATE",
        "published_n": 99,
        "source_platform": "Affymetrix U133 Plus 2",
        "sample_scope": "Published mutation-annotated LUAD subset",
        "eligibility": "PENDING_PATIENT_ASSAY_MAPPING",
        "rationale": "Independent same-platform development candidate",
        "source_url":
            "https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0078614",
    },
    {
        "cohort": "GSE37745",
        "role": "DEVELOPMENT_CANDIDATE",
        "published_n": 106,
        "source_platform": "GPL570 Affymetrix U133 Plus 2",
        "sample_scope": "106 LUAD subset of 196 mixed NSCLC",
        "eligibility": "PENDING_EXACT_GSM_GENOTYPE_MAPPING",
        "rationale": "Additional LUAD development cohort",
        "source_url":
            "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE37745",
    },
    {
        "cohort": "GSE66863",
        "role": "PRIMARY_EXTERNAL_CANDIDATE_LOCKED",
        "published_n": 121,
        "source_platform": "GPL14550 Agilent 8x60K",
        "sample_scope": "121 LUAD expression records",
        "eligibility": "PENDING_BOTH_GENE_AND_INDEPENDENCE_AUDIT",
        "rationale": "Primary independent cross-platform test candidate",
        "source_url":
            "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE66863",
    },
    {
        "cohort": "GSE32863",
        "role": "SECONDARY_EXTERNAL_CANDIDATE_LOCKED",
        "published_n": 58,
        "source_platform": "GPL6884 Illumina HumanWG-6 v3",
        "sample_scope": "58 tumors plus 58 matched normal records",
        "eligibility": "PENDING_TUMOR_GENOTYPE_MAPPING",
        "rationale": "Small independent technology-shift test",
        "source_url":
            "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE32863",
    },
    {
        "cohort": "GSE13213",
        "role": "EXCLUDED_PRIMARY_TECHNOLOGY_DOMAIN",
        "published_n": 117,
        "source_platform": "Agilent two-color reference ratio",
        "sample_scope": "Published LUAD subset",
        "eligibility": "EXCLUDED_BY_ASSAY_SEMANTICS",
        "rationale": "Not equivalent to single-channel expression",
        "source_url":
            "https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0078614",
    },
    {
        "cohort": "GSE26939",
        "role": "EXCLUDED_PRIMARY_TECHNOLOGY_DOMAIN",
        "published_n": 85,
        "source_platform": "Agilent two-color reference ratio",
        "sample_scope": "Published subset; historical counts differ",
        "eligibility": "EXCLUDED_BY_ASSAY_SEMANTICS",
        "rationale": "Reference-ratio and count discrepancies",
        "source_url":
            "https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0078614",
    },
    {
        "cohort": "TCGA-LUAD",
        "role": "OPTIONAL_FUTURE_RNASEQ_DOMAIN",
        "published_n": "",
        "source_platform": "RNA-seq GDC",
        "sample_scope": "Matched primary tumor RNA/genotype required",
        "eligibility": "PENDING_ASSAY_AND_PATIENT_AUDIT",
        "rationale": "Future independently assessed RNA-seq domain",
        "source_url":
            "https://portal.gdc.cancer.gov/projects/TCGA-LUAD",
    },
]

# 4. Architectural literature and overlap register
literature = [
    {
        "work": "MetaKTSP",
        "year": 2016,
        "mechanism": "Cross-study top-scoring gene pairs",
        "overlap_risk": "HIGH_FEATURE_SELECTION",
        "verification": "METHOD_IDENTIFIED_CODE_NOT_AUDITED",
        "url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC6280887/",
    },
    {
        "work": "CPOP",
        "year": 2022,
        "mechanism": "Cross-platform relative expression",
        "overlap_risk": "HIGH_FEATURE_STABILITY",
        "verification": "METHOD_IDENTIFIED_CODE_NOT_AUDITED",
        "url": "https://doi.org/10.1038/s41746-022-00618-5",
    },
    {
        "work": "GAMI-Net",
        "year": 2021,
        "mechanism": "Structured main and pairwise effects",
        "overlap_risk": "PAIRWISE_COMPARATOR",
        "verification": "DETAILED_AUDIT_REQUIRED",
        "url": "https://doi.org/10.1016/j.patcog.2020.107702",
    },
    {
        "work": "StableMate",
        "year": 2024,
        "mechanism": "Stable predictors across environments",
        "overlap_risk": "HIGH_COHORT_CONSTRAINT",
        "verification": "METHOD_IDENTIFIED_CODE_NOT_AUDITED",
        "url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC11437361/",
    },
    {
        "work": "LogicFusion",
        "year": 2026,
        "mechanism": "Differentiable logical AND/OR rules",
        "overlap_risk": "CRITICAL_TRIADIC_LOGIC",
        "verification": "METHOD_IDENTIFIED_CODE_NOT_AUDITED",
        "url": "https://www.ijcai.org/proceedings/2026/609",
    },
    {
        "work": "FiLM",
        "year": 2018,
        "mechanism": "Feature-wise context conditioning",
        "overlap_risk": "CONDITIONAL_EQUIVALENCE",
        "verification": "DETAILED_AUDIT_REQUIRED",
        "url": "https://arxiv.org/abs/1709.07871",
    },
    {
        "work": "2026 LUAD mutation ensemble",
        "year": 2026,
        "mechanism": "Multi-platform driver mutation prediction",
        "overlap_risk": "DIRECT_TASK_COMPETITOR",
        "verification": "BENCHMARK_AND_LEAKAGE_AUDIT_REQUIRED",
        "url": "https://doi.org/10.3390/diagnostics16203413",
    },
]

# 5. Freeze the initial scientific contract
contract = """# LUAD Triadic Ordinal Research Contract

## Research question
Predict EGFR-mutant/KRAS-WT, KRAS-mutant/EGFR-WT
and EGFR/KRAS double-WT LUAD from bulk transcriptomes.

Both-gene mutation verification is mandatory.
Untested status is not wild type.
Double-WT is not genome-wide driver-negative.
Double mutants and ambiguous records require exclusion.
Normal tissue cannot enter tumor classification.
ALK-positive handling must be prespecified.

## Provisional architecture
A regularized hierarchical baseline plus compact
KRAS-specific higher-order ordinal residual learning.

Investigate constrained three-program gene-order motifs.

This is not yet established methodological novelty.

## Development cohort candidates
GSE72094
GSE31210
E-MTAB-923
GSE37745

Published candidate total: 873.
Actual genotype-complete eligible N is NOT verified.

## Locked external candidates
Primary: GSE66863.
Secondary: GSE32863.
Optional separate RNA-seq domain: TCGA-LUAD.

External cohort eligibility must be determined
using source evidence, not model outcomes.

## Provisional validation protocol
Use patient-level deduplication.
Verify individual EGFR and KRAS assay calls.
Use train-only feature selection and preprocessing.
Use four-cohort LOCO development where feasible.
Use group-aware nested tuning.
Disclose previously exposed cohort outcomes.
Do not interpret historical LOCO as untouched evidence.

## Architecture-specific comparators
Regularized multinomial logistic regression.
Regularized hierarchical baseline.
MetaKTSP/CPOP-style rank-based comparators.
Pairwise-only ordinal interaction model.
Same-feature generic triadic logic network.
Same-feature compact MLP.
Full proposed constrained triadic residual.
Matched random biological-program controls.

## Primary predictive endpoint
Equal-cohort mean paired multiclass log-score improvement
against the matched hierarchical baseline.

## Secondary endpoints
KRAS-vs-double-WT conditional NLL.
Macro-F1 and balanced accuracy.
Brier score and calibration.
Cohort stability.
Interaction reproducibility.
CPU runtime and parameter efficiency.

## Mandatory novelty gate
Verify exact mechanism against:
LogicFusion, MetaKTSP, CPOP, StableMate,
GAMI-Net, FiLM and generic higher-order networks.

No architecture novelty is asserted yet.

## Mandatory data gate
Verify source-to-sample genotype provenance,
patient independence, expression-platform semantics
and eligible external samples.

## Current authorization
NOVELTY_CONDITIONAL.
DATA_CONDITIONAL.
CPU_FEASIBLE_UNBENCHMARKED.
REAL_DATA_TRAINING_HOLD.

Do not select cohorts based on favorable observed scores.
Do not change primary endpoints after evaluating outcomes.
Do not guarantee predictive success or publication.
"""

# 6. Immutable research artifacts
recorded = []

recorded.append(write_exclusive(
    run_dir / "DATASET_FEASIBILITY_INVENTORY.csv",
    as_csv(cohorts, list(cohorts[0].keys()))
))

recorded.append(write_exclusive(
    run_dir / "LITERATURE_OVERLAP_REGISTER.csv",
    as_csv(literature, list(literature[0].keys()))
))

recorded.append(write_exclusive(
    run_dir / "RESEARCH_AND_VALIDATION_CONTRACT.md",
    contract
))

recorded.append(write_exclusive(
    run_dir / "DECISION_GATES.json",
    as_json({
        "novelty_status": "NOVELTY_CONDITIONAL",
        "data_status": "DATA_CONDITIONAL",
        "cpu_status": "CPU_FEASIBLE_UNBENCHMARKED",
        "training_authorized": False,
        "next_stage": "PHASE01_METHOD_AND_SOURCE_AUDIT",
        "selection_based_on_test_performance": False,
        "counts_independently_reproduced": False,
    })
))

# 7. Create a real new Jupyter notebook on D:
notebook_path = (
    ROOT / "notebooks" / "00_PHASE00_PREFLIGHT.ipynb"
)

notebook_created = not notebook_path.exists()

if notebook_created:
    starter_notebook = {
        "cells": [
            {
                "cell_type": "markdown",
                "id": uuid.uuid4().hex[:8],
                "metadata": {},
                "source": (
                    "# LUAD Triadic Ordinal Research\n\n"
                    "**Stage:** Phase 00 initialized.\n\n"
                    "**Model training:** HOLD.\n\n"
                    "**Root:** `D:\\LUAD_TRIADIC_ORDINAL_2026`\n\n"
                    "Research manifests are saved under "
                    "`00_admin_and_protocol/run_manifests/`.\n\n"
                    "Paste the next authorized standalone "
                    "research cell below after reviewing "
                    "the Phase 00 execution output."
                ),
            },
            {
                "cell_type": "code",
                "id": uuid.uuid4().hex[:8],
                "metadata": {},
                "execution_count": None,
                "outputs": [],
                "source": "",
            },
        ],
        "metadata": {
            "kernelspec": {
                "display_name": "Python 3",
                "language": "python",
                "name": "python3",
            },
            "language_info": {"name": "python"},
        },
        "nbformat": 4,
        "nbformat_minor": 5,
    }

    recorded.append(write_exclusive(
        notebook_path,
        as_json(starter_notebook)
    ))

# 8. SHA256 research manifest
free_gib = round(
    shutil.disk_usage(ROOT).free / (1024 ** 3), 2
)

manifest = {
    "project_id": PROJECT_ID,
    "run_id": run_id,
    "phase": PHASE,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "platform": platform.platform(),
    "python_version": sys.version.split()[0],
    "disk_free_gib": free_gib,
    "candidate_development_n_published": 873,
    "candidate_development_n_verified": None,
    "external_n_verified": None,
    "novelty_gate": "HOLD",
    "data_gate": "HOLD",
    "training_performed": False,
    "source_downloads_performed": False,
    "created_artifacts": recorded,
}

write_exclusive(
    run_dir / "SHA256_RUN_MANIFEST.json",
    as_json(manifest)
)

# 9. Final execution report
print("=" * 75)
print("PASS_SETUP: LUAD TRIADIC ORDINAL RESEARCH PROJECT")
print("=" * 75)

print("PROJECT_ROOT:", ROOT)
print("RUN_DIR:", run_dir)
print("NOTEBOOK:", notebook_path)
print("NOTEBOOK_CREATED:", notebook_created)

print(
    "DEVELOPMENT_CANDIDATES:",
    ", ".join(
        x["cohort"] for x in cohorts
        if x["role"] == "DEVELOPMENT_CANDIDATE"
    )
)

print("EXTERNAL_PRIMARY: GSE66863")
print("EXTERNAL_SECONDARY: GSE32863")
print("REPORTED_DEVELOPMENT_N: 873")
print("VERIFIED_ELIGIBLE_N: NOT YET VERIFIED")
print("FREE_D_DRIVE_GIB:", free_gib)

print("-" * 75)
print("NOVELTY_GATE: HOLD")
print("DATA_GATE: HOLD")
print("TRAINING_AUTHORIZED: NO")
print("LARGE_DOWNLOADS: NONE")
print("-" * 75)

print(
    "NEXT: Review the complete output, then proceed "
    "to the next source and novelty audit cell."
)
print("=" * 75)

PASS_SETUP: LUAD TRIADIC ORDINAL RESEARCH PROJECT
PROJECT_ROOT: D:\LUAD_TRIADIC_ORDINAL_2026
RUN_DIR: D:\LUAD_TRIADIC_ORDINAL_2026\00_admin_and_protocol\run_manifests\20261023T104101Z_ec62d20f
NOTEBOOK: D:\LUAD_TRIADIC_ORDINAL_2026\notebooks\00_PHASE00_PREFLIGHT.ipynb
NOTEBOOK_CREATED: True
DEVELOPMENT_CANDIDATES: GSE72094, GSE31210, E-MTAB-923, GSE37745
EXTERNAL_PRIMARY: GSE66863
EXTERNAL_SECONDARY: GSE32863
REPORTED_DEVELOPMENT_N: 873
VERIFIED_ELIGIBLE_N: NOT YET VERIFIED
FREE_D_DRIVE_GIB: 178.33
---------------------------------------------------------------------------
NOVELTY_GATE: HOLD
DATA_GATE: HOLD
TRAINING_AUTHORIZED: NO
LARGE_DOWNLOADS: NONE
---------------------------------------------------------------------------
NEXT: Review the complete output, then proceed to the next source and novelty audit cell.


In [2]:

import csv
import hashlib
import io
import itertools
import json
import os
import re
import uuid
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import quote
from urllib.request import Request, urlopen
from xml.etree import ElementTree as ET

import numpy as np

ROOT = Path(r"D:\LUAD_TRIADIC_ORDINAL_2026")
IDENTITY = ROOT / "00_admin_and_protocol/project_identity/PROJECT_ID.json"
MANIFEST_DIR = ROOT / "00_admin_and_protocol/run_manifests"

if os.name != "nt" or not Path("D:\\").is_dir():
    raise RuntimeError("Windows D: drive is unavailable")
if not IDENTITY.is_file():
    raise FileNotFoundError(IDENTITY)
if json.loads(IDENTITY.read_text(encoding="utf-8")).get("project_id") != "LUAD_TRIADIC_ORDINAL_2026":
    raise ValueError("Project identity mismatch")

phase0 = []
for p in sorted(MANIFEST_DIR.glob("*/SHA256_RUN_MANIFEST.json")):
    item = json.loads(p.read_text(encoding="utf-8"))
    if item.get("phase") == "PHASE00_PREFLIGHT":
        phase0.append((p, item))
if not phase0:
    raise RuntimeError("Phase 00 manifest not found")

prior_path, prior = phase0[-1]
for item in prior["created_artifacts"]:
    path = ROOT / item["path"]
    if not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest() != item["sha256"]:
        raise RuntimeError("Phase 00 checksum mismatch: " + str(path))

references = [
    ("MetaKTSP", "10.1093/bioinformatics/btw115", "metaktsp", "PMC6280887"),
    ("CPOP", "10.1038/s41746-022-00618-5", "cross-platform omics prediction", "PMC9253123"),
    ("GAMI-Net", "10.1016/j.patcog.2021.108192", "gami-net", None),
    ("StableMate", "10.1093/nargab/lqae130", "stablemate", "PMC11437361"),
    ("LogicFusion", "10.24963/ijcai.2026/609", "logicfusion", None),
    ("FiLM", "10.1609/aaai.v32i1.11671", "film", None),
    ("LUAD 2026 ensemble", "10.3390/diagnostics16203413", "multi-platform transcriptomic", None),
    ("Planck LUAD", "10.1371/journal.pone.0078614", "genomic and transcriptional", None),
]

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
RUN = ROOT / "01_literature_and_novelty/architecture_overlap" / run_id
RUN.mkdir(parents=True, exist_ok=False)
artifacts = []

def save(relative, payload):
    path = RUN / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    binary = payload if isinstance(payload, bytes) else payload.encode("utf-8")
    with path.open("xb") as f:
        f.write(binary)
    artifacts.append({
        "path": str(path.relative_to(ROOT)).replace("\\", "/"),
        "sha256": hashlib.sha256(binary).hexdigest(),
        "bytes": len(binary),
    })

def csv_text(rows, fields):
    stream = io.StringIO(newline="")
    writer = csv.DictWriter(stream, fieldnames=fields, lineterminator="\n")
    writer.writeheader()
    writer.writerows(rows)
    return stream.getvalue()

def fetch(url, max_bytes):
    req = Request(
        url,
        headers={
            "User-Agent": "LUADResearchAudit/1.0",
            "Accept": "application/json, application/xml, text/xml, */*",
        },
    )
    with urlopen(req, timeout=12) as response:
        data = response.read(max_bytes + 1)
    if len(data) > max_bytes:
        raise ValueError("Source exceeds size limit")
    return data

jobs = {}
with ThreadPoolExecutor(max_workers=3) as pool:
    for name, doi, expected, pmcid in references:
        url = "https://api.crossref.org/works/" + quote(doi, safe="")
        jobs[pool.submit(fetch, url, 1_500_000)] = (
            "metadata", name, doi, expected, url
        )
        if pmcid:
            xml_url = (
                "https://www.ebi.ac.uk/europepmc/webservices/rest/"
                + pmcid + "/fullTextXML"
            )
            jobs[pool.submit(fetch, xml_url, 5_000_000)] = (
                "fulltext", name, doi, pmcid, xml_url
            )

    retrieved = {}
    for future in as_completed(jobs):
        kind, name, doi, expected, url = jobs[future]
        try:
            retrieved[(kind, name)] = (future.result(), "", url)
        except (HTTPError, URLError, TimeoutError, ValueError, OSError) as exc:
            retrieved[(kind, name)] = (None, str(exc)[:250], url)

metadata_rows = []
excerpts = []

for name, doi, expected, pmcid in references:
    raw, error, url = retrieved[("metadata", name)]
    row = {
        "method": name,
        "doi": doi,
        "title": "",
        "year": "",
        "venue": "",
        "status": "NOT_RETRIEVED",
        "source_url": url,
        "error": error,
    }

    if raw is not None:
        try:
            record = json.loads(raw)["message"]
            title = re.sub(
                r"\s+", " ",
                re.sub(r"<[^>]*>", "", (record.get("title") or [""])[0])
            ).strip()
            returned_doi = record.get("DOI", "")

            if returned_doi.casefold() != doi.casefold():
                row["status"] = "DOI_MISMATCH"
            elif expected not in title.casefold():
                row["status"] = "TITLE_MISMATCH"
            else:
                row["status"] = "METADATA_VERIFIED"

            year_parts = (
                record.get("published") or record.get("issued") or {}
            ).get("date-parts", [[]])

            row.update(
                title=title,
                year=year_parts[0][0] if year_parts and year_parts[0] else "",
                venue="; ".join(record.get("container-title", [])),
            )
            save(
                "primary_metadata/" + name.replace(" ", "_") + ".json",
                raw,
            )
        except (KeyError, ValueError, TypeError, IndexError) as exc:
            row["status"] = "PARSE_FAILED"
            row["error"] = str(exc)[:250]

    metadata_rows.append(row)

    if pmcid:
        raw_xml, error_xml, fulltext_url = retrieved[("fulltext", name)]

        if raw_xml is None:
            excerpts.append({
                "method": name,
                "pmcid": pmcid,
                "section": "",
                "excerpt": "",
                "status": "NOT_RETRIEVED",
                "source_url": fulltext_url,
            })
            continue

        try:
            root = ET.fromstring(raw_xml)
            save("primary_fulltext/" + pmcid + ".xml", raw_xml)
            found = 0

            for sec in root.findall(".//sec"):
                heading = " ".join(
                    "".join(node.itertext()).strip()
                    for node in sec.findall("./title")
                ).strip()

                terms = (
                    "method", "algorithm", "approach",
                    "statistic", "predictor", "feature selection",
                )
                if not any(term in heading.casefold() for term in terms):
                    continue

                paragraphs = [
                    " ".join("".join(p.itertext()).split())
                    for p in sec.findall("./p")
                ]

                if paragraphs:
                    excerpts.append({
                        "method": name,
                        "pmcid": pmcid,
                        "section": heading[:200],
                        "excerpt": paragraphs[0][:700],
                        "status": "SOURCE_TEXT_EXTRACTED",
                        "source_url": fulltext_url,
                    })
                    found += 1

                if found >= 3:
                    break

            if found == 0:
                excerpts.append({
                    "method": name,
                    "pmcid": pmcid,
                    "section": "",
                    "excerpt": "",
                    "status": "NO_METHOD_SECTION_EXTRACTED",
                    "source_url": fulltext_url,
                })

        except ET.ParseError:
            excerpts.append({
                "method": name,
                "pmcid": pmcid,
                "section": "",
                "excerpt": "",
                "status": "XML_PARSE_FAILED",
                "source_url": fulltext_url,
            })

x = np.asarray(
    list(itertools.product([0.0, 1.0], repeat=3)),
    dtype=np.float64,
)
alpha = np.array([0.4, 0.7, 0.8], dtype=np.float64)
triadic = np.prod(1 - alpha * (1 - x), axis=1)

def design(max_degree):
    subsets = [
        subset
        for k in range(max_degree + 1)
        for subset in itertools.combinations(range(3), k)
    ]
    matrix = np.column_stack([
        np.prod(x[:, subset], axis=1)
        if subset else np.ones(len(x))
        for subset in subsets
    ])
    return matrix, subsets

pairwise_matrix, _ = design(2)
full_matrix, subsets = design(3)

coefficients = np.linalg.lstsq(
    full_matrix, triadic, rcond=None
)[0]

full_error = float(np.max(np.abs(
    full_matrix @ coefficients - triadic
)))

pairwise_error = float(np.linalg.norm(
    triadic - pairwise_matrix @ np.linalg.lstsq(
        pairwise_matrix, triadic, rcond=None
    )[0]
))

third_coefficient = float(
    coefficients[subsets.index((0, 1, 2))]
)
expected_coefficient = float(np.prod(alpha))

hard_and_error = float(np.max(np.abs(
    np.prod(x, axis=1)
    - np.prod(1 - 1.0 * (1 - x), axis=1)
)))

if (
    full_error > 1e-10
    or abs(third_coefficient - expected_coefficient) > 1e-10
    or pairwise_error < 1e-6
    or hard_and_error > 1e-10
):
    raise AssertionError("Triadic algebra test failed")

math_audit = {
    "proposed_rule": "prod_j [1 - alpha_j (1 - v_j)]",
    "exact_basis": (
        "constant + main + pairwise + three-way ordinal interactions"
    ),
    "polynomial_max_abs_error": full_error,
    "pairwise_only_l2_error": pairwise_error,
    "third_order_coefficient": third_coefficient,
    "expected_third_order_coefficient": expected_coefficient,
    "hard_and_max_abs_error": hard_and_error,
    "logical_conjunction_is_established": True,
    "cross_cohort_training_objective_defined": False,
    "novel_module_demonstrated": False,
    "novelty_status": "HOLD",
}

save(
    "REFERENCE_METADATA.csv",
    csv_text(metadata_rows, list(metadata_rows[0])),
)
save(
    "PRIMARY_METHOD_EXCERPTS.csv",
    csv_text(
        excerpts,
        ["method", "pmcid", "section", "excerpt", "status", "source_url"],
    ),
)
save(
    "TRIADIC_ALGEBRA_AUDIT.json",
    json.dumps(math_audit, indent=2) + "\n",
)

save(
    "ARCHITECTURE_OVERLAP.md",
    """# Architecture overlap

The triadic soft-conjunction module expands exactly into a polynomial containing main effects, pairwise interactions and a three-way interaction. At full selection strength, it equals ordinary Boolean AND on binary ordinal indicators. This establishes an interaction-order distinction from a strictly pairwise additive model, not novelty relative to existing logical-rule or higher-order networks.

MetaKTSP already combines gene-order predictors across studies. CPOP combines gene-expression ratios with cross-platform stability criteria. StableMate selects predictors stable across environments. LogicFusion uses differentiable AND/OR fusion of biological evidence. These are substantive overlap risks. Precise module equivalence cannot be inferred from abstracts or bibliographic metadata alone.

No explicit new cross-cohort constraint, gradient rule or training objective has yet been specified for the candidate. Novelty remains unverified. Real-data model training is not authorized.

GAMI-Net DOI correction: 10.1016/j.patcog.2021.108192.
""",
)

manifest = {
    "project_id": "LUAD_TRIADIC_ORDINAL_2026",
    "phase": "PHASE01A_ARCHITECTURE_AUDIT",
    "run_id": run_id,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "phase00_manifest": str(
        prior_path.relative_to(ROOT)
    ).replace("\\", "/"),
    "phase00_manifest_sha256": hashlib.sha256(
        prior_path.read_bytes()
    ).hexdigest(),
    "metadata_verified": sum(
        r["status"] == "METADATA_VERIFIED"
        for r in metadata_rows
    ),
    "references_total": len(metadata_rows),
    "source_method_sections_extracted": sum(
        r["status"] == "SOURCE_TEXT_EXTRACTED"
        for r in excerpts
    ),
    "mathematical_audit": "PASS",
    "novelty_gate": "HOLD",
    "data_gate": "HOLD",
    "training_performed": False,
    "artifacts": artifacts,
}

save(
    "RUN_MANIFEST.json",
    json.dumps(manifest, indent=2) + "\n",
)

source_status = (
    "PASS"
    if (
        manifest["metadata_verified"] == len(metadata_rows)
        and manifest["source_method_sections_extracted"] >= 3
    )
    else "HOLD"
)

print("PHASE01A:", source_status)
print(
    "REFERENCES:",
    manifest["metadata_verified"],
    "/",
    manifest["references_total"],
)
print(
    "METHOD_SECTIONS:",
    manifest["source_method_sections_extracted"],
)
print("ALGEBRA:", manifest["mathematical_audit"])
print("PAIRWISE_ERROR:", f"{pairwise_error:.8f}")
print("POLYNOMIAL_ERROR:", f"{full_error:.2e}")
print("NOVELTY:", manifest["novelty_gate"])
print("TRAINING:", "NO")
print("RUN:", RUN)

PHASE01A: PASS
REFERENCES: 8 / 8
METHOD_SECTIONS: 6
ALGEBRA: PASS
PAIRWISE_ERROR: 0.07919596
POLYNOMIAL_ERROR: 4.16e-16
NOVELTY: HOLD
TRAINING: NO
RUN: D:\LUAD_TRIADIC_ORDINAL_2026\01_literature_and_novelty\architecture_overlap\20261023T104838Z_e81ecea3


In [3]:

import hashlib
import importlib.util
import itertools
import json
import os
import re
import uuid
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import Request, urlopen
from xml.etree import ElementTree as ET

import numpy as np

ROOT = Path(r"D:\LUAD_TRIADIC_ORDINAL_2026")
IDENTITY = ROOT / "00_admin_and_protocol/project_identity/PROJECT_ID.json"
AUDIT_ROOT = ROOT / "01_literature_and_novelty/architecture_overlap"

if os.name != "nt" or not Path("D:\\").is_dir():
    raise RuntimeError("Windows D: drive unavailable")

if not IDENTITY.is_file():
    raise FileNotFoundError(IDENTITY)

identity = json.loads(IDENTITY.read_text(encoding="utf-8"))
if identity.get("project_id") != "LUAD_TRIADIC_ORDINAL_2026":
    raise ValueError("Project identity mismatch")

previous = []
for path in AUDIT_ROOT.glob("*/RUN_MANIFEST.json"):
    record = json.loads(path.read_text(encoding="utf-8"))
    if record.get("phase") == "PHASE01A_ARCHITECTURE_AUDIT":
        previous.append((path, record))

if not previous:
    raise RuntimeError("Phase 01A manifest not found")

previous_path, previous_record = sorted(previous)[-1]

if previous_record.get("mathematical_audit") != "PASS":
    raise RuntimeError("Phase 01A mathematical audit incomplete")

for artifact in previous_record["artifacts"]:
    path = ROOT / artifact["path"]
    if not path.is_file():
        raise FileNotFoundError(path)
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest != artifact["sha256"]:
        raise RuntimeError("Checksum mismatch: " + str(path))

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run_id += "_" + uuid.uuid4().hex[:8]

RUN = (
    ROOT
    / "01_literature_and_novelty"
    / "module_novelty_contract"
    / run_id
)
RUN.mkdir(parents=True, exist_ok=False)

artifacts = []

def save(name, content):
    path = RUN / name
    path.parent.mkdir(parents=True, exist_ok=True)
    data = content.encode("utf-8") if isinstance(content, str) else content
    with path.open("xb") as handle:
        handle.write(data)
    artifacts.append({
        "path": str(path.relative_to(ROOT)).replace("\\", "/"),
        "sha256": hashlib.sha256(data).hexdigest(),
        "bytes": len(data),
    })

def request_source(url, limit=5_000_000):
    request = Request(
        url,
        headers={
            "User-Agent": "LUAD-Research/1.0",
            "Accept": "application/xml,application/pdf,*/*",
        },
    )
    with urlopen(request, timeout=20) as response:
        data = response.read(limit + 1)
    if len(data) > limit:
        raise ValueError("Source exceeds size limit")
    return data

sources = {
    "TST_2009": {
        "url": (
            "https://www.ebi.ac.uk/europepmc/webservices/rest/"
            "PMC2745389/fullTextXML"
        ),
        "doi": "10.1186/1471-2105-10-256",
        "format": "xml",
    },
    "LOGICFUSION_2026": {
        "url": "https://ijcai.org/proceedings/2026/0609.pdf",
        "doi": "10.24963/ijcai.2026/609",
        "format": "pdf",
    },
}

source_results = {}

for name, source in sources.items():
    result = {
        "doi": source["doi"],
        "url": source["url"],
        "status": "NOT_VERIFIED",
        "sha256": None,
        "error": None,
    }

    try:
        data = request_source(source["url"])

        if source["format"] == "xml":
            root = ET.fromstring(data)
            dois = [
                (node.text or "").strip().lower()
                for node in root.findall(".//article-id")
                if node.get("pub-id-type") == "doi"
            ]
            text = " ".join(root.itertext())
            normalized = re.sub(r"[\W_]+", " ", text.casefold())

            required = [
                "top scoring triplet",
                "six possible orderings",
                "brca1",
            ]

            if source["doi"].lower() not in dois:
                raise ValueError("Original article DOI mismatch")
            if not all(term in normalized for term in required):
                raise ValueError("Required method evidence missing")

            result["status"] = "FULLTEXT_VERIFIED"
            result["evidence"] = required
            save("sources/TST_2009.xml", data)

        else:
            if not data.startswith(b"%PDF"):
                raise ValueError("Invalid PDF header")

            save("sources/LogicFusion_2026.pdf", data)
            result["status"] = "PDF_DOWNLOADED"

            if importlib.util.find_spec("fitz") is not None:
                import fitz

                with fitz.open(stream=data, filetype="pdf") as pdf:
                    text = "\n".join(page.get_text() for page in pdf)
                    result["pages"] = len(pdf)

                terms = [
                    "conjunction operator",
                    "disjunction operator",
                    "gumbel",
                    "rule generation",
                ]

                normalized = re.sub(
                    r"\s+", " ", text.casefold()
                )

                result["evidence"] = {
                    term: term in normalized
                    for term in terms
                }

                if all(result["evidence"].values()):
                    result["status"] = "METHOD_TEXT_VERIFIED"
                else:
                    result["status"] = "PDF_TEXT_INCOMPLETE"

                save("sources/LogicFusion_2026.txt", text)

        result["sha256"] = hashlib.sha256(data).hexdigest()

    except Exception as exc:
        result["error"] = str(exc)[:300]

    source_results[name] = result

alpha = np.array([0.4, 0.7, 0.8], dtype=np.float64)
bits = np.asarray(
    list(itertools.product((0, 1), repeat=3)),
    dtype=np.float64,
)

evidence = 1.0 - (1.0 - bits) * alpha
proposed_rule = np.prod(evidence, axis=1)
product_and = evidence[:, 0] * evidence[:, 1] * evidence[:, 2]

and_error = float(np.max(np.abs(
    proposed_rule - product_and
)))

pairwise_basis = np.column_stack([
    np.ones(len(bits)),
    bits,
    bits[:, 0] * bits[:, 1],
    bits[:, 0] * bits[:, 2],
    bits[:, 1] * bits[:, 2],
])

pairwise_fit = np.linalg.lstsq(
    pairwise_basis,
    proposed_rule,
    rcond=None,
)[0]

pairwise_error = float(np.linalg.norm(
    proposed_rule - pairwise_basis @ pairwise_fit
))

third_order = sum(
    (-1.0) ** (3 - int(b.sum())) * value
    for b, value in zip(bits, proposed_rule)
)

third_order = float(third_order)
expected_third_order = float(np.prod(alpha))

assert and_error < 1e-12
assert abs(third_order - expected_third_order) < 1e-12
assert pairwise_error > 1e-6

audit = {
    "candidate": "Cross-Cohort-Constrained Triadic Ordinal Residual",
    "previous_phase": str(previous_path.relative_to(ROOT)),
    "original_sources": source_results,
    "mathematical_results": {
        "product_and_equivalence_error": and_error,
        "pairwise_projection_error": pairwise_error,
        "third_order_coefficient": third_order,
        "expected_third_order_coefficient": expected_third_order,
    },
    "established_components": [
        "Gene-expression rank comparisons",
        "Top-scoring gene triplets",
        "Differentiable multiplicative conjunction",
        "Hierarchical conditional classification",
        "Cross-study stable predictor selection",
    ],
    "architecture_assessment": {
        "triadic_interaction_distinct_from_pairwise_only": True,
        "multiplicative_and_operator_new": False,
        "gene_order_triplets_new": False,
        "cross_cohort_learning_objective_specified": False,
        "unique_learning_mechanism_verified": False,
        "architecture_novelty_status": "NO_GO_AS_SPECIFIED",
        "evidence_driven_revision_available": True,
        "training_authorized": False,
    },
}

save(
    "MODULE_EQUIVALENCE_AUDIT.json",
    json.dumps(audit, indent=2) + "\n",
)

comparison = [
    {
        "method": "Top-Scoring Triplet",
        "year": 2009,
        "doi": "10.1186/1471-2105-10-256",
        "overlap": "Gene-expression ordering of three genes",
        "distinction": "Three separate pair indicators may use six genes",
    },
    {
        "method": "MetaKTSP",
        "year": 2016,
        "doi": "10.1093/bioinformatics/btw115",
        "overlap": "Cross-study stable ordinal gene-pair selection",
        "distinction": "Does not establish the proposed triadic module",
    },
    {
        "method": "GAMI-Net",
        "year": 2021,
        "doi": "10.1016/j.patcog.2021.108192",
        "overlap": "Structured interaction learning",
        "distinction": "Published architecture emphasizes pairwise effects",
    },
    {
        "method": "StableMate",
        "year": 2024,
        "doi": "10.1093/nargab/lqae130",
        "overlap": "Stable predictors across environments",
        "distinction": "Different selector and prediction formulation",
    },
    {
        "method": "LogicFusion",
        "year": 2026,
        "doi": "10.24963/ijcai.2026/609",
        "overlap": "Learned logical rule selection and AND/OR fusion",
        "distinction": "Different biological task and evidence model",
    },
]

import csv
import io

stream = io.StringIO(newline="")
writer = csv.DictWriter(
    stream,
    fieldnames=list(comparison[0].keys()),
    lineterminator="\n",
)
writer.writeheader()
writer.writerows(comparison)
save("METHOD_OVERLAP.csv", stream.getvalue())

decision = {
    "project_id": identity["project_id"],
    "phase": "PHASE01B_METHOD_OVERLAP",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "novelty": "NO_GO_AS_SPECIFIED",
    "data": "HOLD",
    "training": "HOLD",
    "remaining_architecture_revisions": 1,
    "authorized_work": "ONE_EVIDENCE_DRIVEN_REVISION_AUDIT",
    "artifacts": artifacts,
}

save(
    "DECISION.json",
    json.dumps(decision, indent=2) + "\n",
)

print(
    "PHASE01B:",
    "PASS" if all(
        value["status"] != "NOT_VERIFIED"
        for value in source_results.values()
    ) else "HOLD"
)
print("TST:", source_results["TST_2009"]["status"])
print("LOGICFUSION:", source_results["LOGICFUSION_2026"]["status"])
print("PRODUCT_AND_ERROR:", f"{and_error:.2e}")
print("PAIRWISE_ERROR:", f"{pairwise_error:.8f}")
print("THIRD_ORDER:", f"{third_order:.6f}")
print("NOVELTY:", "NO_GO_AS_SPECIFIED")
print("DATA:", "HOLD")
print("TRAINING:", "HOLD")
print("RUN:", RUN)

PHASE01B: PASS
TST: FULLTEXT_VERIFIED
LOGICFUSION: PDF_DOWNLOADED
PRODUCT_AND_ERROR: 0.00e+00
PAIRWISE_ERROR: 0.07919596
THIRD_ORDER: 0.224000
NOVELTY: NO_GO_AS_SPECIFIED
DATA: HOLD
TRAINING: HOLD
RUN: D:\LUAD_TRIADIC_ORDINAL_2026\01_literature_and_novelty\module_novelty_contract\20261023T105403Z_236d4f9b


In [4]:

import csv
import hashlib
import io
import itertools
import json
import os
import uuid
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

ROOT = Path(r"D:\LUAD_TRIADIC_ORDINAL_2026")
IDENTITY = ROOT / "00_admin_and_protocol/project_identity/PROJECT_ID.json"
AUDIT_ROOT = ROOT / "01_literature_and_novelty/module_novelty_contract"

if os.name != "nt" or not Path("D:\\").is_dir():
    raise RuntimeError("Windows D: drive unavailable")
if not IDENTITY.is_file():
    raise FileNotFoundError(IDENTITY)
if json.loads(IDENTITY.read_text(encoding="utf-8")).get("project_id") != "LUAD_TRIADIC_ORDINAL_2026":
    raise RuntimeError("Project identity mismatch")

prior_runs = []
for path in AUDIT_ROOT.glob("*/DECISION.json"):
    item = json.loads(path.read_text(encoding="utf-8"))
    if item.get("phase") == "PHASE01B_METHOD_OVERLAP":
        prior_runs.append((path, item))
    if item.get("phase") == "PHASE01C_FINAL_REVISION":
        raise RuntimeError("Phase 01C has already been recorded")
if not prior_runs:
    raise RuntimeError("Phase 01B decision not found")

prior_path, prior = sorted(prior_runs)[-1]
if prior.get("novelty") != "NO_GO_AS_SPECIFIED" or prior.get("remaining_architecture_revisions") != 1:
    raise RuntimeError("Phase 01B decision is incompatible")

for entry in prior.get("artifacts", []):
    source = ROOT / entry["path"]
    if not source.is_file():
        raise FileNotFoundError(source)
    if hashlib.sha256(source.read_bytes()).hexdigest() != entry["sha256"]:
        raise RuntimeError("SHA256 mismatch: " + str(source))

pdf = prior_path.parent / "sources/LogicFusion_2026.pdf"
if not pdf.is_file() or not pdf.read_bytes()[:4] == b"%PDF":
    raise RuntimeError("LogicFusion primary PDF missing or invalid")
pdf_sha256 = hashlib.sha256(pdf.read_bytes()).hexdigest()

bits = np.asarray(list(itertools.product((0.0, 1.0), repeat=3)), dtype=np.float64)

def lower_order(b):
    return np.column_stack((
        np.ones(len(b)), b,
        b[:, 0] * b[:, 1],
        b[:, 0] * b[:, 2],
        b[:, 1] * b[:, 2],
    ))

def project_triadic(b, weights):
    low = lower_order(b)
    raw = np.prod(b, axis=1)
    sw = np.sqrt(weights)
    design = low * sw[:, None]
    fit = raw * sw
    beta, _, rank, _ = np.linalg.lstsq(design, fit, rcond=None)
    residual = raw - low @ beta
    return beta, residual, rank, low

cohort_counts = np.array([1, 3, 8], dtype=np.int64)
cohort_features = np.concatenate([
    np.repeat(bits, n, axis=0) for n in cohort_counts
])
cohort_weights = np.concatenate([
    np.full(8 * n, 1.0 / (len(cohort_counts) * 8 * n))
    for n in cohort_counts
])

beta, residual, rank, low = project_triadic(
    cohort_features, cohort_weights
)
ortho_error = float(np.max(np.abs(
    low.T @ (cohort_weights * residual)
)))
variance = float(np.dot(
    cohort_weights, residual * residual
))
reconstruction_error = float(np.max(np.abs(
    np.prod(cohort_features, axis=1) - (low @ beta + residual)
)))

coeff = np.array([0.2, -0.1, 0.3, 0.08, -0.12, 0.04, 0.07])
theta = 0.6
raw = np.prod(cohort_features, axis=1)
reparameterization_error = float(np.max(np.abs(
    (low @ coeff + theta * residual)
    - (low @ (coeff - theta * beta) + theta * raw)
)))

reduced = bits[:-1]
reduced_beta, reduced_residual, reduced_rank, _ = project_triadic(
    reduced, np.full(len(reduced), 1.0 / len(reduced))
)
reduced_variance = float(np.mean(reduced_residual ** 2))

def stable_gate(delta, kras_counts, wt_counts, minimum=15):
    delta = np.asarray(delta, dtype=np.float64)
    kras_counts = np.asarray(kras_counts, dtype=np.int64)
    wt_counts = np.asarray(wt_counts, dtype=np.int64)
    if delta.ndim != 1 or len(delta) < 2:
        return 0
    if delta.shape != kras_counts.shape or delta.shape != wt_counts.shape:
        return 0
    return int(
        np.all(np.isfinite(delta))
        and np.all(delta > 0.0)
        and np.all(kras_counts >= minimum)
        and np.all(wt_counts >= minimum)
    )

counts_k = [20, 24, 31]
counts_w = [55, 80, 42]
stable = stable_gate(
    [0.025, 0.012, 0.018], counts_k, counts_w
)
reversal = stable_gate(
    [0.025, -0.012, 0.018], counts_k, counts_w
)
low_support = stable_gate(
    [0.025, 0.012, 0.018], [20, 8, 31], counts_w
)

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

p_egfr = sigmoid(-0.4)
p_k0 = sigmoid(-0.2)
p_k1 = sigmoid(0.9)
probs0 = np.array([
    p_egfr,
    (1 - p_egfr) * p_k0,
    (1 - p_egfr) * (1 - p_k0),
])
probs1 = np.array([
    p_egfr,
    (1 - p_egfr) * p_k1,
    (1 - p_egfr) * (1 - p_k1),
])

checks = {
    "lower_order_rank": int(rank),
    "weighted_orthogonality_max_error": ortho_error,
    "weighted_residual_variance": variance,
    "raw_product_reconstruction_max_error": reconstruction_error,
    "raw_product_reparameterization_max_error": reparameterization_error,
    "missing_pattern_lower_order_rank": int(reduced_rank),
    "missing_pattern_residual_variance": reduced_variance,
    "consistent_cohort_gate": stable,
    "inconsistent_cohort_gate": reversal,
    "low_support_cohort_gate": low_support,
    "egfr_head_delta": float(abs(probs0[0] - probs1[0])),
    "multiclass_sum_error": float(max(
        abs(probs0.sum() - 1),
        abs(probs1.sum() - 1),
    )),
}

passed = (
    rank == 7
    and ortho_error < 1e-12
    and variance > 1e-4
    and reconstruction_error < 1e-12
    and reparameterization_error < 1e-12
    and reduced_rank == 7
    and reduced_variance < 1e-18
    and (stable, reversal, low_support) == (1, 0, 0)
    and checks["egfr_head_delta"] < 1e-14
    and checks["multiclass_sum_error"] < 1e-14
)
if not passed:
    raise AssertionError(
        "Phase 01C structural test failed: " + json.dumps(checks)
    )

run_id = (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    + "_" + uuid.uuid4().hex[:8]
)
run = AUDIT_ROOT / run_id
run.mkdir(parents=True, exist_ok=False)
outputs = []

def store(filename, value):
    path = run / filename
    payload = value.encode("utf-8") if isinstance(value, str) else value
    with path.open("xb") as handle:
        handle.write(payload)
    outputs.append({
        "path": str(path.relative_to(ROOT)).replace("\\", "/"),
        "sha256": hashlib.sha256(payload).hexdigest(),
        "bytes": len(payload),
    })

contract = """# Final bounded revision: replicated pure triadic residual

## Status
This is a task-specific architecture extension, not an original logical operator or a verified new ML algorithm. The sole allowable architecture revision has been used. Novelty remains conditional on source-code comparison and mechanism-specific empirical evidence.

## Input and contrast
Each candidate comprises three prespecified, distinct ordinal gene-pair indicators b1, b2, b3 from biologically separate expression programs. All candidates, pathway definitions, gene mapping and ordering conventions are selected within the relevant development partition. The external cohorts cannot affect selection.

Only genotype-complete LUAD tumors enter the primary target. Head A predicts EGFR-mutant vs non-EGFR. Head B predicts KRAS-mutant vs EGFR/KRAS double-WT, conditional on non-EGFR.

## Residual layer
For each selected motif t(b)=b1*b2*b3, define L(b)=[1,b1,b2,b3,b1*b2,b1*b3,b2*b3]. Estimate beta=argmin ||sqrt(W)*(t-L*beta)||^2 using the training partition only, with inverse-cohort-size weights W. Set r(b)=t(b)-L(b)*beta. Train-time r is empirically orthogonal to all seven lower-order columns. At inference, compute r with frozen beta.

Empirical orthogonality need not hold under domain shift. If the baseline contains unrestricted lower-order terms, replacing the raw product by the purified residual is an exact reparameterization in function space; its rationale is regularization, interpretability and selection constraints, not additional expressivity.

A rank-7 lower-order design with nonzero residual variance is mandatory; missing states or near-zero variance can invalidate an interaction. No claim of causal independence follows.

## Cohort replication rule
Inside each outer training partition, estimate per-motif cross-fitted, cohort-wise KRAS-conditional log-score gain over the same-input hierarchical baseline. Never use outer-held-out labels for motif selection, residualization, replication gates or tuning.

A fixed gate g_h is 1 only when all eligible inner development cohorts have strictly positive incremental log score and adequate class support. The provisional support floor is 15 KRAS and 15 double-WT samples per source cohort; it must be frozen before any phenotype-dependent audit. A gate of 0 disables the interaction.

The screening procedure uses held-out development labels and must itself be nested inside the outer evaluation. This is conservative screening, not a statistical certificate of true transportability.

## Probability heads
qE=sigmoid(fE(z))
qK=sigmoid(fK0(z)+sum_h g_h*theta_h*r_h(b))
P(EGFR)=qE; P(KRAS)=(1-qE)*qK; P(double-WT)=(1-qE)*(1-qK).

The correction cannot modify qE when the heads are architecturally separate. All motifs and parameters are frozen before a designated external evaluation.

## Distinguishing hypothesis
The proposed inductive bias requires all of the following: KRAS-contrast localization, explicit exclusion of lower-order ordinal components, and fold-internal replication filtering. Each ingredient has prior art. The combined mechanism is a provisional, application-specific extension; it does not establish global functional novelty over generic networks.

## Mandatory comparators
Hierarchical linear model; same triadic raw-product residual without purification; purified residual without cohort gate; cohort gate without purification; matched gene-pair and GAMI-like predictors; matched generic trainable AND network including LogicFusion-style logic when reproducible; StableMate/MetaKTSP/CPOP; matched compact MLP; random pathway groups.

All use the same expression information and comparable tuning budgets.

## Falsification
Reject a methodological superiority claim if gains disappear against matched generic rule networks, cannot be replicated across development cohorts, depend on sparse/unobserved ordinal states, arise only from unbalanced cohort sizes, or fail external validation.

A null residual or all-zero gates means the method reduces to its conventional baseline and does not substantiate the proposed modification.

## Evidence requirements
LogicFusion IJCAI 2026 doi:10.24963/ijcai.2026/609
TST BMC Bioinformatics 2009 doi:10.1186/1471-2105-10-256
MetaKTSP Bioinformatics 2016 doi:10.1093/bioinformatics/btw115
StableMate NAR Genom Bioinform 2024 doi:10.1093/nargab/lqae130
Lengerich et al., AISTATS 2020, PMLR 108:2402-2412
Risk Extrapolation ICML 2021, PMLR 139:5815-5826

## Authorization
Novelty: CONDITIONAL_REVISED.
Dataset audit: AUTHORIZED.
Real-data fitting and model selection: HOLD.
Remaining architecture revisions: 0.
"""

literature = [
    (
        "LogicFusion",
        "10.24963/ijcai.2026/609",
        "Probabilistic AND/OR and learnable view selection",
        "HIGH",
        "Logical rule operator is established",
    ),
    (
        "TST",
        "10.1186/1471-2105-10-256",
        "Gene-order triplet classification",
        "HIGH",
        "Three-gene ordering is established",
    ),
    (
        "MetaKTSP",
        "10.1093/bioinformatics/btw115",
        "Cross-study ordinal feature selection",
        "HIGH",
        "Stability across studies is established",
    ),
    (
        "StableMate",
        "10.1093/nargab/lqae130",
        "Environment-stable predictors",
        "HIGH",
        "Stable predictor selection is established",
    ),
    (
        "Pure interactions",
        "PMLR-v108-lengerich20a",
        "Functional ANOVA purification",
        "HIGH",
        "Orthogonalization is established",
    ),
    (
        "Risk Extrapolation",
        "PMLR-v139-krueger21a",
        "Environment-wise robustness",
        "MODERATE",
        "Domain stability alone is established",
    ),
]

stream = io.StringIO(newline="")
writer = csv.writer(stream, lineterminator="\n")
writer.writerow([
    "method",
    "identifier",
    "nearest_mechanism",
    "overlap",
    "limitation_of_novelty_claim",
])
writer.writerows(literature)

store("REVISION_METHOD_CONTRACT.md", contract)
store("OVERLAP_EVIDENCE.csv", stream.getvalue())
store(
    "STRUCTURAL_TESTS.json",
    json.dumps(checks, indent=2) + "\n",
)

status = {
    "project_id": "LUAD_TRIADIC_ORDINAL_2026",
    "phase": "PHASE01C_FINAL_REVISION",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "previous_decision": str(
        prior_path.relative_to(ROOT)
    ).replace("\\", "/"),
    "logicfusion_pdf_sha256": pdf_sha256,
    "math_and_structure": "PASS",
    "novelty": "CONDITIONAL_REVISED",
    "remaining_architecture_revisions": 0,
    "dataset_source_audit_authorized": True,
    "real_data_training_authorized": False,
    "evaluation_outcomes_inspected": False,
    "artifacts": outputs,
}

store(
    "DECISION.json",
    json.dumps(status, indent=2) + "\n",
)

print("PHASE01C: PASS")
print("ORTHOGONALITY:", f"{ortho_error:.2e}")
print("RESIDUAL_VARIANCE:", f"{variance:.8f}")
print("REPARAM_ERROR:", f"{reparameterization_error:.2e}")
print("MISSING_PATTERN_VARIANCE:", f"{reduced_variance:.2e}")
print("COHORT_GATES:", stable, reversal, low_support)
print(
    "MULTICLASS_SUM_ERROR:",
    f"{checks['multiclass_sum_error']:.2e}",
)
print("NOVELTY: CONDITIONAL_REVISED")
print("DATA_AUDIT: AUTHORIZED")
print("TRAINING: HOLD")
print("RUN:", run)

PHASE01C: PASS
ORTHOGONALITY: 1.03e-16
RESIDUAL_VARIANCE: 0.01562500
REPARAM_ERROR: 3.33e-16
MISSING_PATTERN_VARIANCE: 0.00e+00
COHORT_GATES: 1 0 0
MULTICLASS_SUM_ERROR: 0.00e+00
NOVELTY: CONDITIONAL_REVISED
DATA_AUDIT: AUTHORIZED
TRAINING: HOLD
RUN: D:\LUAD_TRIADIC_ORDINAL_2026\01_literature_and_novelty\module_novelty_contract\20261023T110852Z_4f17630b


In [5]:

import csv
import hashlib
import io
import json
import os
import re
import uuid
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen

ROOT = Path(r"D:\LUAD_DATAFIRST_METHOD_2026")
PROJECT_ID = "LUAD_DATAFIRST_METHOD_2026"

GEO = {
    "GSE72094": 442,
    "GSE31210": 246,
    "GSE37745": 196,
    "GSE66863": 121,
    "GSE32863": 116,
}

EXTERNAL = {"GSE66863", "GSE32863"}

if os.name != "nt" or not Path("D:\\").is_dir():
    raise RuntimeError("Windows D: drive unavailable")

identity = ROOT / "00_admin/project.json"

if ROOT.exists() and any(ROOT.iterdir()) and not identity.is_file():
    raise RuntimeError("Unrecognized existing project")

identity.parent.mkdir(parents=True, exist_ok=True)

if identity.is_file():
    stored = json.loads(identity.read_text(encoding="utf-8"))
    if stored.get("project_id") != PROJECT_ID:
        raise RuntimeError("Project identity mismatch")
else:
    with identity.open("x", encoding="utf-8") as f:
        json.dump({
            "project_id": PROJECT_ID,
            "created_utc": datetime.now(timezone.utc).isoformat()
        }, f)

run_id = (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    + "_" + uuid.uuid4().hex[:8]
)

RUN = ROOT / "01_data_audit" / run_id
RUN.mkdir(parents=True, exist_ok=False)
artifacts = []

def save(filename, content):
    path = RUN / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    data = content.encode("utf-8") if isinstance(content, str) else content

    with path.open("xb") as f:
        f.write(data)

    artifacts.append({
        "path": str(path.relative_to(ROOT)).replace("\\", "/"),
        "sha256": hashlib.sha256(data).hexdigest(),
        "bytes": len(data),
    })

def csv_string(rows, fields):
    out = io.StringIO(newline="")
    writer = csv.DictWriter(
        out,
        fieldnames=fields,
        extrasaction="ignore",
        lineterminator="\n"
    )
    writer.writeheader()
    writer.writerows(rows)
    return out.getvalue()

def retrieve(url, limit=22_000_000):
    req = Request(url, headers={
        "User-Agent": "LUAD-DataAudit/1.0",
        "Accept-Encoding": "identity",
    })

    with urlopen(req, timeout=22) as response:
        blob = response.read(limit + 1)

    if len(blob) > limit or blob[:200].lstrip().lower().startswith(
        (b"<html", b"<!doctype html")
    ):
        raise ValueError("Unexpected metadata response")

    return blob

def read_soft(blob):
    records = []
    current = None

    for line in blob.decode("utf-8-sig", "replace").splitlines():
        if line.startswith("^SAMPLE = "):
            if current is not None:
                records.append(current)

            current = {
                "gsm": line.partition("=")[2].strip(),
                "fields": defaultdict(list),
            }

        elif current is not None and line.startswith("!Sample_") and "=" in line:
            key, _, value = line[1:].partition("=")
            current["fields"][key.strip().casefold()].append(value.strip())

    if current is not None:
        records.append(current)

    return records

def scalar(rec, key):
    return " | ".join(rec["fields"].get(key.casefold(), []))

def characteristics(rec):
    return [
        tuple(map(str.strip, value.split(":", 1)))
        for value in rec["fields"].get("sample_characteristics_ch1", [])
        if ":" in value
    ]

def canon(value):
    return re.sub(r"[^a-z0-9]", "", value.casefold())

def status(value):
    text = re.sub(r"[_-]+", " ", value.casefold()).strip()

    if re.search(
        r"unknown|not tested|untested|not available|"
        r"not assessed|missing|\bn/?a\b",
        text
    ):
        return "UNKNOWN"

    neg = bool(re.search(
        r"\b(wt|wild type|negative|not mutated|"
        r"no mutation|not detected)\b",
        text
    )) or text in {"no", "0"}

    pos = bool(re.search(
        r"\b(mut|mutant|mutated|positive|mutation detected|"
        r"mutation present|l858r|g12[a-z]|g13[a-z])\b",
        text
    )) or text in {"yes", "1"}

    if pos and neg:
        return "CONFLICT"
    if pos:
        return "MUT"
    if neg:
        return "WT"

    return "UNKNOWN"

def mutation(pairs, gene):
    keys = {
        gene + suffix
        for suffix in (
            "", "status", "mutation", "mutationalstatus",
            "mutationstatus", "genotype"
        )
    }

    matches = [
        (key, value)
        for key, value in pairs
        if canon(key) in keys
    ]

    if not matches:
        return "NOT_REPORTED", ""

    calls = {status(value) for _, value in matches}

    result = (
        next(iter(calls))
        if len(calls) == 1
        else "CONFLICT"
    )

    evidence = " | ".join(
        f"{key}: {value}"
        for key, value in matches
    )

    return result, evidence

def histology(rec, pairs):
    keys = {
        "tissue", "histology", "diagnosis",
        "sampletype", "tissuetype"
    }

    text = " ".join(
        [
            scalar(rec, "sample_title"),
            scalar(rec, "sample_source_name_ch1"),
        ]
        + [
            value
            for key, value in pairs
            if canon(key) in keys
        ]
    ).casefold()

    if re.search(r"\b(normal|healthy|adjacent normal)\b", text):
        return "NORMAL"

    if re.search(r"\b(squamous|small cell|large cell)\b", text):
        return "OTHER_HISTOLOGY"

    if re.search(r"\b(adenocarcinoma|adeno|luad)\b", text):
        return "LUAD_REPORTED"

    return "UNVERIFIED"

samples = []
summary = []
ids_by_gsm = defaultdict(set)

for accession, published_n in GEO.items():
    url = (
        "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?"
        + urlencode({
            "acc": accession,
            "targ": "gsm",
            "view": "brief",
            "form": "text",
        })
    )

    records = []
    error = ""

    try:
        raw = retrieve(url)
        records = read_soft(raw)

        if not records or any(
            not record["gsm"].startswith("GSM")
            for record in records
        ):
            raise ValueError("Invalid GEO SOFT records")

        save(f"geo/{accession}.soft", raw)

    except Exception as exc:
        records = []
        error = f"{type(exc).__name__}: {exc}"[:180]

    counts = Counter()
    seen_patients = Counter()

    for rec in records:
        pairs = characteristics(rec)

        egfr, egfr_evidence = mutation(pairs, "egfr")
        kras, kras_evidence = mutation(pairs, "kras")
        stk11, _ = mutation(pairs, "stk11")
        tp53, _ = mutation(pairs, "tp53")
        alk, _ = mutation(pairs, "alk")

        tissue = histology(rec, pairs)

        if (egfr, kras) == ("MUT", "WT"):
            genotype = "EGFR"
        elif (egfr, kras) == ("WT", "MUT"):
            genotype = "KRAS"
        elif (egfr, kras) == ("WT", "WT"):
            genotype = "DWT"
        else:
            genotype = "OTHER"

        patient = next((
            value
            for key, value in pairs
            if canon(key) in {
                "patientid", "patientnumber", "subjectid"
            }
        ), "")

        if patient:
            seen_patients[patient] += 1

        ids_by_gsm[rec["gsm"]].add(accession)

        counts["class_" + genotype] += 1
        counts["histology_" + tissue] += 1

        if (
            tissue == "LUAD_REPORTED"
            and genotype != "OTHER"
            and alk != "MUT"
        ):
            counts["provisional"] += 1

        if (
            stk11 in {"WT", "MUT"}
            and tp53 in {"WT", "MUT"}
        ):
            counts["comut_complete"] += 1

        samples.append({
            "cohort": accession,
            "gsm": rec["gsm"],
            "patient_id": patient,
            "platform": scalar(rec, "sample_platform_id"),
            "histology": tissue,
            "egfr": egfr,
            "kras": kras,
            "stk11": stk11,
            "tp53": tp53,
            "alk": alk,
            "genotype_candidate": genotype,
            "egfr_evidence": egfr_evidence,
            "kras_evidence": kras_evidence,
            "characteristics_json": json.dumps(
                pairs, ensure_ascii=False
            ),
        })

    unique_gsm = len({rec["gsm"] for rec in records})

    summary.append({
        "cohort": accession,
        "role": (
            "external"
            if accession in EXTERNAL
            else "development"
        ),
        "expected_geo_n": published_n,
        "parsed_n": len(records),
        "metadata_status": (
            "PASS"
            if unique_gsm == published_n
            and len(records) == published_n
            else "HOLD"
        ),
        "egfr_candidates": counts["class_EGFR"],
        "kras_candidates": counts["class_KRAS"],
        "dwt_candidates": counts["class_DWT"],
        "unresolved": counts["class_OTHER"],
        "luad_reported": counts["histology_LUAD_REPORTED"],
        "provisional_eligible": counts["provisional"],
        "stk11_tp53_complete": counts["comut_complete"],
        "duplicated_patient_ids": sum(
            n - 1 for n in seen_patients.values()
            if n > 1
        ),
        "error": error,
    })

previous_roots = [
    Path(r"D:\LUAD_DISCORDANCE_FRAMEWORK_2026"),
    Path(r"D:\LUAD_SINGLE_CHANNEL_MUTATION_V2_2026"),
    Path(r"D:\LUAD_TRIADIC_ORDINAL_2026"),
    Path(r"D:\LUAD_RCOR_2026"),
]

pattern = re.compile(
    r"mutation|genotype|label|phenotype|clinical|"
    r"sample|expression|expr|matrix|cohort",
    re.I
)

local = []

for base in previous_roots:
    if not base.is_dir():
        continue

    examined = 0

    for current, dirs, files in os.walk(base):
        depth = len(Path(current).relative_to(base).parts)

        dirs[:] = [
            directory
            for directory in dirs
            if depth < 4
            and not directory.startswith(".")
            and directory not in {
                "__pycache__", "node_modules"
            }
        ]

        for name in files:
            examined += 1

            if examined > 8000 or len(local) >= 100:
                break

            path = Path(current) / name

            if (
                path.suffix.lower() in {
                    ".csv", ".tsv", ".parquet",
                    ".npy", ".npz", ".feather"
                }
                and pattern.search(name)
                and not re.search(
                    r"oof|result|metric|prediction",
                    name, re.I
                )
            ):
                try:
                    local.append({
                        "path": str(path),
                        "size_mb": round(
                            path.stat().st_size / 1e6, 2
                        ),
                    })
                except OSError:
                    pass

        if examined > 8000 or len(local) >= 100:
            break

save(
    "sample_metadata.csv",
    csv_string(samples, [
        "cohort", "gsm", "patient_id", "platform",
        "histology", "egfr", "kras", "stk11",
        "tp53", "alk", "genotype_candidate",
        "egfr_evidence", "kras_evidence",
        "characteristics_json",
    ])
)

save(
    "cohort_summary.csv",
    csv_string(summary, list(summary[0]))
)

save(
    "local_source_inventory.csv",
    csv_string(local, ["path", "size_mb"])
)

shared = {
    key: sorted(value)
    for key, value in ids_by_gsm.items()
    if len(value) > 1
}

save(
    "shared_gsm.json",
    json.dumps(shared, indent=2)
)

ebi_status = "NOT_RETRIEVED"
ebi_error = ""

try:
    data = retrieve(
        "https://www.ebi.ac.uk/biostudies/api/v1/studies/E-MTAB-923",
        5_000_000
    )

    study = json.loads(data)

    if study.get(
        "accno", study.get("accession")
    ) not in ("E-MTAB-923", None):
        raise ValueError("BioStudies accession mismatch")

    save("ebi/E-MTAB-923.json", data)
    ebi_status = "STUDY_METADATA_RETRIEVED"

except Exception as exc:
    ebi_error = f"{type(exc).__name__}: {exc}"[:180]

notebook = ROOT / "notebooks/00_DATA_FIRST.ipynb"
notebook.parent.mkdir(parents=True, exist_ok=True)

if not notebook.exists():
    with notebook.open("x", encoding="utf-8") as f:
        json.dump({
            "cells": [
                {
                    "cell_type": "markdown",
                    "metadata": {},
                    "source": [
                        "# LUAD Data-First Method Development\n"
                    ],
                },
                {
                    "cell_type": "code",
                    "metadata": {},
                    "execution_count": None,
                    "outputs": [],
                    "source": [],
                },
            ],
            "metadata": {
                "kernelspec": {
                    "display_name": "Python 3",
                    "language": "python",
                    "name": "python3",
                }
            },
            "nbformat": 4,
            "nbformat_minor": 5,
        }, f, indent=2)

manifest = {
    "project_id": PROJECT_ID,
    "phase": "SOURCE_AND_LABEL_FEASIBILITY",
    "run_id": run_id,
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "source_summary": summary,
    "ebi": {
        "status": ebi_status,
        "error": ebi_error,
    },
    "local_source_candidates": len(local),
    "architecture": "UNSELECTED",
    "labels_final_verified": False,
    "external_test_metrics_inspected": False,
    "training_performed": False,
    "artifacts": artifacts,
}

save(
    "run_manifest.json",
    json.dumps(manifest, indent=2) + "\n"
)

print(
    "SOURCE_AUDIT:",
    "PASS" if all(
        row["metadata_status"] == "PASS"
        for row in summary
    ) else "HOLD"
)

for row in summary:
    print(
        row["cohort"],
        "N", row["parsed_n"],
        "EGFR", row["egfr_candidates"],
        "KRAS", row["kras_candidates"],
        "DWT", row["dwt_candidates"],
        "LUAD", row["luad_reported"],
        row["metadata_status"],
        row["error"],
    )

print("E-MTAB-923:", ebi_status, ebi_error)
print("SHARED_GSM:", len(shared))
print("LOCAL_DATA_FILES:", len(local))
print("ARCHITECTURE: UNSELECTED")
print("RUN:", RUN)
print("NOTEBOOK:", notebook)

SOURCE_AUDIT: PASS
GSE72094 N 442 EGFR 47 KRAS 154 DWT 241 LUAD 442 PASS 
GSE31210 N 246 EGFR 0 KRAS 0 DWT 0 LUAD 0 PASS 
GSE37745 N 196 EGFR 0 KRAS 0 DWT 0 LUAD 106 PASS 
GSE66863 N 121 EGFR 17 KRAS 39 DWT 58 LUAD 121 PASS 
GSE32863 N 116 EGFR 17 KRAS 22 DWT 19 LUAD 58 PASS 
E-MTAB-923: STUDY_METADATA_RETRIEVED 
SHARED_GSM: 0
LOCAL_DATA_FILES: 33
ARCHITECTURE: UNSELECTED
RUN: D:\LUAD_DATAFIRST_METHOD_2026\01_data_audit\20261023T115223Z_820fa53c
NOTEBOOK: D:\LUAD_DATAFIRST_METHOD_2026\notebooks\00_DATA_FIRST.ipynb


In [6]:

import csv
import hashlib
import io
import json
import os
import re
import uuid
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

ROOT = Path(r"D:\LUAD_DATAFIRST_METHOD_2026")
PROJECT_ID = "LUAD_DATAFIRST_METHOD_2026"
IDENTITY = ROOT / "00_admin/project.json"
AUDIT_ROOT = ROOT / "01_data_audit"

if os.name != "nt" or not Path("D:\\").is_dir():
    raise RuntimeError("Windows D: drive unavailable")

if not IDENTITY.is_file():
    raise FileNotFoundError(IDENTITY)

identity = json.loads(IDENTITY.read_text(encoding="utf-8"))
if identity.get("project_id") != PROJECT_ID:
    raise RuntimeError("Project identity mismatch")

previous_runs = []

for path in AUDIT_ROOT.glob("*/run_manifest.json"):
    record = json.loads(path.read_text(encoding="utf-8"))
    if record.get("phase") == "SOURCE_AND_LABEL_FEASIBILITY":
        previous_runs.append((path, record))

if not previous_runs:
    raise RuntimeError("Source audit manifest not found")

previous_path, previous = sorted(previous_runs)[-1]
source_run = previous_path.parent

for artifact in previous["artifacts"]:
    path = ROOT / artifact["path"]
    if not path.is_file():
        raise FileNotFoundError(path)
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest != artifact["sha256"]:
        raise RuntimeError("Source audit checksum mismatch")

inventory_path = source_run / "local_source_inventory.csv"
sample_path = source_run / "sample_metadata.csv"

with inventory_path.open("r", encoding="utf-8-sig", newline="") as f:
    inventory = list(csv.DictReader(f))

with sample_path.open("r", encoding="utf-8-sig", newline="") as f:
    samples = list(csv.DictReader(f))

gsm_cohort = {
    row["gsm"]: row["cohort"]
    for row in samples
    if row.get("gsm")
}

known_gsm = set(gsm_cohort)
title_cohort = defaultdict(set)

for row in samples:
    for token in re.findall(
        r"\b(?:KH\d{2,5}|GSM\d+)\b",
        row.get("title", ""),
        flags=re.I,
    ):
        title_cohort[token.upper()].add(row["cohort"])

allowed_roots = [
    Path(r"D:\LUAD_DISCORDANCE_FRAMEWORK_2026"),
    Path(r"D:\LUAD_SINGLE_CHANNEL_MUTATION_V2_2026"),
    Path(r"D:\LUAD_TRIADIC_ORDINAL_2026"),
    Path(r"D:\LUAD_RCOR_2026"),
]

allowed_roots = [p.resolve() for p in allowed_roots]

def csv_schema(path):
    result = {
        "format": path.suffix.lower().lstrip("."),
        "rows": "",
        "columns": "",
        "shape": "",
        "dtype": "",
        "gsm_column_hits": 0,
        "gsm_row_examples": 0,
        "label_columns": "",
        "id_columns": "",
        "column_preview": "",
        "row_preview": "",
        "status": "UNRESOLVED",
    }

    with path.open(
        "r", encoding="utf-8-sig",
        errors="replace", newline=""
    ) as f:
        header = f.readline(4_000_000)

        if not header:
            result["status"] = "EMPTY_FILE"
            return result

        if len(header) >= 4_000_000 and not header.endswith(
            ("\n", "\r")
        ):
            result["status"] = "HEADER_LIMIT"
            return result

        delimiter = max(
            (",", "\t", ";"),
            key=lambda d: header.count(d)
        )

        if header.count(delimiter) == 0:
            result["status"] = "DELIMITER_UNRESOLVED"
            return result

        names = next(
            csv.reader([header], delimiter=delimiter)
        )

        first_values = []
        for _ in range(4):
            line = f.readline(1_000_000)
            if not line:
                break
            try:
                fields = next(
                    csv.reader([line], delimiter=delimiter)
                )
                if fields:
                    first_values.append(fields[0].strip())
            except csv.Error:
                break

    result["columns"] = len(names)
    result["column_preview"] = " | ".join(names[:9])
    result["row_preview"] = " | ".join(first_values[:4])

    result["gsm_column_hits"] = sum(
        name.strip().upper() in known_gsm
        for name in names
    )

    result["gsm_row_examples"] = sum(
        value.upper() in known_gsm
        for value in first_values
    )

    label_pattern = re.compile(
        r"(?i)(egfr|kras).*(mutation|mutant|"
        r"genotype|status|call)|"
        r"(mutation|genotype|status).*(egfr|kras)"
    )

    labels = [
        name for name in names
        if label_pattern.search(name)
    ]

    ids = [
        name for name in names
        if re.search(
            r"(?i)(^gsm$|sample.?id|patient.?id|"
            r"subject.?id|source.?name|^geo.?accession$)",
            name
        )
    ]

    result["label_columns"] = " | ".join(labels[:12])
    result["id_columns"] = " | ".join(ids[:12])
    result["status"] = "SCHEMA_READ"

    return result

def binary_schema(path):
    suffix = path.suffix.lower()

    result = {
        "format": suffix.lstrip("."),
        "rows": "",
        "columns": "",
        "shape": "",
        "dtype": "",
        "gsm_column_hits": 0,
        "gsm_row_examples": 0,
        "label_columns": "",
        "id_columns": "",
        "column_preview": "",
        "row_preview": "",
        "status": "UNRESOLVED",
    }

    if suffix == ".npy":
        arr = np.load(
            path, mmap_mode="r", allow_pickle=False
        )
        result["shape"] = str(tuple(arr.shape))
        result["dtype"] = str(arr.dtype)
        result["rows"] = arr.shape[0] if arr.ndim else ""
        result["columns"] = arr.shape[1] if arr.ndim >= 2 else ""
        result["status"] = "METADATA_READ"
        del arr

    elif suffix == ".npz":
        with zipfile.ZipFile(path) as archive:
            entries = [
                item for item in archive.infolist()
                if item.filename.endswith(".npy")
            ]
            result["columns"] = len(entries)
            result["column_preview"] = " | ".join(
                item.filename for item in entries[:10]
            )
            result["status"] = "ARCHIVE_INDEX_READ"

    elif suffix == ".parquet":
        import pyarrow.parquet as pq

        reader = pq.ParquetFile(path)
        result["rows"] = reader.metadata.num_rows
        result["columns"] = len(reader.schema_arrow.names)
        result["column_preview"] = " | ".join(
            reader.schema_arrow.names[:10]
        )
        result["gsm_column_hits"] = sum(
            name.upper() in known_gsm
            for name in reader.schema_arrow.names
        )
        result["status"] = "METADATA_READ"

    elif suffix == ".feather":
        import pyarrow as pa
        import pyarrow.ipc as ipc

        with pa.memory_map(str(path), "r") as source:
            reader = ipc.open_file(source)
            names = reader.schema.names

            result["columns"] = len(names)
            result["column_preview"] = " | ".join(
                names[:10]
            )
            result["gsm_column_hits"] = sum(
                name.upper() in known_gsm
                for name in names
            )
            result["status"] = "SCHEMA_READ"

    return result

rows = []

for item in inventory:
    filename = item.get("path", "")
    if not filename:
        continue

    path = Path(filename)

    row = {
        "path": str(path),
        "cohort_hint": "",
        "size_mb": "",
        "format": path.suffix.lower().lstrip("."),
        "rows": "",
        "columns": "",
        "shape": "",
        "dtype": "",
        "gsm_column_hits": 0,
        "gsm_row_examples": 0,
        "label_columns": "",
        "id_columns": "",
        "column_preview": "",
        "row_preview": "",
        "status": "NOT_INSPECTED",
        "error": "",
    }

    try:
        resolved = path.resolve(strict=True)

        if not any(
            resolved.is_relative_to(base)
            for base in allowed_roots
        ):
            raise ValueError("Source outside approved roots")

        if not resolved.is_file():
            raise ValueError("Source is not a regular file")

        row["size_mb"] = round(
            resolved.stat().st_size / 1_000_000, 2
        )

        cohort_hits = sorted(set(re.findall(
            r"(?i)(GSE\d{4,7}|E-MTAB-\d+|TCGA.LUAD)",
            str(resolved)
        )))

        row["cohort_hint"] = " | ".join(cohort_hits)

        if resolved.suffix.lower() in (".csv", ".tsv"):
            details = csv_schema(resolved)

        elif resolved.suffix.lower() in (
            ".npy", ".npz", ".parquet", ".feather"
        ):
            details = binary_schema(resolved)

        else:
            details = {"status": "UNSUPPORTED_FORMAT"}

        row.update(details)

    except Exception as exc:
        row["status"] = "INSPECTION_FAILED"
        row["error"] = (
            f"{type(exc).__name__}: {exc}"
        )[:240]

    rows.append(row)

cohort_records = []

for cohort in sorted(set(x["cohort"] for x in samples)):
    group = [x for x in samples if x["cohort"] == cohort]

    tissue_counter = Counter(
        x["histology"] for x in group
    )

    source_counter = Counter(
        re.sub(
            r"\s+", " ",
            x.get("source", "").casefold().strip()
        )
        for x in group
    )

    genotype_counter = Counter(
        x["genotype_candidate"] for x in group
    )

    kras_records = [
        x for x in group
        if x["genotype_candidate"] == "KRAS"
    ]

    cohort_records.append({
        "cohort": cohort,
        "records": len(group),
        "egfr_candidates": genotype_counter["EGFR"],
        "kras_candidates": genotype_counter["KRAS"],
        "dwt_candidates": genotype_counter["DWT"],
        "unresolved": genotype_counter["OTHER"],
        "reported_luad": tissue_counter["LUAD_REPORTED"],
        "reported_normal": tissue_counter["NORMAL"],
        "kras_stk11_reported": sum(
            x["stk11"] in ("WT", "MUT")
            for x in kras_records
        ),
        "kras_tp53_reported": sum(
            x["tp53"] in ("WT", "MUT")
            for x in kras_records
        ),
        "source_name_summary": json.dumps(
            source_counter.most_common(5),
            ensure_ascii=False
        ),
    })

run_id = (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    + "_" + uuid.uuid4().hex[:8]
)

RUN = ROOT / "02_source_resolution" / run_id
RUN.mkdir(parents=True, exist_ok=False)

artifacts = []

def save(name, content):
    path = RUN / name
    payload = (
        content.encode("utf-8")
        if isinstance(content, str)
        else content
    )

    with path.open("xb") as f:
        f.write(payload)

    artifacts.append({
        "path": str(path.relative_to(ROOT)).replace("\\", "/"),
        "sha256": hashlib.sha256(payload).hexdigest(),
        "bytes": len(payload),
    })

def make_csv(data, columns):
    buffer = io.StringIO(newline="")
    writer = csv.DictWriter(
        buffer, fieldnames=columns,
        extrasaction="ignore", lineterminator="\n"
    )
    writer.writeheader()
    writer.writerows(data)
    return buffer.getvalue()

fields = [
    "path", "cohort_hint", "size_mb", "format",
    "rows", "columns", "shape", "dtype",
    "gsm_column_hits", "gsm_row_examples",
    "label_columns", "id_columns",
    "column_preview", "row_preview",
    "status", "error",
]

save(
    "LOCAL_SCHEMA_INVENTORY.csv",
    make_csv(rows, fields)
)

save(
    "GEO_COHORT_EVIDENCE.csv",
    make_csv(
        cohort_records,
        list(cohort_records[0].keys())
    )
)

readable = sum(
    r["status"] in {
        "SCHEMA_READ", "METADATA_READ",
        "ARCHIVE_INDEX_READ"
    }
    for r in rows
)

manifest = {
    "project_id": PROJECT_ID,
    "phase": "LOCAL_SCHEMA_AND_SOURCE_RESOLUTION",
    "run_id": run_id,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "parent_manifest": str(
        previous_path.relative_to(ROOT)
    ).replace("\\", "/"),
    "parent_sha256": hashlib.sha256(
        previous_path.read_bytes()
    ).hexdigest(),
    "files_listed": len(rows),
    "files_readable": readable,
    "candidate_gsm_linked_files": sum(
        bool(r["gsm_column_hits"])
        or bool(r["gsm_row_examples"])
        for r in rows
    ),
    "genotype_labels_final_verified": False,
    "expression_values_loaded": False,
    "model_fitting_performed": False,
    "architecture_frozen": False,
    "artifacts": artifacts,
}

save(
    "RUN_MANIFEST.json",
    json.dumps(manifest, indent=2) + "\n"
)

print(
    "SOURCE_RESOLUTION:",
    "PASS" if readable == len(rows) and rows else "PARTIAL"
)
print("LOCAL_FILES:", len(rows))
print("READABLE_SCHEMAS:", readable)
print("GSM_LINKED_FILES:", manifest["candidate_gsm_linked_files"])

for c in cohort_records:
    print(
        c["cohort"],
        "N=", c["records"],
        "EGFR=", c["egfr_candidates"],
        "KRAS=", c["kras_candidates"],
        "DWT=", c["dwt_candidates"],
        "STK11_KRAS=", c["kras_stk11_reported"],
        "TP53_KRAS=", c["kras_tp53_reported"],
    )

print("FILE_INVENTORY:")

for i, row in enumerate(rows, 1):
    name = Path(row["path"]).name

    print(
        f"{i:02d}",
        name[:65],
        "SHAPE=", row["shape"] or (
            "cols:" + str(row["columns"])
        ),
        "GSM=", row["gsm_column_hits"],
        "LABELS=", row["label_columns"][:65] or "-",
        row["status"],
    )

print("RUN:", RUN)

SOURCE_RESOLUTION: PASS
LOCAL_FILES: 33
READABLE_SCHEMAS: 33
GSM_LINKED_FILES: 0
GSE31210 N= 246 EGFR= 0 KRAS= 0 DWT= 0 STK11_KRAS= 0 TP53_KRAS= 0
GSE32863 N= 116 EGFR= 17 KRAS= 22 DWT= 19 STK11_KRAS= 0 TP53_KRAS= 0
GSE37745 N= 196 EGFR= 0 KRAS= 0 DWT= 0 STK11_KRAS= 0 TP53_KRAS= 0
GSE66863 N= 121 EGFR= 17 KRAS= 39 DWT= 58 STK11_KRAS= 0 TP53_KRAS= 38
GSE72094 N= 442 EGFR= 47 KRAS= 154 DWT= 241 STK11_KRAS= 154 TP53_KRAS= 154
FILE_INVENTORY:
01 CELL01B_UNVERIFIED_PROVISIONAL_LABELS.csv SHAPE= cols:16 GSM= 0 LABELS= egfr_status_metadata | kras_status_metadata SCHEMA_READ
02 CELL01G_FOUR_COHORT_CANDIDATES_NOT_FINAL.csv SHAPE= cols:16 GSM= 0 LABELS= - SCHEMA_READ
03 CELL01G_NONCANDIDATE_SAMPLE_AUDIT.csv SHAPE= cols:3 GSM= 0 LABELS= - SCHEMA_READ
04 CELL02B_SAMPLE_NUMERIC_QC.csv SHAPE= cols:11 GSM= 0 LABELS= - SCHEMA_READ
05 CELL02I_GSE26939_SAMPLE_ORDER.csv SHAPE= cols:4 GSM= 0 LABELS= - SCHEMA_READ
06 CELL02I_GSE31210_SAMPLE_ORDER.csv SHAPE= cols:4 GSM= 0 LABELS= - SCHEMA_READ
07 CELL02I_GS

In [7]:

import csv
import gzip
import hashlib
import io
import itertools
import json
import os
import re
import time
import uuid
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

ROOT = Path(r"D:\LUAD_DATAFIRST_METHOD_2026")
IDENTITY = ROOT / "00_admin/project.json"
PREVIOUS = ROOT / "02_source_resolution"

if os.name != "nt" or not Path("D:/").is_dir():
    raise RuntimeError("Windows D: drive unavailable")

if not IDENTITY.is_file():
    raise FileNotFoundError(IDENTITY)

identity = json.loads(IDENTITY.read_text(encoding="utf-8"))
if identity.get("project_id") != "LUAD_DATAFIRST_METHOD_2026":
    raise RuntimeError("Project identity mismatch")

prior = sorted(PREVIOUS.glob("*/RUN_MANIFEST.json"))
if not prior:
    raise RuntimeError("Previous source inventory not found")

prior_path = prior[-1]
prior_manifest = json.loads(prior_path.read_text(encoding="utf-8"))

if prior_manifest.get("phase") != "LOCAL_SCHEMA_AND_SOURCE_RESOLUTION":
    raise RuntimeError("Unexpected previous phase")

for entry in prior_manifest["artifacts"]:
    p = ROOT / entry["path"]
    if not p.is_file():
        raise FileNotFoundError(p)
    if hashlib.sha256(p.read_bytes()).hexdigest() != entry["sha256"]:
        raise RuntimeError("Previous audit checksum mismatch")

with (prior_path.parent / "LOCAL_SCHEMA_INVENTORY.csv").open(
    "r", encoding="utf-8-sig", newline=""
) as f:
    old_files = list(csv.DictReader(f))

geo_files = sorted(
    (ROOT / "01_data_audit").glob("*/sample_metadata.csv")
)
if not geo_files:
    raise RuntimeError("GEO sample mapping not found")

with geo_files[-1].open(
    "r", encoding="utf-8-sig", newline=""
) as f:
    known_gsm = {
        row["gsm"].upper()
        for row in csv.DictReader(f)
        if row.get("gsm")
    }

priority = {
    "LUAD_DISCORDANCE_FRAMEWORK_2026",
    "LUAD_SINGLE_CHANNEL_MUTATION_V2_2026",
    "LUAD_PDRF_2026_v003",
    "LUAD_RIGDR_2026",
}

roots = [
    p for p in Path("D:/").iterdir()
    if p.is_dir()
    and p.name.upper().startswith("LUAD_")
    and p.resolve() != ROOT.resolve()
]

roots.sort(
    key=lambda p: (
        0 if p.name in priority else 1,
        p.name.lower(),
    )
)

formats = {
    ".csv", ".tsv", ".txt", ".gz",
    ".parquet", ".feather", ".npy", ".npz",
    ".rds", ".rdata", ".rda", ".h5",
    ".hdf5", ".mtx",
}

name_re = re.compile(
    r"expr|expression|matrix|assay|feature|"
    r"rank|normalized|normalised|processed|"
    r"microarray|gene|gse\d+|metab|"
    r"sample.?order|mapping|mutation|"
    r"genotype|clinical|label",
    re.I,
)

derived_re = re.compile(
    r"prediction|calibrat|metric|replay|"
    r"sensitivity|meta.?fusion|result|"
    r"checkpoint|bootstrap",
    re.I,
)

start = time.monotonic()
scan_limit = 60000
seconds_limit = 40.0

examined = 0
candidates = []
limited = False

for base in roots:
    for folder, dirs, names in os.walk(
        base, followlinks=False
    ):
        depth = len(Path(folder).relative_to(base).parts)

        dirs[:] = [
            d for d in sorted(dirs)
            if depth < 9
            and d not in {
                ".git", ".venv", "venv",
                "node_modules", "__pycache__",
                ".ipynb_checkpoints", "site-packages",
            }
        ]

        for name in names:
            examined += 1

            if (
                examined > scan_limit
                or time.monotonic() - start > seconds_limit
            ):
                limited = True
                break

            p = Path(folder) / name

            if p.suffix.lower() not in formats or p.is_symlink():
                continue

            try:
                size = p.stat().st_size
            except OSError:
                continue

            if size == 0:
                continue

            score = 0

            if name_re.search(name):
                score += 7

            if re.search(
                r"GSE\d{4,7}|E.MTAB.923",
                str(p), re.I
            ):
                score += 5

            if p.suffix.lower() in {
                ".npy", ".npz", ".parquet",
                ".feather", ".rds", ".rdata",
                ".rda", ".h5", ".hdf5",
            }:
                score += 4

            if size > 2_000_000:
                score += 3

            if size > 20_000_000:
                score += 2

            if derived_re.search(name):
                score -= 9

            candidates.append((score, size, str(p)))

        if limited:
            break

    if limited:
        break

original_paths = {
    str(Path(item["path"]))
    for item in old_files
    if item.get("path")
}

shortlist = sorted(
    candidates,
    key=lambda item: (-item[0], -item[1], item[2])
)[:200]

selected = sorted(
    original_paths | {item[2] for item in shortlist}
)

npy_header = np.lib.format

def npy_meta(file_obj):
    version = npy_header.read_magic(file_obj)

    if version == (1, 0):
        shape, fortran, dtype = (
            npy_header.read_array_header_1_0(file_obj)
        )
    else:
        shape, fortran, dtype = (
            npy_header.read_array_header_2_0(file_obj)
        )

    return tuple(int(v) for v in shape), str(dtype)

def inspect(path):
    info = {
        "path": str(path),
        "size_mb": round(path.stat().st_size / 1e6, 3),
        "format": "".join(path.suffixes[-2:]).lower(),
        "shape": "",
        "columns": 0,
        "column_preview": "",
        "id_preview": "",
        "gsm_hits": 0,
        "label_fields": "",
        "status": "UNINSPECTED",
        "source_type": "UNRESOLVED",
        "error": "",
    }

    suffix = path.suffix.lower()

    if suffix in {".csv", ".tsv", ".txt", ".gz", ".mtx"}:
        opener = gzip.open if suffix == ".gz" else open

        with opener(
            path, "rt", encoding="utf-8-sig",
            errors="replace", newline=""
        ) as f:
            first = f.readline(2_000_000)

            if not first or (
                len(first) >= 2_000_000
                and not first.endswith(("\r", "\n"))
            ):
                raise ValueError("Missing or oversized first row")

            delim = max(
                ("\t", ",", ";"),
                key=first.count
            )

            if first.count(delim) == 0:
                info["status"] = "NO_TABULAR_HEADER"
                return info

            reader = csv.reader(
                itertools.chain([first], f),
                delimiter=delim
            )

            header = next(reader)
            preview = list(itertools.islice(reader, 24))

        info["columns"] = len(header)
        info["column_preview"] = " | ".join(
            header[:8]
        )[:280]

        info["id_preview"] = " | ".join(
            r[0] for r in preview[:5] if r
        )[:180]

        text = first + " ".join(
            " ".join(r[:16])
            for r in preview
        )

        tokens = set(re.findall(
            r"GSM\d+", text, re.I
        ))

        info["gsm_hits"] = len(
            {v.upper() for v in tokens} & known_gsm
        )

        fields = [
            v for v in header
            if re.search(
                r"(?i)egfr|kras|mutation|genotype|"
                r"patient.?id|sample.?id|geo.?accession",
                v
            )
        ]

        info["label_fields"] = " | ".join(
            fields[:8]
        )[:280]

        if len(header) >= 80 or (
            len(header) >= 30
            and path.stat().st_size > 1_000_000
        ):
            info["source_type"] = (
                "EXPRESSION_OR_FEATURE_MATRIX"
            )

        elif fields or re.search(
            r"sample.?order|mutation|genotype|"
            r"clinical|mapping|label",
            path.name, re.I
        ):
            info["source_type"] = (
                "MAPPING_OR_LABEL_CANDIDATE"
            )

        info["status"] = "HEADER_READ"

    elif suffix == ".npy":
        with path.open("rb") as f:
            shape, dtype = npy_meta(f)

        info["shape"] = str(shape) + " " + dtype

        if (
            len(shape) == 2
            and min(shape) >= 30
            and max(shape) >= 200
        ):
            info["source_type"] = (
                "EXPRESSION_OR_FEATURE_MATRIX"
            )

        info["status"] = "ARRAY_HEADER_READ"

    elif suffix == ".npz":
        with zipfile.ZipFile(path) as z:
            arrays = []

            for member in z.namelist()[:12]:
                if not member.endswith(".npy"):
                    continue

                with z.open(member) as f:
                    shape, dtype = npy_meta(f)

                arrays.append(
                    member + ":" + str(shape) + ":" + dtype
                )

                if (
                    len(shape) == 2
                    and min(shape) >= 30
                    and max(shape) >= 200
                ):
                    info["source_type"] = (
                        "EXPRESSION_OR_FEATURE_MATRIX"
                    )

        info["shape"] = " | ".join(
            arrays[:6]
        )[:350]

        info["status"] = "ARCHIVE_HEADER_READ"

    elif suffix == ".parquet":
        import pyarrow.parquet as pq

        pf = pq.ParquetFile(path)
        names = pf.schema_arrow.names

        info["columns"] = len(names)
        info["shape"] = str((
            pf.metadata.num_rows, len(names)
        ))

        info["column_preview"] = " | ".join(
            names[:8]
        )[:280]

        info["gsm_hits"] = len(
            {v.upper() for v in names} & known_gsm
        )

        if (
            min(pf.metadata.num_rows, len(names)) >= 30
            and max(pf.metadata.num_rows, len(names)) >= 200
        ):
            info["source_type"] = (
                "EXPRESSION_OR_FEATURE_MATRIX"
            )

        info["status"] = "PARQUET_METADATA_READ"

    elif suffix == ".feather":
        import pyarrow as pa
        import pyarrow.ipc as ipc

        with pa.memory_map(str(path), "r") as mm:
            reader = ipc.open_file(mm)
            names = reader.schema.names

            info["columns"] = len(names)
            info["column_preview"] = " | ".join(
                names[:8]
            )[:280]

            info["gsm_hits"] = len(
                {v.upper() for v in names} & known_gsm
            )

            info["status"] = "FEATHER_SCHEMA_READ"

            if len(names) >= 80:
                info["source_type"] = (
                    "EXPRESSION_OR_FEATURE_MATRIX"
                )

    else:
        info["status"] = (
            "FORMAT_REQUIRES_SPECIALIZED_READER"
        )

        if name_re.search(path.name):
            info["source_type"] = "POTENTIAL_SOURCE"

    if (
        derived_re.search(path.name)
        and info["source_type"]
        == "EXPRESSION_OR_FEATURE_MATRIX"
    ):
        info["source_type"] = (
            "MODEL_DERIVED_REVIEW_REQUIRED"
        )

    return info

rows = []

for name in selected:
    p = Path(name)

    try:
        row = inspect(p)

    except Exception as exc:
        row = {
            "path": str(p),
            "size_mb": "",
            "format": "",
            "shape": "",
            "columns": "",
            "column_preview": "",
            "id_preview": "",
            "gsm_hits": 0,
            "label_fields": "",
            "source_type": "UNRESOLVED",
            "status": "INSPECTION_FAILED",
            "error": str(exc)[:220],
        }

    rows.append(row)

run_id = (
    datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%SZ"
    )
    + "_" + uuid.uuid4().hex[:8]
)

RUN = ROOT / "03_expression_source_inventory" / run_id
RUN.mkdir(parents=True, exist_ok=False)

artifacts = []

def save(name, data):
    p = RUN / name
    b = data.encode("utf-8") if isinstance(data, str) else data

    with p.open("xb") as f:
        f.write(b)

    artifacts.append({
        "path": str(p.relative_to(ROOT)).replace("\\", "/"),
        "sha256": hashlib.sha256(b).hexdigest(),
        "bytes": len(b),
    })

fields = [
    "path", "size_mb", "format", "shape",
    "columns", "column_preview", "id_preview",
    "gsm_hits", "label_fields", "source_type",
    "status", "error",
]

out = io.StringIO(newline="")
writer = csv.DictWriter(
    out, fieldnames=fields,
    lineterminator="\n"
)
writer.writeheader()
writer.writerows(rows)

save("SOURCE_DISCOVERY.csv", out.getvalue())

matrix_rows = [
    r for r in rows
    if r["source_type"] == "EXPRESSION_OR_FEATURE_MATRIX"
]

mapping_rows = [
    r for r in rows
    if (
        r["source_type"] == "MAPPING_OR_LABEL_CANDIDATE"
        or r["gsm_hits"]
    )
]

manifest = {
    "project_id": "LUAD_DATAFIRST_METHOD_2026",
    "phase": "RAW_EXPRESSION_SOURCE_DISCOVERY",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "scan_roots": [str(p) for p in roots],
    "files_examined": examined,
    "scan_limited": limited,
    "candidate_files": len(candidates),
    "inspected_files": len(rows),
    "matrix_candidates": len(matrix_rows),
    "mapping_candidates": len(mapping_rows),
    "matrices_validated": 0,
    "genotype_labels_verified": False,
    "model_fitted": False,
    "previous_manifest_sha256": hashlib.sha256(
        prior_path.read_bytes()
    ).hexdigest(),
    "artifacts": artifacts,
}

save(
    "RUN_MANIFEST.json",
    json.dumps(manifest, indent=2) + "\n"
)

print(
    "SOURCE_SCAN:",
    "PARTIAL" if limited else "COMPLETE"
)
print("FILES_SCANNED:", examined)
print("FILES_INSPECTED:", len(rows))
print("MATRIX_CANDIDATES:", len(matrix_rows))
print("MAPPING_CANDIDATES:", len(mapping_rows))
print(
    "GSM_MATCHED_FILES:",
    sum(bool(r["gsm_hits"]) for r in rows)
)

for group, data in (
    ("MATRIX", matrix_rows),
    ("MAPPING", mapping_rows),
):
    ranked = sorted(
        data,
        key=lambda x: (
            -int(x["gsm_hits"]),
            -float(x["size_mb"] or 0)
        )
    )

    for r in ranked[:15]:
        print(
            group,
            Path(r["path"]).name[:58],
            r["shape"] or "cols=" + str(r["columns"]),
            "GSM=" + str(r["gsm_hits"]),
            r["path"],
        )

print("RUN:", RUN)

SOURCE_SCAN: COMPLETE
FILES_SCANNED: 9841
FILES_INSPECTED: 231
MATRIX_CANDIDATES: 39
MAPPING_CANDIDATES: 45
GSM_MATCHED_FILES: 35
MATRIX FRMA_GSE31210_226_PROBESET_LOG2.tsv.gz cols=227 GSM=226 D:\LUAD_CAME_Method_2026\04_processed_data\CELL04H_FRMA_GSE31210_226_v001\FRMA_GSE31210_226_PROBESET_LOG2.tsv.gz
MATRIX FRMA_GSE31210_226_PROBESET_LOG2.tsv.gz cols=227 GSM=226 D:\LUAD_PDRF_2026_v002\01_raw_input_copies\CELL00R1_20261021T130120_145481Z\source_expression\FRMA_GSE31210_226_PROBESET_LOG2.tsv.gz
MATRIX FRMA_GSE31210_226_PROBESET_LOG2.tsv.gz cols=227 GSM=226 D:\LUAD_PDRF_2026_v003\01_raw_input_copies\CELL00R1_20261021T130220_661344Z\source_expression\FRMA_GSE31210_226_PROBESET_LOG2.tsv.gz
MATRIX tcga_luad_ascat3_copy_number_raw_matrix_v2.npy (487, 60623) float32 GSM=0 D:\LUAD_PRIVREC_Q1_V1\04_preprocessing\cnv\raw_matrix_v1\tcga_luad_ascat3_copy_number_raw_matrix_v2.npy
MATRIX cmr_gene_expression.npz X.npy:(843, 13016):float64 | y.npy:(843,):int64 | cohort.npy:(843,):<U8 | gsm.npy:(843

In [8]:

import csv
import gzip
import hashlib
import io
import itertools
import json
import uuid
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
from scipy.stats import spearmanr

ROOT = Path(r"D:\LUAD_DATAFIRST_METHOD_2026")

SOURCES = {
    "cmr": Path(
        r"D:\LUAD_CMR_2026\CELL03_DATA_20261021T090947_420335Z_d6aed8ec\cmr_gene_expression.npz"
    ),
    "came": Path(
        r"D:\LUAD_CAME_Method_2026\04_processed_data\CELL03A_R1_COMMON_GENE_RANKS_20261019_v001.npz"
    ),
    "gse72094": Path(
        r"D:\LUAD_RIGDR_2026\04_data_snapshots\STEP04D_R6_MEDIAN_11479_20261021T193114935203Z_6bfab901\GSE72094_RAW_MEDIAN_11479_UNLABELED.npz"
    ),
    "gse31210": Path(
        r"D:\LUAD_CAME_Method_2026\04_processed_data\CELL04H_FRMA_GSE31210_226_v001\FRMA_GSE31210_226_PROBESET_LOG2.tsv.gz"
    ),
}

identity_file = ROOT / "00_admin/project.json"

if (
    not identity_file.is_file()
    or json.loads(identity_file.read_text(encoding="utf-8")).get("project_id")
    != "LUAD_DATAFIRST_METHOD_2026"
):
    raise RuntimeError("Project identity mismatch")

for name, path in SOURCES.items():
    if not path.is_file():
        raise FileNotFoundError(f"{name}: {path}")

prior_files = sorted(
    (ROOT / "03_expression_source_inventory").glob("*/RUN_MANIFEST.json")
)
metadata_files = sorted(
    (ROOT / "01_data_audit").glob("*/sample_metadata.csv")
)

if not prior_files or not metadata_files:
    raise RuntimeError("Prior source inventories are missing")

prior_path = prior_files[-1]
prior = json.loads(prior_path.read_text(encoding="utf-8"))

if prior.get("phase") != "RAW_EXPRESSION_SOURCE_DISCOVERY":
    raise RuntimeError("Unexpected parent phase")

with metadata_files[-1].open(
    "r", encoding="utf-8-sig", newline=""
) as f:
    geo = {
        (r["cohort"].upper(), r["gsm"].upper()): r
        for r in csv.DictReader(f)
    }

def load_npz(path, keys, matrix_key=None):
    with np.load(path, allow_pickle=False) as archive:
        if not set(keys).issubset(archive.files):
            raise ValueError(f"Missing NPZ arrays: {path.name}")

        result = {
            k: np.asarray(archive[k])
            for k in keys
        }

        if matrix_key is not None:
            result[matrix_key] = np.asarray(
                archive[matrix_key], dtype=np.float32
            )

    return result

cmr = load_npz(
    SOURCES["cmr"],
    ["y", "cohort", "gsm", "genes", "patient_ids"],
    "X"
)

X = cmr["X"]
y = np.asarray(cmr["y"])

cohort = np.char.upper(
    np.char.strip(cmr["cohort"].astype(str))
)
gsm = np.char.upper(
    np.char.strip(cmr["gsm"].astype(str))
)
genes = np.char.upper(
    np.char.strip(cmr["genes"].astype(str))
)
patients = np.char.strip(
    cmr["patient_ids"].astype(str)
)

n, p = X.shape

if (
    n != len(y)
    or n != len(cohort)
    or n != len(gsm)
    or n != len(patients)
    or p != len(genes)
):
    raise ValueError("CMR array dimensions are inconsistent")

if np.unique(gsm).size != n:
    raise ValueError("Duplicate CMR sample identifiers")

if len(set(genes.tolist())) != p:
    raise ValueError("Duplicate CMR gene identifiers")

if (
    np.unique(y).size != 3
    or not np.issubdtype(y.dtype, np.integer)
):
    raise ValueError("Unexpected genotype codes")

candidate_codes = sorted(
    int(v) for v in np.unique(y)
)

map_ref = [
    (
        i,
        geo[(str(cohort[i]), str(gsm[i]))]["genotype_candidate"]
    )
    for i in range(n)
    if str(cohort[i]) == "GSE72094"
    and (str(cohort[i]), str(gsm[i])) in geo
    and geo[(str(cohort[i]), str(gsm[i]))]["genotype_candidate"]
    in ("EGFR", "KRAS", "DWT")
]

permutations = []

for perm in itertools.permutations(
    ("EGFR", "KRAS", "DWT")
):
    mapping = dict(zip(candidate_codes, perm))
    agreement = sum(
        mapping[int(y[i])] == label
        for i, label in map_ref
    )
    permutations.append((agreement, mapping))

permutations.sort(key=lambda item: -item[0])

agreement, label_map = permutations[0]

agreement_rate = (
    agreement / len(map_ref)
    if map_ref else None
)

label_map_usable = (
    len(map_ref) >= 100
    and agreement_rate >= 0.98
)

labels = {
    int(code): (
        label_map[int(code)]
        if label_map_usable
        else f"CODE_{code}"
    )
    for code in candidate_codes
}

came = load_npz(
    SOURCES["came"],
    ["y", "cohort", "gsm", "genes"]
)

came_keys = list(zip(
    np.char.upper(
        np.char.strip(came["cohort"].astype(str))
    ),
    np.char.upper(
        np.char.strip(came["gsm"].astype(str))
    )
))

cmr_keys = list(zip(cohort, gsm))
cmr_index = {
    key: i for i, key in enumerate(cmr_keys)
}

came_pairs = [
    (cmr_index[key], j)
    for j, key in enumerate(came_keys)
    if key in cmr_index
]

came_label_agreement = (
    sum(
        int(y[i]) == int(came["y"][j])
        for i, j in came_pairs
    ) / len(came_pairs)
    if came_pairs else None
)

came_gene_overlap = len(
    set(genes.tolist())
    & set(
        np.char.upper(
            came["genes"].astype(str)
        ).tolist()
    )
)

raw = load_npz(
    SOURCES["gse72094"],
    ["genes", "gsm"],
    "X_gene"
)

Xraw = raw["X_gene"]

raw_genes = np.char.upper(
    np.char.strip(raw["genes"].astype(str))
)
raw_gsm = np.char.upper(
    np.char.strip(raw["gsm"].astype(str))
)

raw_index = {
    sample: i
    for i, sample in enumerate(raw_gsm)
}

raw_matched = [
    (i, raw_index[str(gsm[i])])
    for i in range(n)
    if str(cohort[i]) == "GSE72094"
    and str(gsm[i]) in raw_index
]

gene_positions = {
    gene: j for j, gene in enumerate(genes)
}

raw_positions = [
    (gene_positions[gene], j)
    for j, gene in enumerate(raw_genes)
    if gene in gene_positions
]

raw_spearman = []

if len(raw_positions) >= 200:
    a = np.array([
        j for j, _ in raw_positions
    ])
    b = np.array([
        j for _, j in raw_positions
    ])

    for i, j in raw_matched[:8]:
        v1 = X[i, a]
        v2 = Xraw[j, b]

        finite = (
            np.isfinite(v1)
            & np.isfinite(v2)
        )

        if finite.sum() >= 200:
            rho = spearmanr(
                v1[finite], v2[finite]
            ).statistic

            if np.isfinite(rho):
                raw_spearman.append(float(rho))

with gzip.open(
    SOURCES["gse31210"],
    "rt",
    encoding="utf-8-sig",
    errors="replace",
    newline=""
) as f:
    header = next(
        csv.reader(f, delimiter="\t")
    )

frma_gsm = {
    value.strip().upper()
    for value in header[1:]
    if value.strip().upper().startswith("GSM")
}

frma_overlap = sum(
    str(cohort[i]) == "GSE31210"
    and str(gsm[i]) in frma_gsm
    for i in range(n)
)

cohort_rows = []
discordances = []

for c in sorted(np.unique(cohort).tolist()):
    idx = np.flatnonzero(cohort == c)
    Z = X[idx]

    finite = np.isfinite(Z)
    Zfinite = np.where(finite, Z, np.nan)

    sample_median = np.nanmedian(
        Zfinite, axis=1
    )

    spread = (
        np.nanpercentile(Zfinite, 90, axis=1)
        - np.nanpercentile(Zfinite, 10, axis=1)
    )

    gene_var = np.nanstd(
        Zfinite, axis=0
    )

    observed = [
        geo.get(
            (c, str(gsm[i])), {}
        ).get("genotype_candidate", "")
        for i in idx
    ]

    if label_map_usable:
        for j, i in enumerate(idx):
            if (
                observed[j] in ("EGFR", "KRAS", "DWT")
                and labels[int(y[i])] != observed[j]
            ):
                discordances.append({
                    "cohort": c,
                    "gsm": str(gsm[i]),
                    "archived_code": int(y[i]),
                    "inferred_class": labels[int(y[i])],
                    "geo_class": observed[j],
                })

    counts = Counter(
        labels[int(y[i])] for i in idx
    )

    cohort_rows.append({
        "cohort": c,
        "n": len(idx),
        "class_counts": json.dumps(
            dict(sorted(counts.items()))
        ),
        "geo_mutation_records": sum(
            value in ("EGFR", "KRAS", "DWT")
            for value in observed
        ),
        "nonfinite_fraction": round(
            1.0 - float(finite.mean()), 8
        ),
        "sample_median_p50": round(
            float(np.nanmedian(sample_median)), 5
        ),
        "sample_p90_p10_p50": round(
            float(np.nanmedian(spread)), 5
        ),
        "variable_genes": int(
            (gene_var > 1e-5).sum()
        ),
        "unique_patients": len({
            str(patients[i])
            for i in idx
            if str(patients[i])
        }),
    })

strat_rows = []

if label_map_usable:
    kras_idx = [
        i for i in range(n)
        if (
            str(cohort[i]) == "GSE72094"
            and labels[int(y[i])] == "KRAS"
        )
    ]

    for gene in ("stk11", "tp53"):
        mutant = [
            i for i in kras_idx
            if geo.get(
                ("GSE72094", str(gsm[i])), {}
            ).get(gene, "") == "MUT"
        ]

        wildtype = [
            i for i in kras_idx
            if geo.get(
                ("GSE72094", str(gsm[i])), {}
            ).get(gene, "") == "WT"
        ]

        row = {
            "gene": gene.upper(),
            "kras_mutant": len(mutant),
            "kras_wt": len(wildtype),
            "kras_unresolved": (
                len(kras_idx)
                - len(mutant)
                - len(wildtype)
            ),
            "median_abs_standardized_difference": None,
        }

        if len(mutant) >= 8 and len(wildtype) >= 8:
            scale = np.nanstd(
                X[kras_idx], axis=0
            )

            effect = (
                np.nanmean(X[mutant], axis=0)
                - np.nanmean(X[wildtype], axis=0)
            ) / np.maximum(scale, 1e-6)

            row["median_abs_standardized_difference"] = (
                float(np.nanmedian(np.abs(effect)))
            )

        strat_rows.append(row)

contrast_rows = []

if label_map_usable:
    effects = {}

    development_cohorts = (
        "GSE72094",
        "GSE31210",
        "GSE13213",
        "E-MTAB-923",
        "GSE37745",
    )

    for c in development_cohorts:
        idx = np.flatnonzero(cohort == c)

        if len(idx) == 0:
            continue

        Z = X[idx]
        group = np.array([
            labels[int(y[i])] for i in idx
        ])

        contrasts = (
            (
                "EGFR_vs_nonEGFR",
                group == "EGFR",
                group != "EGFR",
            ),
            (
                "KRAS_vs_DWT",
                group == "KRAS",
                group == "DWT",
            ),
        )

        for contrast, positive, negative in contrasts:
            if (
                positive.sum() < 12
                or negative.sum() < 12
            ):
                continue

            scale = np.nanstd(Z, axis=0)

            effect = (
                np.nanmean(Z[positive], axis=0)
                - np.nanmean(Z[negative], axis=0)
            ) / np.maximum(scale, 1e-6)

            effects[(c, contrast)] = effect

    for contrast in (
        "EGFR_vs_nonEGFR",
        "KRAS_vs_DWT",
    ):
        keys = [
            key for key in effects
            if key[1] == contrast
        ]

        for (a, _), (b, _) in itertools.combinations(
            keys, 2
        ):
            va = effects[(a, contrast)]
            vb = effects[(b, contrast)]

            mask = (
                np.isfinite(va)
                & np.isfinite(vb)
            )

            rho = (
                spearmanr(
                    va[mask], vb[mask]
                ).statistic
                if mask.sum() >= 500
                else np.nan
            )

            contrast_rows.append({
                "contrast": contrast,
                "cohort_a": a,
                "cohort_b": b,
                "common_genes": int(mask.sum()),
                "effect_spearman": (
                    float(rho)
                    if np.isfinite(rho)
                    else None
                ),
                "exploratory_only": True,
            })

run_id = (
    datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%SZ"
    )
    + "_" + uuid.uuid4().hex[:8]
)

RUN = ROOT / "04_actual_expression_audit" / run_id
RUN.mkdir(parents=True, exist_ok=False)

artifacts = []

def save(name, content):
    path = RUN / name

    data = (
        content.encode("utf-8")
        if isinstance(content, str)
        else content
    )

    with path.open("xb") as f:
        f.write(data)

    artifacts.append({
        "path": str(
            path.relative_to(ROOT)
        ).replace("\\", "/"),
        "sha256": hashlib.sha256(data).hexdigest(),
    })

def write_csv(name, rows, fields):
    buffer = io.StringIO(newline="")

    writer = csv.DictWriter(
        buffer,
        fieldnames=fields,
        lineterminator="\n",
    )

    writer.writeheader()
    writer.writerows(rows)

    save(name, buffer.getvalue())

write_csv(
    "COHORT_EXPRESSION_QC.csv",
    cohort_rows,
    list(cohort_rows[0])
)

write_csv(
    "LABEL_DISCORDANCES.csv",
    discordances,
    [
        "cohort", "gsm", "archived_code",
        "inferred_class", "geo_class",
    ]
)

write_csv(
    "DEVELOPMENT_EFFECT_CONCORDANCE.csv",
    contrast_rows,
    [
        "contrast", "cohort_a", "cohort_b",
        "common_genes", "effect_spearman",
        "exploratory_only",
    ]
)

write_csv(
    "KRAS_COCMUTATION_QC.csv",
    strat_rows,
    [
        "gene",
        "kras_mutant",
        "kras_wt",
        "kras_unresolved",
        "median_abs_standardized_difference",
    ]
)

validation = {
    "cmr_shape": [int(n), int(p)],
    "cohort_counts": dict(Counter(cohort.tolist())),
    "label_map_inferred_from_GSE72094": {
        str(k): v for k, v in label_map.items()
    },
    "label_mapping_usable": bool(label_map_usable),
    "gse72094_label_comparisons": len(map_ref),
    "gse72094_label_agreement": agreement_rate,
    "geo_label_discordances": len(discordances),
    "came_matched_samples": len(came_pairs),
    "came_raw_code_agreement": came_label_agreement,
    "came_common_genes": came_gene_overlap,
    "raw72094_matched_samples": len(raw_matched),
    "raw72094_common_genes": len(raw_positions),
    "raw72094_spearman_p50": (
        float(np.median(raw_spearman))
        if raw_spearman else None
    ),
    "frma31210_header_samples": len(frma_gsm),
    "frma31210_matched_samples": frma_overlap,
    "gse32863_in_archived_843": bool(
        (cohort == "GSE32863").any()
    ),
    "gse66863_in_archived_843": bool(
        (cohort == "GSE66863").any()
    ),
    "assay_completeness_verified": False,
    "untouched_external_validation_verified": False,
    "supervised_contrasts_exploratory_only": True,
    "model_training_performed": False,
}

save(
    "CROSS_SOURCE_VALIDATION.json",
    json.dumps(validation, indent=2) + "\n"
)

def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as f:
        for block in iter(
            lambda: f.read(4 * 1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()

save(
    "RUN_MANIFEST.json",
    json.dumps({
        "phase": "REAL_EXPRESSION_AND_LABEL_DIAGNOSTIC",
        "project_id": "LUAD_DATAFIRST_METHOD_2026",
        "created_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "parent_manifest_sha256": sha256_file(prior_path),
        "inputs": {
            key: {
                "path": str(path),
                "sha256": sha256_file(path),
            }
            for key, path in SOURCES.items()
        },
        "artifacts": artifacts,
        "model_training_performed": False,
    }, indent=2) + "\n"
)

print("EXPRESSION_AUDIT: PASS")
print("CMR:", n, p)
print(
    "LABEL_MAP:",
    label_map if label_map_usable else "UNVERIFIED"
)
print(
    "GEO_CONCORDANCE:",
    f"{agreement}/{len(map_ref)}",
    "DISCORDANCES:",
    len(discordances),
)

for row in cohort_rows:
    print(
        "COHORT:",
        row["cohort"],
        "N=", row["n"],
        "CLASS=", row["class_counts"],
        "FINITE=",
        f"{1-row['nonfinite_fraction']:.5f}",
        "SCALE=",
        row["sample_median_p50"],
        row["sample_p90_p10_p50"],
    )

print(
    "CAME:",
    len(came_pairs),
    "GENES=", came_gene_overlap,
    "LABEL_MATCH=",
    (
        f"{came_label_agreement:.3f}"
        if came_label_agreement is not None
        else "NA"
    ),
)

print(
    "GSE72094_SOURCE:",
    len(raw_matched),
    "GENES=", len(raw_positions),
    "SPEARMAN=",
    (
        f"{np.median(raw_spearman):.4f}"
        if raw_spearman else "NA"
    ),
)

print(
    "GSE31210_FRMA:",
    frma_overlap,
    "/",
    len(frma_gsm),
)

for row in contrast_rows:
    rho = row["effect_spearman"]

    print(
        "EFFECT:",
        row["contrast"],
        row["cohort_a"],
        row["cohort_b"],
        f"RHO={rho:.4f}" if rho is not None else "RHO=NA",
    )

for row in strat_rows:
    effect = row["median_abs_standardized_difference"]

    print(
        "KRAS_CONTEXT:",
        row["gene"],
        "MUT=", row["kras_mutant"],
        "WT=", row["kras_wt"],
        "UNRESOLVED=", row["kras_unresolved"],
        "EFFECT=",
        round(effect, 4) if effect is not None else "NA",
    )

print(
    "EXTERNAL_EXPOSURE:",
    "GSE32863=" +
    str(validation["gse32863_in_archived_843"]),
    "GSE66863=" +
    str(validation["gse66863_in_archived_843"]),
)

print("RUN:", RUN)

EXPRESSION_AUDIT: PASS
CMR: 843 13016
LABEL_MAP: {0: 'EGFR', 1: 'KRAS', 2: 'DWT'}
GEO_CONCORDANCE: 442/442 DISCORDANCES: 0
COHORT: GSE13213 N= 117 CLASS= {"DWT": 57, "EGFR": 45, "KRAS": 15} FINITE= 1.00000 SCALE= -0.221 3.78908
COHORT: GSE31210 N= 226 CLASS= {"DWT": 79, "EGFR": 127, "KRAS": 20} FINITE= 1.00000 SCALE= 301.84436 2230.9126
COHORT: GSE32863 N= 58 CLASS= {"DWT": 19, "EGFR": 17, "KRAS": 22} FINITE= 1.00000 SCALE= 7.76584 3.65825
COHORT: GSE72094 N= 442 CLASS= {"DWT": 241, "EGFR": 47, "KRAS": 154} FINITE= 1.00000 SCALE= 7.74369 5.92613
CAME: 843 GENES= 13016 LABEL_MATCH= 1.000
GSE72094_SOURCE: 442 GENES= 11478 SPEARMAN= 1.0000
GSE31210_FRMA: 226 / 226
EFFECT: EGFR_vs_nonEGFR GSE72094 GSE31210 RHO=0.6332
EFFECT: EGFR_vs_nonEGFR GSE72094 GSE13213 RHO=0.5319
EFFECT: EGFR_vs_nonEGFR GSE31210 GSE13213 RHO=0.5046
EFFECT: KRAS_vs_DWT GSE72094 GSE31210 RHO=0.3369
EFFECT: KRAS_vs_DWT GSE72094 GSE13213 RHO=0.3596
EFFECT: KRAS_vs_DWT GSE31210 GSE13213 RHO=0.3099
KRAS_CONTEXT: STK11 MUT=

In [9]:

import csv
import hashlib
import io
import json
import os
import uuid
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, balanced_accuracy_score, log_loss
from sklearn.preprocessing import StandardScaler

ROOT = Path(r"D:\LUAD_DATAFIRST_METHOD_2026")
SOURCE = Path(
    r"D:\LUAD_CMR_2026\CELL03_DATA_20261021T090947_420335Z_d6aed8ec\cmr_gene_expression.npz"
)
IDENTITY = ROOT / "00_admin/project.json"
AUDITS = ROOT / "04_actual_expression_audit"

DEV = ("GSE13213", "GSE31210", "GSE72094")
EXCLUDED = ("GSE32863", "GSE66863")

SEED = 20261023
TOP_PER_CONTRAST = 96
EPOCHS = 110
HIDDEN = 10
LR = 0.012
MASK_FLOOR = 0.25

if not IDENTITY.is_file() or not SOURCE.is_file():
    raise FileNotFoundError("Project identity or expression source missing")

identity = json.loads(IDENTITY.read_text(encoding="utf-8"))
if identity.get("project_id") != "LUAD_DATAFIRST_METHOD_2026":
    raise RuntimeError("Project identity mismatch")

validation_files = sorted(
    AUDITS.glob("*/CROSS_SOURCE_VALIDATION.json")
)
if not validation_files:
    raise RuntimeError("Cross-source validation audit missing")

validation_path = validation_files[-1]
validation = json.loads(
    validation_path.read_text(encoding="utf-8")
)

label_map = {
    int(k): v
    for k, v in validation[
        "label_map_inferred_from_GSE72094"
    ].items()
}

if (
    not validation.get("label_mapping_usable")
    or validation.get("gse72094_label_agreement", 0) < 0.98
    or label_map != {0: "EGFR", 1: "KRAS", 2: "DWT"}
):
    raise RuntimeError("Mutation-label ontology verification failed")

torch.manual_seed(SEED)
torch.set_num_threads(min(2, max(1, os.cpu_count() or 1)))
np.random.seed(SEED)

with np.load(SOURCE, allow_pickle=False) as z:
    required = {
        "X", "y", "cohort", "gsm", "genes", "patient_ids"
    }
    if not required.issubset(z.files):
        raise RuntimeError("Incomplete expression archive")

    domain_all = np.char.upper(
        np.char.strip(z["cohort"].astype(str))
    )
    devmask = np.isin(domain_all, DEV)

    x = np.asarray(z["X"][devmask], dtype=np.float32)
    y = np.asarray(z["y"][devmask], dtype=np.int64)
    domain = np.asarray(domain_all[devmask], dtype=str)
    gsm = np.char.upper(
        np.char.strip(z["gsm"][devmask].astype(str))
    )
    patients = np.char.strip(
        z["patient_ids"][devmask].astype(str)
    )
    genes = np.char.upper(
        np.char.strip(z["genes"].astype(str))
    )

if x.shape != (785, 13016):
    raise RuntimeError("Unexpected development matrix dimensions")

if set(domain.tolist()) != set(DEV):
    raise RuntimeError("Development cohort composition mismatch")

if set(y.tolist()) != {0, 1, 2}:
    raise RuntimeError("Unexpected genotype classes")

if len(set(gsm.tolist())) != len(gsm):
    raise RuntimeError("Duplicate GSM identifiers")

if len(set(genes.tolist())) != len(genes):
    raise RuntimeError("Duplicate gene identifiers")

if not np.isfinite(x).all():
    raise RuntimeError("Nonfinite expression measurements")

for cohort in DEV:
    idx = np.flatnonzero(domain == cohort)
    counts = np.bincount(y[idx], minlength=3)

    if counts.min() < 10:
        raise RuntimeError(
            f"Insufficient genotype observations: {cohort}"
        )

patient_cohorts = {}

for cohort, patient in zip(domain, patients):
    if patient and patient.casefold() not in {
        "na", "nan", "none", "unknown"
    }:
        patient_cohorts.setdefault(patient, set()).add(cohort)

if any(len(v) > 1 for v in patient_cohorts.values()):
    raise RuntimeError("Cross-cohort patient identifier overlap")

ranks = (
    rankdata(x, axis=1, method="average").astype(np.float32)
    - 1.0
) / (x.shape[1] - 1)

del x


def select_features(train_idx):
    effects = {0: [], 1: []}

    for cohort in sorted(set(domain[train_idx].tolist())):
        idx = train_idx[domain[train_idx] == cohort]
        r = ranks[idx]
        labels = y[idx]

        scale = np.maximum(np.std(r, axis=0), 0.035)

        for contrast in (0, 1):
            pos = r[labels == contrast]
            neg = (
                r[labels != 0]
                if contrast == 0
                else r[labels == 2]
            )

            if min(len(pos), len(neg)) < 10:
                raise RuntimeError("Insufficient contrast samples")

            effect = (
                pos.mean(axis=0) - neg.mean(axis=0)
            ) / scale

            effects[contrast].append(effect)

    selected = []

    for contrast in (0, 1):
        values = np.stack(effects[contrast])

        agreement = np.abs(
            np.sign(values).mean(axis=0)
        )

        strength = (
            np.abs(values.mean(axis=0))
            * (0.4 + 0.6 * agreement)
        )

        selected.extend(
            np.argsort(-strength, kind="stable")[
                :TOP_PER_CONTRAST
            ]
        )

    return np.sort(np.unique(selected))


class ContrastNet(nn.Module):
    def __init__(self, n_features, nonlinear):
        super().__init__()

        self.egfr = nn.Linear(n_features, 1)
        self.kras = nn.Linear(n_features, 1)

        self.context = (
            nn.Sequential(
                nn.Linear(n_features, HIDDEN, bias=False),
                nn.Tanh(),
                nn.Linear(HIDDEN, 1, bias=False),
            )
            if nonlinear
            else None
        )

        nn.init.normal_(self.egfr.weight, std=0.015)
        nn.init.normal_(self.kras.weight, std=0.015)
        nn.init.zeros_(self.egfr.bias)
        nn.init.zeros_(self.kras.bias)

        if self.context is not None:
            for layer in self.context:
                if isinstance(layer, nn.Linear):
                    nn.init.normal_(layer.weight, std=0.025)

    def forward(self, x):
        e = self.egfr(x).squeeze(-1)
        k = self.kras(x).squeeze(-1)

        if self.context is not None:
            k = k + 0.25 * self.context(x).squeeze(-1)

        return e, k

    def kras_parameters(self):
        parameters = list(self.kras.parameters())

        if self.context is not None:
            parameters += list(self.context.parameters())

        return parameters


def cohort_losses(model, x, labels):
    e, k = model(x)
    egfr_positive = labels == 0

    loss_e = F.binary_cross_entropy_with_logits(
        e, egfr_positive.float()
    )

    non_egfr = ~egfr_positive

    loss_k = F.binary_cross_entropy_with_logits(
        k[non_egfr],
        (labels[non_egfr] == 1).float(),
    )

    loss = (
        loss_e
        + non_egfr.float().mean() * loss_k
    )

    return loss, loss_e, loss_k


def mask_gradients(parameters, gradients):
    for j, param in enumerate(parameters):
        if param.grad is None:
            continue

        values = torch.stack([
            group[j] for group in gradients
        ])

        consistent = (
            (values > 1e-9).all(dim=0)
            | (values < -1e-9).all(dim=0)
        )

        mask = (
            MASK_FLOOR
            + (1.0 - MASK_FLOOR) * consistent.float()
        )

        param.grad.mul_(mask)


def train_model(x, labels, cohort_names, nonlinear, route):
    torch.manual_seed(SEED)

    model = ContrastNet(x.shape[1], nonlinear)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=0.03,
    )

    groups = [
        (
            x[cohort_names == cohort],
            labels[cohort_names == cohort],
        )
        for cohort in sorted(set(cohort_names.tolist()))
    ]

    for _ in range(EPOCHS):
        optimizer.zero_grad(set_to_none=True)

        cohort_objectives = []
        gradients_e = []
        gradients_k = []

        for xx, yy in groups:
            loss, loss_e, loss_k = cohort_losses(
                model, xx, yy
            )

            cohort_objectives.append(loss)

            if route == "both":
                gradients_k.append(
                    torch.autograd.grad(
                        loss_k,
                        model.kras_parameters(),
                        retain_graph=True,
                    )
                )

                gradients_e.append(
                    torch.autograd.grad(
                        loss_e,
                        list(model.egfr.parameters()),
                        retain_graph=True,
                    )
                )

            elif route == "kras":
                gradients_k.append(
                    torch.autograd.grad(
                        loss_k,
                        list(model.context.parameters()),
                        retain_graph=True,
                    )
                )

        objective = torch.stack(
            cohort_objectives
        ).mean()

        if not torch.isfinite(objective):
            raise RuntimeError("Nonfinite training objective")

        objective.backward()

        if route == "both":
            mask_gradients(
                model.kras_parameters(),
                gradients_k,
            )
            mask_gradients(
                list(model.egfr.parameters()),
                gradients_e,
            )

        elif route == "kras":
            mask_gradients(
                list(model.context.parameters()),
                gradients_k,
            )

        optimizer.step()

    return model


def predict_model(model, x):
    with torch.no_grad():
        e, k = model(x)

        pe = torch.sigmoid(e).numpy()
        pk = torch.sigmoid(k).numpy()

        probabilities = np.stack(
            (
                pe,
                (1.0 - pe) * pk,
                (1.0 - pe) * (1.0 - pk),
            ),
            axis=1,
        )

    if np.max(
        np.abs(probabilities.sum(axis=1) - 1.0)
    ) > 1e-6:
        raise RuntimeError("Probability normalization failure")

    probabilities = np.maximum(
        probabilities, 1e-8
    )

    return (
        probabilities
        / probabilities.sum(axis=1, keepdims=True)
    )


def evaluate(truth, probabilities):
    prediction = probabilities.argmax(axis=1)

    non_egfr = truth != 0
    pk = (
        probabilities[non_egfr, 1]
        / (
            probabilities[non_egfr, 1]
            + probabilities[non_egfr, 2]
        )
    )

    true_kras = (truth[non_egfr] == 1).astype(int)
    pk = np.clip(pk, 1e-10, 1.0 - 1e-10)

    conditional_nll = -np.mean(
        true_kras * np.log(pk)
        + (1 - true_kras) * np.log(1 - pk)
    )

    return {
        "n": len(truth),
        "nll": float(
            log_loss(truth, probabilities, labels=[0, 1, 2])
        ),
        "macro_f1": float(
            f1_score(
                truth,
                prediction,
                labels=[0, 1, 2],
                average="macro",
                zero_division=0,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(truth, prediction)
        ),
        "kras_conditional_nll": float(conditional_nll),
        "egfr_recall": float(
            np.mean(prediction[truth == 0] == 0)
        ),
        "kras_recall": float(
            np.mean(prediction[truth == 1] == 1)
        ),
        "dwt_recall": float(
            np.mean(prediction[truth == 2] == 2)
        ),
    }


model_specs = {
    "flat_lr": None,
    "hier_linear": (False, "none"),
    "k_residual": (True, "none"),
    "generic_mask": (True, "both"),
    "contrast_mask": (True, "kras"),
}

metrics = []
predictions = []
selected_genes = {}
checkpoints = {}

for held_cohort in DEV:
    train = np.flatnonzero(domain != held_cohort)
    test = np.flatnonzero(domain == held_cohort)

    columns = select_features(train)

    selected_genes[held_cohort] = genes[columns].tolist()
    checkpoints[held_cohort] = {
        "selected_genes": genes[columns].tolist()
    }

    scaler = StandardScaler().fit(
        ranks[train][:, columns]
    )

    x_train = scaler.transform(
        ranks[train][:, columns]
    ).astype(np.float32)

    x_test = scaler.transform(
        ranks[test][:, columns]
    ).astype(np.float32)

    checkpoints[held_cohort]["scale_mean"] = (
        scaler.mean_.astype(np.float32)
    )
    checkpoints[held_cohort]["scale_std"] = (
        scaler.scale_.astype(np.float32)
    )

    xt = torch.from_numpy(x_train)
    yt = torch.from_numpy(y[train])
    xv = torch.from_numpy(x_test)

    train_domains = domain[train]
    n_domains = len(set(train_domains.tolist()))

    sample_weights = np.asarray([
        len(train)
        / (
            n_domains
            * np.sum(train_domains == cohort)
        )
        for cohort in train_domains
    ])

    for name, specification in model_specs.items():
        if name == "flat_lr":
            model = LogisticRegression(
                C=0.2,
                max_iter=350,
                solver="lbfgs",
                random_state=SEED,
            )

            model.fit(
                x_train,
                y[train],
                sample_weight=sample_weights,
            )

            probabilities = model.predict_proba(x_test)

            checkpoints[held_cohort][name] = {
                "coef": model.coef_.copy(),
                "intercept": model.intercept_.copy(),
                "classes": model.classes_.copy(),
            }

        else:
            nonlinear, route = specification

            model = train_model(
                xt,
                yt,
                train_domains,
                nonlinear,
                route,
            )

            probabilities = predict_model(model, xv)

            checkpoints[held_cohort][name] = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

        scores = evaluate(y[test], probabilities)

        metrics.append({
            "held_cohort": held_cohort,
            "model": name,
            "n_features": len(columns),
            **scores,
        })

        for j, idx in enumerate(test):
            predictions.append({
                "held_cohort": held_cohort,
                "gsm": str(gsm[idx]),
                "true_class": int(y[idx]),
                "model": name,
                "p_egfr": float(probabilities[j, 0]),
                "p_kras": float(probabilities[j, 1]),
                "p_dwt": float(probabilities[j, 2]),
            })


run_id = (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    + "_"
    + uuid.uuid4().hex[:8]
)

RUN = ROOT / "05_bounded_method_development" / run_id
RUN.mkdir(parents=True, exist_ok=False)
artifacts = []


def save(filename, content):
    destination = RUN / filename
    destination.parent.mkdir(parents=True, exist_ok=True)

    if isinstance(content, str):
        content = content.encode("utf-8")

    with destination.open("xb") as f:
        f.write(content)

    artifacts.append({
        "path": str(
            destination.relative_to(ROOT)
        ).replace("\\", "/"),
        "sha256": hashlib.sha256(content).hexdigest(),
    })


def csv_output(rows):
    buffer = io.StringIO(newline="")

    writer = csv.DictWriter(
        buffer,
        fieldnames=list(rows[0]),
        lineterminator="\n",
    )

    writer.writeheader()
    writer.writerows(rows)

    return buffer.getvalue()


save("DEV_LOCO_METRICS.csv", csv_output(metrics))
save("DEV_LOCO_PREDICTIONS.csv", csv_output(predictions))

save(
    "TRAIN_ONLY_SELECTED_GENES.json",
    json.dumps(selected_genes, indent=2) + "\n",
)

for cohort, state in checkpoints.items():
    buffer = io.BytesIO()
    torch.save(state, buffer)

    save(
        f"fold_checkpoints/{cohort}.pt",
        buffer.getvalue(),
    )

summary = {}

for name in model_specs:
    subset = [
        row for row in metrics
        if row["model"] == name
    ]

    summary[name] = {
        "mean_nll": float(np.mean([
            row["nll"] for row in subset
        ])),
        "mean_macro_f1": float(np.mean([
            row["macro_f1"] for row in subset
        ])),
        "mean_kras_conditional_nll": float(np.mean([
            row["kras_conditional_nll"] for row in subset
        ])),
        "cohort_nll": {
            row["held_cohort"]: row["nll"]
            for row in subset
        },
    }

save(
    "EQUAL_COHORT_SUMMARY.json",
    json.dumps(summary, indent=2) + "\n",
)

comparison = {
    "nll_gain_over_same_architecture": (
        summary["k_residual"]["mean_nll"]
        - summary["contrast_mask"]["mean_nll"]
    ),
    "nll_gain_over_generic_mask": (
        summary["generic_mask"]["mean_nll"]
        - summary["contrast_mask"]["mean_nll"]
    ),
    "kras_conditional_nll_gain": (
        summary["k_residual"]["mean_kras_conditional_nll"]
        - summary["contrast_mask"]["mean_kras_conditional_nll"]
    ),
    "positive_cohorts_vs_same_architecture": sum(
        summary["contrast_mask"]["cohort_nll"][c]
        < summary["k_residual"]["cohort_nll"][c]
        for c in DEV
    ),
}

comparison["development_screen"] = (
    "PASS"
    if (
        comparison["nll_gain_over_same_architecture"] >= 0.01
        and comparison["nll_gain_over_generic_mask"] >= 0.005
        and comparison["kras_conditional_nll_gain"] > 0
        and comparison["positive_cohorts_vs_same_architecture"] >= 2
    )
    else "HOLD"
)

save(
    "SCREENING_DECISION.json",
    json.dumps(comparison, indent=2) + "\n",
)

with SOURCE.open("rb") as f:
    source_hash = hashlib.file_digest(
        f, "sha256"
    ).hexdigest()

manifest = {
    "phase": "DEVELOPMENT_ONLY_CONTRAST_ROUTING_PILOT",
    "run_id": run_id,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "input": str(SOURCE),
    "input_sha256": source_hash,
    "cross_source_validation_sha256": hashlib.sha256(
        validation_path.read_bytes()
    ).hexdigest(),
    "development_cohorts": DEV,
    "excluded_from_current_fit": EXCLUDED,
    "gse32863_historically_exposed": True,
    "gse66863_eligibility_unverified": True,
    "label_provenance": "OTHER_COHORT_LABELS_PROVISIONAL",
    "feature_schema": "ARCHIVED_SHARED_GENE_UNIVERSE_EXPOSED_843",
    "epochs": EPOCHS,
    "features_per_contrast": TOP_PER_CONTRAST,
    "mask_floor": MASK_FLOOR,
    "model_novelty": "NOT_ESTABLISHED",
    "no_external_model_evaluation": True,
    "screening_threshold_nll": 0.01,
    "screening_threshold_vs_generic": 0.005,
    "screening_predeclared_exploratory": True,
    "artifacts": artifacts,
}

save(
    "RUN_MANIFEST.json",
    json.dumps(manifest, indent=2) + "\n",
)

print("DEVELOPMENT_PILOT: COMPLETE")

for cohort in DEV:
    print(
        "LOCO", cohort,
        "N=", int(np.sum(domain == cohort)),
        "CLASS=", dict(
            Counter(y[domain == cohort].tolist())
        ),
    )

    for name in model_specs:
        row = next(
            r for r in metrics
            if (
                r["held_cohort"] == cohort
                and r["model"] == name
            )
        )

        print(
            name,
            "NLL=", round(row["nll"], 4),
            "F1=", round(row["macro_f1"], 4),
            "K_NLL=", round(
                row["kras_conditional_nll"], 4
            ),
        )

for name, row in summary.items():
    print(
        "MEAN", name,
        "NLL=", round(row["mean_nll"], 4),
        "F1=", round(row["mean_macro_f1"], 4),
        "K_NLL=", round(
            row["mean_kras_conditional_nll"], 4
        ),
    )

print("SCREEN:", comparison["development_screen"])
print(
    "DELTA_NLL_MATCHED:",
    round(comparison["nll_gain_over_same_architecture"], 4),
)
print(
    "DELTA_NLL_GENERIC:",
    round(comparison["nll_gain_over_generic_mask"], 4),
)
print("RUN:", RUN)

DEVELOPMENT_PILOT: COMPLETE
LOCO GSE13213 N= 117 CLASS= {0: 45, 2: 57, 1: 15}
flat_lr NLL= 4.7051 F1= 0.4534 K_NLL= 0.6878
hier_linear NLL= 4.5096 F1= 0.2754 K_NLL= 1.8124
k_residual NLL= 4.434 F1= 0.3003 K_NLL= 2.0707
generic_mask NLL= 2.211 F1= 0.4475 K_NLL= 1.4355
contrast_mask NLL= 4.4654 F1= 0.3003 K_NLL= 2.1219
LOCO GSE31210 N= 226 CLASS= {1: 20, 2: 79, 0: 127}
flat_lr NLL= 6.0698 F1= 0.1727 K_NLL= 1.3185
hier_linear NLL= 1.9708 F1= 0.3092 K_NLL= 0.4367
k_residual NLL= 2.1404 F1= 0.2459 K_NLL= 0.5256
generic_mask NLL= 2.3175 F1= 0.3064 K_NLL= 0.4733
contrast_mask NLL= 2.0972 F1= 0.3039 K_NLL= 0.4271
LOCO GSE72094 N= 442 CLASS= {0: 47, 2: 241, 1: 154}
flat_lr NLL= 1.5387 F1= 0.4852 K_NLL= 1.1829
hier_linear NLL= 1.8811 F1= 0.4495 K_NLL= 1.6543
k_residual NLL= 1.8642 F1= 0.4382 K_NLL= 1.6063
generic_mask NLL= 1.6276 F1= 0.447 K_NLL= 1.3869
contrast_mask NLL= 1.8992 F1= 0.4385 K_NLL= 1.6456
MEAN flat_lr NLL= 4.1045 F1= 0.3705 K_NLL= 1.0631
MEAN hier_linear NLL= 2.7872 F1= 0.3447 K_N

In [10]:

import csv
import hashlib
import io
import json
import uuid
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch

ROOT = Path(r"D:\LUAD_DATAFIRST_METHOD_2026")
IDENTITY = ROOT / "00_admin/project.json"
DEV = ("GSE13213", "GSE31210", "GSE72094")
MODELS = (
    "flat_lr",
    "hier_linear",
    "k_residual",
    "generic_mask",
    "contrast_mask",
)

if not IDENTITY.is_file():
    raise FileNotFoundError(IDENTITY)

identity = json.loads(IDENTITY.read_text(encoding="utf-8"))
if identity.get("project_id") != "LUAD_DATAFIRST_METHOD_2026":
    raise RuntimeError("Project identity mismatch")

runs = []
for path in (ROOT / "05_bounded_method_development").glob(
    "*/RUN_MANIFEST.json"
):
    doc = json.loads(path.read_text(encoding="utf-8"))
    if doc.get("phase") == "DEVELOPMENT_ONLY_CONTRAST_ROUTING_PILOT":
        runs.append((path, doc))

if not runs:
    raise RuntimeError("Development pilot run not found")

parent, manifest = sorted(runs)[-1]

for item in manifest.get("artifacts", []):
    src = ROOT / item["path"]
    if (
        not src.is_file()
        or hashlib.sha256(src.read_bytes()).hexdigest()
        != item["sha256"]
    ):
        raise RuntimeError("Artifact checksum mismatch: " + str(src))

with (parent.parent / "DEV_LOCO_PREDICTIONS.csv").open(
    "r", encoding="utf-8-sig", newline=""
) as handle:
    incoming = list(csv.DictReader(handle))

records = {}

for row in incoming:
    cohort = row["held_cohort"]
    model = row["model"]
    gsm = row["gsm"]

    if cohort not in DEV or model not in MODELS:
        raise ValueError("Unexpected cohort or model")

    key = (cohort, model, gsm)

    if key in records:
        raise ValueError("Duplicate prediction key")

    label = int(row["true_class"])
    prob = np.array([
        float(row[f"p_{g}"])
        for g in ("egfr", "kras", "dwt")
    ], dtype=np.float64)

    if (
        label not in (0, 1, 2)
        or not np.isfinite(prob).all()
        or (prob < 0).any()
        or abs(prob.sum() - 1.0) > 1e-5
    ):
        raise ValueError("Invalid prediction: " + str(key))

    records[key] = (label, prob)

expected = None

for model in MODELS:
    keys = {
        (cohort, gsm)
        for cohort, method, gsm in records
        if method == model
    }

    if expected is None:
        expected = keys
    elif keys != expected:
        raise RuntimeError("Model sample sets differ")

if len(expected) != 785:
    raise RuntimeError("Unexpected development sample count")

for cohort, gsm in expected:
    observed = {
        records[(cohort, model, gsm)][0]
        for model in MODELS
    }

    if len(observed) != 1:
        raise RuntimeError("Cross-model label disagreement")

metric_rows = []
pair_rows = []
loss_by_model = {}

for cohort in DEV:
    sample_ids = sorted(
        gsm for c, gsm in expected if c == cohort
    )

    if not sample_ids:
        raise RuntimeError("Empty cohort")

    for model in MODELS:
        truth = np.array([
            records[(cohort, model, gsm)][0]
            for gsm in sample_ids
        ], dtype=np.int64)

        prob = np.stack([
            records[(cohort, model, gsm)][1]
            for gsm in sample_ids
        ])

        ptrue = np.clip(
            prob[np.arange(len(truth)), truth],
            1e-12, 1.0
        )

        loss = -np.log(ptrue)
        loss_by_model[(cohort, model)] = loss
        prediction = prob.argmax(axis=1)

        egfr = truth == 0

        pe = np.clip(prob[:, 0], 1e-12, 1 - 1e-12)

        loss_e = -np.mean(np.where(
            egfr,
            np.log(pe),
            np.log(1 - pe),
        ))

        non_egfr = ~egfr

        pk = np.clip(
            prob[non_egfr, 1]
            / (
                prob[non_egfr, 1]
                + prob[non_egfr, 2]
            ),
            1e-12,
            1 - 1e-12,
        )

        kras = truth[non_egfr] == 1

        loss_k = -np.mean(np.where(
            kras,
            np.log(pk),
            np.log(1 - pk),
        ))

        tail_n = max(
            1, int(np.ceil(0.10 * len(truth)))
        )

        tail_share = (
            np.sort(loss)[-tail_n:].sum()
            / loss.sum()
        )

        metric_rows.append({
            "cohort": cohort,
            "model": model,
            "n": len(truth),
            "nll": round(float(loss.mean()), 6),
            "egfr_head_nll": round(float(loss_e), 6),
            "conditional_kras_nll": round(float(loss_k), 6),
            "observed_egfr_fraction": round(
                float(egfr.mean()), 6
            ),
            "predicted_egfr_mean": round(
                float(prob[:, 0].mean()), 6
            ),
            "median_true_probability": round(
                float(np.median(ptrue)), 6
            ),
            "true_probability_lt_0_001": int(
                np.sum(ptrue < 0.001)
            ),
            "overconfident_wrong_gt_0_95": int(
                np.sum(
                    (prediction != truth)
                    & (prob.max(axis=1) > 0.95)
                )
            ),
            "worst_10pct_loss_share": round(
                float(tail_share), 6
            ),
            "max_individual_nll": round(
                float(loss.max()), 6
            ),
        })

    for comparator in (
        "hier_linear", "k_residual", "generic_mask"
    ):
        diff = (
            loss_by_model[(cohort, comparator)]
            - loss_by_model[(cohort, "contrast_mask")]
        )

        pair_rows.append({
            "cohort": cohort,
            "comparator": comparator,
            "mean_gain_for_contrast_mask": round(
                float(diff.mean()), 6
            ),
            "positive_sample_fraction": round(
                float(np.mean(diff > 0)), 6
            ),
            "median_gain_for_contrast_mask": round(
                float(np.median(diff)), 6
            ),
        })

def adam_constant_gradient(factor):
    parameter = torch.nn.Parameter(
        torch.tensor(0.0, dtype=torch.float64)
    )

    optimizer = torch.optim.AdamW(
        [parameter],
        lr=0.01,
        weight_decay=0.0,
    )

    for _ in range(25):
        optimizer.zero_grad(set_to_none=True)

        parameter.grad = torch.tensor(
            float(factor),
            dtype=torch.float64,
        )

        optimizer.step()

    return float(parameter.detach())

adam_unmasked = adam_constant_gradient(1.0)
adam_masked = adam_constant_gradient(0.25)

adam_difference = abs(
    adam_masked - adam_unmasked
)

if adam_difference > 1e-6:
    raise RuntimeError("Unexpected Adam scaling test")

run_id = (
    datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%SZ"
    )
    + "_" + uuid.uuid4().hex[:8]
)

RUN = ROOT / "06_model_forensics" / run_id
RUN.mkdir(parents=True, exist_ok=False)

artifacts = []

def save(name, content):
    data = (
        content.encode("utf-8")
        if isinstance(content, str)
        else content
    )

    path = RUN / name

    with path.open("xb") as handle:
        handle.write(data)

    artifacts.append({
        "path": str(
            path.relative_to(ROOT)
        ).replace("\\", "/"),
        "sha256": hashlib.sha256(data).hexdigest(),
    })

def csv_output(items):
    buffer = io.StringIO(newline="")

    writer = csv.DictWriter(
        buffer,
        fieldnames=list(items[0]),
        lineterminator="\n",
    )

    writer.writeheader()
    writer.writerows(items)

    return buffer.getvalue()

save(
    "COHORT_PROBABILITY_FORENSICS.csv",
    csv_output(metric_rows),
)

save(
    "PAIRED_LOSS_COMPARISONS.csv",
    csv_output(pair_rows),
)

summary = {
    "phase": "DEVELOPMENT_PREDICTION_FORENSICS",
    "source_manifest": str(
        parent.relative_to(ROOT)
    ).replace("\\", "/"),
    "source_manifest_sha256": hashlib.sha256(
        parent.read_bytes()
    ).hexdigest(),
    "samples": len(expected),
    "models": list(MODELS),
    "adam_constant_grad_unmasked": adam_unmasked,
    "adam_constant_grad_masked_025": adam_masked,
    "adam_step_difference": adam_difference,
    "training_performed": False,
    "architecture_approved": False,
    "artifacts": artifacts,
}

save(
    "RUN_MANIFEST.json",
    json.dumps(summary, indent=2) + "\n",
)

print("FORENSIC_AUDIT: PASS")
print("SAMPLES:", len(expected))
print(
    "ADAM_MASK_UPDATE_DIFFERENCE:",
    f"{adam_difference:.2e}",
)

for cohort in DEV:
    print("COHORT:", cohort)

    for row in metric_rows:
        if row["cohort"] == cohort:
            print(
                row["model"],
                "NLL=", row["nll"],
                "EGFR_NLL=", row["egfr_head_nll"],
                "K_NLL=", row["conditional_kras_nll"],
                "TINY_TRUE_P=",
                row["true_probability_lt_0_001"],
                "WRONG_CONFIDENT=",
                row["overconfident_wrong_gt_0_95"],
            )

    for row in pair_rows:
        if row["cohort"] == cohort:
            print(
                "DELTA:",
                row["comparator"],
                row["mean_gain_for_contrast_mask"],
            )

print("TRAINING: NO")
print("RUN:", RUN)

FORENSIC_AUDIT: PASS
SAMPLES: 785
ADAM_MASK_UPDATE_DIFFERENCE: 7.50e-09
COHORT: GSE13213
flat_lr NLL= 4.840116 EGFR_NLL= 4.723652 K_NLL= 0.687795 TINY_TRUE_P= 41 WRONG_CONFIDENT= 50
hier_linear NLL= 4.534356 EGFR_NLL= 3.419007 K_NLL= 1.812441 TINY_TRUE_P= 39 WRONG_CONFIDENT= 50
k_residual NLL= 4.456393 EGFR_NLL= 3.182144 K_NLL= 2.070655 TINY_TRUE_P= 38 WRONG_CONFIDENT= 48
generic_mask NLL= 2.211017 EGFR_NLL= 1.32765 K_NLL= 1.435471 TINY_TRUE_P= 9 WRONG_CONFIDENT= 21
contrast_mask NLL= 4.487909 EGFR_NLL= 3.182144 K_NLL= 2.121868 TINY_TRUE_P= 37 WRONG_CONFIDENT= 48
DELTA: hier_linear 0.046446
DELTA: k_residual -0.031516
DELTA: generic_mask -2.276893
COHORT: GSE31210
flat_lr NLL= 6.070369 EGFR_NLL= 5.49281 K_NLL= 1.318468 TINY_TRUE_P= 125 WRONG_CONFIDENT= 146
hier_linear NLL= 1.970812 EGFR_NLL= 1.779494 K_NLL= 0.436746 TINY_TRUE_P= 1 WRONG_CONFIDENT= 10
k_residual NLL= 2.140357 EGFR_NLL= 1.910136 K_NLL= 0.525557 TINY_TRUE_P= 1 WRONG_CONFIDENT= 46
generic_mask NLL= 2.317478 EGFR_NLL= 2.110

In [11]:

import csv
import hashlib
import io
import json
import os
import time
import uuid
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import rankdata
from sklearn.metrics import f1_score, balanced_accuracy_score

ROOT = Path(r"D:\LUAD_DATAFIRST_METHOD_2026")
SOURCE = Path(
    r"D:\LUAD_CMR_2026\CELL03_DATA_20261021T090947_420335Z_d6aed8ec\cmr_gene_expression.npz"
)
IDENTITY = ROOT / "00_admin/project.json"
DEV = ("GSE13213", "GSE31210", "GSE72094")

SEED = 20261023
GENES_PER_HEAD = 24
HIDDEN = 6
EPOCHS = 180
LR = 0.018
EGFR_CAP = 2.8
KRAS_CAP = 2.2
MIXTURE = 0.08

if os.name != "nt" or not Path("D:/").is_dir():
    raise RuntimeError("Windows D: drive unavailable")

if not IDENTITY.is_file() or not SOURCE.is_file():
    raise FileNotFoundError("Project identity or expression source missing")

identity = json.loads(IDENTITY.read_text(encoding="utf-8"))

if identity.get("project_id") != "LUAD_DATAFIRST_METHOD_2026":
    raise RuntimeError("Project identity mismatch")

audit_files = sorted(
    (ROOT / "04_actual_expression_audit").glob(
        "*/CROSS_SOURCE_VALIDATION.json"
    )
)

if not audit_files:
    raise RuntimeError("Expression audit missing")

audit = json.loads(
    audit_files[-1].read_text(encoding="utf-8")
)

if (
    not audit.get("label_mapping_usable")
    or audit.get("gse72094_label_agreement") != 1.0
):
    raise RuntimeError("Existing sample-label audit is incomplete")

label_map = {
    int(k): v
    for k, v in audit[
        "label_map_inferred_from_GSE72094"
    ].items()
}

if label_map != {0: "EGFR", 1: "KRAS", 2: "DWT"}:
    raise RuntimeError("Label ontology mismatch")

with np.load(SOURCE, allow_pickle=False) as source:
    required = {
        "X", "y", "cohort", "gsm", "genes", "patient_ids"
    }

    if not required.issubset(set(source.files)):
        raise RuntimeError("Source array schema mismatch")

    all_cohorts = np.char.upper(
        np.char.strip(source["cohort"].astype(str))
    )

    take = np.isin(all_cohorts, DEV)

    X = np.asarray(
        source["X"][take], dtype=np.float32
    )
    y = np.asarray(
        source["y"][take], dtype=np.int64
    )
    cohort = np.asarray(all_cohorts[take], dtype=str)
    gsm = np.char.upper(
        np.char.strip(source["gsm"][take].astype(str))
    )
    patients = np.char.strip(
        source["patient_ids"][take].astype(str)
    )
    genes = np.char.upper(
        np.char.strip(source["genes"].astype(str))
    )

if X.shape != (785, 13016):
    raise RuntimeError("Development matrix composition mismatch")

if len(set(gsm.tolist())) != 785:
    raise RuntimeError("Duplicate sample identifiers")

if not np.isfinite(X).all() or set(y.tolist()) != {0, 1, 2}:
    raise RuntimeError("Expression or label validation failed")

if np.unique(genes).size != len(genes):
    raise RuntimeError("Nonunique gene identifiers")

if any(
    (cohort == c).sum() != n
    for c, n in zip(DEV, (117, 226, 442))
):
    raise RuntimeError("Unexpected cohort sizes")

patient_domains = {}

for c, p in zip(cohort, patients):
    if p and p.casefold() not in {
        "na", "nan", "none", "unknown"
    }:
        patient_domains.setdefault(p, set()).add(c)

if any(len(v) > 1 for v in patient_domains.values()):
    raise RuntimeError("Repeated patient IDs across cohorts")

ranks = (
    rankdata(
        X, axis=1, method="average"
    ).astype(np.float32) - 1.0
) / (X.shape[1] - 1)

del X

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(
    min(2, max(1, os.cpu_count() or 1))
)

def features_for_fold(train_idx):
    selections = []

    training_cohorts = sorted(
        set(cohort[train_idx].tolist())
    )

    if len(training_cohorts) != 2:
        raise RuntimeError("Expected exactly two source cohorts")

    for cls in (0, 1):
        effects = []

        for source_cohort in training_cohorts:
            idx = train_idx[
                cohort[train_idx] == source_cohort
            ]

            r = ranks[idx]
            yy = y[idx]

            positive = r[yy == cls]

            negative = (
                r[yy != 0]
                if cls == 0
                else r[yy == 2]
            )

            if min(len(positive), len(negative)) < 10:
                raise RuntimeError(
                    "Insufficient cases for feature selection"
                )

            diff = (
                positive.mean(0) - negative.mean(0)
            ) / np.maximum(r.std(0), 0.04)

            effects.append(diff)

        effect = np.stack(effects)

        consistent = (
            (effect > 0).all(0)
            | (effect < 0).all(0)
        )

        score = np.where(
            consistent,
            np.min(np.abs(effect), axis=0),
            -np.inf,
        )

        chosen = np.argsort(
            -score, kind="stable"
        )[:GENES_PER_HEAD]

        if not np.isfinite(score[chosen]).all():
            raise RuntimeError(
                "Insufficient cohort-consistent genes"
            )

        selections.append((
            chosen,
            np.sign(effect[0, chosen]).astype(np.float32),
            score[chosen],
        ))

    return selections

class EvidenceNet(nn.Module):
    def __init__(
        self,
        sign_e,
        sign_k,
        prior,
        signed,
        bounded,
        residual,
        mixture,
    ):
        super().__init__()

        self.signed = signed
        self.bounded = bounded
        self.residual = residual
        self.mixture = float(mixture)

        self.register_buffer(
            "sign_e",
            torch.tensor(sign_e, dtype=torch.float32),
        )
        self.register_buffer(
            "sign_k",
            torch.tensor(sign_k, dtype=torch.float32),
        )
        self.register_buffer(
            "log_prior",
            torch.log(
                torch.tensor(prior, dtype=torch.float32)
            ),
        )

        if signed:
            self.we = nn.Parameter(
                torch.full((len(sign_e),), -2.7)
            )
            self.wk = nn.Parameter(
                torch.full((len(sign_k),), -2.7)
            )
        else:
            self.we = nn.Parameter(
                torch.empty(len(sign_e)).normal_(0, 0.03)
            )
            self.wk = nn.Parameter(
                torch.empty(len(sign_k)).normal_(0, 0.03)
            )

        self.be = nn.Parameter(torch.zeros(()))
        self.bk = nn.Parameter(torch.zeros(()))

        if residual:
            self.hidden = nn.Linear(
                len(sign_k), HIDDEN, bias=False
            )
            self.output = nn.Linear(
                HIDDEN, 1, bias=False
            )

            nn.init.normal_(
                self.hidden.weight, std=0.14
            )
            nn.init.normal_(
                self.output.weight, std=0.08
            )

    def effective_weights(self):
        if self.signed:
            return (
                self.sign_e * F.softplus(self.we),
                self.sign_k * F.softplus(self.wk),
            )

        return self.we, self.wk

    def forward(self, xe, xk):
        we, wk = self.effective_weights()

        e = (xe * we).sum(1) + self.be
        k = (xk * wk).sum(1) + self.bk

        if self.residual:
            h = torch.tanh(self.hidden(xk))

            k = k + torch.tanh(
                self.output(h).squeeze(1)
            )

        if self.bounded:
            e = EGFR_CAP * torch.tanh(
                e / EGFR_CAP
            )
            k = KRAS_CAP * torch.tanh(
                k / KRAS_CAP
            )

        logp = torch.stack((
            F.logsigmoid(e),
            F.logsigmoid(-e) + F.logsigmoid(k),
            F.logsigmoid(-e) + F.logsigmoid(-k),
        ), dim=1)

        if self.mixture > 0:
            logp = torch.logaddexp(
                logp + np.log1p(-self.mixture),
                self.log_prior[None, :]
                + np.log(self.mixture),
            )

        return logp

SPECS = {
    "free_linear": (
        False, False, False, 0.0
    ),
    "free_bounded": (
        False, True, False, 0.0
    ),
    "signed_bounded": (
        True, True, False, 0.0
    ),
    "signed_k_residual": (
        True, True, True, 0.0
    ),
    "free_k_residual": (
        False, True, True, 0.0
    ),
    "signed_k_residual_prior": (
        True, True, True, MIXTURE
    ),
}

def train_one(
    name, signs, prior, ztrain, labels, groups
):
    signed, bounded, residual, mixture = SPECS[name]

    torch.manual_seed(SEED)

    model = EvidenceNet(
        *signs,
        prior,
        signed,
        bounded,
        residual,
        mixture,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=0.005,
    )

    xe, xk = ztrain
    yt = torch.as_tensor(
        labels, dtype=torch.long
    )

    group_masks = [
        torch.as_tensor(groups == c)
        for c in sorted(set(groups.tolist()))
    ]

    for _ in range(EPOCHS):
        optimizer.zero_grad(set_to_none=True)

        logp = model(xe, xk)

        loss = torch.stack([
            F.nll_loss(
                logp[mask], yt[mask]
            )
            for mask in group_masks
        ]).mean()

        we, wk = model.effective_weights()

        loss = loss + 0.02 * (
            we.square().sum()
            + wk.square().sum()
        )

        if residual:
            loss = loss + 0.008 * (
                model.hidden.weight.square().sum()
                + model.output.weight.square().sum()
            )

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Nonfinite training loss: {name}"
            )

        loss.backward()
        optimizer.step()

    with torch.no_grad():
        end_loss = torch.stack([
            F.nll_loss(
                model(xe, xk)[mask],
                yt[mask],
            )
            for mask in group_masks
        ]).mean().item()

    return model, float(end_loss)

def scores(true, probability):
    if (
        np.max(
            np.abs(probability.sum(1) - 1.0)
        ) > 3e-5
        or not np.isfinite(probability).all()
    ):
        raise RuntimeError(
            "Invalid prediction probabilities"
        )

    prob = np.clip(
        probability, 1e-12, 1.0
    )
    pred = prob.argmax(1)

    nll = float(
        -np.log(
            prob[np.arange(len(true)), true]
        ).mean()
    )

    mask = true != 0

    q = np.clip(
        prob[mask, 1]
        / (
            prob[mask, 1]
            + prob[mask, 2]
        ),
        1e-12,
        1 - 1e-12,
    )

    ktruth = (
        true[mask] == 1
    ).astype(float)

    return {
        "nll": nll,
        "macro_f1": float(
            f1_score(
                true,
                pred,
                labels=[0, 1, 2],
                average="macro",
                zero_division=0,
            )
        ),
        "balanced_acc": float(
            balanced_accuracy_score(
                true, pred
            )
        ),
        "kras_nll": float(
            -(
                ktruth * np.log(q)
                + (1 - ktruth) * np.log(1 - q)
            ).mean()
        ),
        "brier": float(
            (
                (
                    prob - np.eye(3)[true]
                ) ** 2
            ).sum(1).mean()
        ),
        "tiny_true_p": int(
            (
                prob[
                    np.arange(len(true)), true
                ] < 1e-3
            ).sum()
        ),
        "confident_wrong": int(
            (
                (pred != true)
                & (prob.max(1) > 0.95)
            ).sum()
        ),
    }

def as_csv(rows):
    mem = io.StringIO(newline="")

    writer = csv.DictWriter(
        mem,
        fieldnames=list(rows[0]),
        lineterminator="\n",
    )

    writer.writeheader()
    writer.writerows(rows)

    return mem.getvalue()

run_id = (
    datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%SZ"
    )
    + "_"
    + uuid.uuid4().hex[:8]
)

RUN = (
    ROOT / "07_signed_bounded_evidence_pilot"
    / run_id
)
RUN.mkdir(parents=True, exist_ok=False)

written = []

def save(name, payload):
    path = RUN / name
    path.parent.mkdir(
        parents=True, exist_ok=True
    )

    data = (
        payload.encode("utf-8")
        if isinstance(payload, str)
        else payload
    )

    with path.open("xb") as handle:
        handle.write(data)

    written.append({
        "path": str(
            path.relative_to(ROOT)
        ).replace("\\", "/"),
        "sha256": hashlib.sha256(
            data
        ).hexdigest(),
    })

protocol = {
    "phase": "SIGNED_BOUNDED_EVIDENCE_DEVELOPMENT",
    "run_id": run_id,
    "historical_outcome_exposure": True,
    "source": str(SOURCE),
    "development_cohorts": list(DEV),
    "gse32863_not_untouched": True,
    "gse66863_prediction_evaluation_this_run": False,
    "genotype_ontology": {
        "0": "EGFR",
        "1": "KRAS",
        "2": "Double-WT",
    },
    "per_head_features": GENES_PER_HEAD,
    "hidden": HIDDEN,
    "epochs": EPOCHS,
    "learning_rate": LR,
    "cap_egfr": EGFR_CAP,
    "cap_kras": KRAS_CAP,
    "mixture": MIXTURE,
    "architecture_variants": SPECS,
    "methodological_novelty": "NOT_ESTABLISHED",
    "primary_model": "signed_k_residual",
    "screen_min_gain_nll": 0.02,
    "screen_positive_cohorts_min": 2,
    "clinical_assay_provenance": "PROVISIONAL",
    "feature_universe": "ARCHIVED_843_COHORT_SHARED_PANEL",
}

save(
    "PROTOCOL.json",
    json.dumps(protocol, indent=2) + "\n",
)

rows = []
examples = []
train_stats = []
start = time.monotonic()

for held in DEV:
    train = np.flatnonzero(
        cohort != held
    )
    test = np.flatnonzero(
        cohort == held
    )

    selections = features_for_fold(train)

    prepared = []
    transformed = []

    for chosen, direction, reliability in selections:
        base = ranks[train][:, chosen]

        mu = base.mean(0)
        sd = np.maximum(
            base.std(0), 0.08
        )

        z = np.tanh(
            (ranks[:, chosen] - mu)
            / (2.0 * sd)
        ).astype(np.float32)

        prepared.append({
            "indices": chosen.tolist(),
            "genes": genes[chosen].tolist(),
            "signs": direction.tolist(),
            "consensus_strength": reliability.tolist(),
            "mean": mu.tolist(),
            "sd": sd.tolist(),
        })

        transformed.append(z)

    signs = (
        selections[0][1],
        selections[1][1],
    )

    train_cohorts = sorted(
        set(cohort[train].tolist())
    )

    prior = np.mean([
        np.bincount(
            y[train][
                cohort[train] == source
            ],
            minlength=3,
        )
        / np.sum(
            cohort[train] == source
        )
        for source in train_cohorts
    ], axis=0)

    if np.any(prior <= 0):
        raise RuntimeError(
            "Empty class in source prior"
        )

    save(
        f"folds/{held}_features.json",
        json.dumps({
            "egfr": prepared[0],
            "kras": prepared[1],
            "equal_cohort_train_prior": prior.tolist(),
        }, indent=2) + "\n",
    )

    xtrain = (
        torch.from_numpy(
            transformed[0][train]
        ),
        torch.from_numpy(
            transformed[1][train]
        ),
    )

    xtest = (
        torch.from_numpy(
            transformed[0][test]
        ),
        torch.from_numpy(
            transformed[1][test]
        ),
    )

    for name in SPECS:
        model, train_loss = train_one(
            name,
            signs,
            prior,
            xtrain,
            y[train],
            cohort[train],
        )

        with torch.no_grad():
            prob = model(
                *xtest
            ).exp().numpy()

            effective = model.effective_weights()

            weights = (
                effective[0].cpu().numpy(),
                effective[1].cpu().numpy(),
            )

        s = scores(y[test], prob)

        rows.append({
            "cohort": held,
            "model": name,
            "n": len(test),
            "train_nll_equal_cohort": train_loss,
            **s,
        })

        train_stats.append({
            "cohort": held,
            "model": name,
            "l1_egfr_weights": float(
                np.abs(weights[0]).sum()
            ),
            "l1_kras_weights": float(
                np.abs(weights[1]).sum()
            ),
        })

        for j, sample in enumerate(test):
            examples.append({
                "cohort": held,
                "gsm": str(gsm[sample]),
                "class": int(y[sample]),
                "model": name,
                "p_egfr": float(prob[j, 0]),
                "p_kras": float(prob[j, 1]),
                "p_dwt": float(prob[j, 2]),
            })

        buffer = io.BytesIO()

        torch.save({
            "state_dict": model.state_dict(),
            "spec": name,
            "features": prepared,
            "prior": prior.tolist(),
            "training_cohorts": train_cohorts,
        }, buffer)

        save(
            f"models/{held}_{name}.pt",
            buffer.getvalue(),
        )

save(
    "LOCO_METRICS.csv",
    as_csv(rows),
)
save(
    "LOCO_PREDICTIONS.csv",
    as_csv(examples),
)
save(
    "MODEL_COEFFICIENT_SUMMARY.csv",
    as_csv(train_stats),
)

summary = {}

for name in SPECS:
    scores_ = [
        row for row in rows
        if row["model"] == name
    ]

    summary[name] = {
        "mean_nll": float(
            np.mean([
                r["nll"] for r in scores_
            ])
        ),
        "mean_macro_f1": float(
            np.mean([
                r["macro_f1"] for r in scores_
            ])
        ),
        "mean_kras_nll": float(
            np.mean([
                r["kras_nll"] for r in scores_
            ])
        ),
        "cohort_nll": {
            r["cohort"]: r["nll"]
            for r in scores_
        },
    }

candidate = "signed_k_residual"

comparators = (
    "free_linear",
    "free_bounded",
    "signed_bounded",
    "free_k_residual",
    "signed_k_residual_prior",
)

gains = {
    name: (
        summary[name]["mean_nll"]
        - summary[candidate]["mean_nll"]
    )
    for name in comparators
}

pass_count = sum(
    summary["free_k_residual"][
        "cohort_nll"
    ][c]
    >
    summary[candidate][
        "cohort_nll"
    ][c]
    for c in DEV
)

status = (
    "PASS_SCREEN"
    if (
        gains["free_bounded"] >= 0.02
        and gains["free_k_residual"] >= 0.02
        and pass_count >= 2
        and summary[candidate][
            "mean_kras_nll"
        ]
        <
        summary["free_k_residual"][
            "mean_kras_nll"
        ]
    )
    else "HOLD"
)

result = {
    "summary": summary,
    "gains_for_candidate": gains,
    "positive_cohorts_vs_free_residual": pass_count,
    "screen": status,
    "exploratory_only": True,
}

save(
    "EQUAL_COHORT_SUMMARY.json",
    json.dumps(result, indent=2) + "\n",
)

hash_ = hashlib.sha256()

with SOURCE.open("rb") as handle:
    for block in iter(
        lambda: handle.read(4_194_304),
        b"",
    ):
        hash_.update(block)

save(
    "RUN_MANIFEST.json",
    json.dumps({
        "phase": "SIGNED_BOUNDED_EVIDENCE_DEVELOPMENT",
        "source_sha256": hash_.hexdigest(),
        "audit_sha256": hashlib.sha256(
            audit_files[-1].read_bytes()
        ).hexdigest(),
        "artifacts": written,
        "runtime_seconds": round(
            time.monotonic() - start, 2
        ),
        "external_test_accessed": False,
        "novelty_established": False,
    }, indent=2) + "\n",
)

print("DEVELOPMENT: COMPLETE")

for c in DEV:
    print(
        "LOCO", c,
        "N=", sum(cohort == c),
        "CLASS=", dict(
            Counter(
                y[cohort == c].tolist()
            )
        ),
    )

    for name in SPECS:
        row = next(
            r for r in rows
            if r["cohort"] == c
            and r["model"] == name
        )

        print(
            name,
            "NLL=", round(row["nll"], 4),
            "F1=", round(row["macro_f1"], 4),
            "K_NLL=", round(row["kras_nll"], 4),
            "OVERCONF=", row["confident_wrong"],
        )

for name, r in summary.items():
    print(
        "MEAN", name,
        "NLL=", round(r["mean_nll"], 4),
        "F1=", round(r["mean_macro_f1"], 4),
        "K_NLL=", round(r["mean_kras_nll"], 4),
    )

print("SCREEN:", status)
print(
    "GAIN_VS_FREE_RESIDUAL:",
    round(
        gains["free_k_residual"], 4
    ),
)
print(
    "GAIN_VS_BOUNDED_LINEAR:",
    round(
        gains["free_bounded"], 4
    ),
)
print("RUN:", RUN)

DEVELOPMENT: COMPLETE
LOCO GSE13213 N= 117 CLASS= {0: 45, 2: 57, 1: 15}
free_linear NLL= 0.8924 F1= 0.5838 K_NLL= 0.5165 OVERCONF= 0
free_bounded NLL= 0.9019 F1= 0.5814 K_NLL= 0.5265 OVERCONF= 0
signed_bounded NLL= 0.9303 F1= 0.5251 K_NLL= 0.5359 OVERCONF= 0
signed_k_residual NLL= 0.9392 F1= 0.5319 K_NLL= 0.5505 OVERCONF= 0
free_k_residual NLL= 0.9135 F1= 0.573 K_NLL= 0.5455 OVERCONF= 0
signed_k_residual_prior NLL= 0.9304 F1= 0.53 K_NLL= 0.5424 OVERCONF= 0
LOCO GSE31210 N= 226 CLASS= {1: 20, 2: 79, 0: 127}
free_linear NLL= 1.6357 F1= 0.1727 K_NLL= 0.4313 OVERCONF= 0
free_bounded NLL= 1.3768 F1= 0.1727 K_NLL= 0.4451 OVERCONF= 0
signed_bounded NLL= 1.3735 F1= 0.205 K_NLL= 0.4447 OVERCONF= 0
signed_k_residual NLL= 1.3736 F1= 0.2019 K_NLL= 0.4449 OVERCONF= 0
free_k_residual NLL= 1.3769 F1= 0.205 K_NLL= 0.4453 OVERCONF= 0
signed_k_residual_prior NLL= 1.3354 F1= 0.2019 K_NLL= 0.4469 OVERCONF= 0
LOCO GSE72094 N= 442 CLASS= {0: 47, 2: 241, 1: 154}
free_linear NLL= 1.1455 F1= 0.3465 K_NLL= 0.75

In [12]:

import csv
import hashlib
import io
import json
import os
import uuid
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.special import expit
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

ROOT = Path(r"D:\LUAD_DATAFIRST_METHOD_2026")
SOURCE = Path(
    r"D:\LUAD_CMR_2026\CELL03_DATA_20261021T090947_420335Z_d6aed8ec\cmr_gene_expression.npz"
)
COHORTS = ("GSE13213", "GSE31210", "GSE72094")
MODELS = (
    "free_linear",
    "free_bounded",
    "signed_bounded",
    "signed_k_residual",
    "free_k_residual",
    "signed_k_residual_prior",
)

identity_path = ROOT / "00_admin/project.json"

if os.name != "nt" or not Path("D:/").is_dir():
    raise RuntimeError("Windows D: drive unavailable")

if not identity_path.is_file() or not SOURCE.is_file():
    raise FileNotFoundError("Required project source unavailable")

identity = json.loads(identity_path.read_text(encoding="utf-8"))

if identity.get("project_id") != "LUAD_DATAFIRST_METHOD_2026":
    raise RuntimeError("Project identity mismatch")

run_manifests = sorted(
    (ROOT / "07_signed_bounded_evidence_pilot").glob(
        "*/RUN_MANIFEST.json"
    )
)

if not run_manifests:
    raise RuntimeError("Phase 07 run not found")

parent_path = run_manifests[-1]
parent = json.loads(parent_path.read_text(encoding="utf-8"))

if parent.get("phase") != "SIGNED_BOUNDED_EVIDENCE_DEVELOPMENT":
    raise RuntimeError("Unexpected Phase 07 manifest")

for artifact in parent.get("artifacts", []):
    path = ROOT / artifact["path"]

    if (
        not path.is_file()
        or hashlib.sha256(path.read_bytes()).hexdigest()
        != artifact["sha256"]
    ):
        raise RuntimeError(
            "Artifact checksum failed: " + str(path)
        )

prediction_file = parent_path.parent / "LOCO_PREDICTIONS.csv"
frame = pd.read_csv(prediction_file)

required = {
    "cohort", "gsm", "class", "model",
    "p_egfr", "p_kras", "p_dwt",
}

if not required.issubset(frame.columns):
    raise RuntimeError("Prediction schema mismatch")

if (
    set(frame["cohort"]) != set(COHORTS)
    or set(frame["model"]) != set(MODELS)
):
    raise RuntimeError("Unexpected cohort or model")

if len(frame) != 785 * len(MODELS):
    raise RuntimeError("Incomplete prediction count")

if frame.duplicated(["cohort", "gsm", "model"]).any():
    raise RuntimeError("Duplicate prediction keys")

pcols = ["p_egfr", "p_kras", "p_dwt"]
pall = frame[pcols].to_numpy(dtype=np.float64)

if (
    not np.isfinite(pall).all()
    or (pall < 0).any()
    or (pall > 1).any()
):
    raise RuntimeError("Invalid probabilities")

if not np.allclose(
    pall.sum(axis=1), 1.0, atol=1e-5, rtol=0
):
    raise RuntimeError("Probabilities do not sum to one")

if not frame["class"].isin([0, 1, 2]).all():
    raise RuntimeError("Invalid genotype codes")

base = frame.loc[
    frame["model"] == "free_bounded",
    ["cohort", "gsm", "class"],
].sort_values(["cohort", "gsm"])

for model in MODELS:
    subset = frame.loc[
        frame["model"] == model,
        ["cohort", "gsm", "class"],
    ].sort_values(["cohort", "gsm"])

    if not subset.reset_index(drop=True).equals(
        base.reset_index(drop=True)
    ):
        raise RuntimeError("Sample or label alignment failed")

with np.load(SOURCE, allow_pickle=False) as archive:
    source_y = np.asarray(archive["y"], dtype=np.int64)

    source_domains = np.char.upper(
        np.char.strip(archive["cohort"].astype(str))
    )
    source_gsms = np.char.upper(
        np.char.strip(archive["gsm"].astype(str))
    )

if len(source_y) != 843:
    raise RuntimeError("Unexpected source size")

reference = {
    (d, g): int(label)
    for d, g, label in zip(
        source_domains, source_gsms, source_y
    )
}

if len(reference) != len(source_y):
    raise RuntimeError("Duplicate expression sample keys")

for c, g, label in base.itertuples(index=False, name=None):
    if reference.get((c, g)) != int(label):
        raise RuntimeError(
            "Prediction-to-expression label mismatch"
        )

native_rows = []
shift_rows = []

for held in COHORTS:
    training_cohorts = [
        c for c in COHORTS if c != held
    ]

    prior_e = float(np.mean([
        (source_y[source_domains == c] == 0).mean()
        for c in training_cohorts
    ]))

    prior_k = float(np.mean([
        (
            source_y[
                (source_domains == c)
                & (source_y != 0)
            ] == 1
        ).mean()
        for c in training_cohorts
    ]))

    if not (0 < prior_e < 1 and 0 < prior_k < 1):
        raise RuntimeError("Degenerate training priors")

    offset_e = -np.log(
        prior_e / (1 - prior_e)
    )
    offset_k = -np.log(
        prior_k / (1 - prior_k)
    )

    for model in MODELS:
        subset = frame.loc[
            (frame["cohort"] == held)
            & (frame["model"] == model)
        ].sort_values("gsm")

        truth = subset["class"].to_numpy(dtype=int)
        prob = subset[pcols].to_numpy(dtype=np.float64)
        pred = prob.argmax(axis=1)

        egfr_truth = (truth == 0).astype(int)
        non_egfr = truth != 0

        q_k = (
            prob[non_egfr, 1]
            / (
                prob[non_egfr, 1]
                + prob[non_egfr, 2]
            )
        )

        kras_truth = (
            truth[non_egfr] == 1
        ).astype(int)

        auc_e = float(
            roc_auc_score(egfr_truth, prob[:, 0])
        )
        auc_k = float(
            roc_auc_score(kras_truth, q_k)
        )

        ap_e = float(
            average_precision_score(
                egfr_truth, prob[:, 0]
            )
        )
        ap_k = float(
            average_precision_score(
                kras_truth, q_k
            )
        )

        f1 = float(
            f1_score(
                truth, pred,
                labels=[0, 1, 2],
                average="macro",
                zero_division=0,
            )
        )

        nll = float(
            -np.mean(np.log(np.maximum(
                prob[np.arange(len(truth)), truth],
                1e-12,
            )))
        )

        counts = np.bincount(
            pred, minlength=3
        )

        native_rows.append({
            "held_cohort": held,
            "model": model,
            "n": len(truth),
            "observed_egfr_fraction": float(
                egfr_truth.mean()
            ),
            "source_mean_egfr_fraction": prior_e,
            "egfr_auc": auc_e,
            "egfr_average_precision": ap_e,
            "egfr_median_mut": float(
                np.median(prob[egfr_truth == 1, 0])
            ),
            "egfr_median_other": float(
                np.median(prob[egfr_truth == 0, 0])
            ),
            "conditional_kras_auc": auc_k,
            "conditional_kras_average_precision": ap_k,
            "pred_egfr": int(counts[0]),
            "pred_kras": int(counts[1]),
            "pred_dwt": int(counts[2]),
            "macro_f1": f1,
            "nll": nll,
        })

        odds_e = (
            np.log(np.clip(prob[:, 0], 1e-12, 1.0))
            - np.log(np.clip(
                prob[:, 1] + prob[:, 2],
                1e-12, 1.0
            ))
        )

        odds_k = (
            np.log(np.clip(prob[:, 1], 1e-12, 1.0))
            - np.log(np.clip(prob[:, 2], 1e-12, 1.0))
        )

        ref_e = expit(odds_e + offset_e)
        ref_k = expit(odds_k + offset_k)

        adjusted = np.column_stack((
            ref_e,
            (1 - ref_e) * ref_k,
            (1 - ref_e) * (1 - ref_k),
        ))

        if not np.allclose(
            adjusted.sum(axis=1),
            1.0,
            atol=1e-10,
        ):
            raise RuntimeError(
                "Reference probability normalization failed"
            )

        adjusted_pred = adjusted.argmax(axis=1)

        adjusted_nll = float(
            -np.mean(np.log(np.maximum(
                adjusted[
                    np.arange(len(truth)), truth
                ],
                1e-12,
            )))
        )

        adjusted_f1 = float(
            f1_score(
                truth, adjusted_pred,
                labels=[0, 1, 2],
                average="macro",
                zero_division=0,
            )
        )

        adj_counts = np.bincount(
            adjusted_pred, minlength=3
        )

        shift_rows.append({
            "held_cohort": held,
            "model": model,
            "reference": "neutral_binary_head_prior",
            "source_egfr_fraction": prior_e,
            "source_kras_conditional_fraction": prior_k,
            "original_nll": nll,
            "reference_nll": adjusted_nll,
            "original_macro_f1": f1,
            "reference_macro_f1": adjusted_f1,
            "ref_pred_egfr": int(adj_counts[0]),
            "ref_pred_kras": int(adj_counts[1]),
            "ref_pred_dwt": int(adj_counts[2]),
        })

run_id = (
    datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%SZ"
    )
    + "_"
    + uuid.uuid4().hex[:8]
)

RUN = ROOT / "08_score_transfer_diagnosis" / run_id
RUN.mkdir(parents=True, exist_ok=False)

artifacts = []

def write(name, content):
    payload = (
        content.encode("utf-8")
        if isinstance(content, str)
        else content
    )

    path = RUN / name

    with path.open("xb") as handle:
        handle.write(payload)

    artifacts.append({
        "path": str(
            path.relative_to(ROOT)
        ).replace("\\", "/"),
        "sha256": hashlib.sha256(
            payload
        ).hexdigest(),
    })

def as_csv(rows):
    output = io.StringIO(newline="")

    writer = csv.DictWriter(
        output,
        fieldnames=list(rows[0]),
        lineterminator="\n",
    )

    writer.writeheader()
    writer.writerows(rows)

    return output.getvalue()

write(
    "SCORE_TRANSFER_AUDIT.csv",
    as_csv(native_rows),
)
write(
    "SOURCE_ONLY_PRIOR_COUNTERFACTUAL.csv",
    as_csv(shift_rows),
)

anchor = next(
    row for row in native_rows
    if row["held_cohort"] == "GSE31210"
    and row["model"] == "free_bounded"
)

if (
    anchor["egfr_auc"] >= 0.70
    and anchor["pred_egfr"] <= 5
):
    diagnostic = (
        "SCORE_RANKING_PRESERVED_BUT_DECISIONS_COLLAPSE"
    )
elif anchor["egfr_auc"] <= 0.60:
    diagnostic = "EGFR_REPRESENTATION_TRANSFER_FAILURE"
else:
    diagnostic = "MIXED_RANKING_AND_DECISION_SHIFT"

write(
    "RUN_MANIFEST.json",
    json.dumps({
        "phase": "DECISION_GATE_TRANSFER_AND_CLASS_PRIOR",
        "source_manifest": str(
            parent_path.relative_to(ROOT)
        ).replace("\\", "/"),
        "source_manifest_sha256": hashlib.sha256(
            parent_path.read_bytes()
        ).hexdigest(),
        "samples": 785,
        "models": list(MODELS),
        "gse31210_diagnosis": diagnostic,
        "prior_counterfactual": "DIAGNOSTIC_ONLY",
        "heldout_labels_used_for_fitting": False,
        "heldout_labels_used_for_evaluation": True,
        "model_training": False,
        "external_cohorts_accessed": False,
        "artifacts": artifacts,
    }, indent=2) + "\n",
)

print("TRANSFER_DIAGNOSIS: PASS")
print("SAMPLES: 785")
print("UNIFORM_CLASS_NLL:", round(np.log(3), 4))

for held in COHORTS:
    print("COHORT:", held)

    for model in (
        "free_bounded",
        "free_k_residual",
        "signed_k_residual",
    ):
        row = next(
            r for r in native_rows
            if r["held_cohort"] == held
            and r["model"] == model
        )

        adj = next(
            r for r in shift_rows
            if r["held_cohort"] == held
            and r["model"] == model
        )

        print(
            model,
            "E_AUC=", round(row["egfr_auc"], 4),
            "K_AUC=", round(
                row["conditional_kras_auc"], 4
            ),
            "PRED=", (
                row["pred_egfr"],
                row["pred_kras"],
                row["pred_dwt"],
            ),
            "F1=", round(row["macro_f1"], 4),
            "REF_F1=", round(
                adj["reference_macro_f1"], 4
            ),
            "REF_NLL=", round(
                adj["reference_nll"], 4
            ),
        )

print("GSE31210_GATE:", diagnostic)
print("TRAINING: NO")
print("RUN:", RUN)

TRANSFER_DIAGNOSIS: PASS
SAMPLES: 785
UNIFORM_CLASS_NLL: 1.0986
COHORT: GSE13213
free_bounded E_AUC= 0.7707 K_AUC= 0.8409 PRED= (74, 18, 25) F1= 0.5814 REF_F1= 0.3666 REF_NLL= 1.188
free_k_residual E_AUC= 0.7707 K_AUC= 0.8573 PRED= (72, 21, 24) F1= 0.573 REF_F1= 0.3607 REF_NLL= 1.2099
signed_k_residual E_AUC= 0.7704 K_AUC= 0.855 PRED= (77, 19, 21) F1= 0.5319 REF_F1= 0.3644 REF_NLL= 1.2622
COHORT: GSE31210
free_bounded E_AUC= 0.8611 K_AUC= 0.7753 PRED= (0, 0, 226) F1= 0.1727 REF_F1= 0.3665 REF_NLL= 1.0041
free_k_residual E_AUC= 0.8611 K_AUC= 0.769 PRED= (0, 1, 225) F1= 0.205 REF_F1= 0.3318 REF_NLL= 0.9903
signed_k_residual E_AUC= 0.8621 K_AUC= 0.769 PRED= (0, 2, 224) F1= 0.2019 REF_F1= 0.3318 REF_NLL= 0.9886
COHORT: GSE72094
free_bounded E_AUC= 0.8175 K_AUC= 0.6569 PRED= (125, 1, 316) F1= 0.3381 REF_F1= 0.3733 REF_NLL= 1.1111
free_k_residual E_AUC= 0.8175 K_AUC= 0.652 PRED= (119, 1, 322) F1= 0.3412 REF_F1= 0.3699 REF_NLL= 1.0992
signed_k_residual E_AUC= 0.8174 K_AUC= 0.6488 PRED= (120, 

In [13]:

import csv
import hashlib
import io
import json
import os
import time
import uuid
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import rankdata, ks_2samp
from sklearn.metrics import f1_score, roc_auc_score

ROOT = Path(r"D:\LUAD_DATAFIRST_METHOD_2026")
SOURCE = Path(
    r"D:\LUAD_CMR_2026\CELL03_DATA_20261021T090947_420335Z_d6aed8ec\cmr_gene_expression.npz"
)

DEV = ("GSE13213", "GSE31210", "GSE72094")
SEED = 20261023
EPOCHS = 140
LR = 0.025
PAIR_WEIGHT = 0.15
WEIGHT_L2 = 0.015
NUISANCE_L2 = 0.04
ADAPTER_L2 = 0.08
PRIOR_REF = np.array([0.5, 0.25, 0.25], dtype=np.float64)

identity_file = ROOT / "00_admin/project.json"

if os.name != "nt" or not Path("D:/").is_dir():
    raise RuntimeError("Windows D: drive unavailable")

if not identity_file.is_file() or not SOURCE.is_file():
    raise FileNotFoundError("Project identity or input missing")

identity = json.loads(identity_file.read_text(encoding="utf-8"))

if identity.get("project_id") != "LUAD_DATAFIRST_METHOD_2026":
    raise RuntimeError("Project identity mismatch")

phase7_manifests = sorted(
    (ROOT / "07_signed_bounded_evidence_pilot").glob(
        "*/RUN_MANIFEST.json"
    )
)

phase8_manifests = sorted(
    (ROOT / "08_score_transfer_diagnosis").glob(
        "*/RUN_MANIFEST.json"
    )
)

if not phase7_manifests or not phase8_manifests:
    raise RuntimeError("Previous development evidence missing")

phase7_file = phase7_manifests[-1]
phase8_file = phase8_manifests[-1]

phase7 = json.loads(phase7_file.read_text(encoding="utf-8"))
phase8 = json.loads(phase8_file.read_text(encoding="utf-8"))

if (
    phase7.get("phase") != "SIGNED_BOUNDED_EVIDENCE_DEVELOPMENT"
    or phase8.get("phase") != "DECISION_GATE_TRANSFER_AND_CLASS_PRIOR"
):
    raise RuntimeError("Unexpected source evidence phase")

for item in phase7.get("artifacts", []):
    path = ROOT / item["path"]

    if (
        not path.is_file()
        or hashlib.sha256(path.read_bytes()).hexdigest()
        != item["sha256"]
    ):
        raise RuntimeError("Phase 07 artifact checksum mismatch")

with np.load(SOURCE, allow_pickle=False) as archive:
    required = {
        "X", "y", "cohort", "gsm", "genes", "patient_ids"
    }

    if not required.issubset(archive.files):
        raise RuntimeError("Archive missing required arrays")

    all_cohorts = np.char.upper(
        np.char.strip(archive["cohort"].astype(str))
    )

    development_mask = np.isin(all_cohorts, DEV)

    X = np.asarray(
        archive["X"][development_mask],
        dtype=np.float32,
    )

    y = np.asarray(
        archive["y"][development_mask],
        dtype=np.int64,
    )

    cohorts = np.asarray(
        all_cohorts[development_mask],
        dtype=str,
    )

    gsm = np.char.upper(
        np.char.strip(
            archive["gsm"][development_mask].astype(str)
        )
    )

    genes = np.char.upper(
        np.char.strip(archive["genes"].astype(str))
    )

if X.shape != (785, 13016) or not np.isfinite(X).all():
    raise RuntimeError("Unexpected development matrix")

if (
    set(y.tolist()) != {0, 1, 2}
    or len(set(gsm.tolist())) != len(gsm)
):
    raise RuntimeError("Invalid labels or duplicate GSM")

for cohort_name, expected in zip(DEV, (117, 226, 442)):
    if int(np.sum(cohorts == cohort_name)) != expected:
        raise RuntimeError("Cohort composition mismatch")

ranks = (
    rankdata(X, axis=1, method="average").astype(np.float32)
    - 1.0
) / (X.shape[1] - 1)

del X

torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_num_threads(
    min(2, max(1, os.cpu_count() or 1))
)

with (phase7_file.parent / "LOCO_PREDICTIONS.csv").open(
    "r", encoding="utf-8", newline=""
) as handle:
    old_rows = [
        row for row in csv.DictReader(handle)
        if row["model"] == "free_bounded"
    ]

old_predictions = {}

for row in old_rows:
    key = (row["cohort"], row["gsm"].upper())

    if key in old_predictions:
        raise RuntimeError("Duplicate previous prediction")

    old_predictions[key] = (
        int(row["class"]),
        np.array([
            float(row["p_egfr"]),
            float(row["p_kras"]),
            float(row["p_dwt"]),
        ]),
    )

if len(old_predictions) != 785:
    raise RuntimeError("Incomplete previous comparison")


def fold_features(held):
    path = (
        phase7_file.parent
        / "folds"
        / f"{held}_features.json"
    )

    specification = json.loads(
        path.read_text(encoding="utf-8")
    )

    transformed = []

    for head in ("egfr", "kras"):
        entry = specification[head]
        indices = np.asarray(entry["indices"], dtype=int)

        if (
            len(indices) != 24
            or genes[indices].tolist()
            != [name.upper() for name in entry["genes"]]
        ):
            raise RuntimeError("Feature lineage mismatch")

        mean = np.asarray(entry["mean"], dtype=np.float32)
        sd = np.asarray(entry["sd"], dtype=np.float32)

        if not np.isfinite(mean).all() or np.any(sd <= 0):
            raise RuntimeError("Invalid training-only scaler")

        values = np.tanh(
            (ranks[:, indices] - mean) / (2.0 * sd)
        ).astype(np.float32)

        transformed.append(torch.from_numpy(values))

    return transformed


class CNSE(nn.Module):
    def __init__(self, nuisance, adapter=False):
        super().__init__()

        self.e = nn.Linear(24, 1)
        self.k = nn.Linear(24, 1)
        self.nuisance = bool(nuisance)
        self.adapter = bool(adapter)

        if self.adapter and not self.nuisance:
            raise ValueError("Adapter requires nuisance separation")

        if nuisance:
            self.domain_offset = nn.Parameter(
                torch.zeros(2, 2)
            )

        if adapter:
            self.adapter_e = nn.Parameter(
                torch.zeros(2, 24)
            )
            self.adapter_k = nn.Parameter(
                torch.zeros(2, 24)
            )

        nn.init.normal_(
            self.e.weight, mean=0.0, std=0.025
        )
        nn.init.normal_(
            self.k.weight, mean=0.0, std=0.025
        )
        nn.init.zeros_(self.e.bias)
        nn.init.zeros_(self.k.bias)

    def forward(self, xe, xk, domain=None):
        e = self.e(xe).flatten()
        k = self.k(xk).flatten()

        if domain is not None and self.nuisance:
            centered = (
                self.domain_offset
                - self.domain_offset.mean(
                    dim=1, keepdim=True
                )
            )

            e = e + centered[0, domain]
            k = k + centered[1, domain]

            if self.adapter:
                ae = (
                    self.adapter_e
                    - self.adapter_e.mean(
                        dim=0, keepdim=True
                    )
                )

                ak = (
                    self.adapter_k
                    - self.adapter_k.mean(
                        dim=0, keepdim=True
                    )
                )

                e = e + (
                    xe * ae[domain]
                ).sum(dim=1)

                k = k + (
                    xk * ak[domain]
                ).sum(dim=1)

        return e, k


def binary_contrast_loss(
    scores, positive, negative, rank_weight
):
    pos = scores[positive]
    neg = scores[negative]

    if min(pos.numel(), neg.numel()) < 5:
        raise RuntimeError("Insufficient class support")

    balanced = 0.5 * (
        F.softplus(-pos).mean()
        + F.softplus(neg).mean()
    )

    if rank_weight == 0:
        return balanced

    pairs = F.softplus(
        -(pos[:, None] - neg[None, :])
    ).mean()

    return balanced + rank_weight * pairs


def fit(
    xe,
    xk,
    labels,
    domain_ids,
    nuisance=False,
    pair_weight=0.0,
    adapter=False,
):
    torch.manual_seed(SEED)

    model = CNSE(nuisance, adapter)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=0.0,
    )

    if sorted(set(domain_ids.tolist())) != [0, 1]:
        raise RuntimeError("Expected two training domains")

    masks = [
        torch.from_numpy(domain_ids == domain)
        for domain in (0, 1)
    ]

    yt = torch.from_numpy(labels)
    dt = torch.from_numpy(domain_ids)

    for _ in range(EPOCHS):
        optimizer.zero_grad(set_to_none=True)

        se, sk = model(xe, xk, dt)
        losses = []

        for mask in masks:
            group = yt[mask]

            le = binary_contrast_loss(
                se[mask],
                group == 0,
                group != 0,
                pair_weight,
            )

            lk = binary_contrast_loss(
                sk[mask],
                group == 1,
                group == 2,
                pair_weight,
            )

            losses.append(0.5 * (le + lk))

        loss = torch.stack(losses).mean()

        loss = loss + WEIGHT_L2 * (
            model.e.weight.square().sum()
            + model.k.weight.square().sum()
        )

        if nuisance:
            centered = (
                model.domain_offset
                - model.domain_offset.mean(
                    dim=1, keepdim=True
                )
            )

            loss = loss + (
                NUISANCE_L2 * centered.square().sum()
            )

        if adapter:
            ae = (
                model.adapter_e
                - model.adapter_e.mean(
                    dim=0, keepdim=True
                )
            )

            ak = (
                model.adapter_k
                - model.adapter_k.mean(
                    dim=0, keepdim=True
                )
            )

            loss = loss + ADAPTER_L2 * (
                ae.square().sum()
                + ak.square().sum()
            )

        if not torch.isfinite(loss):
            raise RuntimeError("Nonfinite training loss")

        loss.backward()
        optimizer.step()

    return model


def probabilities(model, xe, xk):
    with torch.no_grad():
        le, lk = model(xe, xk)

        pe = torch.sigmoid(le).cpu().numpy()
        pk = torch.sigmoid(lk).cpu().numpy()

    pe = pe.astype(np.float64)
    pk = pk.astype(np.float64)

    prob = np.stack((
        pe,
        (1.0 - pe) * pk,
        (1.0 - pe) * (1.0 - pk),
    ), axis=1)

    prob = np.clip(prob, 1e-12, 1.0)

    return prob / prob.sum(axis=1, keepdims=True)


def reweight_posteriors(
    prob, source_prior, target_prior
):
    ratio = (
        np.asarray(target_prior)
        / np.asarray(source_prior)
    )

    adjusted = prob * ratio[None, :]

    return adjusted / adjusted.sum(
        axis=1, keepdims=True
    )


def head_odds_adjust(
    prob, source_prior, target_prior
):
    prob = np.clip(
        np.asarray(prob, dtype=np.float64),
        1e-10,
        1.0 - 1e-10,
    )

    source_prior = float(
        np.clip(source_prior, 1e-6, 1 - 1e-6)
    )

    target_prior = float(
        np.clip(target_prior, 1e-6, 1 - 1e-6)
    )

    likelihood_odds = (
        prob / (1 - prob)
        * (1 - source_prior) / source_prior
    )

    new_odds = (
        likelihood_odds
        * target_prior / (1 - target_prior)
    )

    return new_odds / (1 + new_odds)


def hierarchical_em(prob, source_prior):
    e = np.asarray(
        prob[:, 0], dtype=np.float64
    )

    k = np.asarray(
        prob[:, 1] / (prob[:, 1] + prob[:, 2]),
        dtype=np.float64,
    )

    src_e = float(source_prior[0])

    src_k = float(
        source_prior[1]
        / (source_prior[1] + source_prior[2])
    )

    pi_e, pi_k = src_e, src_k

    for iteration in range(200):
        q_e = head_odds_adjust(
            e, src_e, pi_e
        )

        q_k = head_odds_adjust(
            k, src_k, pi_k
        )

        target_e = float(
            np.clip(q_e.mean(), 0.005, 0.995)
        )

        target_k = float(
            np.clip(
                np.sum((1 - q_e) * q_k)
                / np.sum(1 - q_e),
                0.005,
                0.995,
            )
        )

        difference = max(
            abs(target_e - pi_e),
            abs(target_k - pi_k),
        )

        pi_e, pi_k = target_e, target_k

        if difference < 1e-9:
            break

    return (pi_e, pi_k), iteration + 1


def hierarchical_adjust(
    prob, source_prior, target_prior
):
    src_e = float(source_prior[0])

    src_k = float(
        source_prior[1]
        / (source_prior[1] + source_prior[2])
    )

    qe = head_odds_adjust(
        prob[:, 0],
        src_e,
        target_prior[0],
    )

    qk = head_odds_adjust(
        prob[:, 1]
        / (prob[:, 1] + prob[:, 2]),
        src_k,
        target_prior[1],
    )

    return np.stack((
        qe,
        (1 - qe) * qk,
        (1 - qe) * (1 - qk),
    ), axis=1)


def estimate_prior_em(prob, source_prior):
    pi = np.asarray(
        source_prior, dtype=np.float64
    ).copy()

    for iteration in range(200):
        q = reweight_posteriors(
            prob, source_prior, pi
        )

        next_pi = np.clip(
            q.mean(axis=0), 0.005, 1.0
        )

        next_pi /= next_pi.sum()

        difference = np.max(
            np.abs(next_pi - pi)
        )

        pi = next_pi

        if difference < 1e-9:
            break

    return pi, iteration + 1


def evaluate(truth, probability):
    if (
        not np.isfinite(probability).all()
        or np.max(
            np.abs(probability.sum(axis=1) - 1)
        ) > 1e-6
    ):
        raise RuntimeError("Invalid probability matrix")

    truth = np.asarray(truth, dtype=int)
    prediction = probability.argmax(axis=1)

    correct = np.clip(
        probability[
            np.arange(len(truth)), truth
        ],
        1e-12,
        1,
    )

    conditional = truth != 0

    kras_truth = (
        truth[conditional] == 1
    ).astype(int)

    kras_prob = np.clip(
        probability[conditional, 1]
        / (
            probability[conditional, 1]
            + probability[conditional, 2]
        ),
        1e-12,
        1 - 1e-12,
    )

    counts = np.bincount(
        prediction, minlength=3
    )

    return {
        "nll": float(-np.log(correct).mean()),
        "f1": float(
            f1_score(
                truth,
                prediction,
                labels=[0, 1, 2],
                average="macro",
                zero_division=0,
            )
        ),
        "egfr_auc": float(
            roc_auc_score(
                truth == 0,
                probability[:, 0],
            )
        ),
        "kras_auc": float(
            roc_auc_score(
                kras_truth, kras_prob
            )
        ),
        "kras_nll": float(
            -np.mean(
                kras_truth * np.log(kras_prob)
                + (1 - kras_truth)
                * np.log(1 - kras_prob)
            )
        ),
        "pred_egfr": int(counts[0]),
        "pred_kras": int(counts[1]),
        "pred_dwt": int(counts[2]),
    }


MODELS = {
    "balanced_ce": (False, 0.0, False),
    "balanced_pair": (False, PAIR_WEIGHT, False),
    "nuisance_ce": (True, 0.0, False),
    "cnse_pair": (True, PAIR_WEIGHT, False),
    "cnse_adapter_pair": (True, PAIR_WEIGHT, True),
}

run_id = (
    datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%SZ"
    )
    + "_"
    + uuid.uuid4().hex[:8]
)

RUN = (
    ROOT
    / "09_nuisance_separated_evidence"
    / run_id
)

RUN.mkdir(parents=True, exist_ok=False)

artifacts = []


def save(name, content):
    path = RUN / name

    path.parent.mkdir(
        parents=True, exist_ok=True
    )

    data = (
        content.encode("utf-8")
        if isinstance(content, str)
        else content
    )

    with path.open("xb") as handle:
        handle.write(data)

    artifacts.append({
        "path": str(
            path.relative_to(ROOT)
        ).replace("\\", "/"),
        "sha256": hashlib.sha256(
            data
        ).hexdigest(),
    })


def csv_output(rows):
    buffer = io.StringIO(newline="")

    writer = csv.DictWriter(
        buffer,
        fieldnames=list(rows[0]),
        lineterminator="\n",
    )

    writer.writeheader()
    writer.writerows(rows)

    return buffer.getvalue()


protocol = {
    "phase": "CNSE_DEVELOPMENT_LOCO",
    "architecture": (
        "Shared contrast-specific odds with centered "
        "cohort intercepts and gene-dependent adapters"
    ),
    "objective": (
        "Equal-domain balanced BCE with optional "
        "within-domain pairwise logistic ranking"
    ),
    "adaptation": (
        "Multiclass EM and hierarchical "
        "EM-like soft assignment"
    ),
    "reference_prior": PRIOR_REF.tolist(),
    "cohorts": list(DEV),
    "seed": SEED,
    "epochs": EPOCHS,
    "learning_rate": LR,
    "pair_weight": PAIR_WEIGHT,
    "weight_l2": WEIGHT_L2,
    "nuisance_l2": NUISANCE_L2,
    "adapter_l2": ADAPTER_L2,
    "historical_development_outcomes_exposed": True,
    "shared_gene_universe_includes_GSE32863": True,
    "external_evaluation": False,
    "method_novelty": "NOT_ESTABLISHED",
}

save(
    "PROTOCOL.json",
    json.dumps(protocol, indent=2) + "\n",
)

results = []
predictions = []
adaptation = []
shift_checks = []

start = time.monotonic()

for held in DEV:
    train = np.flatnonzero(cohorts != held)
    test = np.flatnonzero(cohorts == held)

    source_names = sorted(
        set(cohorts[train].tolist())
    )

    source_domains = np.array([
        source_names.index(name)
        for name in cohorts[train]
    ], dtype=np.int64)

    xe, xk = fold_features(held)
    actual = y[test]

    for name, specification in MODELS.items():
        nuisance, pair_weight, adapter = specification

        model = fit(
            xe[train],
            xk[train],
            y[train],
            source_domains,
            nuisance=nuisance,
            pair_weight=pair_weight,
            adapter=adapter,
        )

        prob = probabilities(
            model, xe[test], xk[test]
        )

        variants = {name: prob}

        if name in (
            "balanced_ce",
            "balanced_pair",
            "cnse_pair",
            "cnse_adapter_pair",
        ):
            joint_prior, joint_steps = (
                estimate_prior_em(
                    prob, PRIOR_REF
                )
            )

            variants[name + "_joint_em"] = (
                reweight_posteriors(
                    prob,
                    PRIOR_REF,
                    joint_prior,
                )
            )

            head_prior, head_steps = (
                hierarchical_em(
                    prob, PRIOR_REF
                )
            )

            variants[name + "_head_em"] = (
                hierarchical_adjust(
                    prob,
                    PRIOR_REF,
                    head_prior,
                )
            )

            if name in (
                "balanced_pair",
                "cnse_pair",
                "cnse_adapter_pair",
            ):
                midpoint = (
                    0.5 * (0.5 + head_prior[0]),
                    0.5 * (0.5 + head_prior[1]),
                )

                variants[name + "_head_shrink"] = (
                    hierarchical_adjust(
                        prob,
                        PRIOR_REF,
                        midpoint,
                    )
                )

            adaptation.append({
                "held_cohort": held,
                "model": name,
                "joint_em_steps": joint_steps,
                "head_em_steps": head_steps,
                "joint_em_prior": json.dumps(
                    joint_prior.tolist()
                ),
                "head_em_prior": json.dumps([
                    head_prior[0],
                    (1 - head_prior[0]) * head_prior[1],
                    (1 - head_prior[0])
                    * (1 - head_prior[1]),
                ]),
                "actual_target_class_fractions_for_reporting_only": (
                    json.dumps(
                        (
                            np.bincount(
                                actual, minlength=3
                            ) / len(actual)
                        ).tolist()
                    )
                ),
            })

        if name == "cnse_adapter_pair":
            with torch.no_grad():
                src_e, src_k = model(
                    xe[train], xk[train]
                )

                src_e = src_e.numpy()
                src_k = src_k.numpy()

            for head, scores, positive, negative in (
                ("egfr", src_e, 0, -1),
                ("kras", src_k, 1, 2),
            ):
                for group_label in (
                    "positive", "negative"
                ):
                    distributions = []

                    for domain_id in (0, 1):
                        if group_label == "positive":
                            class_mask = (
                                y[train] == positive
                            )

                        elif negative == -1:
                            class_mask = (
                                y[train] != 0
                            )

                        else:
                            class_mask = (
                                y[train] == negative
                            )

                        values = scores[
                            (source_domains == domain_id)
                            & class_mask
                        ]

                        distributions.append(values)

                    ks = float(
                        ks_2samp(
                            distributions[0],
                            distributions[1],
                        ).statistic
                    )

                    shift_checks.append({
                        "held_cohort": held,
                        "head": head,
                        "class_group": group_label,
                        "training_sources": "/".join(
                            source_names
                        ),
                        "source_score_ks": ks,
                        "support_n1": len(
                            distributions[0]
                        ),
                        "support_n2": len(
                            distributions[1]
                        ),
                    })

        buffer = io.BytesIO()

        torch.save({
            "model_state": model.state_dict(),
            "nuisance": nuisance,
            "adapter": adapter,
            "pair_weight": pair_weight,
            "source_cohorts": source_names,
            "feature_file": str(
                phase7_file.parent
                / "folds"
                / f"{held}_features.json"
            ),
            "reference_prior": PRIOR_REF.tolist(),
        }, buffer)

        save(
            f"checkpoints/{held}_{name}.pt",
            buffer.getvalue(),
        )

        for variant, variant_prob in variants.items():
            metrics = evaluate(
                actual, variant_prob
            )

            results.append({
                "held_cohort": held,
                "model": variant,
                "n": len(test),
                **metrics,
            })

            for j, index in enumerate(test):
                predictions.append({
                    "held_cohort": held,
                    "gsm": str(gsm[index]),
                    "true_class": int(y[index]),
                    "model": variant,
                    "p_egfr": float(
                        variant_prob[j, 0]
                    ),
                    "p_kras": float(
                        variant_prob[j, 1]
                    ),
                    "p_dwt": float(
                        variant_prob[j, 2]
                    ),
                })

    previous = []

    for index in test:
        key = (held, str(gsm[index]))

        if (
            key not in old_predictions
            or old_predictions[key][0] != int(y[index])
        ):
            raise RuntimeError(
                "Previous prediction sample mismatch"
            )

        previous.append(
            old_predictions[key][1]
        )

    previous = np.stack(previous)

    old_source_prior = np.mean([
        np.bincount(
            y[cohorts == source_name],
            minlength=3,
        ) / np.sum(cohorts == source_name)
        for source_name in source_names
    ], axis=0)

    historic_variants = {
        "old_free_bounded": previous
    }

    old_joint_prior, old_joint_steps = (
        estimate_prior_em(
            previous, old_source_prior
        )
    )

    historic_variants[
        "old_free_bounded_joint_em"
    ] = reweight_posteriors(
        previous,
        old_source_prior,
        old_joint_prior,
    )

    old_head_prior, old_head_steps = (
        hierarchical_em(
            previous, old_source_prior
        )
    )

    historic_variants[
        "old_free_bounded_head_em"
    ] = hierarchical_adjust(
        previous,
        old_source_prior,
        old_head_prior,
    )

    old_e_prior = float(
        old_source_prior[0]
    )

    old_k_prior = float(
        old_source_prior[1]
        / (
            old_source_prior[1]
            + old_source_prior[2]
        )
    )

    old_midpoint = (
        0.5 * (
            old_e_prior + old_head_prior[0]
        ),
        0.5 * (
            old_k_prior + old_head_prior[1]
        ),
    )

    historic_variants[
        "old_free_bounded_head_shrink"
    ] = hierarchical_adjust(
        previous,
        old_source_prior,
        old_midpoint,
    )

    adaptation.append({
        "held_cohort": held,
        "model": "old_free_bounded",
        "joint_em_steps": old_joint_steps,
        "head_em_steps": old_head_steps,
        "joint_em_prior": json.dumps(
            old_joint_prior.tolist()
        ),
        "head_em_prior": json.dumps([
            old_head_prior[0],
            (1 - old_head_prior[0])
            * old_head_prior[1],
            (1 - old_head_prior[0])
            * (1 - old_head_prior[1]),
        ]),
        "actual_target_class_fractions_for_reporting_only": (
            json.dumps(
                (
                    np.bincount(
                        actual, minlength=3
                    ) / len(actual)
                ).tolist()
            )
        ),
    })

    for variant, variant_prob in historic_variants.items():
        metrics = evaluate(
            actual, variant_prob
        )

        results.append({
            "held_cohort": held,
            "model": variant,
            "n": len(test),
            **metrics,
        })

        for j, index in enumerate(test):
            predictions.append({
                "held_cohort": held,
                "gsm": str(gsm[index]),
                "true_class": int(y[index]),
                "model": variant,
                "p_egfr": float(
                    variant_prob[j, 0]
                ),
                "p_kras": float(
                    variant_prob[j, 1]
                ),
                "p_dwt": float(
                    variant_prob[j, 2]
                ),
            })

save(
    "LOCO_METRICS.csv",
    csv_output(results),
)

save(
    "LOCO_PREDICTIONS.csv",
    csv_output(predictions),
)

save(
    "UNLABELED_PRIOR_ESTIMATES.csv",
    csv_output(adaptation),
)

save(
    "SOURCE_CONDITIONAL_SCORE_KS.csv",
    csv_output(shift_checks),
)

summary = {}

for name in dict.fromkeys(
    row["model"] for row in results
):
    subset = [
        row for row in results
        if row["model"] == name
    ]

    if {
        row["held_cohort"] for row in subset
    } != set(DEV):
        raise RuntimeError(
            "Incomplete model cohort coverage"
        )

    summary[name] = {
        "mean_nll": float(
            np.mean([
                row["nll"] for row in subset
            ])
        ),
        "mean_macro_f1": float(
            np.mean([
                row["f1"] for row in subset
            ])
        ),
        "mean_egfr_auc": float(
            np.mean([
                row["egfr_auc"] for row in subset
            ])
        ),
        "mean_kras_auc": float(
            np.mean([
                row["kras_auc"] for row in subset
            ])
        ),
        "mean_kras_nll": float(
            np.mean([
                row["kras_nll"] for row in subset
            ])
        ),
        "cohort_nll": {
            row["held_cohort"]: row["nll"]
            for row in subset
        },
    }

candidate = "cnse_adapter_pair_head_shrink"

controls = (
    "balanced_pair_head_shrink",
    "cnse_pair_head_shrink",
    "old_free_bounded_head_shrink",
    "cnse_adapter_pair",
    "cnse_adapter_pair_head_em",
    "cnse_adapter_pair_joint_em",
)

positive_cohorts = sum(
    summary[candidate]["cohort_nll"][held]
    <
    summary[
        "balanced_pair_head_shrink"
    ]["cohort_nll"][held]
    for held in DEV
)

nll_gain = (
    summary[
        "balanced_pair_head_shrink"
    ]["mean_nll"]
    - summary[candidate]["mean_nll"]
)

f1_gain = (
    summary[candidate]["mean_macro_f1"]
    - summary[
        "balanced_pair_head_shrink"
    ]["mean_macro_f1"]
)

screen = (
    "PASS_EXPLORATORY"
    if (
        nll_gain >= 0.02
        and f1_gain >= 0.02
        and positive_cohorts >= 2
    )
    else "HOLD"
)

comparison = {
    "candidate": candidate,
    "screen": screen,
    "positive_cohorts": positive_cohorts,
    "minimum_mean_nll_gain": 0.02,
    "minimum_mean_f1_gain": 0.02,
    "controls": {
        name: {
            "equal_cohort_nll_gain": (
                summary[name]["mean_nll"]
                - summary[candidate]["mean_nll"]
            ),
            "equal_cohort_f1_gain": (
                summary[candidate]["mean_macro_f1"]
                - summary[name]["mean_macro_f1"]
            ),
        }
        for name in controls
    },
    "novelty_established": False,
    "adaptive_research_stage": True,
}

save(
    "EQUAL_COHORT_COMPARISON.json",
    json.dumps({
        "models": summary,
        "comparison": comparison,
    }, indent=2) + "\n",
)

source_hash = hashlib.sha256()

with SOURCE.open("rb") as handle:
    for chunk in iter(
        lambda: handle.read(4_194_304),
        b"",
    ):
        source_hash.update(chunk)

save(
    "RUN_MANIFEST.json",
    json.dumps({
        "phase": "CNSE_DEVELOPMENT_LOCO",
        "created_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "source_sha256": source_hash.hexdigest(),
        "phase07_sha256": hashlib.sha256(
            phase7_file.read_bytes()
        ).hexdigest(),
        "phase08_sha256": hashlib.sha256(
            phase8_file.read_bytes()
        ).hexdigest(),
        "prediction_records": len(predictions),
        "unlabeled_target_adaptation": True,
        "target_labels_excluded_from_fit_and_adaptation": True,
        "historical_outcomes_exposed": True,
        "source_label_provenance": "PROVISIONAL",
        "external_cohorts_evaluated": False,
        "method_novelty": "NOT_ESTABLISHED",
        "runtime_seconds": round(
            time.monotonic() - start, 2
        ),
        "artifacts": artifacts,
    }, indent=2) + "\n",
)

report_models = (
    "old_free_bounded",
    "old_free_bounded_head_shrink",
    "balanced_ce",
    "balanced_pair",
    "nuisance_ce",
    "cnse_pair",
    "cnse_adapter_pair",
    "balanced_pair_head_shrink",
    "cnse_pair_head_shrink",
    "cnse_adapter_pair_head_shrink",
    "cnse_adapter_pair_head_em",
)

print("CNSE_DEVELOPMENT: COMPLETE")

for held in DEV:
    print(
        "LOCO",
        held,
        "CLASS=",
        dict(Counter(
            y[cohorts == held].tolist()
        )),
    )

    for name in report_models:
        row = next(
            item for item in results
            if item["held_cohort"] == held
            and item["model"] == name
        )

        print(
            name,
            "NLL=", round(row["nll"], 4),
            "F1=", round(row["f1"], 4),
            "E_AUC=", round(row["egfr_auc"], 4),
            "K_AUC=", round(row["kras_auc"], 4),
            "PRED=", (
                row["pred_egfr"],
                row["pred_kras"],
                row["pred_dwt"],
            ),
        )

for name in report_models:
    row = summary[name]

    print(
        "MEAN",
        name,
        "NLL=", round(row["mean_nll"], 4),
        "F1=", round(row["mean_macro_f1"], 4),
        "K_NLL=", round(row["mean_kras_nll"], 4),
    )

for name, row in comparison["controls"].items():
    print(
        "GAIN_VS",
        name,
        "NLL=", round(
            row["equal_cohort_nll_gain"], 4
        ),
        "F1=", round(
            row["equal_cohort_f1_gain"], 4
        ),
    )

for held in DEV:
    estimated = next(
        row for row in adaptation
        if row["held_cohort"] == held
        and row["model"] == "cnse_adapter_pair"
    )

    ks_values = [
        row["source_score_ks"]
        for row in shift_checks
        if row["held_cohort"] == held
    ]

    print(
        "ADAPT",
        held,
        "PRIOR=", estimated["head_em_prior"],
        "SOURCE_KS_MAX=", round(
            max(ks_values), 4
        ),
    )

print(
    "SCREEN:",
    screen,
    "POSITIVE_COHORTS:",
    positive_cohorts,
)

print("NOVELTY: UNVERIFIED")
print("RUN:", RUN)

CNSE_DEVELOPMENT: COMPLETE
LOCO GSE13213 CLASS= {0: 45, 2: 57, 1: 15}
old_free_bounded NLL= 0.9019 F1= 0.5814 E_AUC= 0.7707 K_AUC= 0.8409 PRED= (74, 18, 25)
old_free_bounded_head_shrink NLL= 1.5427 F1= 0.2244 E_AUC= 0.7707 K_AUC= 0.8409 PRED= (112, 5, 0)
balanced_ce NLL= 1.3145 F1= 0.2696 E_AUC= 0.7667 K_AUC= 0.8433 PRED= (109, 6, 2)
balanced_pair NLL= 1.334 F1= 0.2972 E_AUC= 0.7682 K_AUC= 0.8503 PRED= (109, 7, 1)
nuisance_ce NLL= 1.3224 F1= 0.2382 E_AUC= 0.766 K_AUC= 0.8409 PRED= (110, 5, 2)
cnse_pair NLL= 1.3429 F1= 0.2685 E_AUC= 0.7679 K_AUC= 0.8503 PRED= (110, 6, 1)
cnse_adapter_pair NLL= 1.3371 F1= 0.2972 E_AUC= 0.7679 K_AUC= 0.8503 PRED= (109, 7, 1)
balanced_pair_head_shrink NLL= 1.9921 F1= 0.1875 E_AUC= 0.7682 K_AUC= 0.8503 PRED= (115, 2, 0)
cnse_pair_head_shrink NLL= 1.9894 F1= 0.1863 E_AUC= 0.7679 K_AUC= 0.8503 PRED= (116, 1, 0)
cnse_adapter_pair_head_shrink NLL= 1.9775 F1= 0.1863 E_AUC= 0.7679 K_AUC= 0.8503 PRED= (116, 1, 0)
cnse_adapter_pair_head_em NLL= 4.7242 F1= 0.1852 E_

In [14]:

import csv
import hashlib
import io
import json
import os
import time
import uuid
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import rankdata
from sklearn.metrics import f1_score, roc_auc_score

ROOT = Path(r"D:\LUAD_DATAFIRST_METHOD_2026")
SOURCE = Path(
    r"D:\LUAD_CMR_2026\CELL03_DATA_20261021T090947_420335Z_d6aed8ec\cmr_gene_expression.npz"
)

DEV = ("GSE13213", "GSE31210", "GSE72094")
SEED = 20261023
EPOCHS = 125
HIDDEN = 5
LR = 0.018

PAIR_WEIGHT = 0.20
ALIGN_WEIGHT = 0.035
RIDGE = 0.018
RESIDUAL_RIDGE = 0.025
SMOOTH_WORST_TEMP = 0.12
CAP_E, CAP_K = 2.8, 2.3

identity_file = ROOT / "00_admin/project.json"

if os.name != "nt" or not Path("D:/").is_dir():
    raise RuntimeError("Windows D: drive unavailable")

if not identity_file.is_file() or not SOURCE.is_file():
    raise FileNotFoundError("Project identity or source missing")

identity = json.loads(identity_file.read_text(encoding="utf-8"))

if identity.get("project_id") != "LUAD_DATAFIRST_METHOD_2026":
    raise RuntimeError("Project identity mismatch")

phase7_files = sorted(
    (ROOT / "07_signed_bounded_evidence_pilot").glob(
        "*/RUN_MANIFEST.json"
    )
)

if not phase7_files:
    raise RuntimeError("Phase 07 train-fold features unavailable")

phase7_path = phase7_files[-1]
phase7 = json.loads(phase7_path.read_text(encoding="utf-8"))

if phase7.get("phase") != "SIGNED_BOUNDED_EVIDENCE_DEVELOPMENT":
    raise RuntimeError("Unexpected parent phase")

for entry in phase7.get("artifacts", []):
    path = ROOT / entry["path"]

    if (
        not path.is_file()
        or hashlib.sha256(path.read_bytes()).hexdigest()
        != entry["sha256"]
    ):
        raise RuntimeError("Parent artifact checksum mismatch")

with np.load(SOURCE, allow_pickle=False) as z:
    required = {"X", "y", "cohort", "gsm", "genes"}

    if not required.issubset(set(z.files)):
        raise RuntimeError("Source schema mismatch")

    all_domains = np.char.upper(
        np.char.strip(z["cohort"].astype(str))
    )

    chosen = np.isin(all_domains, DEV)

    x = np.asarray(z["X"][chosen], dtype=np.float32)
    y = np.asarray(z["y"][chosen], dtype=np.int64)
    domains = np.asarray(all_domains[chosen], dtype=str)

    gsm = np.char.upper(
        np.char.strip(z["gsm"][chosen].astype(str))
    )

    genes = np.char.upper(
        np.char.strip(z["genes"].astype(str))
    )

if x.shape != (785, 13016) or not np.isfinite(x).all():
    raise RuntimeError("Unexpected development expression matrix")

if set(y.tolist()) != {0, 1, 2} or len(set(gsm.tolist())) != 785:
    raise RuntimeError("Invalid labels or sample identifiers")

for cohort_name, n in zip(DEV, (117, 226, 442)):
    if (domains == cohort_name).sum() != n:
        raise RuntimeError("Cohort sample count mismatch")

ranks = (
    rankdata(x, axis=1, method="average").astype(np.float32)
    - 1.0
) / (x.shape[1] - 1)

del x

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(
    min(2, max(1, os.cpu_count() or 1))
)

historic = {}

with (phase7_path.parent / "LOCO_PREDICTIONS.csv").open(
    "r", encoding="utf-8-sig", newline=""
) as handle:
    for row in csv.DictReader(handle):
        if row["model"] != "free_bounded":
            continue

        key = (row["cohort"], row["gsm"].upper())

        if key in historic:
            raise RuntimeError("Duplicate historic prediction")

        historic[key] = (
            int(row["class"]),
            np.array([
                float(row[f"p_{label}"])
                for label in ("egfr", "kras", "dwt")
            ]),
        )

if len(historic) != 785:
    raise RuntimeError("Incomplete historic predictions")


def fold_features(held):
    path = (
        phase7_path.parent
        / "folds"
        / f"{held}_features.json"
    )

    spec = json.loads(path.read_text(encoding="utf-8"))
    output = []

    for head in ("egfr", "kras"):
        feat = spec[head]
        positions = np.asarray(feat["indices"], dtype=int)

        if (
            positions.shape != (24,)
            or genes[positions].tolist()
            != [g.upper() for g in feat["genes"]]
        ):
            raise RuntimeError("Feature lineage mismatch")

        mu = np.asarray(feat["mean"], dtype=np.float32)
        std = np.asarray(feat["sd"], dtype=np.float32)

        if (
            not np.isfinite(mu).all()
            or not np.isfinite(std).all()
            or np.any(std <= 0)
        ):
            raise RuntimeError("Invalid frozen feature scaler")

        values = np.tanh(
            (ranks[:, positions] - mu) / (2.0 * std)
        ).astype(np.float32)

        output.append(torch.from_numpy(values))

    return output


class Branch(nn.Module):
    def __init__(self, nonlinear, cap):
        super().__init__()

        self.main = nn.Linear(24, 1)
        self.cap = cap
        self.nonlinear = nonlinear

        nn.init.normal_(self.main.weight, std=0.025)
        nn.init.zeros_(self.main.bias)

        if nonlinear:
            self.hidden = nn.Linear(
                24, HIDDEN, bias=False
            )

            self.out = nn.Linear(
                HIDDEN, 1, bias=False
            )

            nn.init.normal_(
                self.hidden.weight, std=0.09
            )

            nn.init.normal_(
                self.out.weight, std=0.07
            )

    def forward(self, value):
        z = self.main(value).squeeze(-1)

        if self.nonlinear:
            h = torch.tanh(self.hidden(value))

            residual = torch.tanh(
                self.out(h).squeeze(-1)
            )

            z = z + 0.3 * residual

        return self.cap * torch.tanh(z / self.cap)


class BridgeModel(nn.Module):
    def __init__(self, nonlinear):
        super().__init__()

        self.egfr = Branch(nonlinear, CAP_E)
        self.kras = Branch(nonlinear, CAP_K)

    def forward(self, xe, xk):
        return self.egfr(xe), self.kras(xk)


def contrast(scores, labels, source_ids, positive, negative):
    positives = []
    negatives = []
    bce = []
    within = []

    for source_id in (0, 1):
        selected = source_ids == source_id

        pos = scores[
            selected & positive(labels)
        ]

        neg = scores[
            selected & negative(labels)
        ]

        if min(pos.numel(), neg.numel()) < 10:
            raise RuntimeError(
                "Insufficient cases per contrast and source"
            )

        positives.append(pos)
        negatives.append(neg)

        bce.append(
            0.5 * (
                F.softplus(-pos).mean()
                + F.softplus(neg).mean()
            )
        )

        within.append(
            F.softplus(
                -(pos[:, None] - neg[None, :])
            ).mean()
        )

    ab = F.softplus(
        -(
            positives[0][:, None]
            - negatives[1][None, :]
        )
    ).mean()

    ba = F.softplus(
        -(
            positives[1][:, None]
            - negatives[0][None, :]
        )
    ).mean()

    hardest = SMOOTH_WORST_TEMP * (
        torch.logsumexp(
            torch.stack((ab, ba))
            / SMOOTH_WORST_TEMP,
            dim=0,
        )
        - np.log(2.0)
    )

    alignment = 0.5 * (
        (
            positives[0].mean()
            - positives[1].mean()
        ).square()
        + (
            negatives[0].mean()
            - negatives[1].mean()
        ).square()
    )

    return (
        torch.stack(bce).mean(),
        torch.stack(within).mean(),
        (ab + ba) / 2,
        hardest,
        alignment,
    )


SPECS = {
    "linear_ce": (False, "none", False),
    "residual_ce": (True, "none", False),
    "residual_within": (True, "within", False),
    "residual_cross_mean": (True, "cross_mean", False),
    "residual_cross_worst": (True, "cross_worst", False),
    "linear_bridge": (False, "cross_worst", True),
    "cmb_bridge": (True, "cross_worst", True),
}


def fit(name, xe, xk, labels, source_ids):
    nonlinear, rank_objective, align = SPECS[name]

    torch.manual_seed(SEED)
    model = BridgeModel(nonlinear)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=0,
    )

    yy = torch.from_numpy(labels)
    dd = torch.from_numpy(source_ids)

    for _ in range(EPOCHS):
        optimizer.zero_grad(set_to_none=True)

        se, sk = model(xe, xk)

        ce_e, within_e, cross_e, worst_e, align_e = contrast(
            se, yy, dd,
            lambda t: t == 0,
            lambda t: t != 0,
        )

        ce_k, within_k, cross_k, worst_k, align_k = contrast(
            sk, yy, dd,
            lambda t: t == 1,
            lambda t: t == 2,
        )

        loss = 0.5 * (ce_e + ce_k)

        if rank_objective == "within":
            loss = loss + PAIR_WEIGHT * 0.5 * (
                within_e + within_k
            )

        elif rank_objective == "cross_mean":
            loss = loss + PAIR_WEIGHT * 0.5 * (
                cross_e + cross_k
            )

        elif rank_objective == "cross_worst":
            loss = loss + PAIR_WEIGHT * 0.5 * (
                worst_e + worst_k
            )

        if align:
            loss = loss + ALIGN_WEIGHT * 0.5 * (
                align_e + align_k
            )

        loss = loss + RIDGE * (
            model.egfr.main.weight.square().sum()
            + model.kras.main.weight.square().sum()
        )

        if nonlinear:
            loss = loss + RESIDUAL_RIDGE * (
                model.egfr.hidden.weight.square().sum()
                + model.egfr.out.weight.square().sum()
                + model.kras.hidden.weight.square().sum()
                + model.kras.out.weight.square().sum()
            )

        if not torch.isfinite(loss):
            raise RuntimeError("Nonfinite model objective")

        loss.backward()
        optimizer.step()

    with torch.no_grad():
        se, sk = model(xe, xk)
        components = []

        for scores, pos, neg in (
            (
                se,
                lambda t: t == 0,
                lambda t: t != 0,
            ),
            (
                sk,
                lambda t: t == 1,
                lambda t: t == 2,
            ),
        ):
            terms = contrast(
                scores, yy, dd, pos, neg
            )

            components.append({
                "ce": float(terms[0]),
                "within": float(terms[1]),
                "cross_mean": float(terms[2]),
                "cross_worst": float(terms[3]),
                "alignment": float(terms[4]),
            })

    return model, components


def predict(model, xe, xk):
    with torch.no_grad():
        se, sk = model(xe, xk)

        e = torch.sigmoid(se).numpy().astype(np.float64)
        k = torch.sigmoid(sk).numpy().astype(np.float64)

    prob = np.column_stack((
        e,
        (1 - e) * k,
        (1 - e) * (1 - k),
    ))

    if (
        not np.isfinite(prob).all()
        or not np.allclose(
            prob.sum(axis=1), 1.0, atol=1e-8
        )
    ):
        raise RuntimeError("Invalid predictive probabilities")

    return prob


def evaluate(truth, probability):
    truth = np.asarray(truth, dtype=np.int64)
    prob = np.clip(probability, 1e-12, 1.0)
    pred = probability.argmax(axis=1)

    non_e = truth != 0

    q = np.clip(
        probability[non_e, 1]
        / (
            probability[non_e, 1]
            + probability[non_e, 2]
        ),
        1e-12,
        1 - 1e-12,
    )

    yk = (truth[non_e] == 1).astype(float)
    counts = np.bincount(pred, minlength=3)

    return {
        "nll": float(
            -np.log(
                prob[np.arange(len(truth)), truth]
            ).mean()
        ),
        "macro_f1": float(
            f1_score(
                truth,
                pred,
                average="macro",
                labels=[0, 1, 2],
                zero_division=0,
            )
        ),
        "egfr_auc": float(
            roc_auc_score(
                truth == 0,
                probability[:, 0],
            )
        ),
        "kras_auc": float(
            roc_auc_score(yk, q)
        ),
        "kras_nll": float(
            -(
                yk * np.log(q)
                + (1 - yk) * np.log1p(-q)
            ).mean()
        ),
        "pred_egfr": int(counts[0]),
        "pred_kras": int(counts[1]),
        "pred_dwt": int(counts[2]),
    }


def csv_output(rows):
    buff = io.StringIO(newline="")

    writer = csv.DictWriter(
        buff,
        fieldnames=list(rows[0]),
        lineterminator="\n",
    )

    writer.writeheader()
    writer.writerows(rows)

    return buff.getvalue()


run_id = (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    + "_"
    + uuid.uuid4().hex[:8]
)

RUN = ROOT / "10_cross_cohort_margin_bridge" / run_id
RUN.mkdir(parents=True, exist_ok=False)

artifacts = []


def save(name, payload):
    path = RUN / name

    path.parent.mkdir(parents=True, exist_ok=True)

    data = (
        payload.encode("utf-8")
        if isinstance(payload, str)
        else payload
    )

    with path.open("xb") as handle:
        handle.write(data)

    artifacts.append({
        "path": str(
            path.relative_to(ROOT)
        ).replace("\\", "/"),
        "sha256": hashlib.sha256(data).hexdigest(),
    })


save(
    "PROTOCOL.json",
    json.dumps({
        "phase": "CROSS_COHORT_MARGIN_BRIDGE_DEVELOPMENT",
        "cohorts": list(DEV),
        "architecture": (
            "Bounded two-head linear or small residual network"
        ),
        "variants": list(SPECS),
        "epochs": EPOCHS,
        "pair_weight": PAIR_WEIGHT,
        "alignment_weight": ALIGN_WEIGHT,
        "smooth_worst_temperature": SMOOTH_WORST_TEMP,
        "feature_source": (
            "Phase07 training-only 24 per contrast"
        ),
        "historically_exposed_cohort_results": True,
        "novelty_established": False,
        "external_validation_performed": False,
    }, indent=2) + "\n",
)

results = []
predictions = []
components = []

start = time.monotonic()

for held in DEV:
    train = np.flatnonzero(domains != held)
    test = np.flatnonzero(domains == held)

    source_names = sorted(
        set(domains[train].tolist())
    )

    source_ids = np.asarray([
        source_names.index(c)
        for c in domains[train]
    ], dtype=np.int64)

    features = fold_features(held)

    train_x = tuple(
        t[train] for t in features
    )

    test_x = tuple(
        t[test] for t in features
    )

    for name in SPECS:
        model, comp = fit(
            name,
            train_x[0],
            train_x[1],
            y[train],
            source_ids,
        )

        prob = predict(
            model,
            test_x[0],
            test_x[1],
        )

        metrics = evaluate(y[test], prob)

        results.append({
            "held_cohort": held,
            "model": name,
            "n": len(test),
            **metrics,
        })

        for head, record in zip(("egfr", "kras"), comp):
            components.append({
                "held_cohort": held,
                "model": name,
                "head": head,
                **record,
            })

        for j, sample in enumerate(test):
            predictions.append({
                "held_cohort": held,
                "gsm": str(gsm[sample]),
                "true_class": int(y[sample]),
                "model": name,
                "p_egfr": float(prob[j, 0]),
                "p_kras": float(prob[j, 1]),
                "p_dwt": float(prob[j, 2]),
            })

        buffer = io.BytesIO()

        torch.save({
            "state_dict": model.state_dict(),
            "architecture": name,
            "source_cohorts": source_names,
            "held_out_cohort": held,
            "feature_spec": str(
                phase7_path.parent
                / "folds"
                / f"{held}_features.json"
            ),
        }, buffer)

        save(
            f"models/{held}_{name}.pt",
            buffer.getvalue(),
        )

    historic_prob = []

    for sample in test:
        key = (held, str(gsm[sample]))

        if (
            key not in historic
            or historic[key][0] != int(y[sample])
        ):
            raise RuntimeError(
                "Previous baseline alignment failure"
            )

        historic_prob.append(
            historic[key][1]
        )

    old_prob = np.stack(historic_prob)

    results.append({
        "held_cohort": held,
        "model": "previous_free_bounded",
        "n": len(test),
        **evaluate(y[test], old_prob),
    })

save("LOCO_METRICS.csv", csv_output(results))
save("LOCO_PREDICTIONS.csv", csv_output(predictions))
save(
    "TRAINING_BRIDGE_COMPONENTS.csv",
    csv_output(components),
)

summary = {}

for name in (*SPECS, "previous_free_bounded"):
    subset = [
        r for r in results
        if r["model"] == name
    ]

    if len(subset) != 3:
        raise RuntimeError(
            "Incomplete cross-cohort evaluation"
        )

    summary[name] = {
        "nll": float(
            np.mean([r["nll"] for r in subset])
        ),
        "f1": float(
            np.mean([r["macro_f1"] for r in subset])
        ),
        "kras_nll": float(
            np.mean([r["kras_nll"] for r in subset])
        ),
        "cohort_nll": {
            r["held_cohort"]: r["nll"]
            for r in subset
        },
    }

candidate = summary["cmb_bridge"]

comparators = (
    "previous_free_bounded",
    "residual_ce",
    "residual_within",
    "residual_cross_mean",
    "residual_cross_worst",
    "linear_bridge",
)

gains = {
    name: {
        "nll": (
            summary[name]["nll"]
            - candidate["nll"]
        ),
        "f1": (
            candidate["f1"]
            - summary[name]["f1"]
        ),
    }
    for name in comparators
}

positive = sum(
    candidate["cohort_nll"][held]
    <
    summary["residual_cross_worst"]["cohort_nll"][held]
    for held in DEV
)

screen = (
    "PASS_EXPLORATORY"
    if (
        gains["previous_free_bounded"]["nll"] >= 0.01
        and gains["residual_cross_worst"]["nll"] >= 0.015
        and gains["residual_cross_worst"]["f1"] >= 0.01
        and positive >= 2
        and candidate["kras_nll"]
        <= summary["residual_cross_worst"]["kras_nll"] + 0.01
    )
    else "HOLD"
)

save(
    "SUMMARY.json",
    json.dumps({
        "models": summary,
        "comparisons": gains,
        "screen": screen,
        "matched_positive_cohorts": positive,
        "method_novelty": "NOT_ESTABLISHED",
        "evaluation": "ADAPTIVE_DEVELOPMENT_ONLY",
    }, indent=2) + "\n",
)

source_hash = hashlib.sha256()

with SOURCE.open("rb") as handle:
    for chunk in iter(
        lambda: handle.read(4 * 1024 * 1024),
        b"",
    ):
        source_hash.update(chunk)

save(
    "RUN_MANIFEST.json",
    json.dumps({
        "phase": "CROSS_COHORT_MARGIN_BRIDGE_DEVELOPMENT",
        "created_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "source_sha256": source_hash.hexdigest(),
        "phase07_manifest_sha256": hashlib.sha256(
            phase7_path.read_bytes()
        ).hexdigest(),
        "source_cohorts": list(DEV),
        "gse32863_historically_exposed": True,
        "gse66863_used": False,
        "external_results_inspected": False,
        "prediction_ontology": {
            "0": "EGFR",
            "1": "KRAS",
            "2": "DWT",
        },
        "mutation_assay_completeness": "NOT_FULLY_VERIFIED",
        "novelty": "UNVERIFIED",
        "runtime_seconds": round(
            time.monotonic() - start, 2
        ),
        "artifacts": artifacts,
    }, indent=2) + "\n",
)

print("CMB_DEVELOPMENT: COMPLETE")

for held in DEV:
    print(
        "LOCO",
        held,
        "CLASS=",
        dict(Counter(
            y[domains == held].tolist()
        )),
    )

    for name in (*SPECS, "previous_free_bounded"):
        row = next(
            r for r in results
            if (
                r["held_cohort"] == held
                and r["model"] == name
            )
        )

        print(
            name,
            "NLL=", round(row["nll"], 4),
            "F1=", round(row["macro_f1"], 4),
            "E_AUC=", round(row["egfr_auc"], 4),
            "K_AUC=", round(row["kras_auc"], 4),
            "PRED=", (
                row["pred_egfr"],
                row["pred_kras"],
                row["pred_dwt"],
            ),
        )

for name, row in summary.items():
    print(
        "MEAN",
        name,
        "NLL=", round(row["nll"], 4),
        "F1=", round(row["f1"], 4),
        "K_NLL=", round(row["kras_nll"], 4),
    )

for name, row in gains.items():
    print(
        "GAIN_VS",
        name,
        "NLL=", round(row["nll"], 4),
        "F1=", round(row["f1"], 4),
    )

print(
    "SCREEN:",
    screen,
    "MATCHED_COHORT_WINS:",
    positive,
)

print("NOVELTY: UNVERIFIED")
print("RUN:", RUN)

CMB_DEVELOPMENT: COMPLETE
LOCO GSE13213 CLASS= {0: 45, 2: 57, 1: 15}
linear_ce NLL= 1.2372 F1= 0.2036 E_AUC= 0.7654 K_AUC= 0.8398 PRED= (111, 4, 2)
residual_ce NLL= 1.2371 F1= 0.2036 E_AUC= 0.7654 K_AUC= 0.8398 PRED= (111, 4, 2)
residual_within NLL= 1.2545 F1= 0.2382 E_AUC= 0.7664 K_AUC= 0.8456 PRED= (110, 5, 2)
residual_cross_mean NLL= 1.2486 F1= 0.2382 E_AUC= 0.767 K_AUC= 0.8468 PRED= (110, 5, 2)
residual_cross_worst NLL= 1.2403 F1= 0.2983 E_AUC= 0.7667 K_AUC= 0.8433 PRED= (108, 7, 2)
linear_bridge NLL= 1.2361 F1= 0.2983 E_AUC= 0.7664 K_AUC= 0.8433 PRED= (108, 7, 2)
cmb_bridge NLL= 1.2361 F1= 0.2983 E_AUC= 0.7664 K_AUC= 0.8433 PRED= (108, 7, 2)
previous_free_bounded NLL= 0.9019 F1= 0.5814 E_AUC= 0.7707 K_AUC= 0.8409 PRED= (74, 18, 25)
LOCO GSE31210 CLASS= {1: 20, 2: 79, 0: 127}
linear_ce NLL= 1.0996 F1= 0.3091 E_AUC= 0.8603 K_AUC= 0.7627 PRED= (0, 9, 217)
residual_ce NLL= 1.0996 F1= 0.3091 E_AUC= 0.8603 K_AUC= 0.7627 PRED= (0, 9, 217)
residual_within NLL= 1.1407 F1= 0.3091 E_AUC= 0.8

In [15]:

import csv
import hashlib
import io
import json
import os
import uuid
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import f1_score, roc_auc_score, balanced_accuracy_score

ROOT = Path(r"D:\LUAD_DATAFIRST_METHOD_2026")
SOURCE = Path(
    r"D:\LUAD_CMR_2026\CELL03_DATA_20261021T090947_420335Z_d6aed8ec\cmr_gene_expression.npz"
)

DEV = ("GSE13213", "GSE31210", "GSE32863", "GSE72094")
SIZES = (24, 128, 512)
SEED = 20261023

if os.name != "nt" or not Path("D:/").is_dir():
    raise RuntimeError("Windows D: drive unavailable")

identity_file = ROOT / "00_admin/project.json"

if not identity_file.is_file() or not SOURCE.is_file():
    raise FileNotFoundError("Project identity or expression archive not found")

identity = json.loads(identity_file.read_text(encoding="utf-8"))

if identity.get("project_id") != "LUAD_DATAFIRST_METHOD_2026":
    raise RuntimeError("Project identity mismatch")

with np.load(SOURCE, allow_pickle=False) as z:
    required = {
        "X", "y", "cohort", "gsm", "genes", "patient_ids"
    }

    if not required.issubset(z.files):
        raise RuntimeError("Expression archive schema mismatch")

    X = np.asarray(z["X"], dtype=np.float32)
    y = np.asarray(z["y"], dtype=np.int64)

    domains = np.char.upper(
        np.char.strip(z["cohort"].astype(str))
    )

    gsm = np.char.upper(
        np.char.strip(z["gsm"].astype(str))
    )

    genes = np.char.upper(
        np.char.strip(z["genes"].astype(str))
    )

    patients = np.char.strip(
        z["patient_ids"].astype(str)
    )

if X.shape != (843, 13016) or not np.isfinite(X).all():
    raise RuntimeError("Unexpected source dimensions or nonfinite values")

if set(domains.tolist()) != set(DEV) or set(y.tolist()) != {0, 1, 2}:
    raise RuntimeError("Development cohorts or label schema mismatch")

if len(set(zip(domains.tolist(), gsm.tolist()))) != len(gsm):
    raise RuntimeError("Duplicate cohort/GSM entries")

if len(set(genes.tolist())) != len(genes):
    raise RuntimeError("Nonunique gene identifiers")

expected = {
    "GSE13213": 117,
    "GSE31210": 226,
    "GSE32863": 58,
    "GSE72094": 442,
}

for c, n in expected.items():
    counts = np.bincount(y[domains == c], minlength=3)

    if np.sum(domains == c) != n or counts.min() < 10:
        raise RuntimeError("Cohort composition or class support mismatch: " + c)

patient_cohorts = {}

for c, patient in zip(domains, patients):
    if patient and patient.lower() not in {
        "na", "nan", "none", "unknown"
    }:
        patient_cohorts.setdefault(patient, set()).add(c)

if any(len(v) > 1 for v in patient_cohorts.values()):
    raise RuntimeError("Patient identifier appears across cohorts")

ranks = (
    rankdata(X, axis=1, method="average").astype(np.float32) - 1.0
) / (X.shape[1] - 1.0)

del X


def supervised_gene_order(indices):
    source_cohorts = sorted(set(domains[indices].tolist()))

    if len(source_cohorts) != 3:
        raise RuntimeError("Exactly three training sources required")

    output = {}

    for head in ("EGFR", "KRAS"):
        effects = []

        for c in source_cohorts:
            ix = indices[domains[indices] == c]
            r = ranks[ix]
            labels = y[ix]

            if head == "EGFR":
                positive = r[labels == 0]
                negative = r[labels != 0]
            else:
                positive = r[labels == 1]
                negative = r[labels == 2]

            if min(len(positive), len(negative)) < 10:
                raise RuntimeError("Insufficient per-source contrast support")

            score = (
                positive.mean(axis=0) - negative.mean(axis=0)
            ) / np.maximum(r.std(axis=0), 0.04)

            effects.append(score)

        effects = np.stack(effects)

        agreement = (
            np.abs(np.sign(effects).sum(axis=0))
            / len(source_cohorts)
        )

        score = (
            np.abs(np.median(effects, axis=0))
            * (0.2 + 0.8 * agreement)
        )

        ranked = np.argsort(-score, kind="stable")

        if not np.isfinite(score[ranked[:max(SIZES)]]).all():
            raise RuntimeError("Nonfinite training-only feature scores")

        output[head] = ranked

    return output


def balanced_source_weights(target, source_ids):
    weights = np.zeros(len(target), dtype=np.float64)

    for cohort_name in sorted(set(source_ids.tolist())):
        for label in (0, 1):
            mask = (
                (source_ids == cohort_name)
                & (target == label)
            )

            count = int(mask.sum())

            if count < 10:
                raise RuntimeError("Insufficient binary group support")

            weights[mask] = 1.0 / count

    weights *= len(weights) / weights.sum()

    return weights


def train_heads(xtrain, ytrain, domaintrain, method):
    non_egfr = ytrain != 0

    e_target = (ytrain == 0).astype(int)
    k_target = (ytrain[non_egfr] == 1).astype(int)

    e_weight = balanced_source_weights(
        e_target, domaintrain
    )

    k_weight = balanced_source_weights(
        k_target, domaintrain[non_egfr]
    )

    if method == "lr":
        e = LogisticRegression(
            C=0.2,
            solver="lbfgs",
            max_iter=400,
            random_state=SEED,
        )

        k = LogisticRegression(
            C=0.2,
            solver="lbfgs",
            max_iter=400,
            random_state=SEED,
        )

    elif method == "rbf":
        e = SVC(
            C=1.0,
            kernel="rbf",
            gamma="scale",
            probability=True,
            random_state=SEED,
        )

        k = SVC(
            C=1.0,
            kernel="rbf",
            gamma="scale",
            probability=True,
            random_state=SEED,
        )

    else:
        raise ValueError("Unknown baseline")

    e.fit(
        xtrain,
        e_target,
        sample_weight=e_weight,
    )

    k.fit(
        xtrain[non_egfr],
        k_target,
        sample_weight=k_weight,
    )

    return e, k


def predict_heads(models, xtest):
    e, k = models

    qe = e.predict_proba(xtest)[
        :, np.flatnonzero(e.classes_ == 1)[0]
    ]

    qk = k.predict_proba(xtest)[
        :, np.flatnonzero(k.classes_ == 1)[0]
    ]

    probability = np.column_stack((
        qe,
        (1.0 - qe) * qk,
        (1.0 - qe) * (1.0 - qk),
    ))

    if (
        not np.isfinite(probability).all()
        or not np.allclose(
            probability.sum(axis=1),
            1.0,
            atol=1e-8,
        )
    ):
        raise RuntimeError("Invalid hierarchical probabilities")

    return np.clip(probability, 1e-12, 1.0)


def evaluate(truth, probability):
    prediction = probability.argmax(axis=1)
    non_egfr = truth != 0

    qk = np.clip(
        probability[non_egfr, 1]
        / (
            probability[non_egfr, 1]
            + probability[non_egfr, 2]
        ),
        1e-12,
        1.0 - 1e-12,
    )

    return {
        "nll": float(
            -np.log(
                probability[np.arange(len(truth)), truth]
            ).mean()
        ),
        "macro_f1": float(
            f1_score(
                truth,
                prediction,
                labels=[0, 1, 2],
                average="macro",
                zero_division=0,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(truth, prediction)
        ),
        "egfr_auc": float(
            roc_auc_score(
                truth == 0,
                probability[:, 0],
            )
        ),
        "kras_auc": float(
            roc_auc_score(
                truth[non_egfr] == 1,
                qk,
            )
        ),
        "pred_e": int((prediction == 0).sum()),
        "pred_k": int((prediction == 1).sum()),
        "pred_w": int((prediction == 2).sum()),
    }


def as_csv(rows):
    buffer = io.StringIO(newline="")

    writer = csv.DictWriter(
        buffer,
        fieldnames=list(rows[0]),
        lineterminator="\n",
    )

    writer.writeheader()
    writer.writerows(rows)

    return buffer.getvalue()


run_id = (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    + "_"
    + uuid.uuid4().hex[:8]
)

RUN = ROOT / "11_representation_capacity_gate" / run_id
RUN.mkdir(parents=True, exist_ok=False)

artifacts = []


def save(name, content):
    path = RUN / name
    path.parent.mkdir(parents=True, exist_ok=True)

    payload = (
        content.encode("utf-8")
        if isinstance(content, str)
        else content
    )

    with path.open("xb") as handle:
        handle.write(payload)

    artifacts.append({
        "path": str(path.relative_to(ROOT)).replace("\\", "/"),
        "sha256": hashlib.sha256(payload).hexdigest(),
    })


protocol = {
    "phase": "FOUR_COHORT_CAPACITY_DIAGNOSTIC",
    "source": str(SOURCE),
    "development_cohorts": list(DEV),
    "historically_exposed_cohort": "GSE32863",
    "external_reserved": "GSE66863",
    "per_head_gene_counts": list(SIZES),
    "models": [
        "hier_lr24",
        "hier_lr128",
        "hier_lr512",
        "hier_rbf128",
    ],
    "primary_metric": "equal_cohort_multiclass_nll",
    "secondary_metrics": [
        "macro_f1",
        "egfr_auc",
        "kras_auc",
    ],
    "training_only_feature_selection": True,
    "shared_gene_schema_previously_harmonized": True,
    "source_genotype_provenance": "PROVISIONAL",
    "historical_development_exposure": True,
    "method_novelty": "NOT_TESTED",
}

save(
    "PROTOCOL.json",
    json.dumps(protocol, indent=2) + "\n",
)

rows = []
predictions = []
feature_records = []

for held in DEV:
    train = np.flatnonzero(domains != held)
    test = np.flatnonzero(domains == held)

    ordering = supervised_gene_order(train)

    for k in SIZES:
        columns = np.sort(np.unique(np.r_[
            ordering["EGFR"][:k],
            ordering["KRAS"][:k],
        ]))

        source_values = ranks[train][:, columns]

        mu = source_values.mean(axis=0)
        sd = np.maximum(source_values.std(axis=0), 0.05)

        xtrain = np.tanh(
            (source_values - mu) / (2.0 * sd)
        ).astype(np.float32)

        xtest = np.tanh(
            (ranks[test][:, columns] - mu) / (2.0 * sd)
        ).astype(np.float32)

        feature_records.append({
            "held_cohort": held,
            "per_head_k": k,
            "n_unique_genes": len(columns),
            "gene_list": genes[columns].tolist(),
        })

        methods = (
            ("lr", "rbf") if k == 128 else ("lr",)
        )

        for method in methods:
            name = f"hier_{method}{k}"

            models = train_heads(
                xtrain,
                y[train],
                domains[train],
                method,
            )

            probability = predict_heads(
                models, xtest
            )

            metrics = evaluate(
                y[test], probability
            )

            rows.append({
                "cohort": held,
                "model": name,
                "n": len(test),
                "n_genes": len(columns),
                **metrics,
            })

            for j, sample in enumerate(test):
                predictions.append({
                    "cohort": held,
                    "gsm": str(gsm[sample]),
                    "true_label": int(y[sample]),
                    "model": name,
                    "p_egfr": float(probability[j, 0]),
                    "p_kras": float(probability[j, 1]),
                    "p_dwt": float(probability[j, 2]),
                })

save(
    "LOCO_METRICS.csv",
    as_csv(rows),
)

save(
    "LOCO_PREDICTIONS.csv",
    as_csv(predictions),
)

save(
    "TRAIN_ONLY_FEATURES.json",
    json.dumps(feature_records, indent=2) + "\n",
)

summary = {}

model_names = (
    "hier_lr24",
    "hier_lr128",
    "hier_lr512",
    "hier_rbf128",
)

for name in model_names:
    subset = [
        row for row in rows
        if row["model"] == name
    ]

    if len(subset) != len(DEV):
        raise RuntimeError("Incomplete model coverage")

    summary[name] = {
        metric: float(np.mean([
            row[metric] for row in subset
        ]))
        for metric in (
            "nll",
            "macro_f1",
            "egfr_auc",
            "kras_auc",
        )
    }

save(
    "EQUAL_COHORT_SUMMARY.json",
    json.dumps(summary, indent=2) + "\n",
)

source_hash = hashlib.sha256()

with SOURCE.open("rb") as handle:
    for block in iter(
        lambda: handle.read(4 * 1024 * 1024),
        b"",
    ):
        source_hash.update(block)

save(
    "RUN_MANIFEST.json",
    json.dumps({
        "phase": "FOUR_COHORT_CAPACITY_DIAGNOSTIC",
        "run_id": run_id,
        "source_sha256": source_hash.hexdigest(),
        "artifacts": artifacts,
        "external_data_accessed": False,
        "new_method_established": False,
        "historically_reused_development_data": True,
    }, indent=2) + "\n",
)

print("CAPACITY_GATE: COMPLETE")

for held in DEV:
    print(
        "COHORT",
        held,
        "CLASS",
        dict(Counter(
            y[domains == held].tolist()
        )),
    )

    for name in model_names:
        row = next(
            value for value in rows
            if value["cohort"] == held
            and value["model"] == name
        )

        print(
            name,
            "NLL=", round(row["nll"], 4),
            "F1=", round(row["macro_f1"], 4),
            "E_AUC=", round(row["egfr_auc"], 4),
            "K_AUC=", round(row["kras_auc"], 4),
            "PRED=", (
                row["pred_e"],
                row["pred_k"],
                row["pred_w"],
            ),
        )

for name, result in summary.items():
    print(
        "MEAN",
        name,
        "NLL=", round(result["nll"], 4),
        "F1=", round(result["macro_f1"], 4),
        "E_AUC=", round(result["egfr_auc"], 4),
        "K_AUC=", round(result["kras_auc"], 4),
    )

print("NOVELTY: NOT_TESTED")
print("EXTERNAL: UNTOUCHED")
print("RUN:", RUN)

D:\Anaconda\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
D:\Anaconda\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
D:\Anaconda\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
D:\Anaconda\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` inst

CAPACITY_GATE: COMPLETE
COHORT GSE13213 CLASS {0: 45, 2: 57, 1: 15}
hier_lr24 NLL= 2.5099 F1= 0.2685 E_AUC= 0.7333 K_AUC= 0.8667 PRED= (110, 6, 1)
hier_lr128 NLL= 1.7747 F1= 0.4585 E_AUC= 0.7917 K_AUC= 0.8795 PRED= (92, 18, 7)
hier_lr512 NLL= 3.0415 F1= 0.257 E_AUC= 0.8046 K_AUC= 0.9064 PRED= (112, 4, 1)
hier_rbf128 NLL= 0.9 F1= 0.2184 E_AUC= 0.8034 K_AUC= 0.8971 PRED= (0, 0, 117)
COHORT GSE31210 CLASS {1: 20, 2: 79, 0: 127}
hier_lr24 NLL= 0.6588 F1= 0.6058 E_AUC= 0.8768 K_AUC= 0.7968 PRED= (154, 18, 54)
hier_lr128 NLL= 1.0898 F1= 0.3758 E_AUC= 0.8871 K_AUC= 0.8158 PRED= (32, 2, 192)
hier_lr512 NLL= 2.0697 F1= 0.1898 E_AUC= 0.8877 K_AUC= 0.8424 PRED= (3, 0, 223)
hier_rbf128 NLL= 1.4039 F1= 0.1727 E_AUC= 0.8858 K_AUC= 0.8399 PRED= (0, 0, 226)
COHORT GSE32863 CLASS {0: 17, 2: 19, 1: 22}
hier_lr24 NLL= 0.9614 F1= 0.4825 E_AUC= 0.8393 K_AUC= 0.8158 PRED= (4, 13, 41)
hier_lr128 NLL= 0.8288 F1= 0.643 E_AUC= 0.8881 K_AUC= 0.9211 PRED= (5, 30, 23)
hier_lr512 NLL= 1.751 F1= 0.3981 E_AUC= 0.868 

In [16]:

import csv
import hashlib
import io
import json
import os
import time
import uuid
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.special import expit, lambertw
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score

ROOT = Path(r"D:\LUAD_DATAFIRST_METHOD_2026")
SOURCE = Path(
    r"D:\LUAD_CMR_2026\CELL03_DATA_20261021T090947_420335Z_d6aed8ec\cmr_gene_expression.npz"
)

DEV = ("GSE13213", "GSE31210", "GSE32863", "GSE72094")
RESOLUTIONS = (128, 512)
SEED = 20261023
EPOCHS = 240
LR = 0.045

if os.name != "nt" or not Path("D:/").is_dir():
    raise RuntimeError("Windows D: drive unavailable")

identity_path = ROOT / "00_admin/project.json"

if not identity_path.is_file() or not SOURCE.is_file():
    raise FileNotFoundError("Project identity or expression source missing")

identity = json.loads(
    identity_path.read_text(encoding="utf-8")
)

if identity.get("project_id") != "LUAD_DATAFIRST_METHOD_2026":
    raise RuntimeError("Project identity mismatch")

phase11_files = sorted(
    (ROOT / "11_representation_capacity_gate").glob(
        "*/RUN_MANIFEST.json"
    )
)

if not phase11_files:
    raise RuntimeError("Phase 11 evidence missing")

phase11_path = phase11_files[-1]
phase11_manifest = json.loads(
    phase11_path.read_text(encoding="utf-8")
)

if phase11_manifest.get("phase") != "FOUR_COHORT_CAPACITY_DIAGNOSTIC":
    raise RuntimeError("Unexpected previous phase")

for entry in phase11_manifest.get("artifacts", []):
    path = ROOT / entry["path"]

    if (
        not path.is_file()
        or hashlib.sha256(path.read_bytes()).hexdigest()
        != entry["sha256"]
    ):
        raise RuntimeError("Phase 11 checksum mismatch")

with np.load(SOURCE, allow_pickle=False) as archive:
    required = {
        "X", "y", "cohort", "gsm",
        "genes", "patient_ids"
    }

    if not required.issubset(archive.files):
        raise RuntimeError("Incomplete expression data")

    X = np.asarray(
        archive["X"], dtype=np.float32
    )
    y = np.asarray(
        archive["y"], dtype=np.int64
    )

    cohorts = np.char.upper(
        np.char.strip(archive["cohort"].astype(str))
    )
    gsms = np.char.upper(
        np.char.strip(archive["gsm"].astype(str))
    )
    genes = np.char.upper(
        np.char.strip(archive["genes"].astype(str))
    )
    patients = np.char.strip(
        archive["patient_ids"].astype(str)
    )

expected_sizes = {
    "GSE13213": 117,
    "GSE31210": 226,
    "GSE32863": 58,
    "GSE72094": 442,
}

if (
    X.shape != (843, 13016)
    or not np.isfinite(X).all()
    or len(np.unique(genes)) != len(genes)
):
    raise RuntimeError("Expression panel mismatch")

if (
    set(cohorts.tolist()) != set(DEV)
    or set(y.tolist()) != {0, 1, 2}
    or len(set(zip(cohorts, gsms))) != len(y)
):
    raise RuntimeError("Cohort, label, or sample-ID mismatch")

for c in DEV:
    counts = np.bincount(
        y[cohorts == c], minlength=3
    )

    if (
        int((cohorts == c).sum()) != expected_sizes[c]
        or counts.min() < 10
    ):
        raise RuntimeError("Insufficient cohort support")

patient_sources = {}

for c, patient in zip(cohorts, patients):
    if patient and patient.casefold() not in {
        "unknown", "na", "nan", "none"
    }:
        patient_sources.setdefault(
            patient, set()
        ).add(c)

if any(
    len(values) > 1
    for values in patient_sources.values()
):
    raise RuntimeError("Cross-cohort patient identifier overlap")

ranks = (
    rankdata(
        X, axis=1, method="average"
    ).astype(np.float32) - 1
) / (X.shape[1] - 1)

del X

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(
    min(2, max(1, os.cpu_count() or 1))
)

baseline_csv = (
    phase11_path.parent
    / "LOCO_PREDICTIONS.csv"
)

previous = {}

with baseline_csv.open(
    "r", encoding="utf-8-sig", newline=""
) as handle:
    for row in csv.DictReader(handle):
        if row["model"] not in (
            "hier_lr128", "hier_lr512"
        ):
            continue

        key = (
            row["cohort"],
            row["gsm"].upper(),
            row["model"],
        )

        if key in previous:
            raise RuntimeError("Duplicate Phase 11 prediction")

        previous[key] = (
            int(row["true_label"]),
            np.array([
                float(row["p_egfr"]),
                float(row["p_kras"]),
                float(row["p_dwt"]),
            ], dtype=np.float64),
        )

if len(previous) != 2 * len(y):
    raise RuntimeError("Missing Phase 11 baseline predictions")


def select_orders(train):
    sources = sorted(
        set(cohorts[train].tolist())
    )

    if not 2 <= len(sources) <= 3:
        raise RuntimeError("Invalid training-domain count")

    outcome = {}

    for name in ("EGFR", "KRAS"):
        effects = []

        for cohort in sources:
            idx = train[
                cohorts[train] == cohort
            ]

            r = ranks[idx]
            labels = y[idx]

            if name == "EGFR":
                positive = r[labels == 0]
                negative = r[labels != 0]
            else:
                positive = r[labels == 1]
                negative = r[labels == 2]

            if min(
                len(positive), len(negative)
            ) < 10:
                raise RuntimeError(
                    "Missing per-source contrast support"
                )

            effect = (
                positive.mean(axis=0)
                - negative.mean(axis=0)
            ) / np.maximum(
                r.std(axis=0), 0.04
            )

            effects.append(effect)

        effect = np.stack(effects)

        agreement = (
            np.abs(
                np.sign(effect).sum(axis=0)
            ) / len(sources)
        )

        strength = (
            np.abs(
                np.median(effect, axis=0)
            )
            * (0.2 + 0.8 * agreement)
        )

        order = np.argsort(
            -strength, kind="stable"
        )

        if not np.isfinite(
            strength[order[:max(RESOLUTIONS)]]
        ).all():
            raise RuntimeError(
                "Unstable gene selection"
            )

        outcome[name] = order

    return outcome


def balanced_weights(binary, domains):
    weights = np.zeros(
        len(binary), dtype=np.float64
    )

    for cohort in sorted(
        set(domains.tolist())
    ):
        for label in (0, 1):
            mask = (
                (domains == cohort)
                & (binary == label)
            )

            if mask.sum() < 10:
                raise RuntimeError(
                    "Insufficient group for balanced loss"
                )

            weights[mask] = 1.0 / mask.sum()

    weights *= (
        len(binary) / weights.sum()
    )

    return weights


def fit_bases(train, target, include_train=False):
    orders = select_orders(train)

    target_scores = np.empty(
        (len(target), 4),
        dtype=np.float64,
    )

    train_scores = (
        np.empty(
            (len(train), 4),
            dtype=np.float64,
        )
        if include_train
        else None
    )

    states = {}

    labels = y[train]
    non_egfr = labels != 0

    for k in RESOLUTIONS:
        columns = np.sort(
            np.unique(
                np.r_[
                    orders["EGFR"][:k],
                    orders["KRAS"][:k],
                ]
            )
        )

        training = ranks[train][:, columns]

        mu = training.mean(axis=0)
        sd = np.maximum(
            training.std(axis=0), 0.05
        )

        xtrain = np.tanh(
            (training - mu) / (2.0 * sd)
        ).astype(np.float32)

        xtarget = np.tanh(
            (
                ranks[target][:, columns]
                - mu
            ) / (2.0 * sd)
        ).astype(np.float32)

        target_e = (
            labels == 0
        ).astype(int)

        target_k = (
            labels[non_egfr] == 1
        ).astype(int)

        model_e = LogisticRegression(
            C=0.2,
            solver="lbfgs",
            max_iter=400,
            random_state=SEED,
        )

        model_k = LogisticRegression(
            C=0.2,
            solver="lbfgs",
            max_iter=400,
            random_state=SEED,
        )

        model_e.fit(
            xtrain,
            target_e,
            sample_weight=balanced_weights(
                target_e,
                cohorts[train],
            ),
        )

        model_k.fit(
            xtrain[non_egfr],
            target_k,
            sample_weight=balanced_weights(
                target_k,
                cohorts[train][non_egfr],
            ),
        )

        column = (
            0 if k == 128 else 1
        )

        target_scores[:, column] = (
            model_e.decision_function(xtarget)
        )

        target_scores[:, column + 2] = (
            model_k.decision_function(xtarget)
        )

        if include_train:
            train_scores[:, column] = (
                model_e.decision_function(xtrain)
            )

            train_scores[:, column + 2] = (
                model_k.decision_function(xtrain)
            )

        states[str(k)] = {
            "gene_indices": columns,
            "genes": genes[columns],
            "mu": mu,
            "sd": sd,
            "e_coef": model_e.coef_.ravel(),
            "e_intercept": model_e.intercept_,
            "k_coef": model_k.coef_.ravel(),
            "k_intercept": model_k.intercept_,
        }

    return target_scores, train_scores, states


def hierarchical_probability(scores):
    pe = expit(scores[:, 0])
    pk = expit(scores[:, 1])

    probability = np.column_stack((
        pe,
        (1.0 - pe) * pk,
        (1.0 - pe) * (1.0 - pk),
    ))

    if (
        not np.isfinite(probability).all()
        or not np.allclose(
            probability.sum(axis=1),
            1.0,
            atol=1e-8,
        )
    ):
        raise RuntimeError(
            "Invalid prediction probabilities"
        )

    return probability


def raw_probabilities(scores, mode):
    if mode == "raw128":
        return hierarchical_probability(
            scores[:, [0, 2]]
        )

    if mode == "raw512":
        return hierarchical_probability(
            scores[:, [1, 3]]
        )

    if mode == "raw_half":
        fused = np.column_stack((
            0.5 * (
                scores[:, 0]
                + scores[:, 1]
            ),
            0.5 * (
                scores[:, 2]
                + scores[:, 3]
            ),
        ))

        return hierarchical_probability(fused)

    raise ValueError(mode)


class ResolutionGate(nn.Module):
    def __init__(self, mode):
        super().__init__()

        if mode not in (
            "only128",
            "static",
            "disagreement",
        ):
            raise ValueError(mode)

        self.mode = mode

        self.log_scale = nn.Parameter(
            torch.zeros(2)
        )

        self.bias = nn.Parameter(
            torch.zeros(2)
        )

        if mode != "only128":
            self.mix = nn.Parameter(
                torch.zeros(2)
            )

        if mode == "disagreement":
            self.slope = nn.Parameter(
                torch.full((2,), -3.0)
            )

    def forward(self, scores):
        anchor = scores[:, [0, 2]]
        high = scores[:, [1, 3]]

        if self.mode == "only128":
            fused = anchor

        else:
            delta = high - anchor

            if self.mode == "static":
                alpha = torch.sigmoid(
                    self.mix
                )[None, :]

            else:
                slope = (
                    0.025
                    + F.softplus(self.slope)
                )

                alpha = torch.sigmoid(
                    self.mix[None, :]
                    - slope[None, :]
                    * delta.abs()
                )

            fused = anchor + alpha * delta

        calibrated = (
            6.0 * torch.tanh(fused / 6.0)
            * torch.exp(
                self.log_scale
            ).clamp(0.2, 3.5)
            + self.bias
        )

        e = calibrated[:, 0]
        k = calibrated[:, 1]

        return torch.stack((
            F.logsigmoid(e),
            F.logsigmoid(-e)
            + F.logsigmoid(k),
            F.logsigmoid(-e)
            + F.logsigmoid(-k),
        ), dim=1)


def train_gate(mode, scores, labels, source_names):
    torch.manual_seed(SEED)

    model = ResolutionGate(mode)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LR,
    )

    x = torch.tensor(
        scores, dtype=torch.float32
    )

    yy = torch.tensor(
        labels, dtype=torch.int64
    )

    masks = [
        torch.tensor(
            source_names == cohort
        )
        for cohort in sorted(
            set(source_names.tolist())
        )
    ]

    if len(masks) != 3:
        raise RuntimeError(
            "Calibration requires three source domains"
        )

    for _ in range(EPOCHS):
        optimizer.zero_grad(
            set_to_none=True
        )

        logp = model(x)
        group_losses = []

        for mask in masks:
            group_labels = yy[mask]

            per_sample = -logp[
                mask, group_labels
            ]

            per_class = [
                per_sample[
                    group_labels == c
                ].mean()
                for c in (0, 1, 2)
            ]

            group_losses.append(
                0.7 * per_sample.mean()
                + 0.3 * torch.stack(
                    per_class
                ).mean()
            )

        loss = torch.stack(
            group_losses
        ).mean()

        loss = loss + 0.012 * (
            model.log_scale.square().sum()
            + model.bias.square().sum()
        )

        if mode == "disagreement":
            loss = loss + 0.002 * (
                F.softplus(
                    model.slope
                ).sum()
            )

        if not torch.isfinite(loss):
            raise RuntimeError(
                "Nonfinite calibration loss"
            )

        loss.backward()
        optimizer.step()

    return model


def gate_probability(model, scores):
    with torch.no_grad():
        prediction = model(
            torch.tensor(
                scores,
                dtype=torch.float32,
            )
        ).exp().numpy().astype(np.float64)

    if (
        not np.isfinite(prediction).all()
        or not np.allclose(
            prediction.sum(axis=1),
            1.0,
            atol=1e-6,
        )
    ):
        raise RuntimeError(
            "Nonfinite calibration predictions"
        )

    return prediction


def evaluate(truth, probability):
    probability = np.asarray(
        probability, dtype=np.float64
    )

    if (
        not np.allclose(
            probability.sum(axis=1),
            1.0,
            atol=1e-6,
        )
        or not np.isfinite(
            probability
        ).all()
    ):
        raise RuntimeError(
            "Invalid probability normalization"
        )

    prediction = probability.argmax(axis=1)

    non_egfr = truth != 0

    q = (
        probability[non_egfr, 1]
        / (
            probability[non_egfr, 1]
            + probability[non_egfr, 2]
        )
    )

    q = np.clip(
        q, 1e-12, 1.0 - 1e-12
    )

    kras_truth = (
        truth[non_egfr] == 1
    ).astype(np.float64)

    counts = np.bincount(
        prediction, minlength=3
    )

    return {
        "nll": float(
            -np.log(
                np.clip(
                    probability[
                        np.arange(len(truth)),
                        truth,
                    ],
                    1e-12,
                    1,
                )
            ).mean()
        ),
        "macro_f1": float(
            f1_score(
                truth,
                prediction,
                labels=[0, 1, 2],
                average="macro",
                zero_division=0,
            )
        ),
        "e_auc": float(
            roc_auc_score(
                truth == 0,
                probability[:, 0],
            )
        ),
        "k_auc": float(
            roc_auc_score(
                kras_truth, q
            )
        ),
        "k_nll": float(
            -(
                kras_truth * np.log(q)
                + (1 - kras_truth)
                * np.log1p(-q)
            ).mean()
        ),
        "brier": float(
            np.mean(
                np.sum(
                    (
                        probability
                        - np.eye(3)[truth]
                    ) ** 2,
                    axis=1,
                )
            )
        ),
        "pred_e": int(counts[0]),
        "pred_k": int(counts[1]),
        "pred_w": int(counts[2]),
    }


def to_csv(records):
    buffer = io.StringIO(newline="")

    writer = csv.DictWriter(
        buffer,
        fieldnames=list(records[0]),
        lineterminator="\n",
    )

    writer.writeheader()
    writer.writerows(records)

    return buffer.getvalue()


run_id = (
    datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%SZ"
    )
    + "_"
    + uuid.uuid4().hex[:8]
)

RUN = (
    ROOT
    / "12_multiresolution_ood_evidence"
    / run_id
)

RUN.mkdir(
    parents=True,
    exist_ok=False,
)

artifacts = []


def save(filename, content):
    path = RUN / filename

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    payload = (
        content.encode("utf-8")
        if isinstance(content, str)
        else content
    )

    with path.open("xb") as handle:
        handle.write(payload)

    artifacts.append({
        "path": str(
            path.relative_to(ROOT)
        ).replace("\\", "/"),
        "sha256": hashlib.sha256(
            payload
        ).hexdigest(),
    })


protocol = {
    "phase": "MULTIRESOLUTION_OUTER_LOCO_NESTED_SOURCE_CALIBRATION",
    "development_cohorts": list(DEV),
    "resolutions": list(RESOLUTIONS),
    "architecture": (
        "128/512 hierarchical contrast scores "
        "with a disagreement-dependent evidence gate"
    ),
    "oof_strategy": (
        "Each outer training cohort is predicted "
        "by base models fitted only on the other "
        "two outer training cohorts"
    ),
    "calibration_modes": [
        "only128",
        "static",
        "disagreement",
    ],
    "class_weighted_oof_loss": 0.3,
    "epochs": EPOCHS,
    "seed": SEED,
    "historical_development_outcomes_exposed": True,
    "external_holdout": "GSE66863, not accessed",
    "mutation_assay_provenance": "PARTIALLY_VERIFIED",
    "method_novelty": "UNVERIFIED",
}

save(
    "PROTOCOL.json",
    json.dumps(
        protocol, indent=2
    ) + "\n",
)

rows = []
predictions = []
calibration_parameters = []
reproduction = []

start = time.monotonic()

for held in DEV:
    outer_train = np.flatnonzero(
        cohorts != held
    )

    outer_test = np.flatnonzero(
        cohorts == held
    )

    outer_sources = sorted(
        set(
            cohorts[outer_train].tolist()
        )
    )

    oof = np.empty(
        (len(outer_train), 4),
        dtype=np.float64,
    )

    filled = np.zeros(
        len(outer_train),
        dtype=bool,
    )

    for inner_target in outer_sources:
        inner_hold = (
            cohorts[outer_train]
            == inner_target
        )

        inner_test_ids = outer_train[
            inner_hold
        ]

        inner_train_ids = outer_train[
            ~inner_hold
        ]

        inner_scores, _, _ = fit_bases(
            inner_train_ids,
            inner_test_ids,
        )

        oof[inner_hold] = inner_scores
        filled[inner_hold] = True

    if (
        not filled.all()
        or not np.isfinite(oof).all()
    ):
        raise RuntimeError(
            "Incomplete source-held-out predictions"
        )

    target_scores, source_insample, base_states = (
        fit_bases(
            outer_train,
            outer_test,
            include_train=True,
        )
    )

    if not np.isfinite(
        target_scores
    ).all():
        raise RuntimeError(
            "Invalid held-out score distribution"
        )

    for resolution in RESOLUTIONS:
        state = base_states[
            str(resolution)
        ]

        buffer = io.BytesIO()

        np.savez_compressed(
            buffer, **state
        )

        save(
            f"outer_models/{held}_lr{resolution}.npz",
            buffer.getvalue(),
        )

    buffer = io.BytesIO()

    np.savez_compressed(
        buffer,
        gsm=gsms[outer_train],
        cohort=cohorts[outer_train],
        label=y[outer_train],
        oof_scores=oof,
        source_insample_scores=source_insample,
    )

    save(
        f"nested_oof/{held}.npz",
        buffer.getvalue(),
    )

    all_predictions = {
        mode: raw_probabilities(
            target_scores, mode
        )
        for mode in (
            "raw128",
            "raw512",
            "raw_half",
        )
    }

    fit_settings = (
        (
            "cal128_ood",
            "only128",
            oof,
        ),
        (
            "cal_static_ood",
            "static",
            oof,
        ),
        (
            "cal_disagree_ood",
            "disagreement",
            oof,
        ),
        (
            "cal_disagree_insample",
            "disagreement",
            source_insample,
        ),
    )

    for name, mode, calibration_scores in fit_settings:
        model = train_gate(
            mode,
            calibration_scores,
            y[outer_train],
            cohorts[outer_train],
        )

        all_predictions[name] = gate_probability(
            model,
            target_scores,
        )

        buffer = io.BytesIO()

        torch.save({
            "state_dict": model.state_dict(),
            "mode": mode,
            "outer_held": held,
            "train_source_cohorts": outer_sources,
            "calibration_source": (
                "OOF"
                if calibration_scores is oof
                else "INSAMPLE"
            ),
        }, buffer)

        save(
            f"calibrators/{held}_{name}.pt",
            buffer.getvalue(),
        )

        with torch.no_grad():
            gate_info = {
                "held": held,
                "model": name,
                "log_scale": model.log_scale.tolist(),
                "bias": model.bias.tolist(),
                "mix": (
                    model.mix.tolist()
                    if hasattr(model, "mix")
                    else []
                ),
                "slope": [],
                "residual_bound": [],
            }

            if hasattr(model, "slope"):
                mix = model.mix.detach().numpy()

                slope = (
                    0.025
                    + F.softplus(
                        model.slope
                    )
                ).detach().numpy()

                bound = np.real(
                    lambertw(
                        np.exp(mix - 1.0)
                    )
                ) / slope

                gate_info["slope"] = slope.tolist()
                gate_info["residual_bound"] = (
                    bound.tolist()
                )

                delta = (
                    target_scores[:, [1, 3]]
                    - target_scores[:, [0, 2]]
                )

                influence = (
                    expit(
                        mix[None, :]
                        - slope[None, :]
                        * np.abs(delta)
                    ) * delta
                )

                if np.any(
                    np.max(
                        np.abs(influence),
                        axis=0,
                    ) > bound + 1e-6
                ):
                    raise RuntimeError(
                        "Residual-bound verification failed"
                    )

        calibration_parameters.append(
            gate_info
        )

    for model_name in (
        "raw128", "raw512"
    ):
        historical_name = (
            "hier_lr128"
            if model_name == "raw128"
            else "hier_lr512"
        )

        probability = all_predictions[
            model_name
        ]

        for j, idx in enumerate(outer_test):
            key = (
                held,
                str(gsms[idx]),
                historical_name,
            )

            if (
                key not in previous
                or previous[key][0] != int(y[idx])
            ):
                raise RuntimeError(
                    "Phase 11 baseline sample mismatch"
                )

            reproduction.append(
                float(
                    np.max(
                        np.abs(
                            previous[key][1]
                            - probability[j]
                        )
                    )
                )
            )

    for name, probability in all_predictions.items():
        result = {
            "held_cohort": held,
            "model": name,
            "n": len(outer_test),
            **evaluate(
                y[outer_test],
                probability,
            ),
        }

        rows.append(result)

        for j, idx in enumerate(outer_test):
            predictions.append({
                "held_cohort": held,
                "gsm": str(gsms[idx]),
                "true_class": int(y[idx]),
                "model": name,
                "p_egfr": float(probability[j, 0]),
                "p_kras": float(probability[j, 1]),
                "p_dwt": float(probability[j, 2]),
            })

if max(reproduction) > 1e-4:
    raise RuntimeError(
        "Phase 11 baseline reproduction mismatch: "
        + str(max(reproduction))
    )

save(
    "LOCO_METRICS.csv",
    to_csv(rows),
)

save(
    "LOCO_PREDICTIONS.csv",
    to_csv(predictions),
)

save(
    "CALIBRATION_PARAMETERS.csv",
    to_csv(calibration_parameters),
)

summary = {}

for name in sorted(
    set(row["model"] for row in rows)
):
    subset = [
        row for row in rows
        if row["model"] == name
    ]

    if len(subset) != 4:
        raise RuntimeError(
            "Incomplete outer LOCO evaluation"
        )

    summary[name] = {
        "nll": float(
            np.mean([
                row["nll"] for row in subset
            ])
        ),
        "macro_f1": float(
            np.mean([
                row["macro_f1"] for row in subset
            ])
        ),
        "e_auc": float(
            np.mean([
                row["e_auc"] for row in subset
            ])
        ),
        "k_auc": float(
            np.mean([
                row["k_auc"] for row in subset
            ])
        ),
        "k_nll": float(
            np.mean([
                row["k_nll"] for row in subset
            ])
        ),
        "cohort_nll": {
            row["held_cohort"]: row["nll"]
            for row in subset
        },
    }

candidate = summary[
    "cal_disagree_ood"
]

comparators = (
    "raw128",
    "raw512",
    "raw_half",
    "cal128_ood",
    "cal_static_ood",
    "cal_disagree_insample",
)

gains = {
    name: {
        "nll": (
            summary[name]["nll"]
            - candidate["nll"]
        ),
        "f1": (
            candidate["macro_f1"]
            - summary[name]["macro_f1"]
        ),
    }
    for name in comparators
}

cohort_wins = sum(
    candidate["cohort_nll"][cohort]
    <
    summary["cal_static_ood"][
        "cohort_nll"
    ][cohort]
    for cohort in DEV
)

screen = (
    "PASS_EXPLORATORY"
    if (
        gains["cal_static_ood"]["nll"] >= 0.01
        and gains["cal_static_ood"]["f1"] >= 0.01
        and gains["raw128"]["nll"] > 0
        and cohort_wins >= 3
    )
    else "HOLD"
)

save(
    "EQUAL_COHORT_SUMMARY.json",
    json.dumps({
        "models": summary,
        "candidate": "cal_disagree_ood",
        "gains": gains,
        "screen": screen,
        "cohort_wins_vs_static": cohort_wins,
        "max_baseline_reproduction_error": (
            max(reproduction)
        ),
        "novelty": "NOT_ESTABLISHED",
    }, indent=2) + "\n",
)

source_hash = hashlib.sha256()

with SOURCE.open("rb") as handle:
    for chunk in iter(
        lambda: handle.read(4 * 1024 * 1024),
        b"",
    ):
        source_hash.update(chunk)

save(
    "RUN_MANIFEST.json",
    json.dumps({
        "phase": "MULTIRESOLUTION_OUTER_LOCO_NESTED_SOURCE_CALIBRATION",
        "source_sha256": source_hash.hexdigest(),
        "parent_manifest_sha256": hashlib.sha256(
            phase11_path.read_bytes()
        ).hexdigest(),
        "calibration_labels": "TRAINING_SOURCE_DOMAINS_ONLY",
        "heldout_labels_in_training": False,
        "external_GSE66863_accessed": False,
        "development_reused_multiple_times": True,
        "method_novelty": "NOT_ESTABLISHED",
        "runtime_s": round(
            time.monotonic() - start, 2
        ),
        "artifacts": artifacts,
    }, indent=2) + "\n",
)

print("MULTIRESOLUTION_PILOT: COMPLETE")

print(
    "BASELINE_REPRODUCTION_MAX_ERROR:",
    f"{max(reproduction):.3e}",
)

report_models = (
    "raw128",
    "raw512",
    "cal128_ood",
    "cal_static_ood",
    "cal_disagree_ood",
    "cal_disagree_insample",
)

for held in DEV:
    print(
        "COHORT",
        held,
        "CLASS",
        dict(
            Counter(
                y[cohorts == held].tolist()
            )
        ),
    )

    for name in report_models:
        row = next(
            r for r in rows
            if r["held_cohort"] == held
            and r["model"] == name
        )

        print(
            name,
            "NLL=", round(row["nll"], 4),
            "F1=", round(row["macro_f1"], 4),
            "E_AUC=", round(row["e_auc"], 4),
            "K_AUC=", round(row["k_auc"], 4),
            "PRED=", (
                row["pred_e"],
                row["pred_k"],
                row["pred_w"],
            ),
        )

for name in (
    "raw128",
    "raw512",
    "raw_half",
    "cal128_ood",
    "cal_static_ood",
    "cal_disagree_ood",
    "cal_disagree_insample",
):
    row = summary[name]

    print(
        "MEAN",
        name,
        "NLL=", round(row["nll"], 4),
        "F1=", round(row["macro_f1"], 4),
        "K_NLL=", round(row["k_nll"], 4),
    )

for name, gain in gains.items():
    print(
        "GAIN_VS",
        name,
        "NLL=", round(gain["nll"], 4),
        "F1=", round(gain["f1"], 4),
    )

print(
    "SCREEN:",
    screen,
    "COHORT_WINS_VS_STATIC:",
    cohort_wins,
)

print("NOVELTY: UNVERIFIED")
print("RUN:", RUN)

MULTIRESOLUTION_PILOT: COMPLETE
BASELINE_REPRODUCTION_MAX_ERROR: 1.785e-06
COHORT GSE13213 CLASS {0: 45, 2: 57, 1: 15}
raw128 NLL= 1.7747 F1= 0.4585 E_AUC= 0.7917 K_AUC= 0.8795 PRED= (92, 18, 7)
raw512 NLL= 3.0414 F1= 0.257 E_AUC= 0.8046 K_AUC= 0.9064 PRED= (112, 4, 1)
cal128_ood NLL= 1.7619 F1= 0.4405 E_AUC= 0.7917 K_AUC= 0.8795 PRED= (94, 18, 5)
cal_static_ood NLL= 2.1983 F1= 0.3202 E_AUC= 0.8056 K_AUC= 0.8842 PRED= (107, 8, 2)
cal_disagree_ood NLL= 2.186 F1= 0.3202 E_AUC= 0.8037 K_AUC= 0.883 PRED= (107, 8, 2)
cal_disagree_insample NLL= 4.0622 F1= 0.2695 E_AUC= 0.804 K_AUC= 0.9064 PRED= (111, 4, 2)
COHORT GSE31210 CLASS {1: 20, 2: 79, 0: 127}
raw128 NLL= 1.0898 F1= 0.3758 E_AUC= 0.8871 K_AUC= 0.8158 PRED= (32, 2, 192)
raw512 NLL= 2.0697 F1= 0.1898 E_AUC= 0.8877 K_AUC= 0.8424 PRED= (3, 0, 223)
cal128_ood NLL= 1.0379 F1= 0.1727 E_AUC= 0.8871 K_AUC= 0.8158 PRED= (0, 0, 226)
cal_static_ood NLL= 1.2577 F1= 0.1727 E_AUC= 0.8896 K_AUC= 0.8399 PRED= (0, 0, 226)
cal_disagree_ood NLL= 1.2569 F

In [17]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import uuid
import sys

import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy.optimize import minimize_scalar
from scipy.special import expit, lambertw
from sklearn.metrics import f1_score, roc_auc_score

ROOT = Path(r"D:\LUAD_DATAFIRST_METHOD_2026")
SOURCE = ROOT / "12_multiresolution_ood_evidence" / "20261023T151555Z_638a3c88"
TARGET_PARENT = ROOT / "13_phase12_failure_forensics"

COHORTS = ("GSE13213", "GSE31210", "GSE32863", "GSE72094")
MODELS = (
    "raw128", "raw512", "raw_half", "cal128_ood",
    "cal_static_ood", "cal_disagree_ood", "cal_disagree_insample"
)
CHANNELS = ("EGFR128", "EGFR512", "KRAS128", "KRAS512")
EXPECTED = {
    "GSE13213": (45, 15, 57),
    "GSE31210": (127, 20, 79),
    "GSE32863": (17, 22, 19),
    "GSE72094": (47, 154, 241)
}

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def portable(x):
    if isinstance(x, Path):
        return str(x)
    if isinstance(x, dict):
        return {str(k): portable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [portable(v) for v in x]
    if isinstance(x, np.integer):
        return int(x)
    if isinstance(x, (float, np.floating)):
        return float(x) if np.isfinite(x) else None
    if isinstance(x, np.bool_):
        return bool(x)
    return x

def save_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(portable(data), f, indent=2, ensure_ascii=False, allow_nan=False)

def effective_logits(p):
    eps = 1e-12
    e = np.clip(p[:, 0], eps, 1 - eps)
    k = np.clip(p[:, 1], eps, 1)
    w = np.clip(p[:, 2], eps, 1)
    return np.column_stack((np.log(e / (1 - e)), np.log(k / w)))

def evaluate(y, p):
    eps = 1e-15
    pred = p.argmax(axis=1)
    sub = y != 0
    qc = p[sub, 1] / np.clip(p[sub, 1] + p[sub, 2], eps, None)
    yt = (y[sub] == 1).astype(int)

    return {
        "n": len(y),
        "nll": float(-np.log(np.clip(p[np.arange(len(y)), y], eps, 1)).mean()),
        "macro_f1": float(f1_score(
            y, pred, labels=[0, 1, 2], average="macro", zero_division=0
        )),
        "egfr_auc": float(roc_auc_score((y == 0).astype(int), p[:, 0])),
        "kras_cond_auc": float(roc_auc_score(yt, qc)),
        "kras_cond_nll": float(-np.mean(
            yt * np.log(np.clip(qc, eps, 1)) +
            (1 - yt) * np.log(np.clip(1 - qc, eps, 1))
        )),
        "brier": float(np.mean(np.sum((p - np.eye(3)[y]) ** 2, axis=1))),
        "pred_e": int((pred == 0).sum()),
        "pred_k": int((pred == 1).sum()),
        "pred_w": int((pred == 2).sum())
    }

assert SOURCE.is_dir(), f"Phase 12 source not found: {SOURCE}"

required = [
    SOURCE / "LOCO_PREDICTIONS.csv",
    SOURCE / "LOCO_METRICS.csv",
    SOURCE / "EQUAL_COHORT_SUMMARY.json",
    SOURCE / "RUN_MANIFEST.json"
]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError("Missing Phase 12 artifacts: " + ", ".join(missing))

df = pd.read_csv(SOURCE / "LOCO_PREDICTIONS.csv")
columns = (
    "held_cohort", "gsm", "true_class", "model",
    "p_egfr", "p_kras", "p_dwt"
)
assert set(columns).issubset(df.columns), f"Unexpected schema: {list(df.columns)}"

df = df.loc[:, columns].copy()
df["gsm"] = df["gsm"].astype(str)
df["true_class"] = pd.to_numeric(df["true_class"], errors="raise").astype(int)

pcols = ["p_egfr", "p_kras", "p_dwt"]
for col in pcols:
    df[col] = pd.to_numeric(df[col], errors="raise")

assert set(df.held_cohort) == set(COHORTS)
assert set(df.model) == set(MODELS)
assert df.true_class.isin([0, 1, 2]).all()
assert not df.duplicated(["held_cohort", "gsm", "model"]).any()

all_probs = df[pcols].to_numpy(dtype=float)
assert np.isfinite(all_probs).all()
assert ((all_probs >= 0) & (all_probs <= 1)).all()

simplex_error = float(np.max(np.abs(all_probs.sum(axis=1) - 1)))
assert simplex_error < 1e-5, f"Invalid probability simplex: {simplex_error}"

aligned = {}
for held in COHORTS:
    group = df.loc[df.held_cohort == held]
    aligned[held] = {}

    for model in MODELS:
        frame = (
            group.loc[group.model == model]
            .sort_values("gsm")
            .reset_index(drop=True)
        )
        assert len(frame) == sum(EXPECTED[held])
        counts = tuple(int((frame.true_class == c).sum()) for c in range(3))
        assert counts == EXPECTED[held], f"Label mismatch: {held}/{model}"
        aligned[held][model] = frame

    reference = aligned[held]["raw128"]
    for model in MODELS[1:]:
        other = aligned[held][model]
        assert other.gsm.equals(reference.gsm)
        assert other.true_class.equals(reference.true_class)

TARGET_PARENT.mkdir(parents=True, exist_ok=True)
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ_") + uuid.uuid4().hex[:8]
RUN = TARGET_PARENT / run_id
RUN.mkdir(exist_ok=False)

save_json(RUN / "PROTOCOL.json", {
    "phase": 13,
    "purpose": "Read-only Phase 12 failure-mechanism audit",
    "source_run": SOURCE,
    "cohorts": COHORTS,
    "models": MODELS,
    "channels": CHANNELS,
    "seed": 20261023,
    "training_performed": False,
    "external_test_used": False,
    "development_status": "Exploratory; all four cohorts historically exposed",
    "versions": {
        "python": sys.version,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": scipy.__version__,
        "scikit_learn": sklearn.__version__
    },
    "limitations": [
        "Retrospective diagnostic comparisons are not independent validation",
        "Score-distribution differences do not prove a causal shift mechanism",
        "Effective logits inferred from probabilities may differ from serialized gate inputs"
    ]
})

metrics_rows = []
prevalence_rows = []
paired_rows = []
complement_rows = []
strata_rows = []
outer_score_rows = []

for held in COHORTS:
    frames = aligned[held]
    y = frames["raw128"].true_class.to_numpy(dtype=int)
    indices = np.arange(len(y))
    probs = {}

    for model in MODELS:
        p = frames[model][pcols].to_numpy(dtype=float)
        probs[model] = p

        metrics_rows.append({
            "held_cohort": held,
            "model": model,
            **evaluate(y, p)
        })

        hard = p.argmax(axis=1)
        for cls, name in enumerate(("EGFR", "KRAS", "DoubleWT")):
            true_fraction = float(np.mean(y == cls))
            prevalence_rows.append({
                "held_cohort": held,
                "model": model,
                "class": name,
                "n": len(y),
                "true_fraction": true_fraction,
                "mean_probability": float(p[:, cls].mean()),
                "hard_pred_fraction": float(np.mean(hard == cls)),
                "probability_bias": float(p[:, cls].mean() - true_fraction)
            })

    proposed = probs["cal_disagree_ood"]
    proposed_loss = -np.log(np.clip(proposed[indices, y], 1e-15, 1))

    for comparator in ("raw128", "raw512", "cal128_ood", "cal_static_ood"):
        base = probs[comparator]
        base_loss = -np.log(np.clip(base[indices, y], 1e-15, 1))
        paired_rows.append({
            "held_cohort": held,
            "comparator": comparator,
            "nll_gain_positive_is_better": float((base_loss - proposed_loss).mean()),
            "new_worse_nll_fraction": float(np.mean(proposed_loss > base_loss + 1e-10)),
            "mean_absolute_probability_delta": float(np.abs(proposed - base).mean()),
            "max_absolute_probability_delta": float(np.abs(proposed - base).max()),
            "changed_hard_predictions": int(
                np.sum(proposed.argmax(axis=1) != base.argmax(axis=1))
            )
        })

    p128 = probs["raw128"]
    p512 = probs["raw512"]

    hard128 = p128.argmax(axis=1)
    hard512 = p512.argmax(axis=1)

    loss128 = -np.log(np.clip(p128[indices, y], 1e-15, 1))
    loss512 = -np.log(np.clip(p512[indices, y], 1e-15, 1))

    logit128 = effective_logits(p128)
    logit512 = effective_logits(p512)
    disagreement = np.abs(logit512 - logit128)

    complement_rows.append({
        "held_cohort": held,
        "n": len(y),
        "raw512_lower_sample_nll_fraction": float(np.mean(loss512 < loss128)),
        "raw128_lower_sample_nll_fraction": float(np.mean(loss128 < loss512)),
        "raw512_rescues_raw128_errors": int(np.sum((hard128 != y) & (hard512 == y))),
        "raw128_rescues_raw512_errors": int(np.sum((hard128 == y) & (hard512 != y))),
        "both_wrong": int(np.sum((hard128 != y) & (hard512 != y))),
        "median_disagreement_egfr": float(np.median(disagreement[:, 0])),
        "median_disagreement_kras": float(np.median(disagreement[:, 1])),
        "p90_disagreement_egfr": float(np.percentile(disagreement[:, 0], 90)),
        "p90_disagreement_kras": float(np.percentile(disagreement[:, 1], 90))
    })

    for h, head in enumerate(("EGFR", "KRAS")):
        ranks = pd.Series(disagreement[:, h]).rank(method="first")
        quartiles = pd.qcut(ranks, q=4, labels=False).to_numpy()

        for q in range(4):
            mask = quartiles == q
            strata_rows.append({
                "held_cohort": held,
                "head": head,
                "disagreement_quartile": q + 1,
                "n": int(mask.sum()),
                "median_abs_disagreement": float(np.median(disagreement[mask, h])),
                "raw512_lower_nll_fraction": float(np.mean(loss512[mask] < loss128[mask])),
                "raw512_minus_raw128_mean_nll": float(
                    np.mean(loss512[mask] - loss128[mask])
                )
            })

    for model, logits in (("raw128", logit128), ("raw512", logit512)):
        for h, head in enumerate(("EGFR", "KRAS")):
            s = logits[:, h]
            outer_score_rows.append({
                "held_cohort": held,
                "model": model,
                "head": head,
                "mean_effective_logit": float(s.mean()),
                "sd_effective_logit": float(s.std()),
                "median_effective_logit": float(np.median(s)),
                "p10_effective_logit": float(np.percentile(s, 10)),
                "p90_effective_logit": float(np.percentile(s, 90))
            })

metric_df = pd.DataFrame(metrics_rows)
metric_df.to_csv(RUN / "RECOMPUTED_LOCO_METRICS.csv", index=False)
pd.DataFrame(prevalence_rows).to_csv(
    RUN / "PREVALENCE_AND_PREDICTION_BIAS.csv", index=False
)
pd.DataFrame(paired_rows).to_csv(
    RUN / "PAIRED_PROBABILITY_DIAGNOSTICS.csv", index=False
)
pd.DataFrame(complement_rows).to_csv(
    RUN / "BRANCH_COMPLEMENTARITY.csv", index=False
)
pd.DataFrame(strata_rows).to_csv(
    RUN / "DISAGREEMENT_STRATIFIED_COMPLEMENTARITY.csv", index=False
)
pd.DataFrame(outer_score_rows).to_csv(
    RUN / "OUTER_EFFECTIVE_LOGIT_DISTRIBUTIONS.csv", index=False
)

nested_rows = []
nested_audit = []

for held in COHORTS:
    fp = SOURCE / "nested_oof" / f"{held}.npz"

    if not fp.is_file():
        nested_audit.append({
            "held_cohort": held,
            "status": "MISSING",
            "path": str(fp)
        })
        continue

    try:
        with np.load(fp, allow_pickle=False) as z:
            keys = list(z.files)

            assert "oof_scores" in keys
            assert "source_insample_scores" in keys

            oof = np.asarray(z["oof_scores"], dtype=float)
            ins = np.asarray(z["source_insample_scores"], dtype=float)

            assert oof.ndim == 2 and oof.shape[1] == 4
            assert ins.shape == oof.shape
            assert np.isfinite(oof).all() and np.isfinite(ins).all()

            try:
                source_cohort = np.asarray(z["cohort"]).astype(str)
            except (ValueError, KeyError):
                source_cohort = np.repeat("ALL_TRAINING_SOURCES", len(oof))

            try:
                labels = np.asarray(z["label"]).astype(int)
            except (ValueError, KeyError):
                labels = None

        assert len(source_cohort) == len(oof)

        if labels is not None:
            assert len(labels) == len(oof)
            assert np.isin(labels, [0, 1, 2]).all()

        nested_audit.append({
            "held_cohort": held,
            "status": "PASS",
            "n": len(oof),
            "keys": keys,
            "shape": list(oof.shape),
            "sha256": sha256(fp)
        })

        for train_source in sorted(set(source_cohort)):
            mask = source_cohort == train_source

            for j, channel in enumerate(CHANNELS):
                a = oof[mask, j]
                b = ins[mask, j]
                gap = np.abs(a - b)

                corr = np.nan
                if len(a) > 2 and a.std() > 0 and b.std() > 0:
                    corr = float(np.corrcoef(a, b)[0, 1])

                record = {
                    "held_cohort": held,
                    "training_source": train_source,
                    "channel": channel,
                    "n": int(mask.sum()),
                    "oof_mean": float(a.mean()),
                    "oof_sd": float(a.std()),
                    "insample_mean": float(b.mean()),
                    "insample_sd": float(b.std()),
                    "median_abs_gap": float(np.median(gap)),
                    "p90_abs_gap": float(np.percentile(gap, 90)),
                    "correlation": corr
                }

                if labels is not None:
                    if channel.startswith("EGFR"):
                        valid = mask
                        target = (labels[valid] == 0).astype(int)
                    else:
                        valid = mask & (labels != 0)
                        target = (labels[valid] == 1).astype(int)

                    if len(np.unique(target)) == 2:
                        record["oof_auc"] = float(roc_auc_score(target, oof[valid, j]))
                        record["insample_auc"] = float(roc_auc_score(target, ins[valid, j]))

                nested_rows.append(record)

    except Exception as exc:
        nested_audit.append({
            "held_cohort": held,
            "status": "FAILED",
            "error": f"{type(exc).__name__}: {exc}"
        })

pd.DataFrame(nested_rows).to_csv(
    RUN / "OOF_VS_INSAMPLE_DIAGNOSTICS.csv", index=False
)
save_json(RUN / "NESTED_OOF_AUDIT.json", nested_audit)

calibration_fp = SOURCE / "CALIBRATION_PARAMETERS.csv"
calibration_info = {"exists": calibration_fp.is_file()}

if calibration_fp.is_file():
    calibration = pd.read_csv(calibration_fp)
    calibration_info.update({
        "columns": list(calibration.columns),
        "shape": list(calibration.shape),
        "first_three_rows": json.loads(
            calibration.head(3).to_json(orient="records")
        )
    })

save_json(RUN / "CALIBRATION_SCHEMA.json", calibration_info)

checkpoint_info = []
checkpoint_dir = SOURCE / "calibrators"

if checkpoint_dir.is_dir():
    try:
        import torch

        for fp in sorted(checkpoint_dir.glob("*.pt")):
            record = {
                "file": fp.name,
                "sha256": sha256(fp)
            }

            try:
                obj = torch.load(fp, map_location="cpu", weights_only=True)
                state = obj.get("state_dict", obj) if isinstance(obj, dict) else obj

                record["parameters"] = {}

                if isinstance(state, dict):
                    for name, value in state.items():
                        if torch.is_tensor(value):
                            arr = value.detach().cpu().numpy()
                            record["parameters"][str(name)] = (
                                arr.tolist() if arr.size <= 20
                                else {"shape": list(arr.shape)}
                            )
                        else:
                            record["parameters"][str(name)] = {
                                "type": type(value).__name__
                            }
                else:
                    record["status"] = "UNKNOWN_CHECKPOINT_STRUCTURE"

            except Exception as exc:
                record["status"] = "SAFE_CHECKPOINT_READ_FAILED"
                record["error"] = f"{type(exc).__name__}: {exc}"

            checkpoint_info.append(record)

    except ImportError:
        checkpoint_info.append({"status": "PYTORCH_NOT_AVAILABLE"})

save_json(RUN / "GATE_CHECKPOINT_INSPECTION.json", checkpoint_info)

bound_checks = []

for a, b in [
    (-4.0, 0.1),
    (-1.0, 0.5),
    (0.0, 1.0),
    (2.0, 0.2),
    (4.0, 2.0)
]:
    w = float(np.real(lambertw(np.exp(a - 1))))
    bound = w / b
    peak = (1 + w) / b

    result = minimize_scalar(
        lambda t: -t * expit(a - b * t),
        bounds=(0, max(15 / b, 3 * peak)),
        method="bounded",
        options={"xatol": 1e-12}
    )

    numeric = -float(result.fun)

    bound_checks.append({
        "a": a,
        "b": b,
        "analytic_bound": bound,
        "numerical_maximum": numeric,
        "absolute_error": abs(numeric - bound)
    })

max_bound_error = max(v["absolute_error"] for v in bound_checks)
assert max_bound_error < 1e-8

save_json(RUN / "LAMBERT_BOUND_CHECK.json", {
    "scope": "Pre-calibration disagreement correction",
    "post_calibration": "Correction sensitivity is rescaled by learned gamma",
    "checks": bound_checks
})

summary = (
    metric_df.groupby("model", sort=False)[
        ["nll", "macro_f1", "egfr_auc", "kras_cond_auc", "kras_cond_nll", "brier"]
    ]
    .mean()
    .reset_index()
)

summary.to_csv(RUN / "EQUAL_COHORT_RECOMPUTED.csv", index=False)

lookup = summary.set_index("model")
gate = lookup.loc["cal_disagree_ood"]
static = lookup.loc["cal_static_ood"]
anchor = lookup.loc["raw128"]

decision = {
    "status": "NO_GO_CURRENT_GATE",
    "gate_minus_static_macro_f1": float(gate.macro_f1 - static.macro_f1),
    "static_minus_gate_nll": float(static.nll - gate.nll),
    "raw128_minus_gate_macro_f1": float(anchor.macro_f1 - gate.macro_f1),
    "gate_minus_raw128_nll": float(gate.nll - anchor.nll),
    "probability_simplex_max_error": simplex_error,
    "oof_audit_passes": sum(x["status"] == "PASS" for x in nested_audit),
    "score_semantics": "Outer effective logits are inferred from predicted probabilities"
}

save_json(RUN / "FAILURE_DIAGNOSIS.json", decision)

source_metrics = pd.read_csv(SOURCE / "LOCO_METRICS.csv", nrows=4)
save_json(RUN / "PHASE12_METRICS_SCHEMA.json", {
    "columns": list(source_metrics.columns),
    "first_four_rows": json.loads(source_metrics.to_json(orient="records"))
})

source_files = sorted(
    p for p in SOURCE.rglob("*")
    if p.is_file() and p.suffix.lower() in (".json", ".csv", ".npz", ".pt")
)
source_items = [
    {
        "path": str(p.relative_to(SOURCE)),
        "sha256": sha256(p),
        "bytes": p.stat().st_size
    }
    for p in source_files
]

outputs = [
    {
        "filename": p.name,
        "sha256": sha256(p),
        "bytes": p.stat().st_size
    }
    for p in sorted(RUN.iterdir())
    if p.is_file()
]

save_json(RUN / "RUN_MANIFEST.json", {
    "source_run": str(SOURCE),
    "phase": 13,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "input_artifacts": source_items,
    "output_artifacts": outputs
})

print("PHASE13: FORENSIC_AUDIT_COMPLETE")
print("PREDICTION_ROWS:", len(df))
print("PROBABILITY_SUM_MAX_ERROR:", f"{simplex_error:.3e}")

print("LOCO_METRICS:")
print(
    summary[["model", "nll", "macro_f1"]]
    .round(4)
    .to_string(index=False)
)

print("GATE_MINUS_STATIC_F1:", f"{decision['gate_minus_static_macro_f1']:.4f}")
print("STATIC_MINUS_GATE_NLL:", f"{decision['static_minus_gate_nll']:.4f}")

print("NESTED_OOF_STATUS:", [
    (x["held_cohort"], x["status"]) for x in nested_audit
])

if nested_rows:
    oof_df = pd.DataFrame(nested_rows)
    print("OOF_MEDIAN_ABS_GAP_BY_CHANNEL:", (
        oof_df.groupby("channel")["median_abs_gap"]
        .median()
        .round(4)
        .to_dict()
    ))

print("BRANCH_COMPLEMENTARITY:")
print(
    pd.DataFrame(complement_rows)[[
        "held_cohort",
        "raw512_lower_sample_nll_fraction",
        "raw512_rescues_raw128_errors",
        "raw128_rescues_raw512_errors"
    ]]
    .round(4)
    .to_string(index=False)
)

print("GATE_VS_STATIC_PROBABILITY_DELTA:")
print(
    pd.DataFrame(paired_rows)
    .query("comparator == 'cal_static_ood'")[[
        "held_cohort",
        "mean_absolute_probability_delta",
        "changed_hard_predictions"
    ]]
    .round(6)
    .to_string(index=False)
)

print("CALIBRATION_COLUMNS:", calibration_info.get("columns", []))
print("BOUND_NUMERICAL_MAX_ERROR:", f"{max_bound_error:.3e}")
print("RUN:", RUN)

PHASE13: FORENSIC_AUDIT_COMPLETE
PREDICTION_ROWS: 5901
PROBABILITY_SUM_MAX_ERROR: 8.941e-08
LOCO_METRICS:
                model    nll  macro_f1
               raw128 1.1371    0.5117
               raw512 1.9665    0.3179
             raw_half 1.3906    0.4167
           cal128_ood 1.1387    0.4237
       cal_static_ood 1.3446    0.3790
     cal_disagree_ood 1.3416    0.3824
cal_disagree_insample 3.0897    0.3244
GATE_MINUS_STATIC_F1: 0.0033
STATIC_MINUS_GATE_NLL: 0.0031
NESTED_OOF_STATUS: [('GSE13213', 'PASS'), ('GSE31210', 'PASS'), ('GSE32863', 'PASS'), ('GSE72094', 'PASS')]
OOF_MEDIAN_ABS_GAP_BY_CHANNEL: {'EGFR128': 1.9637, 'EGFR512': 3.3799, 'KRAS128': 1.1036, 'KRAS512': 1.7539}
BRANCH_COMPLEMENTARITY:
held_cohort  raw512_lower_sample_nll_fraction  raw512_rescues_raw128_errors  raw128_rescues_raw512_errors
   GSE13213                            0.4444                             2                            13
   GSE31210                            0.3540                          

In [18]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import uuid
import sys

import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score, f1_score
import torch
import torch.nn as nn
import torch.nn.functional as F

ROOT = Path(r"D:\LUAD_DATAFIRST_METHOD_2026")
SOURCE = ROOT / "12_multiresolution_ood_evidence" / "20261023T151555Z_638a3c88"
DEST = ROOT / "14_quantile_residual_transfer"

SEED = 20261023
COHORTS = ("GSE13213", "GSE31210", "GSE32863", "GSE72094")
COUNTS = {
    "GSE13213": (45, 15, 57),
    "GSE31210": (127, 20, 79),
    "GSE32863": (17, 22, 19),
    "GSE72094": (47, 154, 241)
}
NEW_MODELS = ("q_anchor", "q_static", "q_residual", "q_residual_guard")
BASE_MODELS = ("raw128", "raw512", "cal_disagree_ood")
CHANNELS = ("EGFR128", "EGFR512", "KRAS128", "KRAS512")
PROB_COLS = ("p_egfr", "p_kras", "p_dwt")

EPOCHS = 400
LR = 0.045
CORRECTION_BOUND = 1.5
COEFFICIENT_L2 = 0.02
GROUP_PENALTY = 25.0


def checksum(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def native(x):
    if isinstance(x, Path):
        return str(x)
    if isinstance(x, dict):
        return {str(k): native(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [native(v) for v in x]
    if isinstance(x, np.integer):
        return int(x)
    if isinstance(x, (np.floating, float)):
        return float(x) if np.isfinite(x) else None
    if isinstance(x, np.bool_):
        return bool(x)
    return x


def write_json(path, item):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(native(item), f, indent=2, ensure_ascii=False, allow_nan=False)


def get_array(z, choices):
    for key in choices:
        if key in z.files:
            return np.asarray(z[key])
    return None


def effective_scores(p):
    eps = 1e-12
    pe = np.clip(p[:, 0], eps, 1 - eps)
    pk = np.clip(p[:, 1], eps, 1)
    pw = np.clip(p[:, 2], eps, 1)
    return np.column_stack((
        np.log(pe / (1 - pe)),
        np.log(pk / pw)
    ))


def quantile_evidence(raw, source):
    raw = np.asarray(raw, dtype=np.float64)
    source = np.asarray(source).astype(str)

    assert raw.ndim == 2 and raw.shape[1] == 4
    assert len(source) == len(raw) and np.isfinite(raw).all()

    out = np.zeros_like(raw)

    for group in np.unique(source):
        ix = np.flatnonzero(source == group)
        n = len(ix)

        if n < 25:
            raise ValueError(
                f"Batch too small for cohort rank transform: {group} ({n})"
            )

        for j in range(4):
            u = (rankdata(raw[ix, j], method="average") - 0.5) / n
            u = np.clip(u, 1e-5, 1 - 1e-5)
            out[ix, j] = (np.log(u) - np.log1p(-u)) / 2.5

    return out


def metrics(y, p):
    p = np.asarray(p, dtype=float)

    assert len(y) == len(p) and p.shape[1] == 3
    assert np.isfinite(p).all() and np.all((p >= 0) & (p <= 1))
    assert np.max(np.abs(p.sum(axis=1) - 1)) < 1e-6

    pred = p.argmax(axis=1)
    k = y != 0
    qk = p[k, 1] / np.clip(p[k, 1] + p[k, 2], 1e-12, None)
    yk = (y[k] == 1).astype(int)

    return {
        "nll": float(
            -np.log(np.clip(p[np.arange(len(y)), y], 1e-15, 1)).mean()
        ),
        "macro_f1": float(
            f1_score(
                y, pred, labels=[0, 1, 2],
                average="macro", zero_division=0
            )
        ),
        "egfr_auc": float(
            roc_auc_score((y == 0).astype(int), p[:, 0])
        ),
        "kras_auc": float(roc_auc_score(yk, qk)),
        "kras_cond_nll": float(-np.mean(
            yk * np.log(np.clip(qk, 1e-15, 1)) +
            (1 - yk) * np.log(np.clip(1 - qk, 1e-15, 1))
        )),
        "brier": float(
            np.mean(np.sum((p - np.eye(3)[y]) ** 2, axis=1))
        ),
        "pred_e": int((pred == 0).sum()),
        "pred_k": int((pred == 1).sum()),
        "pred_w": int((pred == 2).sum())
    }


class QuantileResidual(nn.Module):
    def __init__(self, kind):
        super().__init__()

        self.kind = kind
        self.guard_lambda = 1.0

        self.bias = nn.Parameter(
            torch.zeros(2, dtype=torch.float64)
        )

        init = float(np.log(np.expm1(2.0)))
        self.raw_scale = nn.Parameter(
            torch.full((2,), init, dtype=torch.float64)
        )

        n = 0 if kind == "q_anchor" else (
            1 if kind == "q_static" else 3
        )

        self.weights = nn.Parameter(
            torch.zeros((2, n), dtype=torch.float64)
        )

    def forward(self, x):
        a = x[:, (0, 2)]
        d = x[:, (1, 3)] - a

        scores = self.bias + F.softplus(self.raw_scale) * a

        if self.kind == "q_anchor":
            return scores

        if self.kind == "q_static":
            feats = d.unsqueeze(-1)
        else:
            feats = torch.stack((
                d,
                d * torch.tanh(1.5 * a),
                torch.tanh(2.0 * d) *
                (1 - torch.tanh(a).square())
            ), dim=-1)

        correction = (feats * self.weights).sum(dim=-1)

        return scores + (
            self.guard_lambda *
            CORRECTION_BOUND *
            torch.tanh(correction / CORRECTION_BOUND)
        )


def log_prob_from_scores(s):
    se, sk = s[:, 0], s[:, 1]

    return torch.stack((
        F.logsigmoid(se),
        F.logsigmoid(-se) + F.logsigmoid(sk),
        F.logsigmoid(-se) + F.logsigmoid(-sk)
    ), dim=1)


def fit_quantile_model(kind, xt, yt, group_masks, teacher=None):
    model = QuantileResidual(kind)

    if teacher is not None:
        with torch.no_grad():
            model.bias.copy_(teacher.bias)
            model.raw_scale.copy_(teacher.raw_scale)

        model.bias.requires_grad_(False)
        model.raw_scale.requires_grad_(False)

    optimizer = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad],
        lr=LR
    )

    base_group_nll = None

    if teacher is not None:
        with torch.no_grad():
            base_lp = log_prob_from_scores(teacher(xt))
            row = torch.arange(len(yt))
            sample_base = -base_lp[row, yt]
            base_group_nll = torch.stack([
                sample_base[m].mean() for m in group_masks
            ])

    for _ in range(EPOCHS):
        optimizer.zero_grad(set_to_none=True)

        logp = log_prob_from_scores(model(xt))
        sample_nll = -logp[torch.arange(len(yt)), yt]
        group_nll = torch.stack([
            sample_nll[m].mean() for m in group_masks
        ])

        objective = group_nll.mean()

        if kind == "q_anchor":
            objective = objective + (
                0.001 *
                F.softplus(model.raw_scale).square().mean()
            )
        else:
            objective = objective + (
                COEFFICIENT_L2 *
                model.weights.square().mean()
            )

            if kind == "q_residual_guard":
                excess = F.relu(group_nll - base_group_nll)
                objective = objective + (
                    GROUP_PENALTY *
                    excess.square().mean()
                )

        if not torch.isfinite(objective):
            raise FloatingPointError(
                f"Nonfinite optimization objective for {kind}"
            )

        objective.backward()
        optimizer.step()

    return model


def model_predict(model, x):
    with torch.no_grad():
        s = model(torch.tensor(x, dtype=torch.float64))
        prob = torch.exp(log_prob_from_scores(s)).cpu().numpy()

    assert np.all(np.isfinite(prob))
    assert np.max(np.abs(prob.sum(axis=1) - 1)) < 1e-12

    return prob, s.detach().cpu().numpy()


def project_source_noninferiority(model, teacher, x, y, sources):
    with torch.no_grad():
        base = teacher(x).cpu().numpy()
        direction = model(x).cpu().numpy() - base

    labels = y.cpu().numpy()
    source_np = np.asarray(sources).astype(str)

    groups = [
        np.flatnonzero(source_np == s)
        for s in np.unique(source_np)
    ]

    def loss_for_logits(s):
        se, sk = s[:, 0], s[:, 1]

        loss_e = np.logaddexp(0.0, -se)
        loss_non_e = np.logaddexp(0.0, se)
        loss_k = np.logaddexp(0.0, -sk)
        loss_w = np.logaddexp(0.0, sk)

        return np.where(
            labels == 0,
            loss_e,
            loss_non_e + np.where(
                labels == 1, loss_k, loss_w
            )
        )

    anchor_losses = loss_for_logits(base)
    anchor_risks = [
        float(anchor_losses[g].mean()) for g in groups
    ]

    def admissible(scale):
        new_losses = loss_for_logits(
            base + scale * direction
        )

        return all(
            float(new_losses[g].mean()) <= old + 1e-10
            for g, old in zip(groups, anchor_risks)
        )

    if admissible(1.0):
        chosen = 1.0
    else:
        lo, hi = 0.0, 1.0

        for _ in range(48):
            mid = (lo + hi) / 2.0

            if admissible(mid):
                lo = mid
            else:
                hi = mid

        chosen = lo

    model.guard_lambda = float(chosen)
    assert admissible(model.guard_lambda)

    return chosen


assert SOURCE.is_dir(), f"Missing Phase 12 run: {SOURCE}"

outer_path = SOURCE / "LOCO_PREDICTIONS.csv"
assert outer_path.is_file()

for held in COHORTS:
    assert (
        SOURCE / "nested_oof" / f"{held}.npz"
    ).is_file(), held

torch.manual_seed(SEED)
np.random.seed(SEED)

df = pd.read_csv(outer_path)

required_cols = {
    "held_cohort", "gsm", "true_class",
    "model", *PROB_COLS
}
assert required_cols.issubset(df.columns), (
    f"Phase 12 columns: {df.columns.tolist()}"
)

df["gsm"] = df["gsm"].astype(str)
df["true_class"] = pd.to_numeric(
    df["true_class"], errors="raise"
).astype(int)

for c in PROB_COLS:
    df[c] = pd.to_numeric(df[c], errors="raise")

assert not df.duplicated([
    "held_cohort", "gsm", "model"
]).any()

reference = df.loc[df.model == "raw128"].copy()
assert len(reference) == 843

ref_lookup = {
    (str(row.held_cohort), str(row.gsm)): int(row.true_class)
    for row in reference.itertuples(index=False)
}
assert len(ref_lookup) == 843

run_id = (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ_") +
    uuid.uuid4().hex[:8]
)

DEST.mkdir(parents=True, exist_ok=True)
RUN = DEST / run_id
RUN.mkdir(exist_ok=False)

(RUN / "checkpoints").mkdir()
(RUN / "inner_oof").mkdir()

write_json(RUN / "PROTOCOL.json", {
    "phase": 14,
    "design": "Source-relative quantile evidence with frozen-anchor bounded residual heads",
    "source": str(SOURCE),
    "cohorts": COHORTS,
    "seed": SEED,
    "score_channels": CHANNELS,
    "representation": "Logit of within-cohort unlabeled empirical CDF divided by 2.5",
    "training": "Inner source-held-out evidence scores from Phase 12 only",
    "deployment": "Unlabeled whole target cohort required for within-cohort quantile transformation",
    "mode": "Transductive batch inference, not single-patient inductive inference",
    "models": {
        "q_anchor": "Monotonic affine calibration of quantile EGFR128 and KRAS128",
        "q_static": "Frozen q_anchor plus bounded linear difference correction",
        "q_residual": "Frozen q_anchor plus bounded three-basis conditional correction",
        "q_residual_guard": "Same conditional correction with source-wise anchor-excess-NLL penalty and projection"
    },
    "correction_cap_per_head": CORRECTION_BOUND,
    "noninferiority_penalty": "Soft training penalty followed by empirical source-NLL projection",
    "projection": "Maximum multiplier in [0,1] respecting each inner-OOF source NLL; 48 bisection iterations",
    "projection_guarantee_scope": "Empirical meta-training OOF source NLL only; no guarantee for unseen cohorts",
    "source_objective": "Equal-source mean multiclass NLL, without class-balanced reweighting",
    "epochs": EPOCHS,
    "learning_rate": LR,
    "coefficient_l2": COEFFICIENT_L2,
    "group_penalty": GROUP_PENALTY,
    "no_external_test_use": True,
    "classification": "Exploratory development on four historically reused cohorts",
    "known_risk": "Score quantiles depend on unlabeled cohort composition and may be sensitive to label shift",
    "versions": {
        "python": sys.version,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": scipy.__version__,
        "scikit_learn": sklearn.__version__,
        "torch": torch.__version__
    }
})

all_metrics = []
all_predictions = []
training_risks = []
model_parameters = []
quantile_diagnostics = []

source_hashes = [{
    "path": str(outer_path),
    "sha256": checksum(outer_path)
}]

for held in COHORTS:
    train_sources = [s for s in COHORTS if s != held]
    nested_fp = SOURCE / "nested_oof" / f"{held}.npz"

    with np.load(nested_fp, allow_pickle=False) as z:
        keys = list(z.files)
        oof = get_array(z, ("oof_scores",))
        source = get_array(
            z,
            ("cohort", "source_cohort", "source", "train_cohort")
        )
        label = get_array(
            z,
            ("label", "y", "labels", "true_class", "target")
        )
        sample_id = get_array(
            z,
            ("gsm", "sample_gsm", "gsm_ids", "sample_ids")
        )

    if (
        oof is None or
        np.asarray(oof).ndim != 2 or
        oof.shape[1] != 4
    ):
        raise ValueError(
            f"Unexpected OOF score schema for {held}: {keys}"
        )

    oof = np.asarray(oof, dtype=np.float64)
    assert np.isfinite(oof).all()

    if source is not None:
        source = np.asarray(source).astype(str)

    if sample_id is not None:
        sample_id = np.asarray(sample_id).astype(str)

    if label is not None:
        label = np.asarray(label).astype(int)

    if source is None and sample_id is not None:
        unique_gsm = {}

        for (group, gsm), yy in ref_lookup.items():
            unique_gsm.setdefault(gsm, set()).add(group)

        if all(
            len(unique_gsm.get(g, ())) == 1
            for g in sample_id
        ):
            source = np.array([
                next(iter(unique_gsm[g]))
                for g in sample_id
            ])

    if (
        label is None and
        source is not None and
        sample_id is not None
    ):
        missing = [
            (s, g)
            for s, g in zip(source, sample_id)
            if (s, g) not in ref_lookup
        ]

        if missing:
            raise ValueError(
                f"Cannot recover OOF genotype IDs: {missing[:4]}"
            )

        label = np.array([
            ref_lookup[(s, g)]
            for s, g in zip(source, sample_id)
        ])

    if source is None or label is None:
        raise ValueError(
            f"Nested OOF labels/source missing for {held}: "
            f"keys={keys}. Refusing positional label guesses."
        )

    assert len(source) == len(label) == len(oof)
    assert set(source) == set(train_sources), (
        held, set(source)
    )
    assert np.isin(label, [0, 1, 2]).all()

    expected_n = sum(
        sum(COUNTS[s]) for s in train_sources
    )
    assert len(oof) == expected_n

    for s in train_sources:
        actual = tuple(
            int(((source == s) & (label == j)).sum())
            for j in range(3)
        )

        assert actual == COUNTS[s], (
            f"Nested label-count mismatch: {held}/{s}: {actual}"
        )

    if sample_id is not None:
        assert len(sample_id) == len(oof)
        assert len(
            set(zip(source.tolist(), sample_id.tolist()))
        ) == len(sample_id)

        for s, gsm, yy in zip(source, sample_id, label):
            if (s, gsm) in ref_lookup:
                assert ref_lookup[(s, gsm)] == yy, (
                    f"OOF label discordance: {held}/{s}/{gsm}"
                )

    x_train = quantile_evidence(oof, source)

    np.savez_compressed(
        RUN / "inner_oof" / f"{held}.npz",
        rank_evidence=x_train,
        oof_scores=oof,
        label=label,
        source=source
    )

    source_hashes.append({
        "path": str(nested_fp),
        "sha256": checksum(nested_fp)
    })

    xt = torch.tensor(x_train, dtype=torch.float64)
    yt = torch.tensor(label, dtype=torch.long)

    masks = [
        torch.tensor(source == s)
        for s in train_sources
    ]

    anchor = fit_quantile_model(
        "q_anchor", xt, yt, masks
    )

    fitted = {"q_anchor": anchor}

    for kind in NEW_MODELS[1:]:
        fitted[kind] = fit_quantile_model(
            kind, xt, yt, masks, teacher=anchor
        )

    project_source_noninferiority(
        fitted["q_residual_guard"],
        anchor,
        xt,
        yt,
        source
    )

    reference_test = None
    base_probs = {}

    for name in BASE_MODELS:
        frame = (
            df.loc[
                (df.held_cohort == held) &
                (df.model == name)
            ]
            .sort_values("gsm")
            .reset_index(drop=True)
        )

        assert len(frame) == sum(COUNTS[held])

        counts = tuple(
            int((frame.true_class == j).sum())
            for j in range(3)
        )
        assert counts == COUNTS[held]

        if reference_test is None:
            reference_test = frame
        else:
            assert frame.gsm.equals(reference_test.gsm)
            assert frame.true_class.equals(
                reference_test.true_class
            )

        base_probs[name] = frame[
            list(PROB_COLS)
        ].to_numpy(dtype=np.float64)

    raw128 = effective_scores(base_probs["raw128"])
    raw512 = effective_scores(base_probs["raw512"])

    outer_evidence = np.column_stack((
        raw128[:, 0],
        raw512[:, 0],
        raw128[:, 1],
        raw512[:, 1]
    ))

    x_test = quantile_evidence(
        outer_evidence,
        np.repeat(held, len(outer_evidence))
    )

    assert np.isfinite(x_test).all()

    y_test = reference_test.true_class.to_numpy(dtype=int)
    id_test = reference_test.gsm.to_numpy(dtype=str)

    print(
        f"COHORT: {held} "
        f"N={len(y_test)} "
        f"OOF_TRAIN={len(yt)}"
    )

    with torch.no_grad():
        anchor_lp = log_prob_from_scores(anchor(xt))
        anchor_nll = -anchor_lp[
            torch.arange(len(yt)), yt
        ]
        _, anchor_outer_logits = model_predict(
            anchor, x_test
        )

    for kind, model in fitted.items():
        pp, logits = model_predict(model, x_test)
        base_probs[kind] = pp

        checkpoint = (
            RUN / "checkpoints" / f"{held}_{kind}.npz"
        )

        np.savez_compressed(
            checkpoint,
            bias=model.bias.detach().cpu().numpy(),
            raw_scale=model.raw_scale.detach().cpu().numpy(),
            weights=model.weights.detach().cpu().numpy(),
            correction_bound=np.array(
                CORRECTION_BOUND, dtype=np.float64
            ),
            guard_lambda=np.array(
                model.guard_lambda, dtype=np.float64
            )
        )

        with torch.no_grad():
            fitted_lp = log_prob_from_scores(model(xt))
            fitted_nll = -fitted_lp[
                torch.arange(len(yt)), yt
            ]

        for s, mask in zip(train_sources, masks):
            train_source_nll = float(
                fitted_nll[mask].mean()
            )
            train_anchor_nll = float(
                anchor_nll[mask].mean()
            )

            training_risks.append({
                "held_cohort": held,
                "model": kind,
                "training_source": s,
                "nll": train_source_nll,
                "anchor_nll": train_anchor_nll,
                "excess_over_anchor": (
                    train_source_nll - train_anchor_nll
                )
            })

        model_parameters.append({
            "held_cohort": held,
            "model": kind,
            "bias": model.bias.detach().numpy().tolist(),
            "scale": F.softplus(
                model.raw_scale
            ).detach().numpy().tolist(),
            "weights": model.weights.detach().numpy().tolist(),
            "guard_lambda": model.guard_lambda,
            "max_abs_correction_on_outer": (
                float(np.max(np.abs(
                    logits - anchor_outer_logits
                )))
                if kind != "q_anchor" else 0.0
            )
        })

    for j, channel in enumerate(CHANNELS):
        a = oof[:, j]
        b = outer_evidence[:, j]

        quantile_diagnostics.append({
            "held_cohort": held,
            "channel": channel,
            "oof_logit_median": float(np.median(a)),
            "outer_logit_median": float(np.median(b)),
            "oof_logit_iqr": float(
                np.subtract(*np.percentile(a, [75, 25]))
            ),
            "outer_logit_iqr": float(
                np.subtract(*np.percentile(b, [75, 25]))
            ),
            "outer_score_ties_fraction": float(
                1 - len(np.unique(b)) / len(b)
            )
        })

    for name in BASE_MODELS + NEW_MODELS:
        pp = base_probs[name]
        result = metrics(y_test, pp)

        all_metrics.append({
            "held_cohort": held,
            "model": name,
            **result
        })

        for i in range(len(y_test)):
            all_predictions.append({
                "held_cohort": held,
                "gsm": id_test[i],
                "true_class": int(y_test[i]),
                "model": name,
                "p_egfr": float(pp[i, 0]),
                "p_kras": float(pp[i, 1]),
                "p_dwt": float(pp[i, 2]),
                "predicted_class": int(pp[i].argmax())
            })

metric_df = pd.DataFrame(all_metrics)
pred_df = pd.DataFrame(all_predictions)
risk_df = pd.DataFrame(training_risks)

metric_df.to_csv(
    RUN / "LOCO_METRICS.csv", index=False
)
pred_df.to_csv(
    RUN / "LOCO_PREDICTIONS.csv", index=False
)
risk_df.to_csv(
    RUN / "TRAINING_SOURCE_RISK.csv", index=False
)

pd.DataFrame(quantile_diagnostics).to_csv(
    RUN / "QUANTILE_SCORE_DIAGNOSTICS.csv",
    index=False
)

write_json(
    RUN / "MODEL_PARAMETERS.json",
    model_parameters
)

summary = (
    metric_df.groupby("model", sort=False)[
        [
            "nll",
            "macro_f1",
            "egfr_auc",
            "kras_auc",
            "kras_cond_nll",
            "brier"
        ]
    ]
    .mean()
    .reset_index()
)

summary.to_csv(
    RUN / "EQUAL_COHORT_SUMMARY.csv",
    index=False
)

write_json(
    RUN / "EQUAL_COHORT_SUMMARY.json",
    summary.to_dict(orient="records")
)

lookup = summary.set_index("model")

proposed = lookup.loc["q_residual_guard"]
static = lookup.loc["q_static"]
anchor = lookup.loc["q_anchor"]
raw = lookup.loc["raw128"]

flags = {
    "f1_vs_quantile_anchor_ge_0.01": (
        float(proposed.macro_f1 - anchor.macro_f1) >= 0.01
    ),
    "nll_vs_quantile_anchor_ge_0.01": (
        float(anchor.nll - proposed.nll) >= 0.01
    ),
    "f1_vs_quantile_static_ge_0.01": (
        float(proposed.macro_f1 - static.macro_f1) >= 0.01
    ),
    "nll_vs_quantile_static_ge_0.01": (
        float(static.nll - proposed.nll) >= 0.01
    ),
    "f1_vs_raw128_ge_0.01": (
        float(proposed.macro_f1 - raw.macro_f1) >= 0.01
    ),
    "nll_vs_raw128_ge_0.01": (
        float(raw.nll - proposed.nll) >= 0.01
    )
}

write_json(RUN / "SCREENING.json", {
    "criteria": flags,
    "all_pass": all(flags.values()),
    "interpretation": "Exploratory screening only; development cohort reuse prevents confirmatory claims",
    "primary_matched_regime_comparison": [
        "q_residual_guard",
        "q_static",
        "q_anchor"
    ],
    "secondary_inductive_reference": "raw128",
    "no_external_validation": True
})

write_json(RUN / "RUN_MANIFEST.json", {
    "source_sha256": source_hashes,
    "run_created_utc": (
        datetime.now(timezone.utc).isoformat()
    ),
    "artifacts": [
        {
            "path": str(p.relative_to(RUN)),
            "sha256": checksum(p),
            "bytes": p.stat().st_size
        }
        for p in sorted(RUN.rglob("*"))
        if p.is_file() and p.name != "RUN_MANIFEST.json"
    ]
})

print("PHASE14: COMPLETE")
print("MODEL_RESULTS:")
print(
    summary[
        ["model", "nll", "macro_f1", "egfr_auc", "kras_auc"]
    ].round(4).to_string(index=False)
)

print(
    "GUARD_MINUS_STATIC_F1:",
    round(float(
        proposed.macro_f1 - static.macro_f1
    ), 4)
)
print(
    "STATIC_MINUS_GUARD_NLL:",
    round(float(
        static.nll - proposed.nll
    ), 4)
)
print(
    "GUARD_MINUS_RAW128_F1:",
    round(float(
        proposed.macro_f1 - raw.macro_f1
    ), 4)
)
print(
    "RAW128_MINUS_GUARD_NLL:",
    round(float(
        raw.nll - proposed.nll
    ), 4)
)

max_excess = float(
    risk_df.loc[
        risk_df.model == "q_residual_guard",
        "excess_over_anchor"
    ].max()
)

assert max_excess < 1e-7, (
    f"Source risk projection failed: {max_excess}"
)

print("GUARD_LAMBDAS:", {
    x["held_cohort"]: round(x["guard_lambda"], 5)
    for x in model_parameters
    if x["model"] == "q_residual_guard"
})

print(
    "MAX_TRAINING_GUARD_EXCESS:",
    round(max_excess, 10)
)
print("SCREENING:", flags)
print("RUN:", RUN)

COHORT: GSE13213 N=117 OOF_TRAIN=726
COHORT: GSE31210 N=226 OOF_TRAIN=617
COHORT: GSE32863 N=58 OOF_TRAIN=785
COHORT: GSE72094 N=442 OOF_TRAIN=401
PHASE14: COMPLETE
MODEL_RESULTS:
           model    nll  macro_f1  egfr_auc  kras_auc
          raw128 1.1371    0.5117    0.8662    0.8553
          raw512 1.9665    0.3179    0.8679    0.8638
cal_disagree_ood 1.3416    0.3824    0.8704    0.8632
        q_anchor 0.9210    0.5250    0.8662    0.8553
        q_static 0.9131    0.5257    0.8699    0.8612
      q_residual 0.9093    0.5452    0.8696    0.8626
q_residual_guard 0.9093    0.5452    0.8696    0.8626
GUARD_MINUS_STATIC_F1: 0.0196
STATIC_MINUS_GUARD_NLL: 0.0039
GUARD_MINUS_RAW128_F1: 0.0336
RAW128_MINUS_GUARD_NLL: 0.2278
GUARD_LAMBDAS: {'GSE13213': 1.0, 'GSE31210': 1.0, 'GSE32863': 1.0, 'GSE72094': 1.0}
MAX_TRAINING_GUARD_EXCESS: -0.0026062888
SCREENING: {'f1_vs_quantile_anchor_ge_0.01': True, 'nll_vs_quantile_anchor_ge_0.01': True, 'f1_vs_quantile_static_ge_0.01': True, 'nll_vs_qua

In [19]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import sys
import uuid

import numpy as np
import pandas as pd
import scipy
from scipy.special import expit
from scipy.stats import rankdata
from sklearn.metrics import f1_score, roc_auc_score
import sklearn
import torch
import torch.nn.functional as F

ROOT = Path(r"D:\LUAD_DATAFIRST_METHOD_2026")
SOURCE = ROOT / "14_quantile_residual_transfer" / "20261023T160742Z_f9435239"
DEST = ROOT / "15_residual_mechanism_validation"

SEED = 20261023
COHORTS = ("GSE13213", "GSE31210", "GSE32863", "GSE72094")
EXPECTED = {
    "GSE13213": (45, 15, 57),
    "GSE31210": (127, 20, 79),
    "GSE32863": (17, 22, 19),
    "GSE72094": (47, 154, 241),
}
SAVED = ("raw128", "q_anchor", "q_static", "q_residual", "q_residual_guard")
ABLATIONS = ("q_delta_nonlinear", "q_anchor_nonlinear")
ALL = SAVED + ABLATIONS
PCOLS = ("p_egfr", "p_kras", "p_dwt")
CAP = 1.5
EPOCHS = 400
BOOTSTRAPS = 1000
RANDOM_SPLITS = 10


def digest(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def native(obj):
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, dict):
        return {str(k): native(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [native(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return native(obj.tolist())
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, (float, np.floating)):
        return float(obj) if np.isfinite(obj) else None
    if isinstance(obj, np.bool_):
        return bool(obj)
    return obj


def save_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(native(obj), f, indent=2, ensure_ascii=False, allow_nan=False)


def efflogits(p):
    eps = 1e-12
    pe = np.clip(p[:, 0], eps, 1 - eps)
    pk = np.clip(p[:, 1], eps, 1)
    pw = np.clip(p[:, 2], eps, 1)
    return np.column_stack((
        np.log(pe / (1 - pe)),
        np.log(pk / pw)
    ))


def rank_evidence(x, batch):
    x = np.asarray(x, dtype=np.float64)
    batch = np.asarray(batch)
    assert x.ndim == 2 and x.shape[1] == 4 and len(batch) == len(x)
    assert np.isfinite(x).all()

    z = np.empty_like(x)

    for group in np.unique(batch):
        pos = np.flatnonzero(batch == group)
        if len(pos) < 25:
            raise ValueError(f"Insufficient batch size: {group}, n={len(pos)}")

        for j in range(4):
            u = (rankdata(x[pos, j], method="average") - 0.5) / len(pos)
            u = np.clip(u, 1e-5, 1 - 1e-5)
            z[pos, j] = (np.log(u) - np.log1p(-u)) / 2.5

    return z


def basis(x, name):
    a = x[:, (0, 2)]
    d = x[:, (1, 3)] - a

    if name == "q_static":
        return d[:, :, None]

    if name in ("q_residual", "q_residual_guard"):
        return np.stack((
            d,
            d * np.tanh(1.5 * a),
            np.tanh(2 * d) * (1 - np.tanh(a) ** 2)
        ), axis=2)

    if name == "q_delta_nonlinear":
        return np.stack((
            d,
            np.tanh(2 * d),
            d * np.tanh(d)
        ), axis=2)

    if name == "q_anchor_nonlinear":
        return np.stack((
            a,
            np.tanh(2 * a),
            a * np.tanh(1.5 * a)
        ), axis=2)

    raise ValueError(name)


def probabilities(scores):
    e = expit(scores[:, 0])
    k = expit(scores[:, 1])

    p = np.column_stack((
        e,
        (1 - e) * k,
        (1 - e) * (1 - k)
    ))

    assert np.isfinite(p).all()
    assert np.max(np.abs(p.sum(axis=1) - 1)) < 1e-12
    return p


def restore_params(path):
    with np.load(path, allow_pickle=False) as data:
        obj = {
            "bias": np.asarray(data["bias"], dtype=np.float64),
            "raw_scale": np.asarray(data["raw_scale"], dtype=np.float64),
            "weights": np.asarray(data["weights"], dtype=np.float64),
            "guard_lambda": float(np.asarray(data["guard_lambda"]).item()),
        }

    assert obj["bias"].shape == obj["raw_scale"].shape == (2,)
    assert obj["weights"].shape[0] == 2
    return obj


def forward(x, model, name):
    a = x[:, (0, 2)]
    s = model["bias"] + np.logaddexp(0.0, model["raw_scale"]) * a

    if name != "q_anchor":
        features = basis(x, name)
        assert features.shape[-1] == model["weights"].shape[1]
        v = np.einsum("nhk,hk->nh", features, model["weights"])
        s = s + model["guard_lambda"] * CAP * np.tanh(v / CAP)

    return probabilities(s), s


def summary_metrics(y, p):
    y = np.asarray(y, dtype=int)
    h = p.argmax(axis=1)
    valid = y != 0

    qk = p[valid, 1] / np.clip(
        p[valid, 1] + p[valid, 2], 1e-12, None
    )

    return {
        "nll": float(
            -np.log(np.clip(p[np.arange(len(y)), y], 1e-15, 1)).mean()
        ),
        "macro_f1": float(
            f1_score(
                y, h, labels=[0, 1, 2],
                average="macro", zero_division=0
            )
        ),
        "egfr_auc": float(
            roc_auc_score((y == 0).astype(int), p[:, 0])
        ),
        "kras_auc": float(
            roc_auc_score((y[valid] == 1).astype(int), qk)
        ),
        "brier": float(
            np.mean(np.sum((p - np.eye(3)[y]) ** 2, axis=1))
        )
    }


def ablation_fit(x, labels, groups, anchor, name):
    z = torch.tensor(basis(x, name), dtype=torch.float64)
    a = x[:, (0, 2)]

    offset = (
        anchor["bias"] +
        np.logaddexp(0.0, anchor["raw_scale"]) * a
    )

    base = torch.tensor(offset, dtype=torch.float64)
    y = torch.tensor(labels, dtype=torch.long)
    ix = torch.arange(len(y))
    masks = [
        torch.tensor(groups == g)
        for g in np.unique(groups)
    ]

    w = torch.nn.Parameter(
        torch.zeros((2, 3), dtype=torch.float64)
    )
    optimizer = torch.optim.Adam([w], lr=0.045)

    for _ in range(EPOCHS):
        optimizer.zero_grad(set_to_none=True)

        correction = CAP * torch.tanh(
            (z * w).sum(dim=2) / CAP
        )
        score = base + correction
        se, sk = score[:, 0], score[:, 1]

        lp = torch.stack((
            F.logsigmoid(se),
            F.logsigmoid(-se) + F.logsigmoid(sk),
            F.logsigmoid(-se) + F.logsigmoid(-sk)
        ), dim=1)

        sample_loss = -lp[ix, y]
        objective = torch.stack([
            sample_loss[m].mean()
            for m in masks
        ]).mean()

        objective = objective + 0.02 * w.square().mean()

        if not torch.isfinite(objective):
            raise FloatingPointError(name)

        objective.backward()
        optimizer.step()

    return {
        "bias": anchor["bias"].copy(),
        "raw_scale": anchor["raw_scale"].copy(),
        "weights": w.detach().cpu().numpy().copy(),
        "guard_lambda": 1.0
    }


assert SOURCE.is_dir(), SOURCE
pred_path = SOURCE / "LOCO_PREDICTIONS.csv"
assert pred_path.is_file()

for cohort in COHORTS:
    assert (SOURCE / "inner_oof" / f"{cohort}.npz").is_file()

    for model in SAVED[1:]:
        assert (
            SOURCE / "checkpoints" / f"{cohort}_{model}.npz"
        ).is_file()

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(torch.get_num_threads(), 4))

raw_df = pd.read_csv(pred_path)

assert set((
    "held_cohort", "gsm", "true_class", "model", *PCOLS
)).issubset(raw_df.columns)

raw_df["gsm"] = raw_df["gsm"].astype(str)
raw_df["true_class"] = pd.to_numeric(
    raw_df["true_class"], errors="raise"
).astype(int)

for column in PCOLS:
    raw_df[column] = pd.to_numeric(
        raw_df[column], errors="raise"
    )

assert not raw_df.duplicated([
    "held_cohort", "gsm", "model"
]).any()

aligned = {}

for cohort in COHORTS:
    aligned[cohort] = {}

    for model in SAVED:
        frame = raw_df.loc[
            (raw_df.held_cohort == cohort) &
            (raw_df.model == model)
        ]

        frame = frame.sort_values("gsm").reset_index(drop=True)

        assert len(frame) == sum(EXPECTED[cohort])
        assert tuple(
            int((frame.true_class == k).sum())
            for k in range(3)
        ) == EXPECTED[cohort]

        aligned[cohort][model] = frame

    for model in SAVED[1:]:
        assert aligned[cohort][model].gsm.equals(
            aligned[cohort]["raw128"].gsm
        )
        assert aligned[cohort][model].true_class.equals(
            aligned[cohort]["raw128"].true_class
        )

DEST.mkdir(parents=True, exist_ok=True)

RUN = DEST / (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ_") +
    uuid.uuid4().hex[:8]
)
RUN.mkdir(exist_ok=False)
(RUN / "ablation_checkpoints").mkdir()

save_json(RUN / "PROTOCOL.json", {
    "phase": 15,
    "source_run": str(SOURCE),
    "objective": "Residual-mechanism ablation and unlabeled batch-composition sensitivity",
    "training": "Phase14 saved source-held-out OOF quantile features for matched 3-basis ablations only",
    "existing_models": SAVED,
    "new_ablation_models": ABLATIONS,
    "ablations_parameters_per_head": 3,
    "ablation_training": "Frozen Phase14 quantile anchor; Adam lr=.045, 400 epochs, 0.02 mean L2; equal-source NLL",
    "batch_stress": "10 random 2-way partitions plus score-sorted 2-way partitions per held-out cohort",
    "random_partition_seed": SEED,
    "bootstrap": "1000 paired class-stratified within-cohort resamples of saved full-batch predictions; descriptive only",
    "method_type": "Transductive cohort-level score-normalization",
    "external_cohort_used": False,
    "development_only": True,
    "versions": {
        "python": sys.version,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": scipy.__version__,
        "sklearn": sklearn.__version__,
        "torch": torch.__version__
    }
})

metrics_rows = []
gain_rows = []
class_rows = []
stress_rows = []
param_rows = []
reproduction = []
cohort_predictions = {}
input_paths = [pred_path]

for cohort in COHORTS:
    y = aligned[cohort]["raw128"].true_class.to_numpy(dtype=int)
    p128 = aligned[cohort]["raw128"][
        list(PCOLS)
    ].to_numpy(dtype=float)

    p512_frame = raw_df.loc[
        (raw_df.held_cohort == cohort) &
        (raw_df.model == "raw512")
    ]
    p512_frame = p512_frame.sort_values("gsm").reset_index(drop=True)

    assert p512_frame.gsm.equals(
        aligned[cohort]["raw128"].gsm
    )
    assert p512_frame.true_class.equals(
        aligned[cohort]["raw128"].true_class
    )

    p512 = p512_frame[list(PCOLS)].to_numpy(dtype=float)

    s128 = efflogits(p128)
    s512 = efflogits(p512)

    raw_scores = np.column_stack((
        s128[:, 0],
        s512[:, 0],
        s128[:, 1],
        s512[:, 1]
    ))

    whole = rank_evidence(
        raw_scores, np.zeros(len(y), dtype=int)
    )

    oof_path = SOURCE / "inner_oof" / f"{cohort}.npz"
    input_paths.append(oof_path)

    with np.load(oof_path, allow_pickle=False) as z:
        train_x = np.asarray(
            z["rank_evidence"], dtype=np.float64
        )
        train_y = np.asarray(z["label"], dtype=int)
        train_source = np.asarray(z["source"]).astype(str)

    expected_train_n = sum(
        sum(EXPECTED[s])
        for s in COHORTS if s != cohort
    )
    assert train_x.shape == (expected_train_n, 4)
    assert np.isfinite(train_x).all()
    assert set(train_source) == set(COHORTS) - {cohort}

    for s in np.unique(train_source):
        assert tuple(
            int(((train_source == s) & (train_y == k)).sum())
            for k in range(3)
        ) == EXPECTED[s]

    state = {}

    for model in SAVED[1:]:
        path = (
            SOURCE / "checkpoints" /
            f"{cohort}_{model}.npz"
        )
        input_paths.append(path)
        state[model] = restore_params(path)

    for name in ABLATIONS:
        state[name] = ablation_fit(
            train_x, train_y, train_source,
            state["q_anchor"], name
        )

        path = (
            RUN / "ablation_checkpoints" /
            f"{cohort}_{name}.npz"
        )
        np.savez_compressed(path, **state[name])

    cohort_predictions[cohort] = {}

    for model in ALL:
        if model == "raw128":
            prob = p128
        else:
            prob, _ = forward(whole, state[model], model)

            if model in SAVED:
                expected_prob = aligned[cohort][model][
                    list(PCOLS)
                ].to_numpy(dtype=float)

                max_err = float(
                    np.abs(prob - expected_prob).max()
                )

                reproduction.append({
                    "held_cohort": cohort,
                    "model": model,
                    "max_error": max_err
                })

                assert max_err < 1e-8, (
                    f"Phase14 reproduction mismatch: "
                    f"{cohort}/{model}: {max_err}"
                )

        cohort_predictions[cohort][model] = prob

        metrics_rows.append({
            "held_cohort": cohort,
            "model": model,
            **summary_metrics(y, prob)
        })

    static = cohort_predictions[cohort]["q_static"]
    nonlinear = cohort_predictions[cohort]["q_residual"]

    idx = np.arange(len(y))
    loss_s = -np.log(
        np.clip(static[idx, y], 1e-15, 1)
    )
    loss_n = -np.log(
        np.clip(nonlinear[idx, y], 1e-15, 1)
    )

    pred_s = static.argmax(axis=1)
    pred_n = nonlinear.argmax(axis=1)

    gain_rows.append({
        "held_cohort": cohort,
        "n": len(y),
        "mean_nll_gain_nonlinear_vs_static": float(
            (loss_s - loss_n).mean()
        ),
        "mean_abs_p_change": float(
            np.abs(static - nonlinear).mean()
        ),
        "changed_hard_decisions": int(
            (pred_s != pred_n).sum()
        ),
        "nonlinear_rescues_static": int(
            ((pred_s != y) & (pred_n == y)).sum()
        ),
        "nonlinear_damages_static": int(
            ((pred_s == y) & (pred_n != y)).sum()
        )
    })

    for k, name in enumerate(("EGFR", "KRAS", "DoubleWT")):
        mask = y == k

        class_rows.append({
            "held_cohort": cohort,
            "class": name,
            "n": int(mask.sum()),
            "static_recall": float(
                np.mean(pred_s[mask] == k)
            ),
            "nonlinear_recall": float(
                np.mean(pred_n[mask] == k)
            ),
            "mean_nll_gain_nonlinear_vs_static": float(
                np.mean((loss_s - loss_n)[mask])
            )
        })

    for name in ABLATIONS:
        param_rows.append({
            "held_cohort": cohort,
            "model": name,
            "weights": state[name]["weights"].tolist()
        })

    configurations = []

    for repeat in range(RANDOM_SPLITS):
        rng = np.random.default_rng(
            SEED + 101 * COHORTS.index(cohort) + repeat
        )

        order = rng.permutation(len(y))
        partitions = np.zeros(len(y), dtype=int)
        partitions[order[len(y) // 2:]] = 1

        configurations.append((
            "random_half", repeat, partitions
        ))

    for j, label in (
        (0, "egfr_sorted_half"),
        (2, "kras_sorted_half")
    ):
        order = np.argsort(
            raw_scores[:, j], kind="stable"
        )

        partitions = np.zeros(len(y), dtype=int)
        partitions[order[len(y) // 2:]] = 1

        configurations.append((
            label, 0, partitions
        ))

    for scenario, repeat, partitions in configurations:
        z = rank_evidence(raw_scores, partitions)

        for name in (
            "q_anchor", "q_static", "q_residual",
            *ABLATIONS
        ):
            new_p, _ = forward(z, state[name], name)
            old_p = cohort_predictions[cohort][name]

            newer = summary_metrics(y, new_p)
            old = summary_metrics(y, old_p)

            stress_rows.append({
                "held_cohort": cohort,
                "scenario": scenario,
                "repeat": repeat,
                "model": name,
                "min_batch_n": int(
                    np.bincount(partitions).min()
                ),
                "full_nll": old["nll"],
                "stress_nll": newer["nll"],
                "delta_nll": (
                    newer["nll"] - old["nll"]
                ),
                "full_macro_f1": old["macro_f1"],
                "stress_macro_f1": newer["macro_f1"],
                "delta_macro_f1": (
                    newer["macro_f1"] - old["macro_f1"]
                ),
                "mean_abs_probability_change": float(
                    np.mean(np.abs(new_p - old_p))
                ),
                "changed_predictions": int(
                    np.sum(
                        new_p.argmax(axis=1) !=
                        old_p.argmax(axis=1)
                    )
                )
            })

metric_df = pd.DataFrame(metrics_rows)

metric_df.to_csv(
    RUN / "LOCO_METRICS.csv", index=False
)
pd.DataFrame(gain_rows).to_csv(
    RUN / "PAIRED_RESIDUAL_GAINS.csv", index=False
)
pd.DataFrame(class_rows).to_csv(
    RUN / "CLASS_RECALL_AND_NLL.csv", index=False
)
pd.DataFrame(stress_rows).to_csv(
    RUN / "BATCH_STRESS.csv", index=False
)
pd.DataFrame(reproduction).to_csv(
    RUN / "REPRODUCTION.csv", index=False
)

save_json(
    RUN / "ABLATION_PARAMETERS.json",
    param_rows
)

bootstrap_rng = np.random.default_rng(SEED + 404)
bootstrap_rows = []

for iteration in range(BOOTSTRAPS):
    nll_diffs = []
    f1_diffs = []
    nll_raw_diffs = []
    f1_raw_diffs = []

    for cohort in COHORTS:
        y = aligned[cohort]["raw128"].true_class.to_numpy(dtype=int)

        chosen = np.concatenate([
            bootstrap_rng.choice(
                np.flatnonzero(y == k),
                size=int(np.sum(y == k)),
                replace=True
            )
            for k in (0, 1, 2)
        ])

        yy = y[chosen]
        new = cohort_predictions[cohort]["q_residual"][chosen]
        base = cohort_predictions[cohort]["q_static"][chosen]
        raw = cohort_predictions[cohort]["raw128"][chosen]

        ids = np.arange(len(yy))

        nll_diffs.append(float(np.mean(
            np.log(np.clip(new[ids, yy], 1e-15, 1)) -
            np.log(np.clip(base[ids, yy], 1e-15, 1))
        )))

        nll_raw_diffs.append(float(np.mean(
            np.log(np.clip(new[ids, yy], 1e-15, 1)) -
            np.log(np.clip(raw[ids, yy], 1e-15, 1))
        )))

        f1_diffs.append(float(
            f1_score(
                yy, new.argmax(axis=1),
                average="macro", labels=[0, 1, 2],
                zero_division=0
            ) -
            f1_score(
                yy, base.argmax(axis=1),
                average="macro", labels=[0, 1, 2],
                zero_division=0
            )
        ))

        f1_raw_diffs.append(float(
            f1_score(
                yy, new.argmax(axis=1),
                average="macro", labels=[0, 1, 2],
                zero_division=0
            ) -
            f1_score(
                yy, raw.argmax(axis=1),
                average="macro", labels=[0, 1, 2],
                zero_division=0
            )
        ))

    bootstrap_rows.append({
        "iteration": iteration,
        "gain_nll_vs_static": float(np.mean(nll_diffs)),
        "gain_f1_vs_static": float(np.mean(f1_diffs)),
        "gain_nll_vs_raw128": float(np.mean(nll_raw_diffs)),
        "gain_f1_vs_raw128": float(np.mean(f1_raw_diffs))
    })

boot_df = pd.DataFrame(bootstrap_rows)
boot_df.to_csv(
    RUN / "DEVELOPMENT_BOOTSTRAP.csv", index=False
)

bootstrap_ci = {
    col: {
        "p2_5": float(boot_df[col].quantile(0.025)),
        "median": float(boot_df[col].median()),
        "p97_5": float(boot_df[col].quantile(0.975)),
        "fraction_above_zero": float(
            np.mean(boot_df[col] > 0)
        )
    }
    for col in boot_df.columns if col != "iteration"
}

save_json(
    RUN / "DEVELOPMENT_BOOTSTRAP_CI.json",
    bootstrap_ci
)

summary = (
    metric_df.groupby("model", sort=False)[
        ["nll", "macro_f1", "egfr_auc", "kras_auc", "brier"]
    ]
    .mean()
    .reset_index()
)

summary.to_csv(
    RUN / "EQUAL_COHORT_SUMMARY.csv",
    index=False
)

stress_df = pd.DataFrame(stress_rows)

random_part = stress_df.loc[
    (stress_df.scenario == "random_half") &
    (stress_df.model == "q_residual")
]

stress_summary = random_part.groupby("held_cohort")[[
    "delta_nll",
    "delta_macro_f1",
    "mean_abs_probability_change",
    "changed_predictions"
]].agg(["mean", "min", "max"])

stress_summary.columns = [
    "_".join(c) for c in stress_summary.columns
]

stress_summary.reset_index().to_csv(
    RUN / "RANDOM_BATCH_STRESS_SUMMARY.csv",
    index=False
)

save_json(RUN / "RUN_MANIFEST.json", {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "source_run": str(SOURCE),
    "inputs": [
        {
            "path": str(p),
            "sha256": digest(p)
        }
        for p in sorted(set(input_paths))
    ],
    "outputs": [
        {
            "path": str(p.relative_to(RUN)),
            "sha256": digest(p),
            "size": p.stat().st_size
        }
        for p in sorted(RUN.rglob("*"))
        if p.is_file() and p.name != "RUN_MANIFEST.json"
    ]
})

print("PHASE15: COMPLETE")
print(
    "REPRODUCTION_MAX_ERROR:",
    f"{max(v['max_error'] for v in reproduction):.3e}"
)

print("MODEL_RESULTS:")
print(
    summary[
        ["model", "nll", "macro_f1", "egfr_auc", "kras_auc"]
    ].round(4).to_string(index=False)
)

print("RESIDUAL_VS_STATIC_PER_COHORT:")
print(
    pd.DataFrame(gain_rows)
    .round(4)
    .to_string(index=False)
)

print("RANDOM_TWO_BATCH_STRESS:")
print(
    stress_summary.reset_index()
    .round(4)
    .to_string(index=False)
)

print("DEVELOPMENT_PAIRED_BOOTSTRAP_CI:")
print(json.dumps({
    k: {
        a: round(b, 5)
        for a, b in val.items()
    }
    for k, val in bootstrap_ci.items()
}, indent=2))

print("RUN:", RUN)

PHASE15: COMPLETE
REPRODUCTION_MAX_ERROR: 2.220e-16
MODEL_RESULTS:
             model    nll  macro_f1  egfr_auc  kras_auc
            raw128 1.1371    0.5117    0.8662    0.8553
          q_anchor 0.9210    0.5250    0.8662    0.8553
          q_static 0.9131    0.5257    0.8699    0.8612
        q_residual 0.9093    0.5452    0.8696    0.8626
  q_residual_guard 0.9093    0.5452    0.8696    0.8626
 q_delta_nonlinear 0.9142    0.5398    0.8717    0.8599
q_anchor_nonlinear 0.9183    0.5309    0.8662    0.8553
RESIDUAL_VS_STATIC_PER_COHORT:
held_cohort   n  mean_nll_gain_nonlinear_vs_static  mean_abs_p_change  changed_hard_decisions  nonlinear_rescues_static  nonlinear_damages_static
   GSE13213 117                             0.0067             0.0172                       6                         3                         3
   GSE31210 226                             0.0061             0.0204                      11                        10                         1
   GSE32863  58 

In [20]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import sys
import uuid

import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score, f1_score
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F

ROOT = Path(r"D:\LUAD_DATAFIRST_METHOD_2026")
SOURCE = ROOT / "14_quantile_residual_transfer" / "20261023T160742Z_f9435239"
DEST = ROOT / "16_source_referenced_residual"

SEED = 20261023
COHORTS = ("GSE13213", "GSE31210", "GSE32863", "GSE72094")
COUNTS = {
    "GSE13213": (45, 15, 57),
    "GSE31210": (127, 20, 79),
    "GSE32863": (17, 22, 19),
    "GSE72094": (47, 154, 241)
}
PCOLS = ("p_egfr", "p_kras", "p_dwt")
MODELS = (
    "raw128", "q_anchor", "q_static", "q_residual",
    "ref0_residual", "ref50_residual", "ref50_consistency"
)
EPOCHS = 400
LR = 0.045
CAP = 1.5
L2 = 0.02
CONSISTENCY_WEIGHT = 0.5
AUGMENTED_NLL_WEIGHT = 0.25
RANDOM_STRESS_SPLITS = 10


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def native(x):
    if isinstance(x, Path):
        return str(x)
    if isinstance(x, dict):
        return {str(k): native(v) for k, v in x.items()}
    if isinstance(x, (tuple, list)):
        return [native(v) for v in x]
    if isinstance(x, np.ndarray):
        return native(x.tolist())
    if isinstance(x, np.integer):
        return int(x)
    if isinstance(x, (float, np.floating)):
        return float(x) if np.isfinite(x) else None
    if isinstance(x, np.bool_):
        return bool(x)
    return x


def save_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            native(data), f, ensure_ascii=False,
            indent=2, allow_nan=False
        )


def scores_from_proba(p):
    eps = 1e-12
    e = np.clip(p[:, 0], eps, 1 - eps)
    k = np.clip(p[:, 1], eps, 1)
    w = np.clip(p[:, 2], eps, 1)
    return np.column_stack((
        np.log(e / (1 - e)),
        np.log(k / w)
    ))


def four_scores(p128, p512):
    a = scores_from_proba(p128)
    b = scores_from_proba(p512)
    return np.column_stack((
        a[:, 0], b[:, 0], a[:, 1], b[:, 1]
    ))


def make_reference(raw_scores, source):
    refs = {}
    source = np.asarray(source).astype(str)
    for group in sorted(set(source)):
        values = np.asarray(
            raw_scores[source == group], dtype=np.float64
        )
        assert len(values) >= 25 and values.shape[1] == 4
        refs[group] = np.sort(values, axis=0)
    return refs


def source_ecdf(values, refs):
    values = np.asarray(values, dtype=np.float64)
    result = np.zeros_like(values)
    assert len(refs) > 0

    for source_scores in refs.values():
        for channel in range(4):
            vec = source_scores[:, channel]
            low = np.searchsorted(
                vec, values[:, channel], side="left"
            )
            high = np.searchsorted(
                vec, values[:, channel], side="right"
            )
            result[:, channel] += (
                (low + high) /
                (2.0 * len(vec) * len(refs))
            )
    return result


def own_ecdf(values, batch):
    values = np.asarray(values, dtype=np.float64)
    batch = np.asarray(batch)
    result = np.empty_like(values)

    for group in np.unique(batch):
        ix = np.flatnonzero(batch == group)
        for j in range(4):
            result[ix, j] = (
                rankdata(values[ix, j], method="average") - 0.5
            ) / len(ix)
    return result


def transformed(raw, batch, ref, rho):
    assert 0 <= rho <= 1

    source_u = source_ecdf(raw, ref)

    if rho == 0:
        u = source_u
    else:
        target_u = own_ecdf(raw, batch)
        u = rho * target_u + (1.0 - rho) * source_u

    u = np.clip(u, 1e-5, 1 - 1e-5)
    z = (np.log(u) - np.log1p(-u)) / 2.5
    assert np.isfinite(z).all()
    return z


def training_transform(raw, source, refs, rho, partition=None):
    out = np.empty_like(raw, dtype=np.float64)

    for group in sorted(refs):
        ix = np.flatnonzero(source == group)
        other = {
            k: v for k, v in refs.items() if k != group
        }
        if not other:
            raise ValueError("Reference source unavailable")

        batch = (
            np.zeros(len(ix), dtype=int)
            if partition is None
            else partition[ix]
        )
        out[ix] = transformed(
            raw[ix], batch, other, rho
        )

    return out


def tensor_logp(s):
    e, k = s[:, 0], s[:, 1]
    return torch.stack((
        F.logsigmoid(e),
        F.logsigmoid(-e) + F.logsigmoid(k),
        F.logsigmoid(-e) + F.logsigmoid(-k)
    ), dim=1)


class ResidualHead(nn.Module):
    def __init__(self):
        super().__init__()

        init = float(np.log(np.expm1(2.0)))
        self.bias = nn.Parameter(
            torch.zeros(2, dtype=torch.float64)
        )
        self.raw_scale = nn.Parameter(
            torch.full((2,), init, dtype=torch.float64)
        )
        self.weights = nn.Parameter(
            torch.zeros((2, 3), dtype=torch.float64)
        )

    def forward(self, z, residual=True):
        a = z[:, (0, 2)]
        score = (
            self.bias +
            F.softplus(self.raw_scale) * a
        )

        if residual:
            d = z[:, (1, 3)] - a
            phi = torch.stack((
                d,
                d * torch.tanh(1.5 * a),
                torch.tanh(2.0 * d) *
                (1.0 - torch.tanh(a).square())
            ), dim=-1)

            correction = (
                phi * self.weights
            ).sum(dim=-1)

            score = score + CAP * torch.tanh(
                correction / CAP
            )

        return score


def equal_source_mean(losses, masks):
    return torch.stack([
        losses[m].mean() for m in masks
    ]).mean()


def train_model(
    x, labels, source,
    consistent=False, augments=None
):
    xt = torch.as_tensor(x, dtype=torch.float64)
    yt = torch.as_tensor(labels, dtype=torch.long)
    masks = [
        torch.as_tensor(source == g)
        for g in sorted(set(source))
    ]
    idx = torch.arange(len(yt))
    model = ResidualHead()

    optimizer = torch.optim.Adam(
        (model.bias, model.raw_scale), lr=LR
    )

    for _ in range(EPOCHS):
        optimizer.zero_grad(set_to_none=True)

        logp = tensor_logp(
            model(xt, residual=False)
        )
        loss = equal_source_mean(
            -logp[idx, yt], masks
        )
        loss = loss + (
            0.001 *
            F.softplus(model.raw_scale).square().mean()
        )

        if not torch.isfinite(loss):
            raise FloatingPointError(
                "Anchor loss not finite"
            )

        loss.backward()
        optimizer.step()

    model.bias.requires_grad_(False)
    model.raw_scale.requires_grad_(False)

    optimizer = torch.optim.Adam(
        (model.weights,), lr=LR
    )

    augmented = None
    if consistent:
        assert augments is not None and len(augments) > 0
        augmented = [
            torch.as_tensor(a, dtype=torch.float64)
            for a in augments
        ]

    for epoch in range(EPOCHS):
        optimizer.zero_grad(set_to_none=True)

        logp = tensor_logp(model(xt))
        loss = equal_source_mean(
            -logp[idx, yt], masks
        )
        loss = loss + L2 * model.weights.square().mean()

        if consistent:
            alternate_lp = tensor_logp(
                model(augmented[epoch % len(augmented)])
            )
            augmented_loss = equal_source_mean(
                -alternate_lp[idx, yt], masks
            )

            p = logp.exp()
            q = alternate_lp.exp()

            sym_kl = 0.5 * (
                (p * (logp - alternate_lp)).sum(dim=1) +
                (q * (alternate_lp - logp)).sum(dim=1)
            )

            loss = (
                loss +
                AUGMENTED_NLL_WEIGHT * augmented_loss +
                CONSISTENCY_WEIGHT *
                equal_source_mean(sym_kl, masks)
            )

        if not torch.isfinite(loss):
            raise FloatingPointError(
                "Residual loss not finite"
            )

        loss.backward()
        optimizer.step()

    return model


def predict(model, x, residual=True):
    with torch.no_grad():
        t = torch.as_tensor(
            x, dtype=torch.float64
        )
        scores = model(
            t, residual=residual
        )
        prob = tensor_logp(scores).exp().cpu().numpy()

    assert np.isfinite(prob).all()
    assert np.max(
        np.abs(prob.sum(axis=1) - 1.0)
    ) < 1e-12

    return prob


def restore_phase14(path):
    with np.load(path, allow_pickle=False) as z:
        obj = ResidualHead()

        with torch.no_grad():
            obj.bias.copy_(
                torch.as_tensor(
                    z["bias"], dtype=torch.float64
                )
            )
            obj.raw_scale.copy_(
                torch.as_tensor(
                    z["raw_scale"], dtype=torch.float64
                )
            )
            obj.weights = nn.Parameter(
                torch.as_tensor(
                    np.asarray(z["weights"]),
                    dtype=torch.float64
                ),
                requires_grad=False
            )

        obj.guard_lambda = float(
            np.asarray(z["guard_lambda"]).item()
        )

    return obj


def old_predict(model, z, name):
    if name == "q_anchor":
        return predict(
            model, z, residual=False
        )

    with torch.no_grad():
        t = torch.as_tensor(
            z, dtype=torch.float64
        )
        a = t[:, (0, 2)]
        d = t[:, (1, 3)] - a

        base = (
            model.bias +
            F.softplus(model.raw_scale) * a
        )

        if name == "q_residual":
            phi = torch.stack((
                d,
                d * torch.tanh(1.5 * a),
                torch.tanh(2 * d) *
                (1 - torch.tanh(a).square())
            ), dim=-1)
            corr = (
                phi * model.weights
            ).sum(dim=-1)

        elif name == "q_static":
            corr = d * model.weights[:, 0]

        else:
            raise ValueError(name)

        s = (
            base +
            model.guard_lambda *
            CAP * torch.tanh(corr / CAP)
        )

        return tensor_logp(s).exp().cpu().numpy()


def performance(y, p):
    y = np.asarray(y, dtype=int)
    ix = np.arange(len(y))
    cond = y != 0

    k = p[cond, 1] / np.clip(
        p[cond, 1] + p[cond, 2],
        1e-12, None
    )
    target_k = (y[cond] == 1).astype(int)
    pred = p.argmax(axis=1)

    return {
        "nll": float(
            -np.log(
                np.clip(p[ix, y], 1e-15, 1.0)
            ).mean()
        ),
        "macro_f1": float(
            f1_score(
                y, pred,
                average="macro",
                labels=[0, 1, 2],
                zero_division=0
            )
        ),
        "egfr_auc": float(
            roc_auc_score(
                (y == 0).astype(int), p[:, 0]
            )
        ),
        "kras_auc": float(
            roc_auc_score(target_k, k)
        ),
        "conditional_kras_nll": float(
            -np.mean(
                target_k *
                np.log(np.clip(k, 1e-15, 1)) +
                (1 - target_k) *
                np.log(np.clip(1 - k, 1e-15, 1))
            )
        ),
        "brier": float(
            np.mean(
                np.sum(
                    (p - np.eye(3)[y]) ** 2,
                    axis=1
                )
            )
        ),
        "pred_e": int((pred == 0).sum()),
        "pred_k": int((pred == 1).sum()),
        "pred_w": int((pred == 2).sum())
    }


def make_partitions(raw, source, seed):
    source = np.asarray(source)
    arrangements = {}

    for kind in (
        "random", "egfr_sorted", "kras_sorted"
    ):
        partition = np.zeros(
            len(source), dtype=int
        )

        for j, g in enumerate(sorted(set(source))):
            ix = np.flatnonzero(source == g)

            if kind == "random":
                order = np.random.default_rng(
                    seed + 101 * j
                ).permutation(len(ix))

            else:
                channel = (
                    0 if kind == "egfr_sorted" else 2
                )
                order = np.argsort(
                    raw[ix, channel], kind="stable"
                )

            partition[
                ix[order[len(ix) // 2:]]
            ] = 1

        arrangements[kind] = partition

    return arrangements


assert SOURCE.is_dir(), SOURCE

input_pred = SOURCE / "LOCO_PREDICTIONS.csv"
assert input_pred.is_file()

raw_df = pd.read_csv(input_pred)

assert set((
    "held_cohort", "gsm", "true_class",
    "model", *PCOLS
)).issubset(raw_df.columns)

raw_df["gsm"] = raw_df["gsm"].astype(str)
raw_df["true_class"] = pd.to_numeric(
    raw_df["true_class"],
    errors="raise"
).astype(int)

for c in PCOLS:
    raw_df[c] = pd.to_numeric(
        raw_df[c], errors="raise"
    )

assert not raw_df.duplicated([
    "held_cohort", "gsm", "model"
]).any()

torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_num_threads(
    min(4, torch.get_num_threads())
)

DEST.mkdir(parents=True, exist_ok=True)

RUN = DEST / (
    datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%SZ_"
    ) + uuid.uuid4().hex[:8]
)
RUN.mkdir(exist_ok=False)

for folder in (
    "checkpoints", "references"
):
    (RUN / folder).mkdir()

save_json(RUN / "PROTOCOL.json", {
    "phase": 16,
    "source": str(SOURCE),
    "cohorts": COHORTS,
    "seed": SEED,
    "purpose": (
        "Training-source-referenced score CDF "
        "and batch-consistent bounded residual"
    ),
    "models": {
        "q_residual": (
            "Unchanged Phase14 full-target-quantile comparator"
        ),
        "ref0_residual": (
            "Training-source-reference CDF only"
        ),
        "ref50_residual": (
            "0.5 source CDF plus 0.5 target CDF, ordinary loss"
        ),
        "ref50_consistency": (
            "Matched ref50 residual with augmented supervised "
            "and symmetric-KL consistency objective"
        )
    },
    "reference": (
        "Equal-weight source ECDFs of OOF scores; "
        "leave-source-out for meta-training"
    ),
    "target_normalization": (
        "Unlabeled target-batch ECDF; no target labels"
    ),
    "augmentation": (
        "Random, EGFR128-sorted, KRAS128-sorted "
        "two-way source partitions"
    ),
    "epochs_per_stage": EPOCHS,
    "learning_rate": LR,
    "residual_l2": L2,
    "augmentation_nll_weight": AUGMENTED_NLL_WEIGHT,
    "symmetric_kl_weight": CONSISTENCY_WEIGHT,
    "outer_labels": "Evaluation only",
    "status": "Exploratory reused development cohorts",
    "external_cohort_used": False,
    "versions": {
        "python": sys.version,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "torch": torch.__version__,
        "sklearn": sklearn.__version__
    }
})

metrics_rows = []
pred_rows = []
stress_rows = []
training_rows = []
reproduction_rows = []
reference_rows = []
inputs = [input_pred]

for held_index, held in enumerate(COHORTS):
    frames = {}

    for model in (
        "raw128", "raw512",
        "q_anchor", "q_static", "q_residual"
    ):
        frame = raw_df.loc[
            (raw_df.held_cohort == held) &
            (raw_df.model == model)
        ]
        frame = frame.sort_values(
            "gsm"
        ).reset_index(drop=True)

        assert len(frame) == sum(COUNTS[held])
        assert tuple(
            int((frame.true_class == j).sum())
            for j in range(3)
        ) == COUNTS[held]

        frames[model] = frame

    for model in frames:
        assert frames[model].gsm.equals(
            frames["raw128"].gsm
        )
        assert frames[model].true_class.equals(
            frames["raw128"].true_class
        )

    y = frames["raw128"].true_class.to_numpy(
        dtype=int
    )
    gsm = frames["raw128"].gsm.to_numpy(
        dtype=str
    )

    raw_outer = four_scores(
        frames["raw128"][list(PCOLS)].to_numpy(),
        frames["raw512"][list(PCOLS)].to_numpy()
    )

    oof_path = (
        SOURCE / "inner_oof" / f"{held}.npz"
    )
    assert oof_path.is_file()
    inputs.append(oof_path)

    with np.load(
        oof_path, allow_pickle=False
    ) as z:
        train_raw = np.asarray(
            z["oof_scores"], dtype=np.float64
        )
        train_source = np.asarray(
            z["source"]
        ).astype(str)
        train_label = np.asarray(
            z["label"], dtype=int
        )
        existing_rank = np.asarray(
            z["rank_evidence"],
            dtype=np.float64
        )

    expected_n = sum(
        sum(COUNTS[s])
        for s in COHORTS if s != held
    )

    assert train_raw.shape == (
        expected_n, 4
    )
    assert existing_rank.shape == train_raw.shape
    assert np.isfinite(train_raw).all()
    assert np.isfinite(existing_rank).all()
    assert set(train_source) == (
        set(COHORTS) - {held}
    )
    assert np.isin(
        train_label, [0, 1, 2]
    ).all()

    for g in set(train_source):
        assert tuple(
            int(
                (
                    (train_source == g) &
                    (train_label == j)
                ).sum()
            )
            for j in range(3)
        ) == COUNTS[g]

    refs = make_reference(
        train_raw, train_source
    )

    previous_rank = training_transform(
        train_raw, train_source, refs, 1.0
    )
    original_err = float(
        np.max(
            np.abs(previous_rank - existing_rank)
        )
    )

    assert original_err < 1e-11, (
        f"Phase14 rank mismatch: "
        f"{held}: {original_err}"
    )

    refs_path = (
        RUN / "references" / f"{held}.npz"
    )
    np.savez_compressed(
        refs_path,
        **{
            f"ref_{g}": values
            for g, values in refs.items()
        }
    )

    reference_rows.append({
        "held_cohort": held,
        "sources": sorted(refs),
        "sizes": {
            k: len(v) for k, v in refs.items()
        },
        "previous_rank_max_error": original_err
    })

    old_models = {}

    for name in (
        "q_anchor", "q_static", "q_residual"
    ):
        path = (
            SOURCE / "checkpoints" /
            f"{held}_{name}.npz"
        )
        assert path.is_file()
        inputs.append(path)
        old_models[name] = restore_phase14(path)

    target_group = np.zeros(
        len(y), dtype=int
    )

    full_old_z = transformed(
        raw_outer, target_group, refs, 1.0
    )

    out_p = {
        "raw128": frames["raw128"][
            list(PCOLS)
        ].to_numpy()
    }

    for name in old_models:
        out_p[name] = old_predict(
            old_models[name], full_old_z, name
        )

        saved = frames[name][
            list(PCOLS)
        ].to_numpy()

        err = float(
            np.max(
                np.abs(out_p[name] - saved)
            )
        )

        reproduction_rows.append({
            "held_cohort": held,
            "model": name,
            "max_error": err
        })

        assert err < 1e-8, (
            f"Phase14 reproduction failed: "
            f"{held}/{name}/{err}"
        )

    views = make_partitions(
        train_raw,
        train_source,
        SEED + held_index * 1013
    )

    trained = {}
    train_cache = {}

    specifications = (
        ("ref0_residual", 0.0),
        ("ref50_residual", 0.5),
        ("ref50_consistency", 0.5)
    )

    for name, rho in specifications:
        if rho not in train_cache:
            x = training_transform(
                train_raw,
                train_source,
                refs,
                rho
            )

            aug = [
                training_transform(
                    train_raw,
                    train_source,
                    refs,
                    rho,
                    part
                )
                for part in views.values()
            ]
            train_cache[rho] = (x, aug)

        x, aug = train_cache[rho]

        trained[name] = train_model(
            x,
            train_label,
            train_source,
            consistent=(
                name == "ref50_consistency"
            ),
            augments=aug
        )

        out_z = transformed(
            raw_outer,
            target_group,
            refs,
            rho
        )
        out_p[name] = predict(
            trained[name], out_z
        )

        checkpoint = (
            RUN / "checkpoints" /
            f"{held}_{name}.npz"
        )

        np.savez_compressed(
            checkpoint,
            bias=trained[name].bias.detach().cpu().numpy(),
            raw_scale=trained[name].raw_scale.detach().cpu().numpy(),
            weights=trained[name].weights.detach().cpu().numpy(),
            rho=np.float64(rho),
            reference_file=str(refs_path.name),
            consistency=np.int8(
                name == "ref50_consistency"
            )
        )

        train_prediction = predict(
            trained[name], x
        )

        for g in sorted(refs):
            mask = train_source == g
            training_rows.append({
                "held_cohort": held,
                "model": name,
                "source": g,
                "nll": performance(
                    train_label[mask],
                    train_prediction[mask]
                )["nll"]
            })

    for parameter in (
        "bias", "raw_scale"
    ):
        a = getattr(
            trained["ref50_residual"],
            parameter
        ).detach().numpy()

        b = getattr(
            trained["ref50_consistency"],
            parameter
        ).detach().numpy()

        assert np.max(
            np.abs(a - b)
        ) < 1e-12

    for name in MODELS:
        p = out_p[name]
        result = performance(y, p)

        metrics_rows.append({
            "held_cohort": held,
            "model": name,
            **result
        })

        for i in range(len(y)):
            pred_rows.append({
                "held_cohort": held,
                "gsm": gsm[i],
                "true_class": int(y[i]),
                "model": name,
                "p_egfr": float(p[i, 0]),
                "p_kras": float(p[i, 1]),
                "p_dwt": float(p[i, 2]),
                "predicted_class": int(
                    p[i].argmax()
                )
            })

    configurations = []

    for rep in range(RANDOM_STRESS_SPLITS):
        rng = np.random.default_rng(
            SEED + 101 * held_index + rep
        )
        order = rng.permutation(len(y))

        groups = np.zeros(
            len(y), dtype=int
        )
        groups[
            order[len(y) // 2:]
        ] = 1

        configurations.append((
            "random_half", rep, groups
        ))

    for j, scenario in (
        (0, "egfr_sorted_half"),
        (2, "kras_sorted_half")
    ):
        order = np.argsort(
            raw_outer[:, j], kind="stable"
        )
        groups = np.zeros(
            len(y), dtype=int
        )
        groups[
            order[len(y) // 2:]
        ] = 1

        configurations.append((
            scenario, 0, groups
        ))

    for scenario, rep, groups in configurations:
        for name in (
            "q_residual",
            "ref0_residual",
            "ref50_residual",
            "ref50_consistency"
        ):
            if name == "q_residual":
                zp = transformed(
                    raw_outer, groups, refs, 1.0
                )
                p = old_predict(
                    old_models[name],
                    zp,
                    name
                )
            else:
                rho = (
                    0.0 if name == "ref0_residual"
                    else 0.5
                )
                zp = transformed(
                    raw_outer, groups, refs, rho
                )
                p = predict(
                    trained[name], zp
                )

            baseline = out_p[name]
            old_result = performance(y, baseline)
            new_result = performance(y, p)

            stress_rows.append({
                "held_cohort": held,
                "scenario": scenario,
                "repeat": rep,
                "model": name,
                "batch_min_n": int(
                    np.bincount(groups).min()
                ),
                "delta_nll": (
                    new_result["nll"] -
                    old_result["nll"]
                ),
                "delta_macro_f1": (
                    new_result["macro_f1"] -
                    old_result["macro_f1"]
                ),
                "changed_predictions": int(
                    np.sum(
                        p.argmax(axis=1) !=
                        baseline.argmax(axis=1)
                    )
                ),
                "mean_abs_probability_change": float(
                    np.mean(
                        np.abs(p - baseline)
                    )
                )
            })

    print(
        f"COHORT: {held} "
        f"N={len(y)} "
        f"OOF_TRAIN={len(train_label)}"
    )

metrics = pd.DataFrame(metrics_rows)
predictions = pd.DataFrame(pred_rows)
stress = pd.DataFrame(stress_rows)

metrics.to_csv(
    RUN / "LOCO_METRICS.csv",
    index=False
)
predictions.to_csv(
    RUN / "LOCO_PREDICTIONS.csv",
    index=False
)
stress.to_csv(
    RUN / "BATCH_STRESS.csv",
    index=False
)

pd.DataFrame(training_rows).to_csv(
    RUN / "SOURCE_OOF_TRAIN_RISK.csv",
    index=False
)
pd.DataFrame(reproduction_rows).to_csv(
    RUN / "REPRODUCTION.csv",
    index=False
)

save_json(
    RUN / "REFERENCE_AUDIT.json",
    reference_rows
)

summary = (
    metrics.groupby("model", sort=False)[
        [
            "nll",
            "macro_f1",
            "egfr_auc",
            "kras_auc",
            "conditional_kras_nll",
            "brier"
        ]
    ]
    .mean()
    .reset_index()
)

summary.to_csv(
    RUN / "EQUAL_COHORT_SUMMARY.csv",
    index=False
)

stress_means = (
    stress.groupby(
        ["model", "scenario"],
        sort=False
    )[
        [
            "delta_nll",
            "delta_macro_f1",
            "changed_predictions",
            "mean_abs_probability_change"
        ]
    ]
    .mean()
    .reset_index()
)

stress_means.to_csv(
    RUN / "EQUAL_COHORT_STRESS_SUMMARY.csv",
    index=False
)

lookup = summary.set_index("model")
primary = lookup.loc["ref50_consistency"]
old = lookup.loc["q_residual"]
plain = lookup.loc["ref50_residual"]

stress_lookup = stress_means.loc[
    stress_means.scenario == "random_half"
].set_index("model")

stability_gain = float(
    stress_lookup.loc[
        "ref50_consistency",
        "delta_macro_f1"
    ] -
    stress_lookup.loc[
        "q_residual",
        "delta_macro_f1"
    ]
)

screen = {
    "macro_f1_gain_vs_original_at_least_0_01": (
        float(
            primary.macro_f1 -
            old.macro_f1
        ) >= 0.01
    ),
    "nll_gain_vs_original_at_least_0_01": (
        float(
            old.nll -
            primary.nll
        ) >= 0.01
    ),
    "macro_f1_gain_vs_matched_plain_at_least_0_01": (
        float(
            primary.macro_f1 -
            plain.macro_f1
        ) >= 0.01
    ),
    "random_half_stability_gain_at_least_0_01": (
        stability_gain >= 0.01
    )
}

save_json(RUN / "SCREENING.json", {
    "exploratory_only": True,
    "prespecified_candidate": "ref50_consistency",
    "matched_objective_control": "ref50_residual",
    "source_only_control": "ref0_residual",
    "original_control": "q_residual",
    "results": screen,
    "no_generalization_or_novelty_guarantee": True
})

save_json(RUN / "RUN_MANIFEST.json", {
    "phase": 16,
    "created_utc": (
        datetime.now(timezone.utc).isoformat()
    ),
    "inputs": [
        {
            "path": str(p),
            "sha256": sha256(p)
        }
        for p in sorted(set(inputs))
    ],
    "outputs": [
        {
            "path": str(
                p.relative_to(RUN)
            ),
            "sha256": sha256(p),
            "bytes": p.stat().st_size
        }
        for p in sorted(RUN.rglob("*"))
        if p.is_file()
        and p.name != "RUN_MANIFEST.json"
    ]
})

print("PHASE16: COMPLETE")

print(
    "REPRODUCTION_MAX_ERROR:",
    f"{max(x['max_error'] for x in reproduction_rows):.3e}"
)

print("MODEL_RESULTS:")
print(
    summary[
        [
            "model",
            "nll",
            "macro_f1",
            "egfr_auc",
            "kras_auc"
        ]
    ].round(4).to_string(index=False)
)

print("EQUAL_COHORT_BATCH_STRESS:")
print(
    stress_means.loc[
        stress_means.scenario == "random_half",
        [
            "model",
            "delta_nll",
            "delta_macro_f1",
            "changed_predictions"
        ]
    ].round(4).to_string(index=False)
)

print("SCORE_SORTED_STRESS:")
print(
    stress_means.loc[
        stress_means.scenario != "random_half",
        [
            "model",
            "scenario",
            "delta_nll",
            "delta_macro_f1"
        ]
    ].round(4).to_string(index=False)
)

print("SCREENING:", screen)
print("RUN:", RUN)

COHORT: GSE13213 N=117 OOF_TRAIN=726
COHORT: GSE31210 N=226 OOF_TRAIN=617
COHORT: GSE32863 N=58 OOF_TRAIN=785
COHORT: GSE72094 N=442 OOF_TRAIN=401
PHASE16: COMPLETE
REPRODUCTION_MAX_ERROR: 1.110e-16
MODEL_RESULTS:
            model    nll  macro_f1  egfr_auc  kras_auc
           raw128 1.1371    0.5117    0.8662    0.8553
         q_anchor 0.9210    0.5250    0.8662    0.8553
         q_static 0.9131    0.5257    0.8699    0.8612
       q_residual 0.9093    0.5452    0.8696    0.8626
    ref0_residual 1.4003    0.2331    0.5366    0.7869
   ref50_residual 0.9386    0.4263    0.8506    0.8631
ref50_consistency 0.9426    0.4252    0.8497    0.8630
EQUAL_COHORT_BATCH_STRESS:
            model  delta_nll  delta_macro_f1  changed_predictions
       q_residual     0.0030         -0.0164                  7.2
    ref0_residual     0.0000          0.0000                  0.0
   ref50_residual     0.0020         -0.0073                  4.0
ref50_consistency     0.0021         -0.0124           

In [21]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import sys
import uuid

import numpy as np
import pandas as pd
from scipy.special import expit
from scipy.stats import rankdata
from sklearn.metrics import f1_score
import scipy
import sklearn

ROOT = Path(r"D:\LUAD_DATAFIRST_METHOD_2026")
SOURCE = ROOT / "14_quantile_residual_transfer" / "20261023T160742Z_f9435239"
OUTPUT_ROOT = ROOT / "17_frozen_reference_inference"

COHORTS = ("GSE13213", "GSE31210", "GSE32863", "GSE72094")
EXPECTED = {
    "GSE13213": (45, 15, 57),
    "GSE31210": (127, 20, 79),
    "GSE32863": (17, 22, 19),
    "GSE72094": (47, 154, 241),
}
MODELS = ("q_anchor", "q_static", "q_residual")
PCOLS = ("p_egfr", "p_kras", "p_dwt")
SEED = 20261023
REPEATS = 20
CAP = 1.5


def digest(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for part in iter(lambda: f.read(1024 * 1024), b""):
            h.update(part)
    return h.hexdigest()


def make_native(x):
    if isinstance(x, Path):
        return str(x)
    if isinstance(x, dict):
        return {str(k): make_native(v) for k, v in x.items()}
    if isinstance(x, (tuple, list)):
        return [make_native(v) for v in x]
    if isinstance(x, np.integer):
        return int(x)
    if isinstance(x, (np.floating, float)):
        return float(x) if np.isfinite(x) else None
    if isinstance(x, np.bool_):
        return bool(x)
    return x


def write_json(path, x):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            make_native(x), f, indent=2,
            ensure_ascii=False, allow_nan=False
        )


def effective_logit(p):
    e = np.clip(p[:, 0], 1e-12, 1 - 1e-12)
    k = np.clip(p[:, 1], 1e-12, 1)
    w = np.clip(p[:, 2], 1e-12, 1)
    return np.column_stack((
        np.log(e / (1 - e)),
        np.log(k / w)
    ))


def make_scores(p128, p512):
    a = effective_logit(p128)
    b = effective_logit(p512)
    return np.column_stack((
        a[:, 0], b[:, 0],
        a[:, 1], b[:, 1]
    ))


def to_quantile(u):
    u = np.clip(u, 1e-5, 1 - 1e-5)
    return (np.log(u) - np.log1p(-u)) / 2.5


def full_rank_features(scores):
    n = len(scores)
    u = np.column_stack([
        (rankdata(scores[:, j], method="average") - 0.5) / n
        for j in range(4)
    ])
    return to_quantile(u)


def frozen_bank_features(query_scores, reference_scores):
    ref = np.asarray(reference_scores, dtype=np.float64)
    query = np.asarray(query_scores, dtype=np.float64)

    assert ref.ndim == query.ndim == 2
    assert ref.shape[1] == query.shape[1] == 4
    assert len(ref) >= 25
    assert np.isfinite(ref).all() and np.isfinite(query).all()

    u = np.empty_like(query)

    for j in range(4):
        sorted_ref = np.sort(ref[:, j])
        left = np.searchsorted(
            sorted_ref, query[:, j], side="left"
        )
        right = np.searchsorted(
            sorted_ref, query[:, j], side="right"
        )
        u[:, j] = (
            left + right + 1.0
        ) / (2.0 * (len(ref) + 1))

    return to_quantile(u)


def load_checkpoint(path, name):
    with np.load(path, allow_pickle=False) as z:
        state = {
            "bias": np.asarray(z["bias"], dtype=np.float64),
            "raw_scale": np.asarray(
                z["raw_scale"], dtype=np.float64
            ),
            "weights": np.asarray(
                z["weights"], dtype=np.float64
            ),
            "lam": float(
                np.asarray(z["guard_lambda"]).item()
            ),
        }

    dimensions = {
        "q_anchor": 0,
        "q_static": 1,
        "q_residual": 3
    }

    assert state["bias"].shape == (2,)
    assert state["raw_scale"].shape == (2,)
    assert state["weights"].shape == (
        2, dimensions[name]
    )

    return state


def predict(z, state, name):
    a = z[:, (0, 2)]
    s = (
        state["bias"] +
        np.logaddexp(0, state["raw_scale"]) * a
    )

    if name != "q_anchor":
        d = z[:, (1, 3)] - a

        if name == "q_static":
            phi = d[:, :, None]
        else:
            phi = np.stack((
                d,
                d * np.tanh(1.5 * a),
                np.tanh(2 * d) *
                (1 - np.tanh(a) ** 2)
            ), axis=2)

        correction = np.einsum(
            "nhk,hk->nh",
            phi,
            state["weights"]
        )

        s += (
            state["lam"] *
            CAP *
            np.tanh(correction / CAP)
        )

    e = expit(s[:, 0])
    k = expit(s[:, 1])

    p = np.column_stack((
        e,
        (1 - e) * k,
        (1 - e) * (1 - k)
    ))

    assert np.isfinite(p).all()
    assert np.max(
        np.abs(p.sum(axis=1) - 1)
    ) < 1e-12

    return p


def performance(y, p):
    idx = np.arange(len(y))
    loss = -np.log(
        np.clip(p[idx, y], 1e-15, 1)
    )
    pred = p.argmax(axis=1)

    return {
        "nll": float(loss.mean()),
        "macro_f1": float(f1_score(
            y, pred,
            average="macro",
            labels=[0, 1, 2],
            zero_division=0
        )),
        "accuracy": float(np.mean(y == pred))
    }


assert SOURCE.is_dir(), SOURCE

input_pred = SOURCE / "LOCO_PREDICTIONS.csv"
assert input_pred.is_file(), input_pred

records = pd.read_csv(input_pred)

required = {
    "held_cohort", "gsm", "model",
    "true_class", *PCOLS
}
assert required.issubset(records.columns)

records["gsm"] = records["gsm"].astype(str)
records["true_class"] = pd.to_numeric(
    records["true_class"], errors="raise"
).astype(int)

for pcol in PCOLS:
    records[pcol] = pd.to_numeric(
        records[pcol], errors="raise"
    )

assert not records.duplicated([
    "held_cohort", "gsm", "model"
]).any()

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

run_id = (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ_") +
    uuid.uuid4().hex[:8]
)
RUN = OUTPUT_ROOT / run_id
RUN.mkdir(exist_ok=False)

write_json(RUN / "PROTOCOL.json", {
    "phase": 17,
    "status": "Exploratory; historically reused development cohorts",
    "source_run": str(SOURCE),
    "method": "Frozen unlabeled target-reference empirical-CDF inference",
    "score_channels": [
        "EGFR128", "EGFR512", "KRAS128", "KRAS512"
    ],
    "cdf": "(L+R+1)/(2*(n_ref+1))",
    "reference_designs": [
        "25 random samples",
        "50 percent of target cohort",
        "75 percent of target cohort",
        "Score-selected lower/upper halves",
        "All-other-sample leave-one-out verification"
    ],
    "repeats": REPEATS,
    "seed": SEED,
    "selection": "Unlabeled scores only",
    "evaluation": "Queries excluded from reference bank",
    "inference": "Fixed reference across query batches",
    "full_cohort_comparator": "Phase14 q_residual",
    "no_training": True,
    "external_cohort_used": False,
    "not_inductive": True,
    "versions": {
        "python": sys.version,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": scipy.__version__,
        "sklearn": sklearn.__version__
    }
})

inputs = [input_pred]
reproduction = []
run_rows = []
query_prediction_rows = []
reference_records = []
loo_errors = []
partition_errors = []
full_results = []

for fold_idx, cohort in enumerate(COHORTS):
    frames = {}

    for name in ("raw128", "raw512", *MODELS):
        sub = records.loc[
            (records.held_cohort == cohort) &
            (records.model == name)
        ]
        sub = sub.sort_values(
            "gsm"
        ).reset_index(drop=True)

        assert len(sub) == sum(EXPECTED[cohort])
        assert tuple(
            int((sub.true_class == k).sum())
            for k in range(3)
        ) == EXPECTED[cohort]

        frames[name] = sub

    for name in frames:
        assert frames[name].gsm.equals(
            frames["raw128"].gsm
        )
        assert frames[name].true_class.equals(
            frames["raw128"].true_class
        )

    labels = frames["raw128"].true_class.to_numpy(
        dtype=int
    )
    gsm = frames["raw128"].gsm.to_numpy(
        dtype=str
    )
    raw128 = frames["raw128"][
        list(PCOLS)
    ].to_numpy(dtype=float)
    raw512 = frames["raw512"][
        list(PCOLS)
    ].to_numpy(dtype=float)

    scores = make_scores(raw128, raw512)
    z_full = full_rank_features(scores)

    states = {}
    full_p = {}

    for name in MODELS:
        ckpt = (
            SOURCE / "checkpoints" /
            f"{cohort}_{name}.npz"
        )
        assert ckpt.is_file()
        inputs.append(ckpt)

        states[name] = load_checkpoint(
            ckpt, name
        )
        full_p[name] = predict(
            z_full, states[name], name
        )

        saved_p = frames[name][
            list(PCOLS)
        ].to_numpy(dtype=float)

        err = float(np.max(
            np.abs(full_p[name] - saved_p)
        ))
        assert err < 1e-8, (
            cohort, name, err
        )

        reproduction.append({
            "held_cohort": cohort,
            "model": name,
            "max_error": err
        })
        full_results.append({
            "held_cohort": cohort,
            "model": name,
            **performance(labels, full_p[name])
        })

    max_loo_error = 0.0

    for ix in range(len(gsm)):
        mask = np.ones(
            len(gsm), dtype=bool
        )
        mask[ix] = False

        z_loo = frozen_bank_features(
            scores[ix:ix + 1],
            scores[mask]
        )
        max_loo_error = max(
            max_loo_error,
            float(np.max(
                np.abs(z_loo[0] - z_full[ix])
            ))
        )

    assert max_loo_error < 1e-10

    loo_errors.append({
        "held_cohort": cohort,
        "max_quantile_error": max_loo_error
    })

    n = len(gsm)
    reference_counts = sorted(set((
        25,
        n // 2,
        (3 * n) // 4
    )))
    assert all(
        25 <= size < n
        for size in reference_counts
    )

    configurations = []

    for rsize in reference_counts:
        for rep in range(REPEATS):
            rng = np.random.default_rng(
                SEED +
                fold_idx * 100003 +
                rsize * 1009 +
                rep
            )
            bank = np.sort(
                rng.choice(
                    n, size=rsize,
                    replace=False
                )
            )
            configurations.append((
                f"random_n{rsize}",
                rep,
                bank
            ))

    for channel, prefix in (
        (0, "egfr"),
        (2, "kras")
    ):
        ordered = np.argsort(
            scores[:, channel],
            kind="stable"
        )
        rsize = n // 2

        configurations.append((
            f"{prefix}_low_half",
            0,
            np.sort(ordered[:rsize])
        ))
        configurations.append((
            f"{prefix}_high_half",
            0,
            np.sort(ordered[-rsize:])
        ))

    for scenario, rep, bank in configurations:
        bankmask = np.zeros(
            n, dtype=bool
        )
        bankmask[bank] = True

        query_ids = np.flatnonzero(
            ~bankmask
        )

        assert len(
            np.intersect1d(bank, query_ids)
        ) == 0

        reference_records.append({
            "held_cohort": cohort,
            "scenario": scenario,
            "repeat": rep,
            "n_reference": len(bank),
            "n_query": len(query_ids),
            "reference_gsms": gsm[bank].tolist()
        })

        reference_scores = scores[bank]

        query_features = frozen_bank_features(
            scores[query_ids],
            reference_scores
        )
        query_y = labels[query_ids]

        order = np.argsort(
            scores[query_ids, 0],
            kind="stable"
        )

        first = query_ids[
            order[:len(order) // 2]
        ]
        second = query_ids[
            order[len(order) // 2:]
        ]

        assert len(first) and len(second)

        reordered_z = np.empty_like(
            query_features
        )
        local_index = {
            int(q): j
            for j, q in enumerate(query_ids)
        }

        for group in (first, second):
            group_features = frozen_bank_features(
                scores[group],
                reference_scores
            )
            for idx, feat in zip(
                group, group_features
            ):
                reordered_z[
                    local_index[int(idx)]
                ] = feat

        rank_diff = float(np.max(
            np.abs(
                reordered_z - query_features
            )
        ))
        assert rank_diff < 1e-12
        partition_errors.append(rank_diff)

        full_reference_p = (
            full_p["q_residual"][query_ids]
        )
        raw_reference_p = raw128[query_ids]

        full_metrics = performance(
            query_y, full_reference_p
        )
        raw_metrics = performance(
            query_y, raw_reference_p
        )

        for name in MODELS:
            bank_p = predict(
                query_features,
                states[name],
                name
            )
            bank_metrics = performance(
                query_y, bank_p
            )

            run_rows.append({
                "held_cohort": cohort,
                "scenario": scenario,
                "repeat": rep,
                "model": name,
                "n_reference": len(bank),
                "n_query": len(query_ids),
                "nll": bank_metrics["nll"],
                "macro_f1": bank_metrics["macro_f1"],
                "full_q_residual_nll_on_same_queries": full_metrics["nll"],
                "full_q_residual_f1_on_same_queries": full_metrics["macro_f1"],
                "raw128_nll_on_same_queries": raw_metrics["nll"],
                "raw128_f1_on_same_queries": raw_metrics["macro_f1"],
                "nll_excess_over_full_residual": (
                    bank_metrics["nll"] -
                    full_metrics["nll"]
                ),
                "f1_change_vs_full_residual": (
                    bank_metrics["macro_f1"] -
                    full_metrics["macro_f1"]
                ),
                "mean_abs_probability_delta_vs_full": float(
                    np.mean(np.abs(
                        bank_p - full_reference_p
                    ))
                ),
                "changed_class_count_vs_full": int(
                    np.sum(
                        bank_p.argmax(axis=1) !=
                        full_reference_p.argmax(axis=1)
                    )
                ),
                "mean_abs_quantile_delta": float(
                    np.abs(
                        query_features -
                        z_full[query_ids]
                    ).mean()
                )
            })

            for index, p in zip(
                query_ids, bank_p
            ):
                query_prediction_rows.append({
                    "held_cohort": cohort,
                    "scenario": scenario,
                    "repeat": rep,
                    "model": name,
                    "gsm": gsm[index],
                    "true_class": int(
                        labels[index]
                    ),
                    "p_egfr": float(p[0]),
                    "p_kras": float(p[1]),
                    "p_dwt": float(p[2])
                })

    print(
        f"COHORT: {cohort} "
        f"N={n} "
        f"REFERENCE_COUNTS={reference_counts} "
        f"LOO_ERROR={max_loo_error:.3e}"
    )

result = pd.DataFrame(run_rows)

result.to_csv(
    RUN / "BANK_EVALUATION.csv",
    index=False
)
pd.DataFrame(
    query_prediction_rows
).to_csv(
    RUN / "BANK_PREDICTIONS.csv",
    index=False
)
pd.DataFrame(
    full_results
).to_csv(
    RUN / "REPRODUCED_FULL_COHORT_METRICS.csv",
    index=False
)
pd.DataFrame(
    reproduction
).to_csv(
    RUN / "REPRODUCTION.csv",
    index=False
)

write_json(
    RUN / "REFERENCE_BANK_MEMBERSHIPS.json",
    reference_records
)
write_json(
    RUN / "INVARIANCE_CHECK.json",
    {
        "loo_quantile_errors": loo_errors,
        "max_query_partition_feature_error": max(
            partition_errors
        ),
        "scope": "Fixed reference bank only"
    }
)

random_rows = result.loc[
    result.scenario.str.startswith("random_n")
].copy()

cohort_means = random_rows.groupby(
    ["held_cohort", "n_reference", "model"],
    as_index=False
)[[
    "nll_excess_over_full_residual",
    "f1_change_vs_full_residual",
    "mean_abs_probability_delta_vs_full",
    "changed_class_count_vs_full"
]].mean()

cohort_means.to_csv(
    RUN / "RANDOM_BANK_PER_COHORT.csv",
    index=False
)

random_means = cohort_means.copy()

random_means["reference_level"] = [
    (
        "n25"
        if row.n_reference == 25
        else "half"
        if row.n_reference ==
        sum(EXPECTED[row.held_cohort]) // 2
        else "three_quarters"
    )
    for row in random_means.itertuples(
        index=False
    )
]

equal_cohort = random_means.groupby(
    ["reference_level", "model"],
    as_index=False
)[[
    "nll_excess_over_full_residual",
    "f1_change_vs_full_residual",
    "mean_abs_probability_delta_vs_full"
]].mean()

equal_cohort.to_csv(
    RUN / "EQUAL_COHORT_BANK_SUMMARY.csv",
    index=False
)

stress = result.loc[
    ~result.scenario.str.startswith("random_n")
]

stress.to_csv(
    RUN / "SCORE_SELECTED_REFERENCE_STRESS.csv",
    index=False
)

primary = equal_cohort.loc[
    (equal_cohort.reference_level == "three_quarters") &
    (equal_cohort.model == "q_residual")
].iloc[0]

screen = {
    "quarter_holdout_75pct_reference_f1_loss_no_worse_than_0_02": bool(
        primary.f1_change_vs_full_residual >= -0.02
    ),
    "quarter_holdout_75pct_reference_nll_increase_at_most_0_02": bool(
        primary.nll_excess_over_full_residual <= 0.02
    ),
    "frozen_bank_batch_invariance_pass": bool(
        max(partition_errors) <= 1e-12
    ),
    "reproduces_phase14": bool(
        max(
            x["max_error"]
            for x in reproduction
        ) < 1e-8
    ),
    "note": (
        "Exploratory feasibility thresholds, "
        "not independent confirmation"
    )
}

write_json(
    RUN / "SCREENING.json",
    screen
)

write_json(RUN / "RUN_MANIFEST.json", {
    "phase": 17,
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "inputs": [
        {
            "path": str(p),
            "sha256": digest(p)
        }
        for p in sorted(set(inputs))
    ],
    "outputs": [
        {
            "path": str(
                p.relative_to(RUN)
            ),
            "sha256": digest(p),
            "bytes": p.stat().st_size
        }
        for p in sorted(RUN.iterdir())
        if p.is_file()
        and p.name != "RUN_MANIFEST.json"
    ]
})

print("PHASE17: COMPLETE")

print(
    "REPRODUCTION_MAX_ERROR:",
    f"{max(x['max_error'] for x in reproduction):.3e}"
)
print(
    "FROZEN_REFERENCE_INVARIANCE_ERROR:",
    f"{max(partition_errors):.3e}"
)

print("RANDOM_REFERENCE_SUMMARY:")
print(
    equal_cohort.loc[
        equal_cohort.model == "q_residual"
    ].round(4).to_string(index=False)
)

print("PERCENT75_PER_COHORT:")
selected_75 = cohort_means.loc[
    (cohort_means.model == "q_residual") &
    (
        cohort_means.apply(
            lambda row: (
                row.n_reference ==
                (3 * sum(EXPECTED[row.held_cohort])) // 4
            ),
            axis=1
        )
    )
]
print(
    selected_75.round(4).to_string(index=False)
)

print("SCORE_SELECTED_REFERENCE_STRESS:")
print(
    stress.loc[
        stress.model == "q_residual",
        [
            "held_cohort",
            "scenario",
            "n_reference",
            "n_query",
            "nll_excess_over_full_residual",
            "f1_change_vs_full_residual"
        ]
    ].round(4).to_string(index=False)
)

print("SCREENING:", screen)
print("RUN:", RUN)

COHORT: GSE13213 N=117 REFERENCE_COUNTS=[25, 58, 87] LOO_ERROR=0.000e+00
COHORT: GSE31210 N=226 REFERENCE_COUNTS=[25, 113, 169] LOO_ERROR=0.000e+00
COHORT: GSE32863 N=58 REFERENCE_COUNTS=[25, 29, 43] LOO_ERROR=0.000e+00
COHORT: GSE72094 N=442 REFERENCE_COUNTS=[25, 221, 331] LOO_ERROR=0.000e+00
PHASE17: COMPLETE
REPRODUCTION_MAX_ERROR: 2.220e-16
FROZEN_REFERENCE_INVARIANCE_ERROR: 0.000e+00
RANDOM_REFERENCE_SUMMARY:
reference_level      model  nll_excess_over_full_residual  f1_change_vs_full_residual  mean_abs_probability_delta_vs_full
           half q_residual                        -0.0002                     -0.0099                              0.0187
            n25 q_residual                         0.0222                     -0.0252                              0.0410
 three_quarters q_residual                        -0.0007                     -0.0134                              0.0119
PERCENT75_PER_COHORT:
held_cohort  n_reference      model  nll_excess_over_full_residual  f1_c

In [22]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import sys
import uuid

import numpy as np
import pandas as pd
from scipy.stats import rankdata
import scipy
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F

ROOT = Path(r"D:\LUAD_DATAFIRST_METHOD_2026")
SOURCE = ROOT / "14_quantile_residual_transfer" / "20261023T160742Z_f9435239"
DEST = ROOT / "18_frozen_method_training"

COHORTS = ("GSE13213", "GSE31210", "GSE32863", "GSE72094")
COUNTS = {
    "GSE13213": (45, 15, 57),
    "GSE31210": (127, 20, 79),
    "GSE32863": (17, 22, 19),
    "GSE72094": (47, 154, 241)
}

PCOLS = ("p_egfr", "p_kras", "p_dwt")
SEED = 20261023
EPOCHS = 400
LR = 0.045
CAP = 1.5
L2 = 0.02

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def native(x):
    if isinstance(x, Path):
        return str(x)
    if isinstance(x, dict):
        return {str(k): native(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [native(v) for v in x]
    if isinstance(x, np.ndarray):
        return native(x.tolist())
    if isinstance(x, np.integer):
        return int(x)
    if isinstance(x, (np.floating, float)):
        return float(x) if np.isfinite(x) else None
    if isinstance(x, np.bool_):
        return bool(x)
    return x

def write_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            native(obj), f, indent=2,
            ensure_ascii=False, allow_nan=False
        )

def extract_scores(p):
    e = np.clip(p[:, 0], 1e-12, 1 - 1e-12)
    k = np.clip(p[:, 1], 1e-12, 1)
    w = np.clip(p[:, 2], 1e-12, 1)
    return np.column_stack((
        np.log(e / (1 - e)),
        np.log(k / w)
    ))

def quantile_transform(raw):
    n = len(raw)
    u = np.column_stack([
        (rankdata(raw[:, j], method="average") - 0.5) / n
        for j in range(4)
    ])
    u = np.clip(u, 1e-5, 1 - 1e-5)
    return (np.log(u) - np.log1p(-u)) / 2.5

class FinalResidualModel(nn.Module):
    def __init__(self):
        super().__init__()
        init = float(np.log(np.expm1(2.0)))

        self.bias = nn.Parameter(
            torch.zeros(2, dtype=torch.float64)
        )
        self.raw_scale = nn.Parameter(
            torch.full((2,), init, dtype=torch.float64)
        )
        self.weights = nn.Parameter(
            torch.zeros((2, 3), dtype=torch.float64)
        )

    def forward(self, x, residual=True):
        a = x[:, (0, 2)]
        score = (
            self.bias +
            F.softplus(self.raw_scale) * a
        )

        if residual:
            d = x[:, (1, 3)] - a

            features = torch.stack((
                d,
                d * torch.tanh(1.5 * a),
                torch.tanh(2.0 * d) *
                (1.0 - torch.tanh(a).square())
            ), dim=-1)

            correction = (
                features * self.weights
            ).sum(dim=-1)

            score = score + CAP * torch.tanh(
                correction / CAP
            )

        return score

def hierarchical_logp(s):
    e, k = s[:, 0], s[:, 1]
    return torch.stack((
        F.logsigmoid(e),
        F.logsigmoid(-e) + F.logsigmoid(k),
        F.logsigmoid(-e) + F.logsigmoid(-k)
    ), dim=1)

assert SOURCE.is_dir()
source_file = SOURCE / "LOCO_PREDICTIONS.csv"
assert source_file.is_file()

df = pd.read_csv(source_file)
required = {
    "held_cohort", "gsm", "true_class",
    "model", *PCOLS
}
assert required.issubset(df.columns)

df["gsm"] = df["gsm"].astype(str)
df["true_class"] = pd.to_numeric(
    df["true_class"], errors="raise"
).astype(int)

for col in PCOLS:
    df[col] = pd.to_numeric(
        df[col], errors="raise"
    )

assert not df.duplicated([
    "held_cohort", "gsm", "model"
]).any()

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(
    min(4, torch.get_num_threads())
)

features = []
labels = []
groups = []
sample_records = []

for cohort in COHORTS:
    frames = {}

    for model in ("raw128", "raw512"):
        frame = df.loc[
            (df.held_cohort == cohort) &
            (df.model == model)
        ].sort_values("gsm").reset_index(drop=True)

        assert len(frame) == sum(COUNTS[cohort])
        assert tuple(
            int((frame.true_class == c).sum())
            for c in range(3)
        ) == COUNTS[cohort]

        frames[model] = frame

    a = frames["raw128"]
    b = frames["raw512"]

    assert a.gsm.equals(b.gsm)
    assert a.true_class.equals(b.true_class)

    pa = a[list(PCOLS)].to_numpy(dtype=np.float64)
    pb = b[list(PCOLS)].to_numpy(dtype=np.float64)

    assert np.isfinite(pa).all()
    assert np.isfinite(pb).all()
    assert np.max(np.abs(pa.sum(axis=1) - 1)) < 1e-5
    assert np.max(np.abs(pb.sum(axis=1) - 1)) < 1e-5

    sa = extract_scores(pa)
    sb = extract_scores(pb)

    raw = np.column_stack((
        sa[:, 0], sb[:, 0],
        sa[:, 1], sb[:, 1]
    ))

    transformed = quantile_transform(raw)
    y = a.true_class.to_numpy(dtype=int)

    features.append(transformed)
    labels.append(y)
    groups.extend([cohort] * len(y))

    for i, gsm in enumerate(a.gsm):
        sample_records.append({
            "cohort": cohort,
            "gsm": gsm,
            "label": int(y[i]),
            "egfr128_score": float(raw[i, 0]),
            "egfr512_score": float(raw[i, 1]),
            "kras128_score": float(raw[i, 2]),
            "kras512_score": float(raw[i, 3]),
            "egfr128_quantile": float(transformed[i, 0]),
            "egfr512_quantile": float(transformed[i, 1]),
            "kras128_quantile": float(transformed[i, 2]),
            "kras512_quantile": float(transformed[i, 3])
        })

X = np.vstack(features)
y = np.concatenate(labels)
groups = np.asarray(groups)

assert X.shape == (843, 4)
assert len(y) == len(groups) == 843
assert np.isfinite(X).all()
assert len(set(
    (r["cohort"], r["gsm"])
    for r in sample_records
)) == 843

DEST.mkdir(parents=True, exist_ok=True)

RUN = DEST / (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ_") +
    uuid.uuid4().hex[:8]
)
RUN.mkdir(exist_ok=False)

write_json(RUN / "FROZEN_PROTOCOL.json", {
    "phase": 18,
    "candidate": "Contrast-Conditional Quantile Residual Model",
    "status": "Final meta-model training; final base learners still pending",
    "source_run": str(SOURCE),
    "seed": SEED,
    "source_cohorts": COHORTS,
    "n_samples": 843,
    "class_mapping": {
        "0": "EGFR mutant",
        "1": "KRAS mutant",
        "2": "EGFR/KRAS double wild type"
    },
    "evidence_channels": [
        "EGFR128", "EGFR512",
        "KRAS128", "KRAS512"
    ],
    "base_evidence": "Outer-LOCO source-held-out predictions from Phase12/14",
    "normalization": "Within-source empirical percentile ranks and logit divided by 2.5",
    "architecture": "Frozen-anchor three-basis bounded contrast-conditional residual",
    "residual_bound": CAP,
    "anchor_regularization": 0.001,
    "residual_l2": L2,
    "optimizer": "Adam",
    "learning_rate": LR,
    "epochs_per_stage": EPOCHS,
    "objective": "Equal-source multiclass NLL",
    "training_folds": "Each source's base evidence generated without training base learners on that source",
    "primary_external_inference": "Unlabeled full-cohort quantile transformation; explicitly transductive",
    "secondary_inference": "Representative frozen unlabeled target-reference bank",
    "target_reference_fraction_for_feasibility": 0.75,
    "external_labels_used_for_training": False,
    "external_test_used": False,
    "historical_development_exposure": True,
    "final_genotype_source_verification": "Pending",
    "final_four_source_base_refit": "Pending",
    "external_validation": "Pending",
    "novelty": "Not established",
    "versions": {
        "python": sys.version,
        "numpy": np.__version__,
        "scipy": scipy.__version__,
        "pandas": pd.__version__,
        "sklearn": sklearn.__version__,
        "torch": torch.__version__
    }
})

pd.DataFrame(sample_records).to_csv(
    RUN / "FOUR_SOURCE_OOF_META_INPUTS.csv",
    index=False
)

xt = torch.as_tensor(X, dtype=torch.float64)
yt = torch.as_tensor(y, dtype=torch.long)
idx = torch.arange(len(y))

masks = [
    torch.as_tensor(groups == cohort)
    for cohort in COHORTS
]

model = FinalResidualModel()

optimizer = torch.optim.Adam(
    (model.bias, model.raw_scale),
    lr=LR
)

for epoch in range(EPOCHS):
    optimizer.zero_grad(set_to_none=True)

    logp = hierarchical_logp(
        model(xt, residual=False)
    )

    losses = -logp[idx, yt]
    source_loss = torch.stack([
        losses[mask].mean()
        for mask in masks
    ])

    objective = (
        source_loss.mean() +
        0.001 *
        F.softplus(model.raw_scale).square().mean()
    )

    if not torch.isfinite(objective):
        raise FloatingPointError(
            f"Anchor stage epoch {epoch}"
        )

    objective.backward()
    optimizer.step()

model.bias.requires_grad_(False)
model.raw_scale.requires_grad_(False)

with torch.no_grad():
    anchor_logits = model(
        xt, residual=False
    )
    anchor_logp = hierarchical_logp(
        anchor_logits
    )
    anchor_losses = -anchor_logp[idx, yt]

optimizer = torch.optim.Adam(
    (model.weights,), lr=LR
)

for epoch in range(EPOCHS):
    optimizer.zero_grad(set_to_none=True)

    logp = hierarchical_logp(
        model(xt, residual=True)
    )

    losses = -logp[idx, yt]
    source_loss = torch.stack([
        losses[mask].mean()
        for mask in masks
    ])

    objective = (
        source_loss.mean() +
        L2 * model.weights.square().mean()
    )

    if not torch.isfinite(objective):
        raise FloatingPointError(
            f"Residual stage epoch {epoch}"
        )

    objective.backward()
    optimizer.step()

with torch.no_grad():
    final_logits = model(
        xt, residual=True
    )

    final_logp = hierarchical_logp(
        final_logits
    )

    final_losses = -final_logp[idx, yt]
    final_p = final_logp.exp().cpu().numpy()

    correction = (
        final_logits - anchor_logits
    ).cpu().numpy()

assert np.isfinite(final_p).all()
assert np.max(np.abs(final_p.sum(axis=1) - 1)) < 1e-12

max_correction = float(
    np.max(np.abs(correction))
)
assert max_correction <= CAP + 1e-10

risk_rows = []

for cohort, mask in zip(COHORTS, masks):
    risk_rows.append({
        "cohort": cohort,
        "n": int(mask.sum()),
        "anchor_training_nll": float(
            anchor_losses[mask].mean()
        ),
        "residual_training_nll": float(
            final_losses[mask].mean()
        )
    })

pd.DataFrame(risk_rows).to_csv(
    RUN / "META_TRAINING_SOURCE_RISK.csv",
    index=False
)

np.savez_compressed(
    RUN / "FINAL_META_MODEL.npz",
    bias=model.bias.detach().cpu().numpy(),
    raw_scale=model.raw_scale.detach().cpu().numpy(),
    weights=model.weights.detach().cpu().numpy(),
    correction_bound=np.float64(CAP),
    quantile_divisor=np.float64(2.5),
    seed=np.int64(SEED),
    source_cohorts=np.asarray(COHORTS)
)

write_json(RUN / "MODEL_DIAGNOSTICS.json", {
    "n": len(y),
    "n_sources": len(COHORTS),
    "trainable_residual_parameters": 6,
    "maximum_absolute_residual_correction": max_correction,
    "anchor_training_nll_equal_source": float(
        np.mean([
            anchor_losses[m].mean().item()
            for m in masks
        ])
    ),
    "residual_training_nll_equal_source": float(
        np.mean([
            final_losses[m].mean().item()
            for m in masks
        ])
    ),
    "training_nll_not_validation": True,
    "source_risks": risk_rows,
    "final_base_learners_available": False
})

write_json(RUN / "RUN_MANIFEST.json", {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "input_files": [{
        "path": str(source_file),
        "sha256": sha256(source_file)
    }],
    "output_files": [{
        "path": p.name,
        "sha256": sha256(p),
        "bytes": p.stat().st_size
    } for p in sorted(RUN.iterdir())
      if p.is_file() and p.name != "RUN_MANIFEST.json"]
})

print("PHASE18: FINAL_META_MODEL_FITTED")
print("SAMPLES:", len(y))
print("SOURCE_COUNTS:", {
    c: int(np.sum(groups == c))
    for c in COHORTS
})
print("META_TRAINING_RISK:")
print(
    pd.DataFrame(risk_rows)
    .round(4)
    .to_string(index=False)
)
print(
    "MAX_ABS_RESIDUAL_CORRECTION:",
    f"{max_correction:.6f}"
)
print("FINAL_BASE_REFIT: PENDING")
print("EXTERNAL_VALIDATION: NOT_STARTED")
print("RUN:", RUN)

PHASE18: FINAL_META_MODEL_FITTED
SAMPLES: 843
SOURCE_COUNTS: {'GSE13213': 117, 'GSE31210': 226, 'GSE32863': 58, 'GSE72094': 442}
META_TRAINING_RISK:
  cohort   n  anchor_training_nll  residual_training_nll
GSE13213 117               0.7839                 0.7580
GSE31210 226               0.8215                 0.8071
GSE32863  58               0.7618                 0.7472
GSE72094 442               0.9020                 0.8906
MAX_ABS_RESIDUAL_CORRECTION: 0.761903
FINAL_BASE_REFIT: PENDING
EXTERNAL_VALIDATION: NOT_STARTED
RUN: D:\LUAD_DATAFIRST_METHOD_2026\18_frozen_method_training\20261023T175603Z_c0d033f9


In [24]:

from pathlib import Path
from datetime import datetime, timezone
import json
import re
import ast
import hashlib
import uuid

ROOT = Path(r"D:\LUAD_DATAFIRST_METHOD_2026")
NOTEBOOK = ROOT / "notebooks" / "00_DATA_FIRST.ipynb"

RUNS = {
    "PHASE11": ROOT / "11_representation_capacity_gate" / "20261023T145952Z_8c40de00",
    "PHASE12": ROOT / "12_multiresolution_ood_evidence" / "20261023T151555Z_638a3c88"
}

PATTERNS = {
    "PHASE11": {
        "11_representation_capacity_gate": 25,
        "REPRESENTATION_CAPACITY_GATE": 20,
        "hier_lr128": 8,
        "hier_lr512": 8,
        "LogisticRegression(": 2
    },
    "PHASE12": {
        "12_multiresolution_ood_evidence": 25,
        "MULTIRESOLUTION_PILOT": 20,
        "BASELINE_REPRODUCTION_MAX_ERROR": 15,
        "cal_disagree_ood": 8,
        "nested_oof": 5
    }
}

CORE = re.compile(
    r"^\s*def\s+.*(rank|select|feature|effect|"
    r"weight|base|fit|scale|model|contrast)|"
    r"LogisticRegression|sample_weight|rankdata|"
    r"argsort|tanh|standardiz|sign_agree|"
    r"selection|relevance|effect_size|"
    r"max_iter|solver\s*=|C\s*=",
    re.IGNORECASE
)

assert NOTEBOOK.is_file(), f"Notebook missing: {NOTEBOOK}"

with NOTEBOOK.open("r", encoding="utf-8-sig") as f:
    notebook = json.load(f)

cells = []

for i, cell in enumerate(notebook.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    source = cell.get("source", "")
    if isinstance(source, list):
        source = "".join(source)

    if source.strip():
        cells.append((i, source))

OUT_ROOT = ROOT / "19_original_implementation_recovery"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

RUN = OUT_ROOT / (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ_") +
    uuid.uuid4().hex[:8]
)
RUN.mkdir(exist_ok=False)

selected = {}

print("SOURCE_RECOVERY: START")
print("NOTEBOOK_CODE_CELLS:", len(cells))
print("NOTEBOOK_SHA256:", hashlib.sha256(
    NOTEBOOK.read_bytes()
).hexdigest())

for phase, markers in PATTERNS.items():
    scored = []

    for index, source in cells:
        score = sum(
            weight
            for token, weight in markers.items()
            if token.lower() in source.lower()
        )

        if score > 0:
            scored.append((score, index, source))

    scored.sort(key=lambda x: (-x[0], x[1]))
    chosen = scored[:2]
    selected[phase] = []

    print(f"\n{phase}_CANDIDATES:", len(scored))

    for score, index, source in chosen:
        path = RUN / f"{phase}_CELL_{index:03d}.txt"
        path.write_text(source, encoding="utf-8")

        selected[phase].append(str(path))

        try:
            tree = ast.parse(source)
            functions = [
                node.name
                for node in ast.walk(tree)
                if isinstance(
                    node, (ast.FunctionDef, ast.AsyncFunctionDef)
                )
            ]
        except SyntaxError:
            functions = ["AST_PARSE_UNAVAILABLE"]

        print(
            "CELL:",
            index,
            "SCORE:", score,
            "LINES:", len(source.splitlines()),
            "SHA256:", hashlib.sha256(
                source.encode("utf-8")
            ).hexdigest()[:16]
        )
        print("FUNCTIONS:", functions)
        print("SOURCE_FILE:", path)

        if (score, index, source) != chosen[0]:
            continue

        lines = source.splitlines()
        interesting = set()

        for j, line in enumerate(lines):
            if CORE.search(line):
                for k in range(
                    max(0, j - 1),
                    min(len(lines), j + 3)
                ):
                    interesting.add(k)

        positions = sorted(interesting)
        limit = 110

        print(f"{phase}_IMPLEMENTATION_EXCERPT:")

        for j in positions[:limit]:
            print(f"{j + 1:04d}: {lines[j][:180]}")

        if len(positions) > limit:
            print(
                "ADDITIONAL_MATCHED_LINES:",
                len(positions) - limit
            )

artifact_inventory = {}

for phase, directory in RUNS.items():
    if not directory.is_dir():
        artifact_inventory[phase] = {
            "status": "MISSING",
            "path": str(directory)
        }
        continue

    files = sorted(
        str(p.relative_to(directory))
        for p in directory.rglob("*")
        if p.is_file()
    )

    artifact_inventory[phase] = {
        "status": "FOUND",
        "path": str(directory),
        "files": files
    }

    print(f"\n{phase}_ARTIFACT_COUNT:", len(files))
    print(f"{phase}_ARTIFACTS:")

    for filename in files[:45]:
        print(" ", filename)

    protocol = directory / "PROTOCOL.json"
    if protocol.is_file():
        try:
            obj = json.loads(
                protocol.read_text(encoding="utf-8")
            )
            print(f"{phase}_PROTOCOL_KEYS:", list(obj))
        except (ValueError, UnicodeError) as exc:
            print(f"{phase}_PROTOCOL_READ_ERROR:", str(exc))

report = {
    "notebook": str(NOTEBOOK),
    "notebook_sha256": hashlib.sha256(
        NOTEBOOK.read_bytes()
    ).hexdigest(),
    "selected_code_cells": selected,
    "artifacts": artifact_inventory,
    "execution": "READ_ONLY_SOURCE_INSPECTION",
    "model_training": False,
    "external_validation": False
}

(RUN / "SOURCE_AUDIT.json").write_text(
    json.dumps(
        report,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("\nSOURCE_RECOVERY: COMPLETE")
print("RUN:", RUN)

SOURCE_RECOVERY: START
NOTEBOOK_CODE_CELLS: 0
NOTEBOOK_SHA256: 92b25ee98d8bdd3e20d99909d75b3df12e27dae8e6b3f727e3c2f28709e1c0fe

PHASE11_CANDIDATES: 0

PHASE12_CANDIDATES: 0

PHASE11_ARTIFACT_COUNT: 6
PHASE11_ARTIFACTS:
  EQUAL_COHORT_SUMMARY.json
  LOCO_METRICS.csv
  LOCO_PREDICTIONS.csv
  PROTOCOL.json
  RUN_MANIFEST.json
  TRAIN_ONLY_FEATURES.json
PHASE11_PROTOCOL_KEYS: ['phase', 'source', 'development_cohorts', 'historically_exposed_cohort', 'external_reserved', 'per_head_gene_counts', 'models', 'primary_metric', 'secondary_metrics', 'training_only_feature_selection', 'shared_gene_schema_previously_harmonized', 'source_genotype_provenance', 'historical_development_exposure', 'method_novelty']

PHASE12_ARTIFACT_COUNT: 34
PHASE12_ARTIFACTS:
  CALIBRATION_PARAMETERS.csv
  EQUAL_COHORT_SUMMARY.json
  LOCO_METRICS.csv
  LOCO_PREDICTIONS.csv
  PROTOCOL.json
  RUN_MANIFEST.json
  calibrators\GSE13213_cal128_ood.pt
  calibrators\GSE13213_cal_disagree_insample.pt
  calibrators\GSE13213_cal_

In [27]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import shutil
import uuid

import numpy as np
import pandas as pd
from scipy.special import expit
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression

ROOT = Path(r"D:\LUAD_DATAFIRST_METHOD_2026")
SOURCE = Path(r"D:\LUAD_CMR_2026\CELL03_DATA_20261021T090947_420335Z_d6aed8ec\cmr_gene_expression.npz")
P12 = ROOT / "12_multiresolution_ood_evidence" / "20261023T151555Z_638a3c88"
P18 = ROOT / "18_frozen_method_training" / "20261023T175603Z_c0d033f9"
OUT = ROOT / "19_exact_source_base_refit"
COHORTS = ("GSE13213", "GSE31210", "GSE32863", "GSE72094")
EXPECTED = {
    "GSE13213": (45, 15, 57),
    "GSE31210": (127, 20, 79),
    "GSE32863": (17, 22, 19),
    "GSE72094": (47, 154, 241),
}
RESOLUTIONS = (128, 512)
SEED = 20261023
ARCHIVE_TOL = 1e-4
ROUNDTRIP_TOL = 2e-5


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(4 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def write_json(path, value):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(value, f, indent=2, ensure_ascii=False, allow_nan=False)


def hierarchical_probability(scores):
    scores = np.asarray(scores, dtype=np.float64)
    if scores.ndim != 2 or scores.shape[1] != 2 or not np.isfinite(scores).all():
        raise RuntimeError("Invalid hierarchical decision scores")
    e = expit(scores[:, 0])
    k = expit(scores[:, 1])
    p = np.column_stack((e, (1.0 - e) * k, (1.0 - e) * (1.0 - k)))
    if not np.isfinite(p).all() or not np.allclose(p.sum(axis=1), 1.0, rtol=1e-12, atol=1e-12):
        raise RuntimeError("Invalid hierarchical probabilities")
    return p


def select_orders(training_ids):
    names = sorted(set(cohorts[training_ids].tolist()))
    if not 2 <= len(names) <= 4:
        raise RuntimeError("Unexpected source count")
    orders = {}
    for contrast in ("EGFR", "KRAS"):
        effects = []
        for name in names:
            ix = training_ids[cohorts[training_ids] == name]
            r, t = ranks[ix], y[ix]
            positive = r[t == (0 if contrast == "EGFR" else 1)]
            negative = r[t != 0] if contrast == "EGFR" else r[t == 2]
            if min(len(positive), len(negative)) < 10:
                raise RuntimeError(f"Insufficient contrast support: {name}/{contrast}")
            effect = (positive.mean(axis=0) - negative.mean(axis=0)) / np.maximum(r.std(axis=0), 0.04)
            effects.append(effect)
        effect = np.stack(effects)
        agreement = np.abs(np.sign(effect).sum(axis=0)) / len(names)
        strength = np.abs(np.median(effect, axis=0)) * (0.2 + 0.8 * agreement)
        if not np.isfinite(strength).all():
            raise RuntimeError(f"Nonfinite feature strength: {contrast}")
        orders[contrast] = np.argsort(-strength, kind="stable")
    return orders


def balanced_weights(binary, domains):
    w = np.zeros(len(binary), dtype=np.float64)
    for cohort in sorted(set(domains.tolist())):
        for cls in (0, 1):
            mask = (domains == cohort) & (binary == cls)
            if int(mask.sum()) < 10:
                raise RuntimeError(f"Insufficient weighted stratum: {cohort}/{cls}")
            w[mask] = 1.0 / mask.sum()
    w *= len(binary) / w.sum()
    return w


def fit_bases(training_ids, target_ids):
    orders = select_orders(training_ids)
    scores = np.empty((len(target_ids), 4), dtype=np.float64)
    models = {}
    labels = y[training_ids]
    non_e = labels != 0
    for resolution in RESOLUTIONS:
        columns = np.sort(np.unique(np.r_[orders["EGFR"][:resolution], orders["KRAS"][:resolution]]))
        training = ranks[training_ids][:, columns]
        mu = training.mean(axis=0)
        sd = np.maximum(training.std(axis=0), 0.05)
        xt = np.tanh((training - mu) / (2.0 * sd)).astype(np.float32)
        xv = np.tanh((ranks[target_ids][:, columns] - mu) / (2.0 * sd)).astype(np.float32)
        target_e = (labels == 0).astype(int)
        target_k = (labels[non_e] == 1).astype(int)
        e = LogisticRegression(C=0.2, solver="lbfgs", max_iter=400, random_state=SEED)
        k = LogisticRegression(C=0.2, solver="lbfgs", max_iter=400, random_state=SEED)
        e.fit(xt, target_e, sample_weight=balanced_weights(target_e, cohorts[training_ids]))
        k.fit(xt[non_e], target_k, sample_weight=balanced_weights(target_k, cohorts[training_ids][non_e]))
        j = 0 if resolution == 128 else 1
        scores[:, j] = e.decision_function(xv)
        scores[:, j + 2] = k.decision_function(xv)
        models[str(resolution)] = {
            "gene_indices": columns,
            "genes": genes[columns],
            "mu": mu,
            "sd": sd,
            "e_coef": e.coef_.ravel(),
            "e_intercept": e.intercept_,
            "k_coef": k.coef_.ravel(),
            "k_intercept": k.intercept_,
        }
    return scores, models


def load_state(path):
    fields = ("gene_indices", "genes", "mu", "sd", "e_coef", "e_intercept", "k_coef", "k_intercept")
    with np.load(path, allow_pickle=False) as z:
        if not set(fields).issubset(z.files):
            raise RuntimeError(f"Checkpoint fields missing: {path}")
        state = {k: np.array(z[k], copy=True) for k in fields}
    cols = state["gene_indices"]
    if cols.ndim != 1 or cols.size == 0 or np.any(cols < 0) or np.any(cols >= len(genes)):
        raise RuntimeError(f"Invalid feature indices: {path}")
    if not np.array_equal(cols, np.unique(cols)) or not np.array_equal(state["genes"].astype(str), genes[cols]):
        raise RuntimeError(f"Feature ID mismatch: {path}")
    for field in ("mu", "sd", "e_coef", "k_coef"):
        if state[field].shape != (len(cols),) or not np.isfinite(state[field]).all():
            raise RuntimeError(f"Bad {field}: {path}")
    for field in ("e_intercept", "k_intercept"):
        if state[field].size != 1 or not np.isfinite(state[field]).all():
            raise RuntimeError(f"Bad {field}: {path}")
    if np.any(state["sd"] <= 0):
        raise RuntimeError(f"Nonpositive scales: {path}")
    return state


def replay_scores(state, indices):
    z = np.tanh((ranks[indices][:, state["gene_indices"]] - state["mu"]) / (2 * state["sd"])).astype(np.float32)
    return np.column_stack((
        z @ state["e_coef"] + state["e_intercept"],
        z @ state["k_coef"] + state["k_intercept"],
    ))


needed = [SOURCE, ROOT / "00_admin" / "project.json", P12 / "LOCO_PREDICTIONS.csv",
          P18 / "FINAL_META_MODEL.npz", P18 / "FOUR_SOURCE_OOF_META_INPUTS.csv"]
needed += [P12 / "outer_models" / f"{c}_lr{r}.npz" for c in COHORTS for r in RESOLUTIONS]
for p in needed:
    if not p.is_file():
        raise FileNotFoundError(str(p))
identity = json.loads((ROOT / "00_admin" / "project.json").read_text(encoding="utf-8"))
if identity.get("project_id") != "LUAD_DATAFIRST_METHOD_2026":
    raise RuntimeError("Project identity mismatch")
with np.load(P18 / "FINAL_META_MODEL.npz", allow_pickle=False) as z:
    for field, shape in (("bias", (2,)), ("raw_scale", (2,)), ("weights", (2, 3))):
        if field not in z.files or z[field].shape != shape or not np.isfinite(z[field]).all():
            raise RuntimeError(f"Invalid Phase18 meta-head: {field}")
with np.load(SOURCE, allow_pickle=False) as z:
    X = np.asarray(z["X"], dtype=np.float32)
    y = np.asarray(z["y"], dtype=np.int64)
    cohorts = np.char.upper(np.char.strip(z["cohort"].astype(str)))
    gsms = np.char.upper(np.char.strip(z["gsm"].astype(str)))
    genes = np.char.upper(np.char.strip(z["genes"].astype(str)))
    patients = np.char.strip(z["patient_ids"].astype(str))
if X.shape != (843, 13016) or not np.isfinite(X).all():
    raise RuntimeError("Expression source shape or values mismatch")
if set(cohorts) != set(COHORTS) or set(y) != {0, 1, 2}:
    raise RuntimeError("Cohort/label mismatch")
if len(set(zip(cohorts, gsms))) != 843 or len(set(genes)) != 13016:
    raise RuntimeError("Duplicate sample or gene")
for cohort in COHORTS:
    observed = tuple(int(((cohorts == cohort) & (y == j)).sum()) for j in (0, 1, 2))
    if observed != EXPECTED[cohort]:
        raise RuntimeError(f"Incorrect cohort counts: {cohort} {observed}")
patient_sources = {}
for cohort, pid in zip(cohorts, patients):
    if pid and pid.lower() not in {"na", "nan", "none", "unknown"}:
        patient_sources.setdefault(pid, set()).add(cohort)
if any(len(v) > 1 for v in patient_sources.values()):
    raise RuntimeError("Cross-cohort patient overlap")

meta_roster = pd.read_csv(P18 / "FOUR_SOURCE_OOF_META_INPUTS.csv", dtype={"cohort": str, "gsm": str})
if not {"cohort", "gsm", "label"}.issubset(meta_roster.columns):
    raise RuntimeError("Phase18 roster schema mismatch")
prior_roster = {(str(r.cohort).upper().strip(), str(r.gsm).upper().strip()): int(r.label)
                for r in meta_roster.itertuples(index=False)}
actual_roster = {(c, g): int(label) for c, g, label in zip(cohorts, gsms, y)}
if len(prior_roster) != 843 or prior_roster != actual_roster:
    raise RuntimeError("Phase18 roster mismatch")

ref = pd.read_csv(P12 / "LOCO_PREDICTIONS.csv", dtype={"gsm": str})
required = {"held_cohort", "gsm", "model", "true_class", "p_egfr", "p_kras", "p_dwt"}
if not required.issubset(ref.columns):
    raise RuntimeError("Phase12 prediction schema mismatch")
ref = ref.loc[ref["model"].isin(("raw128", "raw512"))].copy()
ref["gsm"] = ref["gsm"].astype(str).str.upper().str.strip()
for c in ("p_egfr", "p_kras", "p_dwt"):
    ref[c] = pd.to_numeric(ref[c], errors="raise")
if len(ref) != 1686 or ref.duplicated(["held_cohort", "gsm", "model"]).any():
    raise RuntimeError("Phase12 prediction row mismatch")
ref_lookup = ref.set_index(["held_cohort", "gsm", "model"])

ranks = (rankdata(X, axis=1, method="average").astype(np.float32) - 1) / (X.shape[1] - 1)
del X

OUT.mkdir(parents=True, exist_ok=True)
RUN = OUT / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ_") + uuid.uuid4().hex[:8])
RUN.mkdir(exist_ok=False)
write_json(RUN / "PROTOCOL.json", {
    "phase": "19_original_final_refit", "cohorts": COHORTS, "resolutions": RESOLUTIONS,
    "source": str(SOURCE), "phase12": str(P12), "phase18": str(P18),
    "original_notebook_html_sha256": "62b307f1c09033cbba52688aec12abddc01fc896efed9849d95482c27d069dca",
    "archive_probability_tolerance": ARCHIVE_TOL, "final_score_roundtrip_tolerance": ROUNDTRIP_TOL,
    "float32_feature_path": True, "float64_probability_validation": True,
    "external_validation": "NOT_STARTED", "novelty": "UNVERIFIED"
})

checks = []
for held in COHORTS:
    train = np.flatnonzero(cohorts != held)
    test = np.flatnonzero(cohorts == held)
    orders = select_orders(train)
    for resolution in RESOLUTIONS:
        state = load_state(P12 / "outer_models" / f"{held}_lr{resolution}.npz")
        expected_columns = np.sort(np.unique(np.r_[orders["EGFR"][:resolution], orders["KRAS"][:resolution]]))
        feature_match = np.array_equal(state["gene_indices"], expected_columns)
        old_train = ranks[train][:, state["gene_indices"]]
        mu_err = float(np.max(np.abs(old_train.mean(axis=0) - state["mu"])))
        sd_err = float(np.max(np.abs(np.maximum(old_train.std(axis=0), 0.05) - state["sd"])))
        keys = [(str(cohorts[i]), str(gsms[i]), f"raw{resolution}") for i in test]
        if any(k not in ref_lookup.index for k in keys):
            raise RuntimeError(f"Missing archived patients: {held}/{resolution}")
        archived_y = ref_lookup.loc[keys, "true_class"].to_numpy(dtype=int)
        if not np.array_equal(archived_y, y[test]):
            raise RuntimeError(f"Archived label mismatch: {held}/{resolution}")
        archived_p = ref_lookup.loc[keys, ["p_egfr", "p_kras", "p_dwt"]].to_numpy(dtype=float)
        replay_p = hierarchical_probability(replay_scores(state, test))
        diff = float(np.max(np.abs(replay_p - archived_p)))
        ok = feature_match and mu_err <= 1e-5 and sd_err <= 1e-5 and diff <= ARCHIVE_TOL
        checks.append({"held": held, "resolution": resolution, "feature_match": bool(feature_match),
                       "mu_max_error": mu_err, "sd_max_error": sd_err, "probability_max_error": diff, "pass": bool(ok)})
        print(f"CHECKPOINT {held} {resolution} selected={len(np.intersect1d(expected_columns, state['gene_indices']))}/{len(expected_columns)} mu={mu_err:.2e} sd={sd_err:.2e} probability={diff:.2e} {'PASS' if ok else 'FAIL'}")
        if not ok:
            pd.DataFrame(checks).to_csv(RUN / "REPRODUCTION.csv", index=False)
            write_json(RUN / "GATE_STATUS.json", {"status": "CHECKPOINT_REPRODUCTION_FAILED", "final_saved": False})
            raise RuntimeError(f"Archived checkpoint mismatch: {held}/{resolution}")

pd.DataFrame(checks).to_csv(RUN / "REPRODUCTION.csv", index=False)
full = np.arange(len(y), dtype=np.int64)
full_scores, final_models = fit_bases(full, full)
checks_final = []
for resolution in RESOLUTIONS:
    col = 0 if resolution == 128 else 1
    state = final_models[str(resolution)]
    rebuilt = replay_scores(state, full)
    original = full_scores[:, [col, col + 2]]
    score_err = float(np.max(np.abs(rebuilt.astype(np.float64) - original)))
    prob_err = float(np.max(np.abs(hierarchical_probability(rebuilt) - hierarchical_probability(original))))
    ok = score_err <= ROUNDTRIP_TOL and prob_err <= ROUNDTRIP_TOL
    checks_final.append({"resolution": resolution, "genes": int(len(state["gene_indices"])),
                         "score_max_error": score_err, "probability_max_error": prob_err, "pass": bool(ok)})
    print(f"FINAL_VERIFY {resolution} genes={len(state['gene_indices'])} score_error={score_err:.3e} probability_error={prob_err:.3e} {'PASS' if ok else 'FAIL'}")
    if not ok:
        write_json(RUN / "GATE_STATUS.json", {"status": "FINAL_ROUNDTRIP_FAILED", "checks": checks_final, "final_saved": False})
        raise RuntimeError(f"Unexpected final model numerical mismatch: {resolution}")

for resolution in RESOLUTIONS:
    path = RUN / f"FINAL_lr{resolution}.npz"
    np.savez_compressed(path, **final_models[str(resolution)])
    recovered = load_state(path)
    for field, value in final_models[str(resolution)].items():
        if not np.array_equal(np.asarray(recovered[field]), np.asarray(value)):
            raise RuntimeError(f"Saved checkpoint array mismatch: {resolution}/{field}")
    print(f"FINAL_MODEL {resolution} saved={path.name} PASS")

shutil.copy2(P18 / "FINAL_META_MODEL.npz", RUN / "FINAL_META_MODEL.npz")
if sha256(P18 / "FINAL_META_MODEL.npz") != sha256(RUN / "FINAL_META_MODEL.npz"):
    raise RuntimeError("Phase18 meta model copy mismatch")

pd.DataFrame(checks_final).to_csv(RUN / "FINAL_ROUNDTRIP.csv", index=False)
write_json(RUN / "GATE_STATUS.json", {
    "status": "PASS_FINAL_BASE_REFIT", "checkpoint_checks": len(checks),
    "checkpoint_max_probability_error": max(x["probability_max_error"] for x in checks),
    "final_roundtrip_max_score_error": max(x["score_max_error"] for x in checks_final),
    "models": ["FINAL_lr128.npz", "FINAL_lr512.npz", "FINAL_META_MODEL.npz"],
    "external_validation": "NOT_STARTED", "genotype_provenance": "PENDING", "novelty": "UNVERIFIED"
})
write_json(RUN / "RUN_MANIFEST.json", {
    "utc": datetime.now(timezone.utc).isoformat(),
    "inputs": [{"path": str(p), "sha256": sha256(p)} for p in needed],
    "outputs": [{"file": p.name, "sha256": sha256(p)} for p in sorted(RUN.iterdir()) if p.is_file() and p.name != "RUN_MANIFEST.json"]
})
print("PHASE19: COMPLETE")
print("ARCHIVED_REPLAY_MAX_ERROR:", f"{max(x['probability_max_error'] for x in checks):.3e}")
print("FINAL_SCORE_ROUNDTRIP_MAX_ERROR:", f"{max(x['score_max_error'] for x in checks_final):.3e}")
print("FINAL_BASE_MODELS: FINAL_lr128.npz, FINAL_lr512.npz")
print("FINAL_META_MODEL: COPIED_UNCHANGED")
print("EXTERNAL_VALIDATION: NOT_STARTED")
print("RUN:", RUN)

CHECKPOINT GSE13213 128 selected=253/253 mu=0.00e+00 sd=0.00e+00 probability=1.11e-16 PASS
CHECKPOINT GSE13213 512 selected=970/970 mu=0.00e+00 sd=0.00e+00 probability=1.11e-16 PASS
CHECKPOINT GSE31210 128 selected=249/249 mu=0.00e+00 sd=0.00e+00 probability=2.00e-08 PASS
CHECKPOINT GSE31210 512 selected=978/978 mu=0.00e+00 sd=0.00e+00 probability=1.11e-16 PASS
CHECKPOINT GSE32863 128 selected=248/248 mu=0.00e+00 sd=0.00e+00 probability=1.68e-07 PASS
CHECKPOINT GSE32863 512 selected=968/968 mu=0.00e+00 sd=0.00e+00 probability=3.49e-07 PASS
CHECKPOINT GSE72094 128 selected=252/252 mu=0.00e+00 sd=0.00e+00 probability=2.43e-07 PASS
CHECKPOINT GSE72094 512 selected=971/971 mu=0.00e+00 sd=0.00e+00 probability=1.11e-16 PASS
FINAL_VERIFY 128 genes=250 score_error=0.000e+00 probability_error=0.000e+00 PASS
FINAL_VERIFY 512 genes=959 score_error=0.000e+00 probability_error=0.000e+00 PASS
FINAL_MODEL 128 saved=FINAL_lr128.npz PASS
FINAL_MODEL 512 saved=FINAL_lr512.npz PASS
PHASE19: COMPLETE
ARCH

In [28]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import uuid

import numpy as np
import pandas as pd
from scipy.special import expit
from scipy.stats import rankdata

ROOT = Path(r"D:\LUAD_DATAFIRST_METHOD_2026")
MODEL_RUN = ROOT / "19_exact_source_base_refit" / "20261023T190611Z_a22f33e7"
SOURCE = Path(
    r"D:\LUAD_CMR_2026\CELL03_DATA_20261021T090947_420335Z_d6aed8ec\cmr_gene_expression.npz"
)
OUT_ROOT = ROOT / "20_integrated_inference_verification"

EXPECTED_COHORTS = (
    "GSE13213", "GSE31210", "GSE32863", "GSE72094"
)
CLASS_NAMES = (
    "EGFR_MUT", "KRAS_MUT", "EGFR_KRAS_DOUBLE_WT"
)
SEED = 20261023
MIN_BATCH = 25
NUMERIC_TOL = 2e-5


def hash_file(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def write_json(path, data):
    path.write_text(
        json.dumps(
            data, indent=2,
            ensure_ascii=False, allow_nan=False
        ),
        encoding="utf-8"
    )


def gene_keys(values):
    return np.char.upper(
        np.char.strip(
            np.asarray(values).astype(str)
        )
    )


def read_base(path):
    required = (
        "gene_indices", "genes", "mu", "sd",
        "e_coef", "e_intercept",
        "k_coef", "k_intercept"
    )

    with np.load(path, allow_pickle=False) as z:
        if not set(required).issubset(z.files):
            raise ValueError(
                f"Incomplete checkpoint: {path}"
            )

        state = {
            key: np.array(z[key], copy=True)
            for key in required
        }

    indices = state["gene_indices"]

    if (
        indices.ndim != 1
        or len(indices) == 0
        or not np.array_equal(
            indices, np.unique(indices)
        )
    ):
        raise ValueError(
            f"Invalid gene indices: {path}"
        )

    n = len(indices)

    for name in ("mu", "sd", "e_coef", "k_coef"):
        if (
            state[name].shape != (n,)
            or not np.isfinite(state[name]).all()
        ):
            raise ValueError(
                f"Invalid checkpoint array: {name}/{path}"
            )

    for name in ("e_intercept", "k_intercept"):
        if (
            state[name].size != 1
            or not np.isfinite(state[name]).all()
        ):
            raise ValueError(
                f"Invalid checkpoint intercept: {name}/{path}"
            )

    if np.any(state["sd"] <= 0):
        raise ValueError(
            f"Nonpositive SD: {path}"
        )

    return state


def score_to_probability(e, k):
    e = np.asarray(e, dtype=np.float64)
    k = np.asarray(k, dtype=np.float64)

    pe = expit(e)
    pk = expit(k)

    p = np.column_stack((
        pe,
        (1 - pe) * pk,
        (1 - pe) * (1 - pk)
    ))

    if (
        not np.isfinite(p).all()
        or not np.allclose(
            p.sum(axis=1), 1,
            atol=1e-12, rtol=1e-12
        )
    ):
        raise ArithmeticError(
            "Invalid multiclass probabilities"
        )

    return p


def base_evidence(ranks, models):
    result = np.empty(
        (len(ranks), 4),
        dtype=np.float64
    )

    for capacity in (128, 512):
        model = models[capacity]
        chosen = model["gene_indices"]

        z = np.tanh(
            (
                ranks[:, chosen] - model["mu"]
            ) / (2.0 * model["sd"])
        ).astype(np.float32)

        e = (
            z @ model["e_coef"]
            + model["e_intercept"]
        )
        k = (
            z @ model["k_coef"]
            + model["k_intercept"]
        )

        p = score_to_probability(e, k)

        pe = np.clip(
            p[:, 0], 1e-12, 1 - 1e-12
        )
        pkr = np.clip(
            p[:, 1], 1e-12, 1
        )
        pwt = np.clip(
            p[:, 2], 1e-12, 1
        )

        column = (
            0 if capacity == 128 else 1
        )

        result[:, column] = np.log(
            pe / (1 - pe)
        )

        result[:, column + 2] = np.log(
            pkr / pwt
        )

    if not np.isfinite(result).all():
        raise ArithmeticError(
            "Nonfinite base evidence"
        )

    return result


def quantile_features(raw, cohort_ids):
    cohort_ids = np.asarray(
        cohort_ids
    ).astype(str)

    if len(raw) != len(cohort_ids):
        raise ValueError(
            "Evidence and batch labels differ"
        )

    q = np.empty_like(
        raw, dtype=np.float64
    )

    for cohort in np.unique(cohort_ids):
        positions = np.flatnonzero(
            cohort_ids == cohort
        )

        n = len(positions)

        if n < MIN_BATCH:
            raise ValueError(
                f"Full-cohort quantiles require at least "
                f"{MIN_BATCH} samples: {cohort}"
            )

        for channel in range(4):
            u = (
                rankdata(
                    raw[positions, channel],
                    method="average"
                ) - 0.5
            ) / n

            u = np.clip(
                u, 1e-5, 1 - 1e-5
            )

            q[positions, channel] = (
                np.log(u) - np.log1p(-u)
            ) / 2.5

    return q


def meta_predict(q, meta):
    a = q[:, (0, 2)]
    d = q[:, (1, 3)] - a

    features = np.stack((
        d,
        d * np.tanh(1.5 * a),
        np.tanh(2 * d) * (
            1 - np.tanh(a) ** 2
        )
    ), axis=-1)

    correction = np.einsum(
        "nhk,hk->nh",
        features,
        meta["weights"]
    )

    z = (
        meta["bias"]
        + np.logaddexp(
            0.0, meta["raw_scale"]
        ) * a
        + 1.5 * np.tanh(
            correction / 1.5
        )
    )

    p = score_to_probability(
        z[:, 0], z[:, 1]
    )

    return p, z


def predict_expression(
    expression,
    incoming_genes,
    batch_ids,
    sample_ids,
    expected_genes,
    models,
    meta
):
    expression = np.asarray(
        expression, dtype=np.float32
    )

    batch_ids = np.asarray(
        batch_ids
    ).astype(str)

    sample_ids = np.asarray(
        sample_ids
    ).astype(str)

    incoming = gene_keys(
        incoming_genes
    )

    if (
        expression.ndim != 2
        or expression.shape[1] != len(incoming)
    ):
        raise ValueError(
            "Expression dimensions do not match gene columns"
        )

    if (
        expression.shape[0] != len(batch_ids)
        or expression.shape[0] != len(sample_ids)
    ):
        raise ValueError(
            "Sample, batch, and expression rows do not match"
        )

    if (
        len(incoming) != len(expected_genes)
        or len(set(incoming)) != len(incoming)
    ):
        raise ValueError(
            "Gene schema length or uniqueness mismatch"
        )

    if (
        len(set(zip(batch_ids, sample_ids)))
        != len(sample_ids)
    ):
        raise ValueError(
            "Duplicate sample identifiers within batch"
        )

    if not np.isfinite(expression).all():
        raise ValueError(
            "Nonfinite expression matrix"
        )

    mapping = {
        gene: i
        for i, gene in enumerate(incoming)
    }

    missing = [
        gene
        for gene in expected_genes
        if gene not in mapping
    ]

    if missing:
        raise ValueError(
            f"Missing required genes: {missing[:5]}"
        )

    order = np.array(
        [
            mapping[gene]
            for gene in expected_genes
        ],
        dtype=np.int64
    )

    aligned = expression[:, order]

    ranks = (
        rankdata(
            aligned,
            axis=1,
            method="average"
        ).astype(np.float32) - 1
    ) / (len(expected_genes) - 1)

    evidence = base_evidence(
        ranks, models
    )

    q = quantile_features(
        evidence, batch_ids
    )

    p, z = meta_predict(
        q, meta
    )

    frame = pd.DataFrame({
        "cohort": batch_ids,
        "gsm": sample_ids,
        "p_egfr": p[:, 0],
        "p_kras": p[:, 1],
        "p_dwt": p[:, 2],
        "predicted_class": p.argmax(axis=1)
    })

    return frame, evidence, q, z


required_files = [
    SOURCE,
    MODEL_RUN / "FINAL_lr128.npz",
    MODEL_RUN / "FINAL_lr512.npz",
    MODEL_RUN / "FINAL_META_MODEL.npz",
    MODEL_RUN / "GATE_STATUS.json"
]

for path in required_files:
    if not path.is_file():
        raise FileNotFoundError(str(path))


gate = json.loads(
    (
        MODEL_RUN / "GATE_STATUS.json"
    ).read_text(encoding="utf-8")
)

if gate.get("status") != "PASS_FINAL_BASE_REFIT":
    raise RuntimeError(
        "Phase19 run is not verified"
    )


with np.load(
    SOURCE,
    allow_pickle=False
) as z:
    X = np.asarray(
        z["X"], dtype=np.float32
    )

    source_genes = gene_keys(
        z["genes"]
    )
    source_cohorts = gene_keys(
        z["cohort"]
    )
    source_gsm = gene_keys(
        z["gsm"]
    )


if (
    X.shape != (843, 13016)
    or set(source_cohorts) != set(EXPECTED_COHORTS)
):
    raise RuntimeError(
        "Unexpected development source"
    )

if len(set(source_genes)) != len(source_genes):
    raise RuntimeError(
        "Nonunique training genes"
    )


models = {
    capacity: read_base(
        MODEL_RUN / f"FINAL_lr{capacity}.npz"
    )
    for capacity in (128, 512)
}

for capacity, model in models.items():
    indices = model["gene_indices"]

    if (
        np.any(indices < 0)
        or np.any(indices >= len(source_genes))
    ):
        raise RuntimeError(
            f"Out-of-bounds features: {capacity}"
        )

    if not np.array_equal(
        gene_keys(model["genes"]),
        source_genes[indices]
    ):
        raise RuntimeError(
            f"Saved gene mapping mismatch: {capacity}"
        )


with np.load(
    MODEL_RUN / "FINAL_META_MODEL.npz",
    allow_pickle=False
) as z:
    meta = {
        key: np.asarray(
            z[key], dtype=np.float64
        )
        for key in (
            "bias", "raw_scale", "weights"
        )
    }


if (
    meta["bias"].shape != (2,)
    or meta["raw_scale"].shape != (2,)
    or meta["weights"].shape != (2, 3)
):
    raise RuntimeError(
        "Meta-model schema mismatch"
    )

if not all(
    np.isfinite(value).all()
    for value in meta.values()
):
    raise RuntimeError(
        "Nonfinite meta-model parameters"
    )


predictions, evidence, q, logits = predict_expression(
    X,
    source_genes,
    source_cohorts,
    source_gsm,
    source_genes,
    models,
    meta
)

probabilities = predictions[
    ["p_egfr", "p_kras", "p_dwt"]
].to_numpy()


if not np.allclose(
    probabilities.sum(axis=1),
    1.0,
    atol=1e-12,
    rtol=1e-12
):
    raise RuntimeError(
        "Invalid integrated probability sums"
    )


rng = np.random.default_rng(SEED)
permutation = rng.permutation(len(X))

permuted_quantiles = quantile_features(
    evidence[permutation],
    source_cohorts[permutation]
)

permuted_probabilities, _ = meta_predict(
    permuted_quantiles, meta
)

row_invariance_error = float(
    np.max(np.abs(
        permuted_probabilities
        - probabilities[permutation]
    ))
)

if row_invariance_error > 1e-12:
    raise RuntimeError(
        f"Row-order invariance failed: "
        f"{row_invariance_error}"
    )


sample = np.flatnonzero(
    source_cohorts == EXPECTED_COHORTS[0]
)[:30]

gene_permutation = rng.permutation(
    len(source_genes)
)

part_a, _, _, _ = predict_expression(
    X[sample],
    source_genes,
    source_cohorts[sample],
    source_gsm[sample],
    source_genes,
    models,
    meta
)

part_b, _, _, _ = predict_expression(
    X[sample][:, gene_permutation],
    source_genes[gene_permutation],
    source_cohorts[sample],
    source_gsm[sample],
    source_genes,
    models,
    meta
)

gene_invariance_error = float(
    np.max(np.abs(
        part_a[
            ["p_egfr", "p_kras", "p_dwt"]
        ].to_numpy()
        - part_b[
            ["p_egfr", "p_kras", "p_dwt"]
        ].to_numpy()
    ))
)

if gene_invariance_error > NUMERIC_TOL:
    raise RuntimeError(
        f"Gene-column alignment failed: "
        f"{gene_invariance_error}"
    )


OUT_ROOT.mkdir(
    parents=True, exist_ok=True
)

RUN = OUT_ROOT / (
    datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%SZ_"
    ) + uuid.uuid4().hex[:8]
)
RUN.mkdir(exist_ok=False)


predictions.insert(
    0,
    "role",
    "IN_SAMPLE_INFERENCE_QA_ONLY"
)

predictions.to_csv(
    RUN / "IN_SAMPLE_QA_PREDICTIONS.csv",
    index=False
)


summary = predictions.groupby(
    "cohort"
).agg(
    n=("gsm", "size"),
    predicted_egfr=(
        "predicted_class",
        lambda z: int((z == 0).sum())
    ),
    predicted_kras=(
        "predicted_class",
        lambda z: int((z == 1).sum())
    ),
    predicted_dwt=(
        "predicted_class",
        lambda z: int((z == 2).sum())
    )
).reset_index()

summary.to_csv(
    RUN / "PREDICTION_DISTRIBUTION_QA_ONLY.csv",
    index=False
)


write_json(RUN / "INFERENCE_CONTRACT.json", {
    "status": "FROZEN_INTEGRATED_INFERENCE_VERIFIED",
    "source_model_run": str(MODEL_RUN),
    "candidate": "Contrast-conditional quantile residual classifier",
    "classes": {
        "0": CLASS_NAMES[0],
        "1": CLASS_NAMES[1],
        "2": CLASS_NAMES[2]
    },
    "score_order": [
        "EGFR128",
        "EGFR512",
        "KRAS128",
        "KRAS512"
    ],
    "gene_count": int(len(source_genes)),
    "rank": "(within-sample average rank - 1)/(n_genes - 1), float32",
    "scaling": "tanh((rank-saved_mu)/(2*saved_sd)), float32",
    "evidence": "Phase18 conditional-logit extraction with 1e-12 clipping",
    "quantile": "Within-target-cohort percentile logit divided by 2.5",
    "head": "Phase18 frozen three-basis bounded conditional residual",
    "primary_inference": "Complete unlabeled target cohort transductive inference",
    "minimum_batch_size": MIN_BATCH,
    "gene_alignment": "Exact frozen 13016-gene universe",
    "development_data_role": "Technical QA only; not independent validation",
    "external_genotype_eligibility": "PENDING",
    "novelty": "NOT_ESTABLISHED"
})


write_json(RUN / "INFERENCE_AUDIT.json", {
    "n_samples": int(len(predictions)),
    "gene_count": int(len(source_genes)),
    "cohort_counts": {
        str(row.cohort): int(row.n)
        for row in summary.itertuples(index=False)
    },
    "max_row_order_probability_error": row_invariance_error,
    "max_gene_column_order_probability_error": gene_invariance_error,
    "probability_sum_max_error": float(
        np.max(np.abs(
            probabilities.sum(axis=1) - 1
        ))
    ),
    "min_probability": float(
        probabilities.min()
    ),
    "max_probability": float(
        probabilities.max()
    ),
    "model_checkpoint_sha256": {
        path.name: hash_file(path)
        for path in [
            MODEL_RUN / "FINAL_lr128.npz",
            MODEL_RUN / "FINAL_lr512.npz",
            MODEL_RUN / "FINAL_META_MODEL.npz"
        ]
    }
})


write_json(RUN / "RUN_MANIFEST.json", {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "source": {
        "path": str(SOURCE),
        "sha256": hash_file(SOURCE)
    },
    "model_source": str(MODEL_RUN),
    "output_hashes": {
        p.name: hash_file(p)
        for p in sorted(RUN.glob("*"))
        if p.is_file()
        and p.name != "RUN_MANIFEST.json"
    }
})


print("PHASE20: INTEGRATED_INFERENCE_VERIFIED")
print("MODEL_RUN:", MODEL_RUN)
print(
    "GENES:", len(source_genes),
    "QA_SAMPLES:", len(predictions)
)
print(
    "ROW_ORDER_MAX_ERROR:",
    f"{row_invariance_error:.3e}"
)
print(
    "GENE_COLUMN_ORDER_MAX_ERROR:",
    f"{gene_invariance_error:.3e}"
)
print(
    "PROBABILITY_SUM_MAX_ERROR:",
    f"{np.max(np.abs(probabilities.sum(axis=1) - 1)):.3e}"
)
print("QA_ONLY_PREDICTION_COUNTS:")
print(summary.to_string(index=False))
print("INDEPENDENT_EXTERNAL_VALIDATION: NOT_STARTED")
print("RUN:", RUN)

PHASE20: INTEGRATED_INFERENCE_VERIFIED
MODEL_RUN: D:\LUAD_DATAFIRST_METHOD_2026\19_exact_source_base_refit\20261023T190611Z_a22f33e7
GENES: 13016 QA_SAMPLES: 843
ROW_ORDER_MAX_ERROR: 0.000e+00
GENE_COLUMN_ORDER_MAX_ERROR: 0.000e+00
PROBABILITY_SUM_MAX_ERROR: 2.220e-16
QA_ONLY_PREDICTION_COUNTS:
  cohort   n  predicted_egfr  predicted_kras  predicted_dwt
GSE13213 117              35              19             63
GSE31210 226              70              43            113
GSE32863  58              17              13             28
GSE72094 442             130              84            228
INDEPENDENT_EXTERNAL_VALIDATION: NOT_STARTED
RUN: D:\LUAD_DATAFIRST_METHOD_2026\20_integrated_inference_verification\20261023T191159Z_de09912c


In [30]:

from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import csv
import gzip
import hashlib
import json
import os
import re
import urllib.request
import uuid

import numpy as np
import pandas as pd

ROOT = Path(r"D:\LUAD_DATAFIRST_METHOD_2026")
TRAIN = Path(
    r"D:\LUAD_CMR_2026\CELL03_DATA_20261021T090947_420335Z_d6aed8ec\cmr_gene_expression.npz"
)
MODEL = ROOT / "19_exact_source_base_refit" / "20261023T190611Z_a22f33e7"
P20 = ROOT / "20_integrated_inference_verification" / "20261023T191159Z_de09912c"
OUT_ROOT = ROOT / "21_external_gene_eligibility"
CACHE = OUT_ROOT / "public_geo_cache"

SERIES_FILE = CACHE / "GSE66863_series_matrix.txt.gz"
MAPPING_FILE = CACHE / "GPL14550_probe_gene_map.tsv"
MAPPING_META = CACHE / "GPL14550_probe_gene_map.meta.json"

SERIES_URL = (
    "https://ftp.ncbi.nlm.nih.gov/geo/series/"
    "GSE66nnn/GSE66863/matrix/GSE66863_series_matrix.txt.gz"
)

SELF_URLS = [
    "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi"
    "?acc=GPL14550&targ=self&form=text&view=data",
    "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi"
    "?acc=GPL14550&targ=self&form=text&view=full",
]

FAMILY_URL = (
    "https://ftp.ncbi.nlm.nih.gov/geo/platforms/"
    "GPL14nnn/GPL14550/soft/GPL14550_family.soft.gz"
)

MAX_MATRIX_DOWNLOAD = 75 * 1024**2
MAX_PLATFORM_COMPRESSED = 90 * 1024**2
MAX_PLATFORM_TEXT = 160 * 1024**2
EXPECTED_EXTERNAL_N = 121


def sha256(path):
    h = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(chunk)

    return h.hexdigest()


def dump_json(path, obj):
    with path.open("w", encoding="utf-8") as f:
        json.dump(
            obj, f,
            indent=2,
            ensure_ascii=False,
            allow_nan=False
        )


def normalize_ids(values):
    return np.char.upper(
        np.char.strip(
            np.asarray(values).astype(str)
        )
    )


def normalize_gene(value):
    s = str(value).strip().strip('"').upper()

    valid = re.fullmatch(
        r"[A-Z0-9][A-Z0-9_.-]*", s
    )

    invalid = {
        "NA", "N/A", "NAN",
        "NONE", "UNKNOWN", "NULL"
    }

    if valid and s not in invalid:
        return s

    return None


def split_tsv(line):
    return next(csv.reader(
        [line], delimiter="\t"
    ))


def fetch_matrix_if_missing():
    if (
        SERIES_FILE.is_file()
        and SERIES_FILE.stat().st_size > 0
    ):
        with gzip.open(
            SERIES_FILE, "rb"
        ) as f:
            if not f.read(100):
                raise RuntimeError(
                    "Cached matrix is empty"
                )

            for _ in iter(
                lambda: f.read(1024 * 1024),
                b""
            ):
                pass

        return "CACHED"

    tmp = SERIES_FILE.with_name(
        SERIES_FILE.name + ".partial"
    )

    try:
        req = urllib.request.Request(
            SERIES_URL,
            headers={
                "User-Agent": (
                    "Mozilla/5.0 "
                    "(research integrity audit)"
                )
            }
        )

        with (
            urllib.request.urlopen(
                req, timeout=60
            ) as response,
            tmp.open("wb") as out
        ):
            total = 0

            while True:
                block = response.read(
                    1024 * 1024
                )

                if not block:
                    break

                total += len(block)

                if total > MAX_MATRIX_DOWNLOAD:
                    raise RuntimeError(
                        "Series matrix exceeds "
                        "expected download budget"
                    )

                out.write(block)

        with gzip.open(
            tmp, "rb"
        ) as f:
            if not f.read(100):
                raise RuntimeError(
                    "Downloaded matrix is empty"
                )

            for _ in iter(
                lambda: f.read(1024 * 1024),
                b""
            ):
                pass

        os.replace(
            tmp, SERIES_FILE
        )

        return "DOWNLOADED"

    finally:
        if tmp.exists():
            tmp.unlink()


class CompressedLimit:
    def __init__(self, response, budget):
        self.response = response
        self.budget = budget
        self.total = 0

    def read(self, n=-1):
        if n is None or n < 0:
            n = 8192

        if self.total >= self.budget:
            raise RuntimeError(
                "Platform stream exceeded "
                "compressed-byte budget"
            )

        chunk = self.response.read(
            min(
                n,
                self.budget - self.total
            )
        )

        self.total += len(chunk)
        return chunk


def parse_platform_lines(
    byte_lines,
    require_platform_first
):
    columns = None
    in_table = False
    saw_platform = False
    total_bytes = 0
    table_rows = 0
    ambiguous = 0
    controls = 0
    conflicting = set()
    mapping = {}

    for raw in byte_lines:
        total_bytes += len(raw)

        if total_bytes > MAX_PLATFORM_TEXT:
            raise RuntimeError(
                "Platform table exceeded "
                "uncompressed-byte budget"
            )

        line = raw.decode(
            "utf-8",
            errors="replace"
        ).strip("\r\n")

        tag = line.strip().lower()

        if tag.startswith("^platform ="):
            if "gpl14550" not in tag:
                raise RuntimeError(
                    "Unexpected GPL entry"
                )

            saw_platform = True

        if (
            require_platform_first
            and tag.startswith("^sample =")
            and not in_table
        ):
            raise RuntimeError(
                "Family SOFT started with "
                "sample records rather than "
                "the platform table"
            )

        if tag == "!platform_table_begin":
            in_table = True
            continue

        if tag == "!platform_table_end":
            if columns is not None:
                break
            continue

        if columns is not None and (
            tag.startswith("^sample =")
            or tag.startswith("^series =")
        ):
            break

        if (
            not line
            or line.startswith(
                ("!", "^", "#")
            )
        ):
            continue

        if columns is None:
            header = [
                x.strip().strip('"').upper()
                for x in split_tsv(line)
            ]

            if (
                "ID" in header
                and "GENE_SYMBOL" in header
            ):
                columns = {
                    name: i
                    for i, name
                    in enumerate(header)
                }

                in_table = True

            continue

        if not in_table:
            continue

        fields = split_tsv(line)

        if len(fields) <= max(
            columns["ID"],
            columns["GENE_SYMBOL"]
        ):
            ambiguous += 1
            continue

        table_rows += 1

        probe = fields[
            columns["ID"]
        ].strip().strip('"').upper()

        gene = normalize_gene(
            fields[
                columns["GENE_SYMBOL"]
            ]
        )

        control_col = columns.get(
            "CONTROL_TYPE"
        )

        if (
            control_col is not None
            and control_col < len(fields)
        ):
            control_value = fields[
                control_col
            ].strip().strip('"').upper()

            if control_value not in {
                "", "FALSE", "0",
                "NA", "N/A"
            }:
                controls += 1
                continue

        if gene is None or not probe:
            ambiguous += 1
            continue

        if probe in conflicting:
            continue

        if (
            probe in mapping
            and mapping[probe] != gene
        ):
            conflicting.add(probe)
            del mapping[probe]

        else:
            mapping[probe] = gene

        if table_rows > 160000:
            raise RuntimeError(
                "Unexpected excessive "
                "platform rows"
            )

    if (
        columns is None
        or "ID" not in columns
        or "GENE_SYMBOL" not in columns
    ):
        raise RuntimeError(
            "Could not find GPL14550 "
            "ID/GENE_SYMBOL table header"
        )

    if (
        table_rows < 20000
        or len(mapping) < 15000
    ):
        raise RuntimeError(
            "Incomplete platform table: "
            f"rows={table_rows}, "
            f"mapped={len(mapping)}"
        )

    stats = {
        "platform_table_rows": table_rows,
        "unique_usable_probe_symbols": len(mapping),
        "skipped_control_probes": controls,
        "skipped_ambiguous_probes": ambiguous,
        "conflicting_probes_quarantined": len(
            conflicting
        ),
        "uncompressed_bytes_read": total_bytes,
        "saw_platform_record": saw_platform
    }

    return mapping, stats


def remote_platform(url, compressed):
    req = urllib.request.Request(
        url,
        headers={
            "User-Agent": (
                "Mozilla/5.0 "
                "(academic gene annotation audit)"
            ),
            "Accept-Encoding": "identity"
        }
    )

    with urllib.request.urlopen(
        req, timeout=70
    ) as response:
        content_type = response.headers.get(
            "Content-Type", ""
        )

        if not compressed:
            total = 0

            def source_lines():
                nonlocal total

                for raw in response:
                    total += len(raw)

                    if total > MAX_PLATFORM_TEXT:
                        raise RuntimeError(
                            "Self-only GPL endpoint "
                            "returned an unexpectedly "
                            "large document"
                        )

                    yield raw

            mapping, stats = parse_platform_lines(
                source_lines(),
                require_platform_first=False
            )

            stats["compressed_bytes_read"] = None

        else:
            limited = CompressedLimit(
                response,
                MAX_PLATFORM_COMPRESSED
            )

            with gzip.GzipFile(
                fileobj=limited,
                mode="rb"
            ) as stream:
                mapping, stats = parse_platform_lines(
                    stream,
                    require_platform_first=True
                )

            stats["compressed_bytes_read"] = (
                limited.total
            )

        stats["content_type"] = content_type

        return mapping, stats


def get_annotation():
    if (
        MAPPING_FILE.is_file()
        and MAPPING_META.is_file()
    ):
        meta = json.loads(
            MAPPING_META.read_text(
                encoding="utf-8"
            )
        )

        if (
            meta.get("platform") == "GPL14550"
            and meta.get("mapping_sha256")
            == sha256(MAPPING_FILE)
        ):
            mapping = {}

            with MAPPING_FILE.open(
                "r",
                encoding="utf-8",
                newline=""
            ) as f:
                reader = csv.DictReader(
                    f, delimiter="\t"
                )

                if reader.fieldnames != [
                    "ID", "GENE_SYMBOL"
                ]:
                    raise RuntimeError(
                        "Cached GPL mapping schema mismatch"
                    )

                for row in reader:
                    probe = row["ID"]
                    gene = row["GENE_SYMBOL"]

                    if probe in mapping:
                        raise RuntimeError(
                            "Duplicate cached probe ID"
                        )

                    mapping[probe] = gene

            if len(mapping) < 15000:
                raise RuntimeError(
                    "Cached GPL annotation incomplete"
                )

            return mapping, meta, "CACHED"

    attempts = []

    candidates = (
        [(url, False) for url in SELF_URLS]
        + [(FAMILY_URL, True)]
    )

    for url, compressed in candidates:
        try:
            mapping, stats = remote_platform(
                url, compressed
            )

            tmp = MAPPING_FILE.with_name(
                MAPPING_FILE.name + ".partial"
            )

            with tmp.open(
                "w",
                encoding="utf-8",
                newline=""
            ) as f:
                writer = csv.writer(
                    f,
                    delimiter="\t",
                    lineterminator="\n"
                )

                writer.writerow(
                    ("ID", "GENE_SYMBOL")
                )

                for probe in sorted(mapping):
                    writer.writerow(
                        (probe, mapping[probe])
                    )

            os.replace(
                tmp, MAPPING_FILE
            )

            meta = {
                "platform": "GPL14550",
                "source_url": url,
                "mode": (
                    "streamed_platform_prefix"
                    if compressed
                    else "self_only_platform_table"
                ),
                "retrieved_utc": datetime.now(
                    timezone.utc
                ).isoformat(),
                "mapping_sha256": sha256(
                    MAPPING_FILE
                ),
                "annotation_stats": stats
            }

            dump_json(
                MAPPING_META, meta
            )

            return mapping, meta, "RETRIEVED"

        except Exception as exc:
            attempts.append(
                f"{type(exc).__name__}: {exc}"
                f" | {url}"
            )

            print(
                "GPL_SOURCE_ATTEMPT_FAILED:",
                type(exc).__name__,
                str(exc)[:140]
            )

    raise RuntimeError(
        "All bounded platform-annotation "
        "sources failed:\n"
        + "\n".join(attempts)
    )


def read_expression_matrix(
    path,
    mapping,
    target_genes
):
    matched = defaultdict(list)
    seen = set()
    samples = None
    active = False
    n_probe_rows = 0
    invalid_selected = 0
    mapped_probe_rows = 0

    with gzip.open(
        path,
        "rt",
        encoding="utf-8",
        errors="replace"
    ) as f:
        for line in f:
            tag = line.strip().lower()

            if tag == "!series_matrix_table_begin":
                active = True
                continue

            if tag == "!series_matrix_table_end":
                break

            if not active:
                continue

            fields = split_tsv(
                line.rstrip("\r\n")
            )

            if samples is None:
                if (
                    not fields
                    or fields[0].strip()
                    .strip('"').upper()
                    != "ID_REF"
                ):
                    raise RuntimeError(
                        "GSE66863 matrix "
                        "lacks ID_REF header"
                    )

                samples = [
                    x.strip().strip('"').upper()
                    for x in fields[1:]
                ]

                if (
                    len(samples) != EXPECTED_EXTERNAL_N
                    or len(set(samples)) != len(samples)
                ):
                    raise RuntimeError(
                        "GSM count/uniqueness mismatch: "
                        f"{len(samples)}"
                    )

                if not all(
                    re.fullmatch(r"GSM\d+", x)
                    for x in samples
                ):
                    raise RuntimeError(
                        "Unexpected GSM identifiers"
                    )

                continue

            n_probe_rows += 1

            probe = fields[0].strip(
            ).strip('"').upper()

            if probe in seen:
                raise RuntimeError(
                    f"Repeated matrix probe: {probe}"
                )

            seen.add(probe)

            gene = mapping.get(probe)

            if gene not in target_genes:
                continue

            if len(fields) != len(samples) + 1:
                invalid_selected += 1
                continue

            try:
                values = np.asarray(
                    [
                        float(x.strip().strip('"'))
                        for x in fields[1:]
                    ],
                    dtype=np.float32
                )

            except ValueError:
                invalid_selected += 1
                continue

            if not np.isfinite(values).all():
                invalid_selected += 1
                continue

            matched[gene].append(values)
            mapped_probe_rows += 1

    if (
        samples is None
        or n_probe_rows < 30000
    ):
        raise RuntimeError(
            "Incomplete GSE66863 "
            f"expression table: {n_probe_rows}"
        )

    stats = {
        "matrix_probe_rows": n_probe_rows,
        "mapped_finite_target_probe_rows": (
            mapped_probe_rows
        ),
        "invalid_mapped_probe_rows": (
            invalid_selected
        ),
        "mapped_target_genes": len(matched)
    }

    return samples, matched, stats


OUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

CACHE.mkdir(
    parents=True,
    exist_ok=True
)

RUN = OUT_ROOT / (
    datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%SZ_"
    )
    + uuid.uuid4().hex[:8]
)

RUN.mkdir(exist_ok=False)


required_files = (
    TRAIN,
    MODEL / "GATE_STATUS.json",
    MODEL / "FINAL_lr128.npz",
    MODEL / "FINAL_lr512.npz",
    MODEL / "FINAL_META_MODEL.npz",
    P20 / "INFERENCE_AUDIT.json"
)

for path in required_files:
    if not path.is_file():
        raise FileNotFoundError(path)


gate = json.loads(
    (
        MODEL / "GATE_STATUS.json"
    ).read_text(encoding="utf-8")
)

if gate.get("status") != "PASS_FINAL_BASE_REFIT":
    raise RuntimeError(
        "Final model gate has not passed"
    )


with np.load(
    TRAIN,
    allow_pickle=False
) as z:
    train_genes = normalize_ids(
        z["genes"]
    )

    development_gsms = set(
        normalize_ids(z["gsm"]).tolist()
    )


if (
    len(train_genes) != 13016
    or len(set(train_genes)) != 13016
):
    raise RuntimeError(
        "Frozen 13016-gene universe mismatch"
    )


required_genes = set(
    train_genes.tolist()
)

selected = set()

for capacity in (128, 512):
    with np.load(
        MODEL / f"FINAL_lr{capacity}.npz",
        allow_pickle=False
    ) as z:
        selected.update(
            normalize_ids(
                z["genes"]
            ).tolist()
        )

if not selected.issubset(
    required_genes
):
    raise RuntimeError(
        "Final-model gene IDs outside "
        "the frozen gene universe"
    )


try:
    matrix_status = fetch_matrix_if_missing()

    print(
        "GEO_MATRIX:",
        matrix_status,
        "bytes=",
        SERIES_FILE.stat().st_size
    )

    annotation, annotation_meta, annotation_status = (
        get_annotation()
    )

    print(
        "GPL14550_MAPPING:",
        annotation_status,
        "probes=",
        len(annotation)
    )

    samples, grouped, matrix_stats = (
        read_expression_matrix(
            SERIES_FILE,
            annotation,
            required_genes
        )
    )

except Exception as exc:
    dump_json(
        RUN / "STATUS.json",
        {
            "status": "INPUT_OR_PARSE_BLOCKED",
            "error": (
                f"{type(exc).__name__}: {exc}"
            ),
            "series_matrix_cached": (
                SERIES_FILE.is_file()
            ),
            "mutation_labels_accessed": False,
            "model_changed": False
        }
    )

    print(
        "PHASE21: INPUT_OR_PARSE_BLOCKED"
    )
    print("RUN:", RUN)

    raise


found = set(grouped)

missing = sorted(
    required_genes - found
)

selected_missing = sorted(
    selected - found
)

gsm_overlap = sorted(
    set(samples) & development_gsms
)


coverage = pd.DataFrame({
    "gene": train_genes,
    "usable_probes": [
        len(grouped.get(g, ()))
        for g in train_genes
    ],
    "mapped": [
        g in found
        for g in train_genes
    ],
    "selected_by_final_model": [
        g in selected
        for g in train_genes
    ]
})

coverage.to_csv(
    RUN / "GENE_COVERAGE.csv",
    index=False
)

pd.DataFrame({
    "gsm": samples
}).to_csv(
    RUN / "GSM_ROSTER.csv",
    index=False
)


if not missing and not gsm_overlap:
    status = (
        "GENE_SCHEMA_PASS_OTHER_AUDITS_PENDING"
    )

else:
    status = (
        "GENE_SCHEMA_INCOMPATIBLE_OR_GSM_OVERLAP"
    )


if not missing and not gsm_overlap:
    aligned = np.column_stack([
        np.median(
            np.stack(
                grouped[g],
                axis=0
            ),
            axis=0
        )
        for g in train_genes
    ]).astype(np.float32)

    if (
        aligned.shape != (
            EXPECTED_EXTERNAL_N, 13016
        )
        or not np.isfinite(
            aligned
        ).all()
    ):
        raise RuntimeError(
            "Aligned expression "
            "schema integrity failure"
        )

    np.savez_compressed(
        RUN / "GSE66863_EXPRESSION_ONLY_13016.npz",
        X=aligned,
        gsm=np.asarray(samples),
        genes=train_genes,
        cohort=np.asarray(
            ["GSE66863"] * len(samples)
        )
    )


report = {
    "status": status,
    "dataset": "GSE66863",
    "platform": "GPL14550",
    "observed_expression_samples": len(samples),
    "frozen_gene_universe": len(train_genes),
    "matched_frozen_genes": len(found),
    "missing_frozen_genes": len(missing),
    "missing_gene_examples": missing[:40],
    "selected_final_gene_union": len(selected),
    "selected_genes_missing": len(
        selected_missing
    ),
    "selected_gene_missing_examples": (
        selected_missing[:40]
    ),
    "GSM_overlap": len(gsm_overlap),
    "GSM_overlap_examples": (
        gsm_overlap[:20]
    ),
    "patient_level_overlap": "NOT_VERIFIED",
    "mutation_labels": "NOT_ACCESSED",
    "assay_label_completeness": "PENDING",
    "platform_to_gene": (
        "Single unambiguous GENE_SYMBOL "
        "per probe; control probes excluded"
    ),
    "multi_probe_gene_aggregation": (
        "Median across valid mapped probes"
    ),
    "microarray_to_training_preprocessing_comparability": (
        "NOT_VERIFIED"
    ),
    "historical_heldout_status": "NOT_VERIFIED",
    "genotype_labels_used_for_model_decisions": False,
    "external_model_predictions_generated": False,
    "matrix_source_sha256": sha256(
        SERIES_FILE
    ),
    "platform_annotation": annotation_meta,
    "matrix_stats": matrix_stats,
    "model_sha256": {
        f"FINAL_lr{k}.npz": sha256(
            MODEL / f"FINAL_lr{k}.npz"
        )
        for k in (128, 512)
    }
}


dump_json(
    RUN / "ELIGIBILITY_AUDIT.json",
    report
)

dump_json(
    RUN / "RUN_MANIFEST.json",
    {
        "created_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "status": status,
        "outputs": {
            p.name: sha256(p)
            for p in sorted(RUN.iterdir())
            if (
                p.is_file()
                and p.name != "RUN_MANIFEST.json"
            )
        }
    }
)


print("PHASE21:", status)
print("GEO_SAMPLES:", len(samples))

print(
    "GENES_MATCHED:",
    len(found),
    "/",
    len(required_genes)
)

print(
    "GENES_MISSING:",
    len(missing)
)

print(
    "FINAL_BASE_SELECTED_MISSING:",
    len(selected_missing)
)

print(
    "GSM_OVERLAP:",
    len(gsm_overlap)
)

print(
    "GPL_SOURCE:",
    annotation_meta.get(
        "source_url"
    )
)

print("GENOTYPE_PROVENANCE: PENDING")
print("EXTERNAL_PREDICTION: NOT_RUN")
print("RUN:", RUN)

GEO_MATRIX: CACHED bytes= 24625035
GPL14550_MAPPING: RETRIEVED probes= 30434
PHASE21: GENE_SCHEMA_INCOMPATIBLE_OR_GSM_OVERLAP
GEO_SAMPLES: 121
GENES_MATCHED: 12989 / 13016
GENES_MISSING: 27
FINAL_BASE_SELECTED_MISSING: 2
GSM_OVERLAP: 0
GPL_SOURCE: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GPL14550&targ=self&form=text&view=data
GENOTYPE_PROVENANCE: PENDING
EXTERNAL_PREDICTION: NOT_RUN
RUN: D:\LUAD_DATAFIRST_METHOD_2026\21_external_gene_eligibility\20261023T194857Z_9fdc35e5


In [31]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import uuid

import numpy as np
import pandas as pd
from scipy.special import expit
from scipy.stats import rankdata
from sklearn.metrics import f1_score

ROOT = Path(r"D:\LUAD_DATAFIRST_METHOD_2026")
DATA = Path(
    r"D:\LUAD_CMR_2026\CELL03_DATA_20261021T090947_420335Z_d6aed8ec\cmr_gene_expression.npz"
)
P12 = ROOT / "12_multiresolution_ood_evidence" / "20261023T151555Z_638a3c88"
P14 = ROOT / "14_quantile_residual_transfer" / "20261023T160742Z_f9435239"
P19 = ROOT / "19_exact_source_base_refit" / "20261023T190611Z_a22f33e7"
P21 = ROOT / "21_external_gene_eligibility" / "20261023T194857Z_9fdc35e5"
OUT = ROOT / "22_assay_missingness_bridge_audit"

COHORTS = (
    "GSE13213",
    "GSE31210",
    "GSE32863",
    "GSE72094"
)
P_COLS = ("p_egfr", "p_kras", "p_dwt")
REPRO_TOL = 1e-4


def digest(p):
    h = hashlib.sha256()

    with p.open("rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(block)

    return h.hexdigest()


def save_json(p, obj):
    p.write_text(
        json.dumps(
            obj,
            indent=2,
            ensure_ascii=False,
            allow_nan=False
        ),
        encoding="utf-8"
    )


def canonical(a):
    return np.char.upper(
        np.char.strip(
            np.asarray(a).astype(str)
        )
    )


def load_base(p, all_genes):
    with np.load(p, allow_pickle=False) as z:
        keys = (
            "gene_indices",
            "genes",
            "mu",
            "sd",
            "e_coef",
            "e_intercept",
            "k_coef",
            "k_intercept"
        )

        if not set(keys).issubset(z.files):
            raise ValueError(
                f"Incomplete base checkpoint: {p}"
            )

        d = {
            k: np.asarray(z[k]).copy()
            for k in keys
        }

    ix = np.asarray(
        d["gene_indices"], dtype=int
    )

    if (
        ix.ndim != 1
        or len(ix) == 0
        or np.any(ix < 0)
        or np.any(ix >= len(all_genes))
        or not np.array_equal(ix, np.unique(ix))
        or not np.array_equal(
            canonical(d["genes"]),
            all_genes[ix]
        )
    ):
        raise ValueError(
            f"Gene mapping mismatch: {p}"
        )

    for k in ("mu", "sd", "e_coef", "k_coef"):
        if (
            d[k].shape != ix.shape
            or not np.isfinite(d[k]).all()
        ):
            raise ValueError(
                f"Invalid {k}: {p}"
            )

    if np.any(d["sd"] <= 0):
        raise ValueError(
            f"Invalid SD: {p}"
        )

    d["gene_indices"] = ix
    return d


def load_head(p):
    with np.load(p, allow_pickle=False) as z:
        keys = (
            "bias",
            "raw_scale",
            "weights"
        )

        if not set(keys).issubset(z.files):
            raise ValueError(
                f"Incomplete meta checkpoint: {p}"
            )

        d = {
            k: np.asarray(
                z[k], dtype=np.float64
            ).copy()
            for k in keys
        }

        d["correction_bound"] = (
            float(
                np.asarray(
                    z["correction_bound"]
                ).item()
            )
            if "correction_bound" in z.files
            else 1.5
        )

        d["guard_lambda"] = (
            float(
                np.asarray(
                    z["guard_lambda"]
                ).item()
            )
            if "guard_lambda" in z.files
            else 1.0
        )

    if (
        d["bias"].shape != (2,)
        or d["raw_scale"].shape != (2,)
        or d["weights"].shape != (2, 3)
        or not all(
            np.isfinite(v).all()
            for v in (
                d["bias"],
                d["raw_scale"],
                d["weights"]
            )
        )
        or not 0 <= d["guard_lambda"] <= 1
        or d["correction_bound"] <= 0
    ):
        raise ValueError(
            f"Invalid meta checkpoint: {p}"
        )

    return d


def to_prob(scores):
    s = np.asarray(
        scores, dtype=np.float64
    )

    pe = expit(s[:, 0])
    pk = expit(s[:, 1])

    p = np.column_stack((
        pe,
        (1 - pe) * pk,
        (1 - pe) * (1 - pk)
    ))

    if (
        not np.isfinite(p).all()
        or not np.allclose(
            p.sum(axis=1), 1,
            atol=1e-12,
            rtol=1e-12
        )
    ):
        raise ArithmeticError(
            "Invalid base probabilities"
        )

    return p


def effective_scores(p):
    e = np.clip(
        p[:, 0], 1e-12, 1 - 1e-12
    )
    k = np.clip(
        p[:, 1], 1e-12, 1
    )
    w = np.clip(
        p[:, 2], 1e-12, 1
    )

    return np.column_stack((
        np.log(e / (1 - e)),
        np.log(k / w)
    ))


def pair_to_four(p128, p512):
    a = effective_scores(p128)
    b = effective_scores(p512)

    return np.column_stack((
        a[:, 0],
        b[:, 0],
        a[:, 1],
        b[:, 1]
    ))


def cohort_quantiles(raw):
    n = len(raw)

    if n < 25:
        raise ValueError(
            "Small quantile target batch"
        )

    u = np.column_stack([
        (
            rankdata(
                raw[:, j],
                method="average"
            ) - 0.5
        ) / n
        for j in range(4)
    ])

    u = np.clip(
        u, 1e-5, 1 - 1e-5
    )

    return (
        np.log(u) - np.log1p(-u)
    ) / 2.5


def head_predict(raw, head):
    q = cohort_quantiles(raw)

    a = q[:, (0, 2)]
    d = q[:, (1, 3)] - a

    basis = np.stack((
        d,
        d * np.tanh(1.5 * a),
        np.tanh(2 * d) * (
            1 - np.tanh(a) ** 2
        )
    ), axis=-1)

    correction = np.sum(
        basis * head["weights"],
        axis=-1
    )

    bound = head["correction_bound"]

    score = (
        head["bias"]
        + np.logaddexp(
            0.0, head["raw_scale"]
        ) * a
        + head["guard_lambda"]
        * bound
        * np.tanh(correction / bound)
    )

    return to_prob(score)


def checkpoint_probability(
    rank_matrix,
    state,
    present_mask=None
):
    ix = state["gene_indices"]

    local_rank = np.asarray(
        rank_matrix[:, ix],
        dtype=np.float32
    ).copy()

    if present_mask is not None:
        unavailable = ~present_mask[ix]

        if unavailable.any():
            local_rank[:, unavailable] = (
                state["mu"][unavailable]
            )

    z = np.tanh(
        (
            local_rank - state["mu"]
        ) / (
            2.0 * state["sd"]
        )
    ).astype(np.float32)

    e = (
        z @ state["e_coef"]
        + state["e_intercept"]
    )

    k = (
        z @ state["k_coef"]
        + state["k_intercept"]
    )

    return to_prob(
        np.column_stack((e, k))
    )


def metrics(y, p):
    y = np.asarray(y, dtype=int)

    nll = -np.log(
        np.clip(
            p[np.arange(len(y)), y],
            1e-15,
            1
        )
    ).mean()

    f1 = f1_score(
        y,
        p.argmax(axis=1),
        labels=[0, 1, 2],
        average="macro",
        zero_division=0
    )

    return {
        "nll": float(nll),
        "macro_f1": float(f1)
    }


def get_saved(df, cohort, model, ids):
    frame = df.loc[
        (df.held_cohort == cohort)
        & (df.model == model)
    ].copy()

    if (
        len(frame) != len(ids)
        or frame.gsm.duplicated().any()
    ):
        raise ValueError(
            f"Prediction roster mismatch: "
            f"{cohort}/{model}"
        )

    frame = frame.set_index(
        "gsm"
    ).loc[ids]

    p = frame[
        list(P_COLS)
    ].to_numpy(dtype=np.float64)

    y_saved = frame.true_class.to_numpy(
        dtype=int
    )

    if (
        not np.isfinite(p).all()
        or not np.allclose(
            p.sum(axis=1),
            1,
            atol=1e-5
        )
    ):
        raise ValueError(
            f"Invalid reference prediction: "
            f"{cohort}/{model}"
        )

    return p, y_saved


required_paths = [
    DATA,
    P21 / "GENE_COVERAGE.csv",
    P21 / "GSM_ROSTER.csv",
    P14 / "LOCO_PREDICTIONS.csv",
    P19 / "FINAL_lr128.npz",
    P19 / "FINAL_lr512.npz",
    P19 / "FINAL_META_MODEL.npz"
]

for c in COHORTS:
    for k in (128, 512):
        required_paths.append(
            P12 / "outer_models"
            / f"{c}_lr{k}.npz"
        )

    required_paths.append(
        P14 / "checkpoints"
        / f"{c}_q_residual.npz"
    )

for p in required_paths:
    if not p.is_file():
        raise FileNotFoundError(p)


with np.load(
    DATA, allow_pickle=False
) as z:
    genes = canonical(z["genes"])

    X = np.asarray(
        z["X"],
        dtype=np.float32
    )

    y = np.asarray(
        z["y"],
        dtype=np.int64
    )

    cohorts = canonical(
        z["cohort"]
    )
    gsms = canonical(z["gsm"])

if (
    X.shape != (843, 13016)
    or len(np.unique(genes)) != 13016
    or not np.isfinite(X).all()
):
    raise ValueError(
        "Original development matrix/schema mismatch"
    )

if (
    set(cohorts.tolist()) != set(COHORTS)
    or set(y.tolist()) != {0, 1, 2}
):
    raise ValueError(
        "Development labels/cohorts mismatch"
    )


coverage = pd.read_csv(
    P21 / "GENE_COVERAGE.csv",
    dtype={"gene": str}
)

if not {
    "gene", "mapped"
}.issubset(coverage.columns):
    raise ValueError(
        "Phase21 coverage columns invalid"
    )

coverage["gene"] = (
    coverage["gene"]
    .str.strip()
    .str.upper()
)

if (
    len(coverage) != len(genes)
    or coverage.gene.duplicated().any()
    or set(coverage.gene) != set(genes)
):
    raise ValueError(
        "Phase21 gene inventory differs "
        "from frozen training genes"
    )

flags = (
    coverage.set_index("gene")
    .loc[genes, "mapped"]
    .astype(str)
    .str.lower()
)

if not flags.isin([
    "true", "false"
]).all():
    raise ValueError(
        "Phase21 mapped field is not boolean"
    )

present_mask = (
    flags.eq("true")
    .to_numpy(dtype=bool)
)

missing = [
    str(g)
    for g in genes[~present_mask]
]

if len(missing) != 27:
    raise ValueError(
        f"Expected 27 missing genes; "
        f"found {len(missing)}"
    )


final_models = {
    k: load_base(
        P19 / f"FINAL_lr{k}.npz",
        genes
    )
    for k in (128, 512)
}

selected_all = set(
    np.concatenate([
        v["gene_indices"]
        for v in final_models.values()
    ]).tolist()
)

selected_missing = [
    str(genes[j])
    for j in sorted(selected_all)
    if not present_mask[j]
]

if len(selected_missing) != 2:
    raise ValueError(
        "Selected-gene inventory differs "
        f"from Phase21: {selected_missing}"
    )


external_ids = pd.read_csv(
    P21 / "GSM_ROSTER.csv",
    dtype=str
)["gsm"].str.upper().tolist()

if (
    len(external_ids) != 121
    or len(set(external_ids)) != 121
    or set(external_ids) & set(gsms.tolist())
):
    raise ValueError(
        "Phase21 expression sample roster mismatch"
    )


archived = pd.read_csv(
    P14 / "LOCO_PREDICTIONS.csv",
    dtype={
        "gsm": str,
        "held_cohort": str
    }
)

archived["gsm"] = (
    archived["gsm"]
    .str.strip()
    .str.upper()
)

if not set(
    P_COLS + (
        "model",
        "true_class",
        "gsm",
        "held_cohort"
    )
).issubset(archived.columns):
    raise ValueError(
        "Phase14 prediction file schema invalid"
    )


OUT.mkdir(
    parents=True,
    exist_ok=True
)

RUN = OUT / (
    datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%SZ_"
    )
    + uuid.uuid4().hex[:8]
)

RUN.mkdir(exist_ok=False)


save_json(
    RUN / "PROTOCOL.json",
    {
        "phase": 22,
        "purpose": (
            "Fixed GSE66863 27-gene absence "
            "stress on historical source-held-out predictions"
        ),
        "no_external_mutation_labels_accessed": True,
        "source_rank_calculation": (
            "Observed 12989-gene within-sample percentile ranks"
        ),
        "missing_predictor_rule": (
            "Set missing selected-gene rank "
            "to archived training mu so "
            "transformed contribution is zero"
        ),
        "parameters_changed": False,
        "label_usage": (
            "Development labels only for "
            "masked-versus-unmasked comparison"
        ),
        "status": (
            "EXPLORATORY_SENSITIVITY_NOT_EXTERNAL_VALIDATION"
        )
    }
)


rows = []
sample_rows = []
max_replay_error = 0.0


for cohort in COHORTS:
    indices = np.flatnonzero(
        cohorts == cohort
    )

    ids = gsms[indices].tolist()

    if (
        len(ids) < 25
        or len(set(ids)) != len(ids)
    ):
        raise ValueError(
            f"Invalid source roster: {cohort}"
        )

    Xc = X[indices]

    ranks_full = (
        rankdata(
            Xc,
            axis=1,
            method="average"
        ).astype(np.float32) - 1
    ) / (len(genes) - 1)

    ranks_observed = (
        rankdata(
            Xc[:, present_mask],
            axis=1,
            method="average"
        ).astype(np.float32) - 1
    ) / (int(present_mask.sum()) - 1)

    ranks_masked = np.empty_like(
        ranks_full
    )

    ranks_masked[:, present_mask] = (
        ranks_observed
    )

    ranks_masked[:, ~present_mask] = np.nan

    original_p = {}
    masked_p = {}
    state_missing = {}
    fold_raw_error = 0.0

    for k in (128, 512):
        checkpoint = load_base(
            P12 / "outer_models"
            / f"{cohort}_lr{k}.npz",
            genes
        )

        base_full = checkpoint_probability(
            ranks_full,
            checkpoint
        )

        base_masked = checkpoint_probability(
            ranks_masked,
            checkpoint,
            present_mask
        )

        saved_raw, reference_y = get_saved(
            archived,
            cohort,
            f"raw{k}",
            ids
        )

        if not np.array_equal(
            reference_y,
            y[indices]
        ):
            raise ValueError(
                f"Genotype roster mismatch: {cohort}"
            )

        raw_error = float(
            np.max(np.abs(
                base_full - saved_raw
            ))
        )

        fold_raw_error = max(
            fold_raw_error,
            raw_error
        )

        max_replay_error = max(
            max_replay_error,
            raw_error
        )

        if raw_error > REPRO_TOL:
            raise RuntimeError(
                "Archived base reproduction failed: "
                f"{cohort}/{k}: {raw_error:.2e}"
            )

        original_p[k] = saved_raw
        masked_p[k] = base_masked

        state_missing[k] = int(
            (
                ~present_mask[
                    checkpoint["gene_indices"]
                ]
            ).sum()
        )

    outer_head = load_head(
        P14 / "checkpoints"
        / f"{cohort}_q_residual.npz"
    )

    p_original = head_predict(
        pair_to_four(
            original_p[128],
            original_p[512]
        ),
        outer_head
    )

    p_archive, reference_y = get_saved(
        archived,
        cohort,
        "q_residual",
        ids
    )

    meta_error = float(
        np.max(np.abs(
            p_original - p_archive
        ))
    )

    max_replay_error = max(
        max_replay_error,
        meta_error
    )

    if meta_error > REPRO_TOL:
        raise RuntimeError(
            "Phase14 q_residual reproduction failed: "
            f"{cohort}: {meta_error:.2e}"
        )

    p_masked = head_predict(
        pair_to_four(
            masked_p[128],
            masked_p[512]
        ),
        outer_head
    )

    original_metric = metrics(
        y[indices],
        p_archive
    )

    masked_metric = metrics(
        y[indices],
        p_masked
    )

    changed = int(
        (
            p_archive.argmax(axis=1)
            != p_masked.argmax(axis=1)
        ).sum()
    )

    row = {
        "cohort": cohort,
        "n": len(indices),
        "missing_128_selected": state_missing[128],
        "missing_512_selected": state_missing[512],
        "baseline_nll": original_metric["nll"],
        "masked_nll": masked_metric["nll"],
        "nll_increase": (
            masked_metric["nll"]
            - original_metric["nll"]
        ),
        "baseline_macro_f1": original_metric["macro_f1"],
        "masked_macro_f1": masked_metric["macro_f1"],
        "macro_f1_change": (
            masked_metric["macro_f1"]
            - original_metric["macro_f1"]
        ),
        "changed_predictions": changed,
        "mean_abs_probability_delta": float(
            np.abs(p_masked - p_archive).mean()
        ),
        "max_abs_probability_delta": float(
            np.abs(p_masked - p_archive).max()
        ),
        "replay_max_error": max(
            fold_raw_error,
            meta_error
        )
    }

    rows.append(row)

    for j, i in enumerate(indices):
        sample_rows.append({
            "cohort": cohort,
            "gsm": str(gsms[i]),
            "true_class": int(y[i]),
            "baseline_pred": int(
                p_archive[j].argmax()
            ),
            "masked_pred": int(
                p_masked[j].argmax()
            ),
            **{
                f"baseline_{key}": float(
                    p_archive[j, c]
                )
                for c, key in enumerate(P_COLS)
            },
            **{
                f"masked_{key}": float(
                    p_masked[j, c]
                )
                for c, key in enumerate(P_COLS)
            }
        })

    print(
        f"LOCO_MASK {cohort} "
        f"n={len(indices)} "
        f"missing128={state_missing[128]} "
        f"missing512={state_missing[512]} "
        f"delta_nll={row['nll_increase']:.5f} "
        f"delta_f1={row['macro_f1_change']:.5f} "
        f"changed={changed} "
        f"replay={meta_error:.2e}"
    )


result = pd.DataFrame(rows)

result.to_csv(
    RUN / "HISTORICAL_LOCO_MASK_STRESS.csv",
    index=False
)

pd.DataFrame(
    sample_rows
).to_csv(
    RUN / "MASKED_DEV_PREDICTIONS.csv",
    index=False
)

pd.DataFrame({
    "gene": missing,
    "selected_by_final_base": [
        g in selected_missing
        for g in missing
    ]
}).to_csv(
    RUN / "MISSING_GENE_INVENTORY.csv",
    index=False
)


summary = {
    "mean_baseline_nll": float(
        result.baseline_nll.mean()
    ),
    "mean_masked_nll": float(
        result.masked_nll.mean()
    ),
    "mean_nll_increase": float(
        result.nll_increase.mean()
    ),
    "mean_baseline_macro_f1": float(
        result.baseline_macro_f1.mean()
    ),
    "mean_masked_macro_f1": float(
        result.masked_macro_f1.mean()
    ),
    "mean_macro_f1_change": float(
        result.macro_f1_change.mean()
    ),
    "total_class_changes": int(
        result.changed_predictions.sum()
    ),
    "total_samples": int(
        result.n.sum()
    ),
    "max_archived_replay_error": (
        max_replay_error
    ),
    "missing_genes": missing,
    "missing_final_selected_genes": selected_missing,
    "warning": (
        "Assay coverage sensitivity on historically reused "
        "development cohorts only; does not assess external "
        "expression-platform shift or mutation-label validity. "
        "Missing predictor mean-rank substitution changes "
        "the original inference procedure."
    )
}

save_json(
    RUN / "SUMMARY.json",
    summary
)

save_json(
    RUN / "RUN_MANIFEST.json",
    {
        "created_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "inputs_sha256": {
            str(p): digest(p)
            for p in required_paths
        },
        "outputs_sha256": {
            p.name: digest(p)
            for p in RUN.iterdir()
            if (
                p.is_file()
                and p.name != "RUN_MANIFEST.json"
            )
        }
    }
)


print("PHASE22: COMPLETE")

print(
    "MISSING_27_GENES:",
    ", ".join(missing)
)

print(
    "MISSING_FINAL_SELECTED_GENES:",
    selected_missing
)

print("EQUAL_COHORT_MASK_STRESS:")

for key in (
    "mean_baseline_nll",
    "mean_masked_nll",
    "mean_nll_increase",
    "mean_baseline_macro_f1",
    "mean_masked_macro_f1",
    "mean_macro_f1_change"
):
    print(
        f"{key}: {summary[key]:.6f}"
    )

print(
    "TOTAL_CHANGED_PREDICTIONS:",
    summary["total_class_changes"]
)

print(
    "MAX_ARCHIVED_REPRO_ERROR:",
    f"{max_replay_error:.3e}"
)

print("INDEPENDENT_VALIDATION: NOT_RUN")
print("RUN:", RUN)

LOCO_MASK GSE13213 n=117 missing128=0 missing512=1 delta_nll=0.00078 delta_f1=0.00263 changed=2 replay=1.67e-16
LOCO_MASK GSE31210 n=226 missing128=1 missing512=3 delta_nll=-0.00138 delta_f1=-0.00445 changed=2 replay=2.22e-16
LOCO_MASK GSE32863 n=58 missing128=1 missing512=4 delta_nll=-0.00208 delta_f1=-0.00425 changed=1 replay=2.22e-16
LOCO_MASK GSE72094 n=442 missing128=1 missing512=5 delta_nll=0.00043 delta_f1=-0.00341 changed=9 replay=2.22e-16
PHASE22: COMPLETE
MISSING_27_GENES: ATP6V0A2, BTN3A3, CPM, CRYBG3, EIF2S3, FBXO45, FCGR3B, FLJ20021, GRSF1, GSTA1, IGHD, LIN7C, LOC100288893, LOC100506098, LOC100506100, LOC100506271, LYZ, MGAT4B, NAPSB, PSG4, PSG6, PSG7, RRN3P1, TMEM70, UGT2B17, ZNF440, ZNF611
MISSING_FINAL_SELECTED_GENES: ['LOC100506098', 'TMEM70']
EQUAL_COHORT_MASK_STRESS:
mean_baseline_nll: 0.909271
mean_masked_nll: 0.908705
mean_nll_increase: -0.000565
mean_baseline_macro_f1: 0.545245
mean_masked_macro_f1: 0.542874
mean_macro_f1_change: -0.002371
TOTAL_CHANGED_PREDICTION

In [32]:

from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import csv
import gzip
import hashlib
import json
import re
import uuid

import numpy as np
import pandas as pd
from scipy.special import expit
from scipy.stats import rankdata

ROOT = Path(r'D:\LUAD_DATAFIRST_METHOD_2026')
SOURCE = Path(
    r'D:\LUAD_CMR_2026\CELL03_DATA_20261021T090947_420335Z_d6aed8ec\cmr_gene_expression.npz'
)
P19 = ROOT / '19_exact_source_base_refit' / '20261023T190611Z_a22f33e7'
P20 = ROOT / '20_integrated_inference_verification' / '20261023T191159Z_de09912c'
P21 = ROOT / '21_external_gene_eligibility' / '20261023T194857Z_9fdc35e5'
P22 = ROOT / '22_assay_missingness_bridge_audit' / '20261023T195608Z_c617ee1c'
CACHE = ROOT / '21_external_gene_eligibility' / 'public_geo_cache'
OUT_ROOT = ROOT / '23_external_prediction_lock'

SERIES = CACHE / 'GSE66863_series_matrix.txt.gz'
ANNOTATION = CACHE / 'GPL14550_probe_gene_map.tsv'
ANNOTATION_META = CACHE / 'GPL14550_probe_gene_map.meta.json'
EXPECTED_N = 121
EXPECTED_GENES = 13016
EXPECTED_MISSING = 27


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def save_json(path, obj):
    path.write_text(
        json.dumps(
            obj, indent=2,
            ensure_ascii=False,
            allow_nan=False
        ),
        encoding='utf-8'
    )


def normalize(x):
    return np.char.upper(
        np.char.strip(np.asarray(x).astype(str))
    )


def parse_fields(line):
    return next(csv.reader(
        [line.rstrip('\r\n')], delimiter='\t'
    ))


def read_annotation(path):
    df = pd.read_csv(
        path, sep='\t',
        dtype=str, keep_default_na=False
    )

    if list(df.columns) != ['ID', 'GENE_SYMBOL']:
        raise RuntimeError(
            'Unexpected GPL14550 annotation schema'
        )

    df['ID'] = df['ID'].str.strip().str.upper()
    df['GENE_SYMBOL'] = (
        df['GENE_SYMBOL'].str.strip().str.upper()
    )

    if df.ID.duplicated().any() or len(df) < 25000:
        raise RuntimeError(
            'Duplicate or incomplete probe annotation'
        )

    return dict(zip(df.ID, df.GENE_SYMBOL))


def read_expression_and_header(
    path, probe_to_gene, desired
):
    metadata = defaultdict(list)
    grouped = defaultdict(list)
    seen = set()
    sample_order = None
    in_table = False
    rows = 0
    invalid = 0

    with gzip.open(
        path, 'rt',
        encoding='utf-8', errors='replace'
    ) as f:
        for line in f:
            if line.startswith(
                '!series_matrix_table_begin'
            ):
                in_table = True
                continue

            if line.startswith(
                '!series_matrix_table_end'
            ):
                break

            if not in_table:
                if (
                    line.startswith('!Sample_')
                    and '\t' in line
                ):
                    key = line.split(
                        '\t', 1
                    )[0].lstrip('!')

                    if key in {
                        'Sample_geo_accession',
                        'Sample_title',
                        'Sample_source_name_ch1',
                        'Sample_characteristics_ch1',
                        'Sample_description'
                    }:
                        metadata[key].append([
                            x.strip().strip('"')
                            for x in parse_fields(line)[1:]
                        ])
                continue

            cells = parse_fields(line)

            if sample_order is None:
                if (
                    cells[0].strip().strip('"').upper()
                    != 'ID_REF'
                ):
                    raise RuntimeError(
                        'GEO matrix missing ID_REF header'
                    )

                sample_order = [
                    x.strip().strip('"').upper()
                    for x in cells[1:]
                ]

                if (
                    len(sample_order) != EXPECTED_N
                    or len(set(sample_order)) != EXPECTED_N
                    or not all(
                        re.fullmatch(r'GSM\d+', s)
                        for s in sample_order
                    )
                ):
                    raise RuntimeError(
                        'GEO matrix GSM roster invalid'
                    )
                continue

            rows += 1
            probe = cells[0].strip().strip('"').upper()

            if probe in seen:
                raise RuntimeError(
                    'Duplicate GEO matrix probe'
                )

            seen.add(probe)
            gene = probe_to_gene.get(probe)

            if gene not in desired:
                continue

            if len(cells) != EXPECTED_N + 1:
                invalid += 1
                continue

            try:
                values = np.asarray(
                    [
                        float(x.strip().strip('"'))
                        for x in cells[1:]
                    ],
                    dtype=np.float32
                )
            except ValueError:
                invalid += 1
                continue

            if not np.isfinite(values).all():
                invalid += 1
                continue

            grouped[gene].append(values)

    if sample_order is None or rows < 30000:
        raise RuntimeError(
            'Incomplete GEO series matrix'
        )

    for key, entries in metadata.items():
        for entry in entries:
            if len(entry) != EXPECTED_N:
                raise RuntimeError(
                    'GEO phenotype header misaligned: '
                    + key
                )

    if 'Sample_geo_accession' in metadata:
        accession = [
            s.upper()
            for s in metadata[
                'Sample_geo_accession'
            ][0]
        ]

        if accession != sample_order:
            raise RuntimeError(
                'GEO metadata GSM order differs '
                'from expression order'
            )

    return (
        sample_order,
        grouped,
        metadata,
        {
            'matrix_probe_rows': rows,
            'invalid_mapped_rows': invalid
        }
    )


def load_base(path, genes):
    with np.load(path, allow_pickle=False) as z:
        fields = (
            'gene_indices', 'genes',
            'mu', 'sd',
            'e_coef', 'e_intercept',
            'k_coef', 'k_intercept'
        )

        if not set(fields).issubset(z.files):
            raise RuntimeError(
                'Incomplete base checkpoint: '
                + str(path)
            )

        a = {
            key: np.array(z[key], copy=True)
            for key in fields
        }

    ix = a['gene_indices'].astype(np.int64)

    if (
        not np.array_equal(ix, np.unique(ix))
        or not np.array_equal(
            normalize(a['genes']), genes[ix]
        )
    ):
        raise RuntimeError(
            'Checkpoint gene identity invalid'
        )

    for name in (
        'mu', 'sd', 'e_coef', 'k_coef'
    ):
        if (
            a[name].shape != ix.shape
            or not np.isfinite(a[name]).all()
        ):
            raise RuntimeError(
                'Checkpoint parameter invalid: '
                + name
            )

    a['gene_indices'] = ix
    return a


def base_logits(observed_ranks, index_map, state):
    selected = state['gene_indices']
    positions = index_map[selected]
    present = positions >= 0

    z = np.zeros(
        (
            len(observed_ranks),
            len(selected)
        ),
        dtype=np.float32
    )

    if present.any():
        r = observed_ranks[
            :, positions[present]
        ]

        z[:, present] = np.tanh(
            (
                r - state['mu'][present]
            ) / (
                2.0 * state['sd'][present]
            )
        ).astype(np.float32)

    e = (
        z @ state['e_coef']
        + state['e_intercept']
    )

    k = (
        z @ state['k_coef']
        + state['k_intercept']
    )

    return (
        np.column_stack((e, k)),
        int((~present).sum())
    )


def four_evidence(raw128, raw512):
    def conditional_logit(s):
        pe = expit(
            np.asarray(
                s[:, 0], dtype=np.float64
            )
        )

        pk = expit(
            np.asarray(
                s[:, 1], dtype=np.float64
            )
        )

        p = np.column_stack((
            pe,
            (1 - pe) * pk,
            (1 - pe) * (1 - pk)
        ))

        e = np.clip(
            p[:, 0], 1e-12, 1 - 1e-12
        )

        k = np.clip(
            p[:, 1], 1e-12, 1
        )

        w = np.clip(
            p[:, 2], 1e-12, 1
        )

        return np.column_stack((
            np.log(e / (1 - e)),
            np.log(k / w)
        ))

    a = conditional_logit(raw128)
    b = conditional_logit(raw512)

    return np.column_stack((
        a[:, 0],
        b[:, 0],
        a[:, 1],
        b[:, 1]
    ))


def predict(raw, meta):
    n = len(raw)

    q = np.column_stack([
        (
            rankdata(
                raw[:, j],
                method='average'
            ) - 0.5
        ) / n
        for j in range(4)
    ])

    q = np.clip(
        q, 1e-5, 1 - 1e-5
    )

    q = (
        np.log(q)
        - np.log1p(-q)
    ) / 2.5

    a = q[:, (0, 2)]
    d = q[:, (1, 3)] - a

    features = np.stack((
        d,
        d * np.tanh(1.5 * a),
        np.tanh(2 * d) * (
            1 - np.tanh(a) ** 2
        )
    ), axis=-1)

    bound = meta['bound']

    z = (
        meta['bias']
        + np.logaddexp(
            0.0, meta['raw_scale']
        ) * a
        + meta['guard'] * bound
        * np.tanh(
            np.einsum(
                'nhk,hk->nh',
                features,
                meta['weights']
            ) / bound
        )
    )

    pe = expit(z[:, 0])
    pk = expit(z[:, 1])

    p = np.column_stack((
        pe,
        (1 - pe) * pk,
        (1 - pe) * (1 - pk)
    ))

    if (
        not np.isfinite(p).all()
        or not np.allclose(
            p.sum(axis=1),
            1.0,
            atol=1e-12
        )
    ):
        raise RuntimeError(
            'Invalid locked probabilities'
        )

    return p


paths = [
    SOURCE,
    SERIES,
    ANNOTATION,
    ANNOTATION_META,
    P19 / 'FINAL_lr128.npz',
    P19 / 'FINAL_lr512.npz',
    P19 / 'FINAL_META_MODEL.npz',
    P19 / 'GATE_STATUS.json',
    P20 / 'INFERENCE_AUDIT.json',
    P21 / 'GENE_COVERAGE.csv',
    P21 / 'GSM_ROSTER.csv',
    P22 / 'SUMMARY.json'
]

for path in paths:
    if not path.is_file():
        raise FileNotFoundError(path)


gate = json.loads(
    (
        P19 / 'GATE_STATUS.json'
    ).read_text(encoding='utf-8')
)

if gate.get('status') != 'PASS_FINAL_BASE_REFIT':
    raise RuntimeError(
        'Final training gate not verified'
    )


with np.load(SOURCE, allow_pickle=False) as z:
    genes = normalize(z['genes'])
    train_gsms = set(
        normalize(z['gsm']).tolist()
    )

if (
    len(genes) != EXPECTED_GENES
    or len(set(genes)) != EXPECTED_GENES
):
    raise RuntimeError(
        'Frozen gene schema invalid'
    )


meta_map = json.loads(
    ANNOTATION_META.read_text(
        encoding='utf-8'
    )
)

if (
    meta_map.get('mapping_sha256')
    != sha256(ANNOTATION)
):
    raise RuntimeError(
        'Cached platform mapping checksum mismatch'
    )


coverage = pd.read_csv(
    P21 / 'GENE_COVERAGE.csv'
)

coverage['gene'] = (
    coverage['gene']
    .astype(str)
    .str.strip()
    .str.upper()
)

coverage = (
    coverage.set_index('gene')
    .loc[genes]
)

present_reference = (
    coverage['mapped']
    .astype(str)
    .str.lower()
    .eq('true')
    .to_numpy()
)

if (
    present_reference.sum()
    != EXPECTED_GENES - EXPECTED_MISSING
):
    raise RuntimeError(
        'Phase21 gene coverage receipt unexpected'
    )


with np.load(
    P19 / 'FINAL_META_MODEL.npz',
    allow_pickle=False
) as z:
    meta = {
        k: np.asarray(
            z[k], dtype=np.float64
        )
        for k in (
            'bias', 'raw_scale', 'weights'
        )
    }

    meta['guard'] = (
        float(
            np.asarray(
                z['guard_lambda']
            ).item()
        )
        if 'guard_lambda' in z.files
        else 1.0
    )

    meta['bound'] = (
        float(
            np.asarray(
                z['correction_bound']
            ).item()
        )
        if 'correction_bound' in z.files
        else 1.5
    )

if (
    meta['bias'].shape != (2,)
    or meta['raw_scale'].shape != (2,)
    or meta['weights'].shape != (2, 3)
    or meta['bound'] <= 0
    or not 0 <= meta['guard'] <= 1
):
    raise RuntimeError(
        'Frozen meta-model schema invalid'
    )


base = {
    k: load_base(
        P19 / f'FINAL_lr{k}.npz',
        genes
    )
    for k in (128, 512)
}

mapping = read_annotation(ANNOTATION)

gsms, gene_groups, metadata, matrix_stats = (
    read_expression_and_header(
        SERIES, mapping, set(genes)
    )
)

if set(gsms) & train_gsms:
    raise RuntimeError(
        'Development GSM overlap'
    )

phase21_gsms = set(
    pd.read_csv(
        P21 / 'GSM_ROSTER.csv',
        dtype=str
    )['gsm'].str.upper()
)

if set(gsms) != phase21_gsms:
    raise RuntimeError(
        'Phase21 GSM roster mismatch'
    )


present = np.asarray(
    [
        g in gene_groups
        for g in genes
    ],
    dtype=bool
)

if not np.array_equal(
    present, present_reference
):
    raise RuntimeError(
        'Phase21 gene-coverage mapping '
        'not reproduced'
    )


ordered_genes = genes[present]

aligned = np.column_stack([
    np.median(
        np.stack(
            gene_groups[g],
            axis=0
        ),
        axis=0
    )
    for g in ordered_genes
]).astype(np.float32)

if (
    aligned.shape != (
        EXPECTED_N,
        EXPECTED_GENES - EXPECTED_MISSING
    )
    or not np.isfinite(aligned).all()
):
    raise RuntimeError(
        'Assay-adapted expression alignment failed'
    )


ranked = (
    rankdata(
        aligned,
        axis=1,
        method='average'
    ).astype(np.float32) - 1
) / (len(ordered_genes) - 1)


index_map = np.full(
    len(genes), -1, dtype=int
)

index_map[np.flatnonzero(present)] = (
    np.arange(present.sum())
)


raw128, missing128 = base_logits(
    ranked, index_map, base[128]
)

raw512, missing512 = base_logits(
    ranked, index_map, base[512]
)

raw = four_evidence(
    raw128, raw512
)

p = predict(raw, meta)


order = np.random.default_rng(
    20261023
).permutation(len(gsms))

invariant = float(
    np.max(np.abs(
        p[order]
        - predict(raw[order], meta)
    ))
)

if invariant > 1e-12:
    raise RuntimeError(
        'External row order invariance failed'
    )


OUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

RUN = OUT_ROOT / (
    datetime.now(timezone.utc).strftime(
        '%Y%m%dT%H%M%SZ_'
    )
    + uuid.uuid4().hex[:8]
)

RUN.mkdir(exist_ok=False)


pred = pd.DataFrame({
    'gsm': gsms,
    'cohort': 'GSE66863',
    'p_egfr': p[:, 0],
    'p_kras': p[:, 1],
    'p_dwt': p[:, 2],
    'predicted_class': p.argmax(axis=1),
    'max_probability': p.max(axis=1)
})

pred_path = (
    RUN / 'FROZEN_EXTERNAL_PREDICTIONS_UNSCORED.csv'
)

pred.to_csv(
    pred_path,
    index=False,
    float_format='%.17g'
)

locked_prediction_sha = sha256(
    pred_path
)


metadata_rows = []

for j, gsm in enumerate(gsms):
    item = {'gsm': gsm}

    for key, arrays in metadata.items():
        parts = [
            arr[j]
            for arr in arrays
        ]

        item[key] = ' | '.join(parts)

    metadata_rows.append(item)


metadata_df = pd.DataFrame(
    metadata_rows
)

metadata_df.to_csv(
    RUN / 'GEO_PUBLIC_SAMPLE_METADATA.csv',
    index=False
)

fields_with_mutation_text = [
    name
    for name in metadata_df.columns
    if (
        name != 'gsm'
        and metadata_df[name]
        .astype(str)
        .str.contains(
            r'\b(?:EGFR|KRAS|MUTATION|GENOTYPE)\b',
            case=False,
            regex=True
        ).any()
    )
]


selected_missing = sorted(
    set(genes[~present])
    & set(
        np.concatenate([
            base[k]['genes']
            for k in (128, 512)
        ]).astype(str)
    )
)

if len(selected_missing) != 2:
    raise RuntimeError(
        'Missing selected gene lineage mismatch'
    )


receipt = {
    'status': (
        'ASSAY_ADAPTED_EXTERNAL_PREDICTIONS_'
        'FROZEN_UNSCORED'
    ),
    'dataset': 'GSE66863',
    'n': len(gsms),
    'observed_genes': int(present.sum()),
    'missing_genes': int(
        (~present).sum()
    ),
    'missing_selected_genes': [
        str(g)
        for g in selected_missing
    ],
    'missing_128_predictors': missing128,
    'missing_512_predictors': missing512,
    'adaptation': (
        'Rank across exactly 12989 observed genes; '
        'absent model predictors set to '
        'training-standardized zero'
    ),
    'probe_aggregation': (
        'Median of all finite GPL14550 '
        'single-symbol mapped probes'
    ),
    'prediction_file_sha256': (
        locked_prediction_sha
    ),
    'row_order_max_error': invariant,
    'external_labels_read_for_scoring': False,
    'external_validation_performed': False,
    'genotype_source_provenance': 'PENDING',
    'genotype_related_metadata_fields': (
        fields_with_mutation_text
    ),
    'model_checksums': {
        f'FINAL_lr{k}.npz': sha256(
            P19 / f'FINAL_lr{k}.npz'
        )
        for k in (128, 512)
    },
    'meta_checksum': sha256(
        P19 / 'FINAL_META_MODEL.npz'
    ),
    'matrix_checksum': sha256(SERIES),
    'annotation_checksum': sha256(ANNOTATION),
    'matrix_stats': matrix_stats
}

save_json(
    RUN / 'FROZEN_PREDICTION_RECEIPT.json',
    receipt
)

save_json(
    RUN / 'RUN_MANIFEST.json',
    {
        'created_utc': datetime.now(
            timezone.utc
        ).isoformat(),
        'input_checksums': {
            str(x): sha256(x)
            for x in paths
        },
        'output_checksums': {
            x.name: sha256(x)
            for x in RUN.iterdir()
            if (
                x.is_file()
                and x.name != 'RUN_MANIFEST.json'
            )
        }
    }
)


counts = np.bincount(
    p.argmax(axis=1),
    minlength=3
)

print(
    'PHASE23: EXTERNAL_PREDICTIONS_FROZEN_UNSCORED'
)

print(
    'GSM:', len(gsms),
    'GENES_MAPPED:', int(present.sum()),
    'GENES_MISSING:', int((~present).sum())
)

print(
    'MODEL_FEATURES_MISSING:',
    [str(g) for g in selected_missing]
)

print(
    'MISSING_MODEL_COLUMNS:',
    'LR128=', missing128,
    'LR512=', missing512
)

print(
    'PREDICTION_COUNTS_UNSCORED:',
    'EGFR=', int(counts[0]),
    'KRAS=', int(counts[1]),
    'DWT=', int(counts[2])
)

print(
    'MEAN_MAX_PROBABILITY:',
    round(
        float(
            p.max(axis=1).mean()
        ),
        6
    )
)

print(
    'GEO_MUTATION_METADATA_FIELDS:',
    fields_with_mutation_text
)

print(
    'ROW_ORDER_ERROR:',
    f'{invariant:.3e}'
)

print(
    'FROZEN_PREDICTION_SHA256:',
    locked_prediction_sha
)

print(
    'GENOTYPE_PROVENANCE: PENDING'
)

print(
    'EXTERNAL_VALIDATION: NOT_RUN'
)

print('RUN:', RUN)

PHASE23: EXTERNAL_PREDICTIONS_FROZEN_UNSCORED
GSM: 121 GENES_MAPPED: 12989 GENES_MISSING: 27
MODEL_FEATURES_MISSING: ['LOC100506098', 'TMEM70']
MISSING_MODEL_COLUMNS: LR128= 1 LR512= 2
PREDICTION_COUNTS_UNSCORED: EGFR= 37 KRAS= 25 DWT= 59
MEAN_MAX_PROBABILITY: 0.623683
GEO_MUTATION_METADATA_FIELDS: ['Sample_characteristics_ch1']
ROW_ORDER_ERROR: 0.000e+00
FROZEN_PREDICTION_SHA256: 8dc5c7eda0a663145116d7481e155ac1877c28392eeb5090c895adb9fcff07f2
GENOTYPE_PROVENANCE: PENDING
EXTERNAL_VALIDATION: NOT_RUN
RUN: D:\LUAD_DATAFIRST_METHOD_2026\23_external_prediction_lock\20261023T200343Z_0abb47df


In [33]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import re
import uuid

import numpy as np
import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    precision_recall_fscore_support,
    f1_score,
    roc_auc_score
)

ROOT = Path(r"D:\LUAD_DATAFIRST_METHOD_2026")
P23 = ROOT / "23_external_prediction_lock" / "20261023T200343Z_0abb47df"
P19 = ROOT / "19_exact_source_base_refit" / "20261023T190611Z_a22f33e7"
CACHE = ROOT / "21_external_gene_eligibility" / "public_geo_cache"
OUT_ROOT = ROOT / "24_locked_external_label_validation"

LOCKED_SHA256 = "8dc5c7eda0a663145116d7481e155ac1877c28392eeb5090c895adb9fcff07f2"
CLASS_NAMES = ["EGFR_MUT", "KRAS_MUT", "EGFR_KRAS_DOUBLE_WT"]
PROB_COLS = ["p_egfr", "p_kras", "p_dwt"]
BOOTSTRAPS = 1200
SEED = 20261023


def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def save_json(path, payload):
    path.write_text(
        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False,
            allow_nan=False
        ),
        encoding="utf-8"
    )


def classify_status(raw):
    value = re.sub(
        r"[\s_\-]+",
        " ",
        raw.strip().casefold()
    ).strip(" .;\t")

    mutant = {
        "mutated",
        "mutant",
        "mutation",
        "positive",
        "mutation detected",
        "yes"
    }

    wild = {
        "wild type",
        "wildtype",
        "wt",
        "not mutated",
        "non mutated",
        "no mutation",
        "negative",
        "mutation not detected",
        "no"
    }

    if value in mutant:
        return "MUT"

    if value in wild:
        return "WT"

    return "UNRESOLVED"


def parse_genes(characteristics):
    evidence = {
        "EGFR": [],
        "KRAS": []
    }

    text = (
        ""
        if pd.isna(characteristics)
        else str(characteristics)
    )

    for token in re.split(r"\s*\|\s*", text):
        matched = re.fullmatch(
            r"\s*(egfr|kras)\s*"
            r"(?:mutation(?:\s+status)?|mutational\s+status)"
            r"\s*:\s*(.*?)\s*",
            token,
            flags=re.IGNORECASE
        )

        if matched:
            gene = matched.group(1).upper()
            raw_value = matched.group(2).strip()

            evidence[gene].append(
                (
                    raw_value,
                    classify_status(raw_value)
                )
            )

    parsed = {}

    for gene in ("EGFR", "KRAS"):
        values = evidence[gene]

        if not values:
            parsed[gene] = ("MISSING", "")

        elif any(
            status == "UNRESOLVED"
            for _, status in values
        ):
            parsed[gene] = (
                "UNRESOLVED",
                " | ".join(
                    raw for raw, _ in values
                )
            )

        elif len(
            set(status for _, status in values)
        ) != 1:
            parsed[gene] = (
                "CONFLICT",
                " | ".join(
                    raw for raw, _ in values
                )
            )

        else:
            parsed[gene] = (
                values[0][1],
                " | ".join(
                    raw for raw, _ in values
                )
            )

    return parsed


def score_metrics(truth, prob):
    pred = prob.argmax(axis=1)

    macro_f1 = f1_score(
        truth,
        pred,
        labels=[0, 1, 2],
        average="macro",
        zero_division=0
    )

    nll = -np.log(
        np.clip(
            prob[np.arange(len(truth)), truth],
            1e-15,
            1
        )
    ).mean()

    recalls = []
    aucs = []

    for k in range(3):
        binary = (truth == k).astype(int)

        if 0 < binary.sum() < len(binary):
            auc = float(
                roc_auc_score(
                    binary,
                    prob[:, k]
                )
            )
        else:
            auc = None

        aucs.append(auc)

        recalls.append(
            float(
                np.mean(
                    pred[binary == 1] == k
                )
            )
            if binary.sum()
            else None
        )

    return {
        "n": int(len(truth)),
        "accuracy": float(
            np.mean(pred == truth)
        ),
        "macro_f1": float(macro_f1),
        "macro_recall": float(
            np.mean([
                value
                for value in recalls
                if value is not None
            ])
        ),
        "nll": float(nll),
        "ovr_auroc": aucs,
        "macro_ovr_auroc": (
            float(np.mean(aucs))
            if all(
                value is not None
                for value in aucs
            )
            else None
        )
    }


pred_path = P23 / "FROZEN_EXTERNAL_PREDICTIONS_UNSCORED.csv"
meta_path = P23 / "GEO_PUBLIC_SAMPLE_METADATA.csv"
receipt_path = P23 / "FROZEN_PREDICTION_RECEIPT.json"

needed = [
    pred_path,
    meta_path,
    receipt_path,
    P19 / "FINAL_lr128.npz",
    P19 / "FINAL_lr512.npz",
    P19 / "FINAL_META_MODEL.npz",
    CACHE / "GSE66863_series_matrix.txt.gz",
    CACHE / "GPL14550_probe_gene_map.tsv"
]

for path in needed:
    if not path.is_file():
        raise FileNotFoundError(path)


receipt = json.loads(
    receipt_path.read_text(
        encoding="utf-8"
    )
)

if receipt.get("status") != (
    "ASSAY_ADAPTED_EXTERNAL_PREDICTIONS_FROZEN_UNSCORED"
):
    raise RuntimeError(
        "Unexpected prediction receipt status"
    )

if not (
    sha256(pred_path)
    == LOCKED_SHA256
    == receipt.get("prediction_file_sha256")
):
    raise RuntimeError(
        "FROZEN PREDICTION HASH MISMATCH"
    )


for k in (128, 512):
    name = f"FINAL_lr{k}.npz"

    if sha256(P19 / name) != (
        receipt["model_checksums"][name]
    ):
        raise RuntimeError(
            f"Base checkpoint integrity failure: {name}"
        )


if sha256(
    P19 / "FINAL_META_MODEL.npz"
) != receipt["meta_checksum"]:
    raise RuntimeError(
        "Meta checkpoint integrity failure"
    )

if sha256(
    CACHE / "GSE66863_series_matrix.txt.gz"
) != receipt["matrix_checksum"]:
    raise RuntimeError(
        "GSE66863 matrix checksum changed"
    )

if sha256(
    CACHE / "GPL14550_probe_gene_map.tsv"
) != receipt["annotation_checksum"]:
    raise RuntimeError(
        "GPL annotation checksum changed"
    )


pred = pd.read_csv(
    pred_path,
    dtype={
        "gsm": str,
        "cohort": str
    }
)

meta = pd.read_csv(
    meta_path,
    dtype=str,
    keep_default_na=False
)


for df, label in (
    (pred, "predictions"),
    (meta, "GEO metadata")
):
    if (
        "gsm" not in df.columns
        or df["gsm"].isna().any()
    ):
        raise RuntimeError(
            f"Invalid {label} GSM"
        )

    df["gsm"] = (
        df["gsm"]
        .str.strip()
        .str.upper()
    )

    if (
        len(df) != 121
        or df["gsm"].duplicated().any()
    ):
        raise RuntimeError(
            f"Invalid {label} GSM count or duplicates"
        )


if set(pred.gsm) != set(meta.gsm):
    raise RuntimeError(
        "Prediction versus GEO metadata GSM mismatch"
    )

if "Sample_characteristics_ch1" not in meta.columns:
    raise RuntimeError(
        "Mutation characteristics field not present"
    )

if not set(
    PROB_COLS + [
        "predicted_class",
        "cohort"
    ]
).issubset(pred.columns):
    raise RuntimeError(
        "Frozen prediction column schema changed"
    )


probs = pred[
    PROB_COLS
].to_numpy(dtype=np.float64)

if (
    not np.isfinite(probs).all()
    or np.any(probs < 0)
    or np.any(probs > 1)
    or not np.allclose(
        probs.sum(axis=1),
        1,
        atol=1e-12,
        rtol=1e-12
    )
    or not np.array_equal(
        probs.argmax(axis=1),
        pred.predicted_class.to_numpy(dtype=int)
    )
    or set(pred.cohort) != {"GSE66863"}
):
    raise RuntimeError(
        "Frozen prediction integrity failure"
    )


frame = pred.merge(
    meta,
    on="gsm",
    how="left",
    validate="one_to_one",
    sort=False
)

records = []

for row in frame.itertuples(index=False):
    parsed = parse_genes(
        getattr(row, "Sample_characteristics_ch1")
    )

    es, er = parsed["EGFR"]
    ks, kr = parsed["KRAS"]

    if es == "MUT" and ks == "WT":
        label = 0
        disposition = "ELIGIBLE"

    elif es == "WT" and ks == "MUT":
        label = 1
        disposition = "ELIGIBLE"

    elif es == "WT" and ks == "WT":
        label = 2
        disposition = "ELIGIBLE"

    elif es == "MUT" and ks == "MUT":
        label = None
        disposition = "DOUBLE_MUTANT_OUT_OF_SCOPE"

    else:
        label = None
        disposition = "INCOMPLETE_OR_CONFLICTING_GENOTYPE"

    records.append({
        "gsm": row.gsm,
        "egfr_status": es,
        "kras_status": ks,
        "egfr_raw": er,
        "kras_raw": kr,
        "label": label,
        "label_status": disposition
    })


provenance = pd.DataFrame(records)

scored = frame.merge(
    provenance,
    on="gsm",
    how="left",
    validate="one_to_one",
    sort=False
)

valid = scored.loc[
    scored.label_status == "ELIGIBLE"
].copy()

valid["label"] = valid["label"].astype(int)


OUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

RUN = OUT_ROOT / (
    datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%SZ_"
    )
    + uuid.uuid4().hex[:8]
)

RUN.mkdir(exist_ok=False)


provenance.to_csv(
    RUN / "GSM_GENOTYPE_PROVENANCE.csv",
    index=False
)


report = {
    "source": (
        "GSE66863 series-matrix "
        "Sample_characteristics_ch1"
    ),
    "labels_from": (
        "Explicit GEO EGFR mutation "
        "and KRAS mutation fields"
    ),
    "not_from": (
        "Predicted probabilities or "
        "inferred absence of mutation"
    ),
    "frozen_prediction_sha256": LOCKED_SHA256,
    "n_total": int(len(frame)),
    "n_eligible": int(len(valid)),
    "label_coverage": float(
        len(valid) / len(frame)
    ),
    "label_status_counts": {
        k: int(v)
        for k, v in provenance.label_status.value_counts().items()
    },
    "egfr_status_counts": {
        k: int(v)
        for k, v in provenance.egfr_status.value_counts().items()
    },
    "kras_status_counts": {
        k: int(v)
        for k, v in provenance.kras_status.value_counts().items()
    },
    "double_mutant_handling": (
        "Quarantine outside 3-class target"
    ),
    "missing_label_handling": (
        "Exclude; unknown is never treated as WT"
    ),
    "model_or_threshold_modified": False,
    "evaluation_type": (
        "Frozen assay-adapted external cohort"
    ),
    "external_platform": "GPL14550",
    "label_parser_version": "explicit-dual-gene-v1"
}


if len(valid) == 0:
    report["status"] = "NO_USABLE_GENOTYPE_LABELS"

    save_json(
        RUN / "LABEL_PROVENANCE_AUDIT.json",
        report
    )

    print(
        "PHASE24: NO_USABLE_GENOTYPE_LABELS"
    )

else:
    valid["true_class"] = (
        valid.label.astype(int)
    )

    valid["true_class_name"] = (
        valid.true_class.map(
            dict(enumerate(CLASS_NAMES))
        )
    )

    valid["predicted_class_name"] = (
        valid.predicted_class.map(
            dict(enumerate(CLASS_NAMES))
        )
    )

    keep = [
        "gsm",
        "cohort",
        "egfr_status",
        "kras_status",
        "true_class",
        "true_class_name",
        "predicted_class",
        "predicted_class_name"
    ] + PROB_COLS

    valid[keep].to_csv(
        RUN / "SCORED_FROZEN_EXTERNAL_PREDICTIONS.csv",
        index=False,
        float_format="%.17g"
    )

    truth = valid.true_class.to_numpy(
        dtype=int
    )

    pr = valid[
        PROB_COLS
    ].to_numpy(dtype=np.float64)

    guess = pr.argmax(axis=1)

    cm = confusion_matrix(
        truth,
        guess,
        labels=[0, 1, 2]
    )

    confusion_df = pd.DataFrame(
        cm,
        index=[
            "true_" + n
            for n in CLASS_NAMES
        ],
        columns=[
            "pred_" + n
            for n in CLASS_NAMES
        ]
    )

    confusion_df.to_csv(
        RUN / "CONFUSION_MATRIX.csv"
    )

    prec, recall, cf1, support = (
        precision_recall_fscore_support(
            truth,
            guess,
            labels=[0, 1, 2],
            zero_division=0
        )
    )

    overall = score_metrics(
        truth,
        pr
    )

    pd.DataFrame({
        "class": CLASS_NAMES,
        "precision": prec,
        "recall": recall,
        "f1": cf1,
        "support": support,
        "one_vs_rest_auroc": (
            overall["ovr_auroc"]
        )
    }).to_csv(
        RUN / "CLASSWISE_METRICS.csv",
        index=False
    )

    overall["true_counts"] = {
        CLASS_NAMES[c]: int(
            (truth == c).sum()
        )
        for c in range(3)
    }

    overall["pred_counts"] = {
        CLASS_NAMES[c]: int(
            (guess == c).sum()
        )
        for c in range(3)
    }

    report["metrics"] = overall

    sufficient = all(
        (truth == c).sum() >= 5
        for c in range(3)
    )

    report["status"] = (
        "FROZEN_ASSAY_ADAPTED_EXTERNAL_VALIDATION_SCORED"
        if sufficient
        else "FROZEN_EXTERNAL_SCORING_LIMITED_CLASS_SUPPORT"
    )

    if sufficient:
        rng = np.random.default_rng(SEED)

        per_class = [
            np.flatnonzero(truth == j)
            for j in range(3)
        ]

        bootstrap = []

        for _ in range(BOOTSTRAPS):
            rows = np.concatenate([
                rng.choice(
                    group,
                    size=len(group),
                    replace=True
                )
                for group in per_class
            ])

            b = score_metrics(
                truth[rows],
                pr[rows]
            )

            bootstrap.append({
                k: b[k]
                for k in (
                    "accuracy",
                    "macro_f1",
                    "macro_recall",
                    "nll",
                    "macro_ovr_auroc"
                )
            })

        boot = pd.DataFrame(
            bootstrap
        )

        ci = []

        for metric in boot.columns:
            lower, upper = np.quantile(
                boot[metric].dropna().to_numpy(),
                [0.025, 0.975]
            )

            ci.append({
                "metric": metric,
                "point": overall[metric],
                "ci95_lower": float(lower),
                "ci95_upper": float(upper)
            })

        pd.DataFrame(ci).to_csv(
            RUN / "STRATIFIED_BOOTSTRAP_95CI.csv",
            index=False
        )

    save_json(
        RUN / "LABEL_PROVENANCE_AUDIT.json",
        report
    )

    print(
        "PHASE24:",
        report["status"]
    )

    print(
        "N_TOTAL:", len(frame),
        "N_ELIGIBLE:", len(valid)
    )

    print(
        "TRUE_CLASS_COUNTS:",
        overall["true_counts"]
    )

    print(
        "ACCURACY:",
        f"{overall['accuracy']:.6f}"
    )

    print(
        "MACRO_F1:",
        f"{overall['macro_f1']:.6f}"
    )

    print(
        "MEAN_NLL:",
        f"{overall['nll']:.6f}"
    )

    print(
        "MACRO_OVR_AUROC:",
        (
            f"{overall['macro_ovr_auroc']:.6f}"
            if overall["macro_ovr_auroc"] is not None
            else "NOT_ESTIMABLE"
        )
    )

    print(
        "CONFUSION_MATRIX:"
    )

    print(
        confusion_df.to_string()
    )


print(
    "LABEL_STATUS_COUNTS:",
    report["label_status_counts"]
)


manifest = {
    "run": str(RUN),
    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "locked_prediction_sha256": (
        LOCKED_SHA256
    ),
    "metadata_sha256": sha256(meta_path),
    "label_audit_status": report["status"],
    "outputs_sha256": {
        p.name: sha256(p)
        for p in RUN.iterdir()
        if p.is_file()
    }
}

save_json(
    RUN / "RUN_MANIFEST.json",
    manifest
)

print("MODEL_RETRAINED: NO")
print("LABEL_SOURCE: EXPLICIT_GEO_CLINICAL_FIELDS")
print("LOCK_SHA256:", LOCKED_SHA256)
print("RUN:", RUN)

PHASE24: FROZEN_ASSAY_ADAPTED_EXTERNAL_VALIDATION_SCORED
N_TOTAL: 121 N_ELIGIBLE: 114
TRUE_CLASS_COUNTS: {'EGFR_MUT': 17, 'KRAS_MUT': 39, 'EGFR_KRAS_DOUBLE_WT': 58}
ACCURACY: 0.666667
MACRO_F1: 0.655480
MEAN_NLL: 0.796132
MACRO_OVR_AUROC: 0.865747
CONFUSION_MATRIX:
                          pred_EGFR_MUT  pred_KRAS_MUT  pred_EGFR_KRAS_DOUBLE_WT
true_EGFR_MUT                        16              0                         1
true_KRAS_MUT                         4             21                        14
true_EGFR_KRAS_DOUBLE_WT             17              2                        39
LABEL_STATUS_COUNTS: {'ELIGIBLE': 114, 'INCOMPLETE_OR_CONFLICTING_GENOTYPE': 7}
MODEL_RETRAINED: NO
LABEL_SOURCE: EXPLICIT_GEO_CLINICAL_FIELDS
LOCK_SHA256: 8dc5c7eda0a663145116d7481e155ac1877c28392eeb5090c895adb9fcff07f2
RUN: D:\LUAD_DATAFIRST_METHOD_2026\24_locked_external_label_validation\20261023T201758Z_8a4f4ab4


In [34]:

from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import csv
import gzip
import hashlib
import json
import uuid

import numpy as np
import pandas as pd
from scipy.special import expit
from scipy.stats import rankdata
from sklearn.metrics import f1_score, roc_auc_score

ROOT = Path(r"D:\LUAD_DATAFIRST_METHOD_2026")
TRAIN = Path(r"D:\LUAD_CMR_2026\CELL03_DATA_20261021T090947_420335Z_d6aed8ec\cmr_gene_expression.npz")
P19 = ROOT / "19_exact_source_base_refit" / "20261023T190611Z_a22f33e7"
P23 = ROOT / "23_external_prediction_lock" / "20261023T200343Z_0abb47df"
P24 = ROOT / "24_locked_external_label_validation" / "20261023T201758Z_8a4f4ab4"
CACHE = ROOT / "21_external_gene_eligibility" / "public_geo_cache"
OUT = ROOT / "25_external_matched_baseline_audit"

LOCK_SHA = "8dc5c7eda0a663145116d7481e155ac1877c28392eeb5090c895adb9fcff07f2"
BOOTSTRAPS = 1000
SEED = 20261023
PROB = ["p_egfr", "p_kras", "p_dwt"]
BASELINES = [
    "LR128",
    "LR512",
    "EQUAL_PROBABILITY_MEAN",
    "FROZEN_META_NO_RESIDUAL"
]


def hash_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def write_json(p, obj):
    p.write_text(
        json.dumps(
            obj,
            ensure_ascii=False,
            indent=2,
            allow_nan=False
        ),
        encoding="utf-8"
    )


def canon(a):
    return np.char.upper(
        np.char.strip(
            np.asarray(a).astype(str)
        )
    )


def load_base(p, genes):
    fields = (
        "gene_indices", "genes",
        "mu", "sd",
        "e_coef", "e_intercept",
        "k_coef", "k_intercept"
    )

    with np.load(p, allow_pickle=False) as z:
        if not set(fields).issubset(z.files):
            raise ValueError(
                "Missing base checkpoint arrays: " + str(p)
            )

        state = {
            key: np.asarray(z[key]).copy()
            for key in fields
        }

    ix = np.asarray(
        state["gene_indices"],
        dtype=np.int64
    )

    if (
        ix.ndim != 1
        or ix.size == 0
        or np.any(ix < 0)
        or np.any(ix >= len(genes))
        or not np.array_equal(ix, np.unique(ix))
        or not np.array_equal(
            canon(state["genes"]),
            genes[ix]
        )
    ):
        raise ValueError(
            "Base gene mapping mismatch: " + str(p)
        )

    for key in ("mu", "sd", "e_coef", "k_coef"):
        if (
            state[key].shape != ix.shape
            or not np.isfinite(state[key]).all()
        ):
            raise ValueError(
                "Invalid parameter " + key + ": " + str(p)
            )

    for key in ("e_intercept", "k_intercept"):
        if (
            state[key].size != 1
            or not np.isfinite(state[key]).all()
        ):
            raise ValueError(
                "Invalid intercept " + key + ": " + str(p)
            )

    if not np.all(state["sd"] > 0):
        raise ValueError("Invalid SD: " + str(p))

    state["gene_indices"] = ix
    return state


def read_expression(p, annotation, target, expected_gsms):
    grouped = defaultdict(list)
    seen = set()
    roster = None
    rows = 0
    within = False

    with gzip.open(
        p, "rt",
        encoding="utf-8",
        errors="replace"
    ) as f:
        for line in f:
            if line.startswith("!series_matrix_table_begin"):
                within = True
                continue

            if line.startswith("!series_matrix_table_end"):
                break

            if not within:
                continue

            fields = next(
                csv.reader(
                    [line.rstrip("\r\n")],
                    delimiter="\t"
                )
            )

            if roster is None:
                if (
                    fields[0].strip().strip('"').upper()
                    != "ID_REF"
                ):
                    raise ValueError(
                        "Missing GEO series matrix ID_REF header"
                    )

                roster = [
                    x.strip().strip('"').upper()
                    for x in fields[1:]
                ]

                if roster != expected_gsms:
                    raise ValueError(
                        "GEO series matrix order differs "
                        "from locked prediction order"
                    )

                continue

            rows += 1

            probe = (
                fields[0].strip().strip('"').upper()
            )

            if probe in seen:
                raise ValueError(
                    "Duplicate GEO probe: " + probe
                )

            seen.add(probe)
            gene = annotation.get(probe)

            if gene not in target:
                continue

            if len(fields) != len(roster) + 1:
                continue

            try:
                vector = np.asarray(
                    [
                        float(x.strip().strip('"'))
                        for x in fields[1:]
                    ],
                    dtype=np.float32
                )
            except ValueError:
                continue

            if np.isfinite(vector).all():
                grouped[gene].append(vector)

    if (
        roster is None
        or len(roster) != 121
        or rows < 30000
    ):
        raise ValueError(
            "GEO expression table incomplete"
        )

    return grouped, rows


def head_probs(s):
    s = np.asarray(s, dtype=np.float64)

    pe = expit(s[:, 0])
    pk = expit(s[:, 1])

    p = np.column_stack((
        pe,
        (1.0 - pe) * pk,
        (1.0 - pe) * (1.0 - pk)
    ))

    if (
        not np.isfinite(p).all()
        or not np.allclose(
            p.sum(axis=1),
            1.0,
            atol=1e-12
        )
    ):
        raise ValueError(
            "Invalid hierarchical probability output"
        )

    return p


def base_scores(ranks, idx_map, state):
    selected = state["gene_indices"]
    positions = idx_map[selected]
    ok = positions >= 0

    z = np.zeros(
        (len(ranks), len(selected)),
        dtype=np.float32
    )

    if ok.any():
        z[:, ok] = np.tanh(
            (
                ranks[:, positions[ok]]
                - state["mu"][ok]
            ) / (
                2.0 * state["sd"][ok]
            )
        ).astype(np.float32)

    return np.column_stack((
        z @ state["e_coef"] + state["e_intercept"],
        z @ state["k_coef"] + state["k_intercept"]
    ))


def eff(s):
    p = head_probs(s)

    pe = np.clip(
        p[:, 0], 1e-12, 1 - 1e-12
    )
    pk = np.clip(
        p[:, 1], 1e-12, 1
    )
    pw = np.clip(
        p[:, 2], 1e-12, 1
    )

    return np.column_stack((
        np.log(pe / (1 - pe)),
        np.log(pk / pw)
    ))


def meta_forward(s128, s512, meta, correction=True):
    a = eff(s128)
    b = eff(s512)

    raw = np.column_stack((
        a[:, 0],
        b[:, 0],
        a[:, 1],
        b[:, 1]
    ))

    n = len(raw)

    u = np.column_stack([
        (
            rankdata(
                raw[:, j],
                method="average"
            ) - 0.5
        ) / n
        for j in range(4)
    ])

    u = np.clip(
        u, 1e-5, 1 - 1e-5
    )

    q = (
        np.log(u) - np.log1p(-u)
    ) / 2.5

    anchor = q[:, (0, 2)]

    z = (
        meta["bias"]
        + np.logaddexp(
            0.0, meta["raw_scale"]
        ) * anchor
    )

    if correction:
        d = q[:, (1, 3)] - anchor

        basis = np.stack((
            d,
            d * np.tanh(1.5 * anchor),
            np.tanh(2.0 * d)
            * (1 - np.tanh(anchor) ** 2)
        ), axis=-1)

        z = z + (
            meta["guard"]
            * meta["bound"]
            * np.tanh(
                np.einsum(
                    "nhk,hk->nh",
                    basis,
                    meta["weights"]
                ) / meta["bound"]
            )
        )

    return head_probs(z)


def evaluate(y, p):
    yh = p.argmax(axis=1)
    conditional = y != 0

    cond_p = (
        p[conditional, 1]
        / np.maximum(
            p[conditional, 1]
            + p[conditional, 2],
            1e-15
        )
    )

    ovrs = [
        roc_auc_score(
            (y == k).astype(int),
            p[:, k]
        )
        for k in range(3)
    ]

    return {
        "accuracy": float(
            (yh == y).mean()
        ),
        "macro_f1": float(
            f1_score(
                y, yh,
                labels=[0, 1, 2],
                average="macro",
                zero_division=0
            )
        ),
        "nll": float(
            -np.log(
                np.clip(
                    p[np.arange(len(y)), y],
                    1e-15, 1
                )
            ).mean()
        ),
        "macro_ovr_auc": float(
            np.mean(ovrs)
        ),
        "egfr_auc": float(
            ovrs[0]
        ),
        "kras_conditional_auc": float(
            roc_auc_score(
                (y[conditional] == 1).astype(int),
                cond_p
            )
        ),
        "brier": float(
            np.mean(
                np.sum(
                    (p - np.eye(3)[y]) ** 2,
                    axis=1
                )
            )
        )
    }


file_pred = (
    P23 / "FROZEN_EXTERNAL_PREDICTIONS_UNSCORED.csv"
)
file_scored = (
    P24 / "SCORED_FROZEN_EXTERNAL_PREDICTIONS.csv"
)
file_receipt = (
    P23 / "FROZEN_PREDICTION_RECEIPT.json"
)
file_audit = (
    P24 / "LABEL_PROVENANCE_AUDIT.json"
)
file_matrix = (
    CACHE / "GSE66863_series_matrix.txt.gz"
)
file_annotation = (
    CACHE / "GPL14550_probe_gene_map.tsv"
)
file_annotation_meta = (
    CACHE / "GPL14550_probe_gene_map.meta.json"
)

files = [
    TRAIN,
    file_pred,
    file_scored,
    file_receipt,
    file_audit,
    file_matrix,
    file_annotation,
    file_annotation_meta
] + [
    P19 / f"FINAL_lr{k}.npz"
    for k in (128, 512)
] + [
    P19 / "FINAL_META_MODEL.npz"
]

for f in files:
    if not f.is_file():
        raise FileNotFoundError(f)


receipt = json.loads(
    file_receipt.read_text(
        encoding="utf-8"
    )
)

prior = json.loads(
    file_audit.read_text(
        encoding="utf-8"
    )
)

if (
    hash_file(file_pred) != LOCK_SHA
    or receipt.get("prediction_file_sha256") != LOCK_SHA
    or prior.get("frozen_prediction_sha256") != LOCK_SHA
):
    raise RuntimeError(
        "External prediction-lock lineage does not match"
    )


for k in (128, 512):
    name = f"FINAL_lr{k}.npz"

    if (
        hash_file(P19 / name)
        != receipt["model_checksums"][name]
    ):
        raise RuntimeError(
            "Changed base checkpoint: " + name
        )


if (
    hash_file(P19 / "FINAL_META_MODEL.npz")
    != receipt["meta_checksum"]
):
    raise RuntimeError(
        "Changed meta checkpoint"
    )

if (
    hash_file(file_matrix) != receipt["matrix_checksum"]
    or hash_file(file_annotation)
    != receipt["annotation_checksum"]
):
    raise RuntimeError(
        "External expression/annotation artifacts changed"
    )


map_receipt = json.loads(
    file_annotation_meta.read_text(
        encoding="utf-8"
    )
)

if (
    map_receipt.get("mapping_sha256")
    != hash_file(file_annotation)
):
    raise RuntimeError(
        "GEO mapping receipt mismatch"
    )


frozen = pd.read_csv(
    file_pred,
    dtype={"gsm": str}
)

scored = pd.read_csv(
    file_scored,
    dtype={"gsm": str}
)

for df in (frozen, scored):
    df["gsm"] = (
        df["gsm"].str.strip().str.upper()
    )

    if (
        df.gsm.duplicated().any()
        or not set(
            PROB + ["gsm"]
        ).issubset(df.columns)
    ):
        raise RuntimeError(
            "Duplicated GSM or malformed scored data"
        )


if (
    len(frozen) != 121
    or len(scored) != 114
    or not set(scored.gsm).issubset(
        set(frozen.gsm)
    )
):
    raise RuntimeError(
        "Unexpected external evaluation cohort roster"
    )


rows = pd.Index(
    frozen.gsm
).get_indexer(
    scored.gsm
)

if (
    (rows < 0).any()
    or not np.allclose(
        scored[PROB].to_numpy(dtype=float),
        frozen.iloc[rows][PROB].to_numpy(dtype=float),
        atol=1e-12,
        rtol=0
    )
):
    raise RuntimeError(
        "Scored predictions differ from locked predictions"
    )


y = scored[
    "true_class"
].to_numpy(dtype=int)

if (
    set(y) != {0, 1, 2}
    or len(y) != 114
):
    raise RuntimeError(
        "Invalid eligible genotype labels"
    )

if tuple(
    np.bincount(y, minlength=3)
) != (17, 39, 58):
    raise RuntimeError(
        "Unexpected genotype label counts"
    )


with np.load(
    TRAIN,
    allow_pickle=False
) as z:
    genes = canon(z["genes"])

if (
    len(genes) != 13016
    or len(set(genes)) != 13016
):
    raise RuntimeError(
        "Frozen training gene schema corrupted"
    )


annotation = pd.read_csv(
    file_annotation,
    sep="\t",
    dtype=str,
    keep_default_na=False
)

if list(annotation.columns) != [
    "ID", "GENE_SYMBOL"
]:
    raise RuntimeError(
        "Unexpected GPL14550 annotation schema"
    )

annotation["ID"] = canon(
    annotation["ID"]
)
annotation["GENE_SYMBOL"] = canon(
    annotation["GENE_SYMBOL"]
)

if (
    annotation.ID.duplicated().any()
    or len(annotation) < 25000
):
    raise RuntimeError(
        "Incomplete platform annotation"
    )

probe_to_gene = dict(zip(
    annotation.ID,
    annotation.GENE_SYMBOL
))


groups, n_probes = read_expression(
    file_matrix,
    probe_to_gene,
    set(genes),
    frozen.gsm.tolist()
)

present = np.asarray([
    g in groups
    for g in genes
], dtype=bool)

if int(present.sum()) != 12989:
    raise RuntimeError(
        "Frozen external gene coverage changed: "
        + str(present.sum())
    )


observed = genes[present]

X = np.column_stack([
    np.median(
        np.stack(
            groups[g],
            axis=0
        ),
        axis=0
    )
    for g in observed
]).astype(np.float32)

if (
    X.shape != (121, 12989)
    or not np.isfinite(X).all()
):
    raise RuntimeError(
        "External expression alignment invalid"
    )


ranked = (
    rankdata(
        X,
        axis=1,
        method="average"
    ).astype(np.float32) - 1
) / (len(observed) - 1)

index_map = np.full(
    len(genes),
    -1,
    dtype=np.int64
)

index_map[
    np.flatnonzero(present)
] = np.arange(
    len(observed),
    dtype=np.int64
)


base = {
    k: load_base(
        P19 / f"FINAL_lr{k}.npz",
        genes
    )
    for k in (128, 512)
}


with np.load(
    P19 / "FINAL_META_MODEL.npz",
    allow_pickle=False
) as z:
    required = {
        "bias",
        "raw_scale",
        "weights"
    }

    if not required.issubset(z.files):
        raise RuntimeError(
            "Incomplete residual meta checkpoint"
        )

    meta = {
        k: np.asarray(
            z[k], dtype=np.float64
        )
        for k in required
    }

    meta["bound"] = (
        float(
            np.asarray(
                z["correction_bound"]
            ).item()
        )
        if "correction_bound" in z.files
        else 1.5
    )

    meta["guard"] = (
        float(
            np.asarray(
                z["guard_lambda"]
            ).item()
        )
        if "guard_lambda" in z.files
        else 1.0
    )


if (
    meta["bias"].shape != (2,)
    or meta["raw_scale"].shape != (2,)
    or meta["weights"].shape != (2, 3)
    or meta["bound"] <= 0
    or not (0 <= meta["guard"] <= 1)
):
    raise RuntimeError(
        "Meta checkpoint parameter schema invalid"
    )


s128 = base_scores(
    ranked,
    index_map,
    base[128]
)

s512 = base_scores(
    ranked,
    index_map,
    base[512]
)

p128 = head_probs(s128)
p512 = head_probs(s512)

predicted_full = meta_forward(
    s128,
    s512,
    meta,
    True
)

lock_probs = frozen[
    PROB
].to_numpy(dtype=np.float64)

replay_error = float(
    np.abs(
        predicted_full - lock_probs
    ).max()
)

if replay_error > 2e-5:
    raise RuntimeError(
        "Phase23 frozen probability reproduction failed: "
        + str(replay_error)
    )


all_models = {
    "LR128": p128[rows],
    "LR512": p512[rows],
    "EQUAL_PROBABILITY_MEAN": (
        0.5 * (p128 + p512)
    )[rows],
    "FROZEN_META_NO_RESIDUAL": (
        meta_forward(
            s128,
            s512,
            meta,
            False
        )[rows]
    ),
    "FROZEN_RESIDUAL": lock_probs[rows]
}


results = []

for name, prob in all_models.items():
    results.append({
        "model": name,
        "n": len(y),
        **evaluate(y, prob)
    })

summary = pd.DataFrame(
    results
).set_index("model")


for metric, reported in (
    ("accuracy", 0.666667),
    ("macro_f1", 0.655480),
    ("nll", 0.796132),
    ("macro_ovr_auc", 0.865747)
):
    observed_value = float(
        summary.loc[
            "FROZEN_RESIDUAL",
            metric
        ]
    )

    if abs(
        observed_value - reported
    ) > 5e-6:
        raise RuntimeError(
            "Phase24 primary external result differs: "
            + metric
        )


def bootstrap_metrics(
    y,
    p,
    boot_indices,
    class_bounds
):
    n_rep = boot_indices.shape[0]
    n = len(y)

    bootstrap_truth = y[boot_indices]
    bootstrap_pred = p.argmax(
        axis=1
    )[boot_indices]

    true_class_log_loss = -np.log(
        np.clip(
            p[np.arange(n), y],
            1e-15,
            1
        )
    )

    nll = true_class_log_loss[
        boot_indices
    ].mean(axis=1)

    f1_vals = []
    auc_vals = []

    for k, (start, end) in enumerate(
        class_bounds
    ):
        tp = np.sum(
            (bootstrap_truth == k)
            & (bootstrap_pred == k),
            axis=1
        )

        fp = np.sum(
            (bootstrap_truth != k)
            & (bootstrap_pred == k),
            axis=1
        )

        fn = (end - start) - tp
        denom = 2 * tp + fp + fn

        f1_vals.append(
            np.divide(
                2 * tp,
                denom,
                out=np.zeros(n_rep),
                where=denom > 0
            )
        )

        positive = p[
            boot_indices[:, start:end],
            k
        ]

        negative_indices = np.concatenate(
            (
                boot_indices[:, :start],
                boot_indices[:, end:]
            ),
            axis=1
        )

        negative = p[
            negative_indices,
            k
        ]

        pair_deltas = (
            positive[:, :, None]
            - negative[:, None, :]
        )

        auc_vals.append(
            np.mean(
                (pair_deltas > 0)
                + 0.5 * (pair_deltas == 0),
                axis=(1, 2)
            )
        )

    return {
        "nll": nll,
        "macro_f1": np.mean(
            f1_vals, axis=0
        ),
        "macro_ovr_auc": np.mean(
            auc_vals, axis=0
        )
    }


rng = np.random.default_rng(SEED)

strata = [
    np.flatnonzero(y == c)
    for c in range(3)
]

boot_ix = np.concatenate(
    [
        rng.choice(
            s,
            size=(BOOTSTRAPS, len(s)),
            replace=True
        )
        for s in strata
    ],
    axis=1
)

sizes = [
    len(s)
    for s in strata
]

class_bounds = list(zip(
    np.cumsum([0] + sizes[:-1]),
    np.cumsum(sizes)
))

boot_metrics = {
    name: bootstrap_metrics(
        y,
        prob,
        boot_ix,
        class_bounds
    )
    for name, prob in all_models.items()
}


paired = []

for baseline in BASELINES:
    differences = {
        "delta_macro_f1": (
            boot_metrics[
                "FROZEN_RESIDUAL"
            ]["macro_f1"]
            - boot_metrics[
                baseline
            ]["macro_f1"]
        ),
        "nll_reduction": (
            boot_metrics[
                baseline
            ]["nll"]
            - boot_metrics[
                "FROZEN_RESIDUAL"
            ]["nll"]
        ),
        "delta_macro_ovr_auc": (
            boot_metrics[
                "FROZEN_RESIDUAL"
            ]["macro_ovr_auc"]
            - boot_metrics[
                baseline
            ]["macro_ovr_auc"]
        )
    }

    point_differences = {
        "delta_macro_f1": float(
            summary.loc[
                "FROZEN_RESIDUAL",
                "macro_f1"
            ] - summary.loc[
                baseline,
                "macro_f1"
            ]
        ),
        "nll_reduction": float(
            summary.loc[
                baseline,
                "nll"
            ] - summary.loc[
                "FROZEN_RESIDUAL",
                "nll"
            ]
        ),
        "delta_macro_ovr_auc": float(
            summary.loc[
                "FROZEN_RESIDUAL",
                "macro_ovr_auc"
            ] - summary.loc[
                baseline,
                "macro_ovr_auc"
            ]
        )
    }

    for metric, arr in differences.items():
        lo, hi = np.quantile(
            arr, [0.025, 0.975]
        )

        paired.append({
            "baseline": baseline,
            "delta_metric": metric,
            "point": point_differences[metric],
            "ci95_low": float(lo),
            "ci95_high": float(hi),
            "positive_favors_residual": True
        })


paired = pd.DataFrame(paired)


OUT.mkdir(
    parents=True,
    exist_ok=True
)

RUN = OUT / (
    datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%SZ_"
    ) + uuid.uuid4().hex[:8]
)

RUN.mkdir(exist_ok=False)


summary.reset_index().to_csv(
    RUN / "MATCHED_MODEL_METRICS.csv",
    index=False,
    float_format="%.12g"
)

paired.to_csv(
    RUN / "PAIRED_STRATIFIED_BOOTSTRAP.csv",
    index=False,
    float_format="%.12g"
)


rows_out = pd.DataFrame({
    "gsm": scored.gsm,
    "true_class": y
})

for name, prob in all_models.items():
    for k, tag in enumerate(PROB):
        rows_out[
            f"{name}_{tag}"
        ] = prob[:, k]


rows_out.to_csv(
    RUN / "MATCHED_SAMPLE_PREDICTIONS.csv",
    index=False,
    float_format="%.17g"
)


write_json(
    RUN / "COMPARISON_PROTOCOL.json",
    {
        "status": (
            "POST_HOC_SECONDARY_MATCHED_EXTERNAL_BENCHMARK"
        ),
        "external_test_already_seen_before_comparator_design": True,
        "development_model_modified": False,
        "external_labels_used_for_fitting": False,
        "prediction_lock_sha256": LOCK_SHA,
        "paired_external_n": int(len(y)),
        "true_class_counts": np.bincount(
            y, minlength=3
        ).tolist(),
        "external_gene_count": int(
            present.sum()
        ),
        "archived_lock_reproduction_max_error": (
            replay_error
        ),
        "bootstrap_replicates": BOOTSTRAPS,
        "bootstrap": (
            "Paired resampling stratified by original "
            "true class, fixed class counts"
        ),
        "baselines": {
            "LR128": (
                "Direct frozen Phase19 LR128 "
                "hierarchical classifier"
            ),
            "LR512": (
                "Direct frozen Phase19 LR512 "
                "hierarchical classifier"
            ),
            "EQUAL_PROBABILITY_MEAN": (
                "Untuned 50/50 probability average "
                "of LR128 and LR512"
            ),
            "FROZEN_META_NO_RESIDUAL": (
                "Frozen Phase18 bias and anchor scales; "
                "nonlinear correction disabled "
                "WITHOUT refitting"
            ),
            "FROZEN_RESIDUAL": (
                "Original Phase23 locked probabilities; "
                "no new threshold or tuning"
            )
        },
        "interpretation": (
            "Exploratory comparison, not a preregistered "
            "superiority test; no novel-method claim inferred."
        )
    }
)


write_json(
    RUN / "RUN_MANIFEST.json",
    {
        "timestamp_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "input_hashes": {
            str(f): hash_file(f)
            for f in files
        },
        "output_hashes": {
            f.name: hash_file(f)
            for f in RUN.iterdir()
            if f.name != "RUN_MANIFEST.json"
        }
    }
)


print(
    "PHASE25: MATCHED_EXTERNAL_COMPARISON_COMPLETE"
)

print(
    "EXTERNAL_LABELLED_N:",
    len(y),
    "LOCK_REPLAY_MAX_ERROR:",
    f"{replay_error:.3e}"
)

print("MODEL_METRICS:")

print(
    summary[
        [
            "accuracy",
            "macro_f1",
            "nll",
            "macro_ovr_auc",
            "egfr_auc",
            "kras_conditional_auc"
        ]
    ].round(5).to_string()
)

print(
    "PAIRED_DELTA_VS_FROZEN_RESIDUAL "
    "(positive favors residual):"
)

print(
    paired.round(5).to_string(index=False)
)

print(
    "STATUS: EXPLORATORY_POST_HOC_NOT_PROOF_OF_NOVELTY"
)

print("RUN:", RUN)

PHASE25: MATCHED_EXTERNAL_COMPARISON_COMPLETE
EXTERNAL_LABELLED_N: 114 LOCK_REPLAY_MAX_ERROR: 1.110e-16
MODEL_METRICS:
                         accuracy  macro_f1      nll  macro_ovr_auc  egfr_auc  kras_conditional_auc
model                                                                                              
LR128                     0.68421   0.67686  1.02943        0.85678   0.90661               0.85765
LR512                     0.70175   0.69190  1.07242        0.87547   0.93633               0.87887
EQUAL_PROBABILITY_MEAN    0.68421   0.67496  0.98170        0.86764   0.91935               0.86693
FROZEN_META_NO_RESIDUAL   0.64912   0.63846  0.83128        0.84570   0.90661               0.85765
FROZEN_RESIDUAL           0.66667   0.65548  0.79613        0.86575   0.92116               0.86605
PAIRED_DELTA_VS_FROZEN_RESIDUAL (positive favors residual):
               baseline        delta_metric    point  ci95_low  ci95_high  positive_favors_residual
                  LR1

In [35]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import uuid

import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from scipy.special import logsumexp
from sklearn.metrics import f1_score, roc_auc_score

ROOT = Path(r"D:\LUAD_DATAFIRST_METHOD_2026")
DATA = Path(
    r"D:\LUAD_CMR_2026\CELL03_DATA_20261021T090947_420335Z_d6aed8ec\cmr_gene_expression.npz"
)
P12 = ROOT / "12_multiresolution_ood_evidence" / "20261023T151555Z_638a3c88"
P23 = ROOT / "23_external_prediction_lock" / "20261023T200343Z_0abb47df"
P24 = ROOT / "24_locked_external_label_validation" / "20261023T201758Z_8a4f4ab4"
P25 = ROOT / "25_external_matched_baseline_audit" / "20261024T052011Z_a71b4922"
OUT = ROOT / "26_development_calibrated_external_controls"

LOCK_SHA = "8dc5c7eda0a663145116d7481e155ac1877c28392eeb5090c895adb9fcff07f2"
PC = ("p_egfr", "p_kras", "p_dwt")
DEV = ("GSE13213", "GSE31210", "GSE32863", "GSE72094")
BOOT = 2000
SEED = 20261023


def digest(p):
    h = hashlib.sha256()

    with p.open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(chunk)

    return h.hexdigest()


def audit_probability(p, name):
    p = np.asarray(p, dtype=np.float64)

    if (
        p.ndim != 2
        or p.shape[1] != 3
        or not np.isfinite(p).all()
        or (p < -1e-12).any()
        or (p > 1 + 1e-12).any()
        or not np.allclose(
            p.sum(axis=1), 1,
            atol=1e-8, rtol=0
        )
    ):
        raise RuntimeError(
            "Invalid probabilities: " + name
        )

    return np.clip(p, 0, 1)


def temperature_apply(p, t):
    lp = np.log(
        np.clip(p, 1e-15, 1.0)
    ) / t

    return np.exp(
        lp - logsumexp(
            lp, axis=1, keepdims=True
        )
    )


def weighted_nll(y, p, w):
    loss = -np.log(
        np.clip(
            p[np.arange(len(y)), y],
            1e-15, 1
        )
    )

    return float(np.sum(w * loss))


def fit_temperature(y, p, cohorts):
    weights = np.zeros(
        len(y), dtype=np.float64
    )

    for cohort in DEV:
        mask = cohorts == cohort

        if mask.sum() < 30:
            raise RuntimeError(
                "Insufficient calibration cohort: "
                + cohort
            )

        weights[mask] = (
            1.0 / (len(DEV) * mask.sum())
        )

    if abs(weights.sum() - 1) > 1e-12:
        raise RuntimeError(
            "Invalid calibration weights"
        )

    def objective(log_t):
        return weighted_nll(
            y,
            temperature_apply(
                p, np.exp(log_t)
            ),
            weights
        )

    lower = np.log(0.05)
    upper = np.log(30.0)

    result = minimize_scalar(
        objective,
        method="bounded",
        bounds=(lower, upper),
        options={
            "xatol": 1e-7,
            "maxiter": 250
        }
    )

    if (
        not result.success
        or not np.isfinite(result.fun)
    ):
        raise RuntimeError(
            "Temperature optimization failed"
        )

    temperature = float(
        np.exp(result.x)
    )

    at_bound = (
        abs(result.x - lower) < 1e-3
        or abs(result.x - upper) < 1e-3
    )

    return (
        temperature,
        float(objective(0.0)),
        float(objective(result.x)),
        bool(at_bound)
    )


def metrics(y, p):
    prediction = p.argmax(axis=1)

    aucs = [
        roc_auc_score(
            (y == k).astype(int),
            p[:, k]
        )
        for k in range(3)
    ]

    return {
        "accuracy": float(
            np.mean(prediction == y)
        ),
        "macro_f1": float(
            f1_score(
                y, prediction,
                labels=[0, 1, 2],
                average="macro",
                zero_division=0
            )
        ),
        "nll": float(
            np.mean(
                -np.log(
                    np.clip(
                        p[np.arange(len(y)), y],
                        1e-15, 1
                    )
                )
            )
        ),
        "brier": float(
            np.mean(
                np.sum(
                    (p - np.eye(3)[y]) ** 2,
                    axis=1
                )
            )
        ),
        "macro_ovr_auc": float(
            np.mean(aucs)
        )
    }


required = [
    DATA,
    P12 / "LOCO_PREDICTIONS.csv",
    P23 / "FROZEN_EXTERNAL_PREDICTIONS_UNSCORED.csv",
    P24 / "SCORED_FROZEN_EXTERNAL_PREDICTIONS.csv",
    P25 / "MATCHED_SAMPLE_PREDICTIONS.csv"
]

for path in required:
    if not path.is_file():
        raise FileNotFoundError(path)


if digest(required[2]) != LOCK_SHA:
    raise RuntimeError(
        "Phase23 prediction lock mismatch"
    )


with np.load(
    DATA, allow_pickle=False
) as z:
    dev = pd.DataFrame({
        "held_cohort": np.char.upper(
            np.char.strip(
                z["cohort"].astype(str)
            )
        ),
        "gsm": np.char.upper(
            np.char.strip(
                z["gsm"].astype(str)
            )
        ),
        "true_class": np.asarray(
            z["y"], dtype=int
        )
    })


if (
    len(dev) != 843
    or dev[
        ["held_cohort", "gsm"]
    ].duplicated().any()
    or set(dev.held_cohort) != set(DEV)
    or tuple(
        dev.true_class.value_counts()
        .reindex([0, 1, 2]).tolist()
    ) != (236, 211, 396)
):
    raise RuntimeError(
        "Development sample roster mismatch"
    )


dev_index = pd.MultiIndex.from_frame(
    dev[["held_cohort", "gsm"]]
)

archive = pd.read_csv(
    P12 / "LOCO_PREDICTIONS.csv",
    dtype={
        "held_cohort": str,
        "gsm": str
    }
)

for key in ("held_cohort", "gsm"):
    archive[key] = (
        archive[key]
        .str.strip().str.upper()
    )


source_probs = {}

for k in (128, 512):
    part = archive.loc[
        archive.model == f"raw{k}"
    ].copy()

    if (
        len(part) != 843
        or part[
            ["held_cohort", "gsm"]
        ].duplicated().any()
        or set(
            map(
                tuple,
                part[
                    ["held_cohort", "gsm"]
                ].values
            )
        ) != set(
            map(
                tuple,
                dev[
                    ["held_cohort", "gsm"]
                ].values
            )
        )
    ):
        raise RuntimeError(
            f"Incomplete source OOF: raw{k}"
        )

    part = part.set_index(
        ["held_cohort", "gsm"]
    ).loc[dev_index]

    if not np.array_equal(
        part.true_class.to_numpy(dtype=int),
        dev.true_class.to_numpy()
    ):
        raise RuntimeError(
            f"OOF genotype mismatch: raw{k}"
        )

    source_probs[f"LR{k}"] = (
        audit_probability(
            part[list(PC)].to_numpy(),
            f"raw{k}"
        )
    )


source_probs["EQUAL_MEAN"] = (
    0.5 * (
        source_probs["LR128"]
        + source_probs["LR512"]
    )
)


scored = pd.read_csv(
    P24 / "SCORED_FROZEN_EXTERNAL_PREDICTIONS.csv",
    dtype={"gsm": str}
)

matched = pd.read_csv(
    P25 / "MATCHED_SAMPLE_PREDICTIONS.csv",
    dtype={"gsm": str}
)

frozen = pd.read_csv(
    P23 / "FROZEN_EXTERNAL_PREDICTIONS_UNSCORED.csv",
    dtype={"gsm": str}
)


for name, frame in (
    ("scored", scored),
    ("matched", matched),
    ("frozen", frozen)
):
    frame["gsm"] = (
        frame.gsm.str.strip().str.upper()
    )

    if (
        frame.gsm.isna().any()
        or frame.gsm.duplicated().any()
    ):
        raise RuntimeError(
            "Invalid GSM keys: " + name
        )


if (
    len(scored) != 114
    or len(matched) != 114
    or len(frozen) != 121
    or set(scored.gsm) != set(matched.gsm)
    or not set(scored.gsm).issubset(
        set(frozen.gsm)
    )
):
    raise RuntimeError(
        "External sample roster mismatch"
    )


matched = (
    matched.set_index("gsm")
    .loc[scored.gsm]
    .reset_index()
)

frozen = (
    frozen.set_index("gsm")
    .loc[scored.gsm]
    .reset_index()
)

external_y = scored.true_class.to_numpy(
    dtype=int
)

if (
    tuple(
        np.bincount(
            external_y, minlength=3
        )
    ) != (17, 39, 58)
    or not np.array_equal(
        matched.true_class.to_numpy(
            dtype=int
        ),
        external_y
    )
):
    raise RuntimeError(
        "External genotype labels differ "
        "from Phase24"
    )


external = {}

for key, prefix in (
    ("LR128", "LR128"),
    ("LR512", "LR512"),
    ("EQUAL_MEAN", "EQUAL_PROBABILITY_MEAN"),
    ("FROZEN_RESIDUAL", "FROZEN_RESIDUAL")
):
    columns = [
        f"{prefix}_{c}"
        for c in PC
    ]

    if not set(columns).issubset(
        matched.columns
    ):
        raise RuntimeError(
            "Missing Phase25 columns: "
            + str(columns)
        )

    external[key] = audit_probability(
        matched[columns].to_numpy(),
        key
    )


if (
    np.max(np.abs(
        external["FROZEN_RESIDUAL"]
        - frozen[list(PC)].to_numpy(
            dtype=float
        )
    )) > 1e-12
    or np.max(np.abs(
        external["FROZEN_RESIDUAL"]
        - scored[list(PC)].to_numpy(
            dtype=float
        )
    )) > 1e-12
):
    raise RuntimeError(
        "Frozen residual prediction mismatch"
    )


reference = metrics(
    external_y,
    external["FROZEN_RESIDUAL"]
)

for field, expected in (
    ("accuracy", 0.666667),
    ("macro_f1", 0.655480),
    ("nll", 0.796132),
    ("macro_ovr_auc", 0.865747)
):
    if abs(
        reference[field] - expected
    ) > 5e-6:
        raise RuntimeError(
            "Phase24 score mismatch: "
            + field
        )


cal_params = []

for name in (
    "LR128",
    "LR512",
    "EQUAL_MEAN"
):
    temperature, before, after, at_bound = (
        fit_temperature(
            dev.true_class.to_numpy(),
            source_probs[name],
            dev.held_cohort.to_numpy()
        )
    )

    external[
        f"TEMP_{name}"
    ] = temperature_apply(
        external[name],
        temperature
    )

    cal_params.append({
        "baseline": name,
        "temperature": temperature,
        "source_equal_cohort_nll_before": before,
        "source_equal_cohort_nll_after": after,
        "temperature_near_bound": at_bound,
        "calibration_n": 843
    })

    print(
        f"SOURCE_TEMPERATURE {name} "
        f"T={temperature:.6f} "
        f"LOCO_NLL={before:.5f}->{after:.5f} "
        f"AT_BOUND={at_bound}"
    )


ordered = (
    "LR128",
    "LR512",
    "EQUAL_MEAN",
    "TEMP_LR128",
    "TEMP_LR512",
    "TEMP_EQUAL_MEAN",
    "FROZEN_RESIDUAL"
)

metric_rows = [
    {
        "model": name,
        **metrics(
            external_y,
            external[name]
        )
    }
    for name in ordered
]

metric_frame = pd.DataFrame(
    metric_rows
).set_index("model")


rng = np.random.default_rng(SEED)

classes = [
    np.flatnonzero(
        external_y == i
    )
    for i in range(3)
]

boots = np.concatenate(
    [
        rng.choice(
            ix,
            size=(BOOT, len(ix)),
            replace=True
        )
        for ix in classes
    ],
    axis=1
)


residual_loss = -np.log(
    np.clip(
        external["FROZEN_RESIDUAL"][
            np.arange(len(external_y)),
            external_y
        ],
        1e-15, 1
    )
)


paired = []

for name in (
    "TEMP_LR128",
    "TEMP_LR512",
    "TEMP_EQUAL_MEAN"
):
    baseline_loss = -np.log(
        np.clip(
            external[name][
                np.arange(len(external_y)),
                external_y
            ],
            1e-15, 1
        )
    )

    difference = (
        baseline_loss - residual_loss
    )

    bootstrap_difference = (
        difference[boots].mean(axis=1)
    )

    low, high = np.quantile(
        bootstrap_difference,
        [0.025, 0.975]
    )

    paired.append({
        "calibrated_baseline": name,
        "nll_advantage_residual": float(
            difference.mean()
        ),
        "ci95_low": float(low),
        "ci95_high": float(high),
        "positive_favors_residual": True
    })


OUT.mkdir(
    parents=True, exist_ok=True
)

RUN = OUT / (
    datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%SZ_"
    )
    + "_"
    + uuid.uuid4().hex[:8]
)

RUN.mkdir(exist_ok=False)


pd.DataFrame(cal_params).to_csv(
    RUN / "DEVELOPMENT_TEMPERATURES.csv",
    index=False
)

metric_frame.reset_index().to_csv(
    RUN / "MATCHED_EXTERNAL_METRICS.csv",
    index=False
)

pd.DataFrame(paired).to_csv(
    RUN / "PAIRED_NLL_VS_CALIBRATED.csv",
    index=False
)


with (
    RUN / "CALIBRATION_PROTOCOL.json"
).open("w", encoding="utf-8") as f:
    json.dump(
        {
            "status": (
                "EXPLORATORY_POST_HOC_EXTERNAL_COMPARISON"
            ),
            "calibration_source": (
                "Phase12 four-cohort LOCO "
                "held-out predictions only"
            ),
            "fitting_objective": (
                "Equal-cohort weighted multiclass NLL"
            ),
            "optimization": (
                "One positive temperature per "
                "baseline, bounded 0.05 to 30"
            ),
            "external_labels_used_to_fit_temperature": False,
            "external_test_historically_exposed": True,
            "bootstrap": (
                "Paired class-stratified, 2000 repetitions"
            ),
            "frozen_prediction_sha256": LOCK_SHA,
            "frozen_final_model_unchanged": True,
            "does_not_prove_method_novelty": True,
            "outputs": {
                p.name: digest(p)
                for p in RUN.glob("*.csv")
            }
        },
        f,
        indent=2
    )


print(
    "PHASE26: "
    "DEVELOPMENT_ONLY_CALIBRATION_COMPARISON_COMPLETE"
)

print("MATCHED_EXTERNAL_METRICS:")

print(
    metric_frame[
        [
            "accuracy",
            "macro_f1",
            "nll",
            "brier",
            "macro_ovr_auc"
        ]
    ].round(6).to_string()
)

print(
    "PAIRED_NLL_ADVANTAGE_OF_RESIDUAL:"
)

print(
    pd.DataFrame(
        paired
    ).round(6).to_string(index=False)
)

print("NO_EXTERNAL_PARAMETER_FITTING: TRUE")
print("NOVELTY_STATUS: NOT_PROVEN")
print("RUN:", RUN)

SOURCE_TEMPERATURE LR128 T=2.627837 LOCO_NLL=1.13707->0.91408 AT_BOUND=False
SOURCE_TEMPERATURE LR512 T=5.937969 LOCO_NLL=1.96649->0.98318 AT_BOUND=False
SOURCE_TEMPERATURE EQUAL_MEAN T=3.173679 LOCO_NLL=1.25912->0.95212 AT_BOUND=False
PHASE26: DEVELOPMENT_ONLY_CALIBRATION_COMPARISON_COMPLETE
MATCHED_EXTERNAL_METRICS:
                 accuracy  macro_f1       nll     brier  macro_ovr_auc
model                                                                 
LR128            0.684211  0.676860  1.029432  0.484627       0.856778
LR512            0.701754  0.691898  1.072417  0.491930       0.875472
EQUAL_MEAN       0.684211  0.674963  0.981700  0.464656       0.867635
TEMP_LR128       0.684211  0.676860  0.778934  0.454601       0.859425
TEMP_LR512       0.701754  0.691898  0.799738  0.472933       0.879669
TEMP_EQUAL_MEAN  0.684211  0.674963  0.768057  0.447364       0.869997
FROZEN_RESIDUAL  0.666667  0.655480  0.796132  0.458756       0.865747
PAIRED_NLL_ADVANTAGE_OF_RESIDUAL:
calibra

In [36]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import uuid

import numpy as np
import pandas as pd
from scipy.special import expit, logsumexp
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score

ROOT = Path(r"D:\LUAD_DATAFIRST_METHOD_2026")
DATA = Path(
    r"D:\LUAD_CMR_2026\CELL03_DATA_20261021T090947_420335Z_d6aed8ec\cmr_gene_expression.npz"
)

P14 = ROOT / "14_quantile_residual_transfer" / "20261023T160742Z_f9435239"
P19 = ROOT / "19_exact_source_base_refit" / "20261023T190611Z_a22f33e7"
P23 = ROOT / "23_external_prediction_lock" / "20261023T200343Z_0abb47df"
P24 = ROOT / "24_locked_external_label_validation" / "20261023T201758Z_8a4f4ab4"
P25 = ROOT / "25_external_matched_baseline_audit" / "20261024T052011Z_a71b4922"
P26 = ROOT / "26_development_calibrated_external_controls" / "20261024T052536Z__b95cc0d9"

OUT = ROOT / "27_resolution_disagreement_information_audit"

LOCK = "8dc5c7eda0a663145116d7481e155ac1877c28392eeb5090c895adb9fcff07f2"

COHORTS = (
    "GSE13213",
    "GSE31210",
    "GSE32863",
    "GSE72094"
)

P = ("p_egfr", "p_kras", "p_dwt")


def file_sha(path):
    h = hashlib.sha256()

    with path.open("rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(block)

    return h.hexdigest()


def write_json(path, obj):
    path.write_text(
        json.dumps(
            obj,
            indent=2,
            allow_nan=False
        ),
        encoding="utf-8"
    )


def check_prob(p, label):
    p = np.asarray(
        p, dtype=np.float64
    )

    if (
        p.ndim != 2
        or p.shape[1] != 3
        or not np.isfinite(p).all()
        or (p < 0).any()
        or (p > 1).any()
        or not np.allclose(
            p.sum(axis=1),
            1,
            atol=1e-9,
            rtol=0
        )
    ):
        raise RuntimeError(
            "Invalid probabilities: " + label
        )

    return p


def to_evidence(p):
    p = check_prob(
        p, "evidence input"
    )

    pe = np.clip(
        p[:, 0], 1e-12, 1 - 1e-12
    )
    pk = np.clip(
        p[:, 1], 1e-12, 1
    )
    pw = np.clip(
        p[:, 2], 1e-12, 1
    )

    return np.column_stack((
        np.log(pe / (1 - pe)),
        np.log(pk / pw)
    ))


def evidence4(p128, p512):
    a = to_evidence(p128)
    b = to_evidence(p512)

    return np.column_stack((
        a[:, 0],
        b[:, 0],
        a[:, 1],
        b[:, 1]
    ))


def cohort_quantile(raw):
    n = len(raw)

    if n < 25:
        raise RuntimeError(
            "Too few samples for frozen quantile transform"
        )

    u = np.column_stack([
        (
            rankdata(
                raw[:, j],
                method="average"
            ) - 0.5
        ) / n
        for j in range(4)
    ])

    u = np.clip(
        u, 1e-5, 1 - 1e-5
    )

    return (
        np.log(u) - np.log1p(-u)
    ) / 2.5


def head_parameters(path):
    with np.load(
        path, allow_pickle=False
    ) as z:
        needed = (
            "bias",
            "raw_scale",
            "weights"
        )

        if not set(needed).issubset(z.files):
            raise RuntimeError(
                "Missing head checkpoint parameters: "
                + str(path)
            )

        h = {
            k: np.asarray(
                z[k], dtype=np.float64
            )
            for k in needed
        }

        h["guard"] = (
            float(
                np.asarray(
                    z["guard_lambda"]
                ).item()
            )
            if "guard_lambda" in z.files
            else 1.0
        )

        h["cap"] = (
            float(
                np.asarray(
                    z["correction_bound"]
                ).item()
            )
            if "correction_bound" in z.files
            else 1.5
        )

    if (
        h["bias"].shape != (2,)
        or h["raw_scale"].shape != (2,)
        or h["weights"].shape != (2, 3)
        or h["cap"] <= 0
        or not 0 <= h["guard"] <= 1
        or not all(
            np.isfinite(h[k]).all()
            for k in needed
        )
    ):
        raise RuntimeError(
            "Invalid frozen head parameters: "
            + str(path)
        )

    return h


def probability_from_logits(s):
    pe = expit(s[:, 0])
    pk = expit(s[:, 1])

    p = np.column_stack((
        pe,
        (1 - pe) * pk,
        (1 - pe) * (1 - pk)
    ))

    return check_prob(p, "head")


def residual_forward(
    q, head,
    residual=True,
    basis_off=None
):
    a = q[:, (0, 2)]
    d = q[:, (1, 3)] - a

    s = (
        head["bias"]
        + np.logaddexp(
            0, head["raw_scale"]
        ) * a
    )

    if residual:
        f = np.stack((
            d,
            d * np.tanh(1.5 * a),
            np.tanh(2 * d)
            * (1 - np.tanh(a) ** 2)
        ), axis=-1)

        if basis_off is not None:
            for j in basis_off:
                f[:, :, j] = 0.0

        v = np.einsum(
            "nhk,hk->nh",
            f,
            head["weights"]
        )

        s = (
            s
            + head["guard"]
            * head["cap"]
            * np.tanh(
                v / head["cap"]
            )
        )

    return probability_from_logits(s)


def loss(y, p):
    return -np.log(
        np.clip(
            p[np.arange(len(y)), y],
            1e-15, 1
        )
    )


def subset_table(
    source, cohort, name, ids
):
    part = source.loc[
        (source.held_cohort == cohort)
        & (source.model == name)
    ]

    if (
        len(part) != len(ids)
        or part.gsm.duplicated().any()
        or set(part.gsm) != set(ids)
    ):
        raise RuntimeError(
            f"Incomplete archived OOF roster "
            f"{cohort}/{name}"
        )

    return (
        part.set_index("gsm")
        .loc[ids]
    )


def error_auc(y, p, magnitude):
    wrong = (
        p.argmax(axis=1) != y
    ).astype(int)

    if len(np.unique(wrong)) < 2:
        return None

    return float(
        roc_auc_score(
            wrong, magnitude
        )
    )


def confidence_matched_error_gap(
    y, baseline, d
):
    confidence = baseline.max(axis=1)

    mistakes = (
        baseline.argmax(axis=1) != y
    )

    gaps = []
    sizes = []

    partitions = np.array_split(
        np.argsort(
            confidence, kind="stable"
        ),
        4
    )

    for pos in partitions:
        ordered = pos[
            np.argsort(
                d[pos], kind="stable"
            )
        ]

        low, high = np.array_split(
            ordered, 2
        )

        gaps.append(
            float(
                mistakes[high].mean()
                - mistakes[low].mean()
            )
        )

        sizes.append(len(pos))

    return float(
        np.average(
            gaps, weights=sizes
        )
    )


def summarize_group(
    cohort,
    y,
    p128,
    p512,
    anchor,
    residual,
    d_raw,
    d_q=None
):
    high = (
        d_raw >= np.quantile(
            d_raw, 0.75
        )
    )

    loss_diff = (
        loss(y, anchor)
        - loss(y, residual)
    )

    r = residual.argmax(axis=1)
    b = p512.argmax(axis=1)

    entry = {
        "cohort": cohort,
        "n": len(y),
        "high_raw_disagreement_n": int(
            high.sum()
        ),
        "residual_gain_vs_anchor_nll": float(
            loss_diff.mean()
        ),
        "gain_high_disagreement": float(
            loss_diff[high].mean()
        ),
        "gain_other_samples": float(
            loss_diff[~high].mean()
        ),
        "gain_high_minus_other": float(
            loss_diff[high].mean()
            - loss_diff[~high].mean()
        ),
        "residual_nll": float(
            loss(y, residual).mean()
        ),
        "lr512_nll": float(
            loss(y, p512).mean()
        ),
        "lr512_error_auc_from_raw_disagreement": (
            error_auc(y, p512, d_raw)
        ),
        "lr512_error_gap_within_confidence_quartiles": (
            confidence_matched_error_gap(
                y, p512, d_raw
            )
        ),
        "lr512_error_high": float(
            (b[high] != y[high]).mean()
        ),
        "lr512_error_other": float(
            (b[~high] != y[~high]).mean()
        ),
        "rescue_vs_lr512": int(
            (
                (b != y)
                & (r == y)
            ).sum()
        ),
        "harm_vs_lr512": int(
            (
                (b == y)
                & (r != y)
            ).sum()
        ),
        "correct_when_both_bases_wrong": int(
            (
                (p128.argmax(axis=1) != y)
                & (b != y)
                & (r == y)
            ).sum()
        ),
        "residual_wrong_when_both_bases_correct": int(
            (
                (p128.argmax(axis=1) == y)
                & (b == y)
                & (r != y)
            ).sum()
        )
    }

    if d_q is not None:
        high_q = (
            d_q >= np.quantile(
                d_q, 0.75
            )
        )

        entry["gain_high_q_disagreement"] = float(
            loss_diff[high_q].mean()
        )

        entry["gain_other_q_disagreement"] = float(
            loss_diff[~high_q].mean()
        )

    return entry


paths = [
    DATA,
    P14 / "LOCO_PREDICTIONS.csv",
    P23 / "FROZEN_EXTERNAL_PREDICTIONS_UNSCORED.csv",
    P24 / "SCORED_FROZEN_EXTERNAL_PREDICTIONS.csv",
    P25 / "MATCHED_SAMPLE_PREDICTIONS.csv",
    P26 / "DEVELOPMENT_TEMPERATURES.csv",
    P19 / "FINAL_lr128.npz",
    P19 / "FINAL_lr512.npz"
]

paths += [
    P14 / "checkpoints"
    / f"{c}_q_residual.npz"
    for c in COHORTS
]

for path in paths:
    if not path.is_file():
        raise FileNotFoundError(path)


if file_sha(
    P23 / "FROZEN_EXTERNAL_PREDICTIONS_UNSCORED.csv"
) != LOCK:
    raise RuntimeError(
        "Frozen external prediction SHA256 mismatch"
    )


archive = pd.read_csv(
    P14 / "LOCO_PREDICTIONS.csv",
    dtype={
        "gsm": str,
        "held_cohort": str
    }
)

archive["gsm"] = (
    archive.gsm
    .str.strip()
    .str.upper()
)

required = {
    "held_cohort",
    "gsm",
    "true_class",
    "model",
    *P
}

if not required.issubset(archive.columns):
    raise RuntimeError(
        "Phase14 prediction columns do not match"
    )


source_metrics = []
source_sample = []


for cohort in COHORTS:
    outer = archive.loc[
        (archive.held_cohort == cohort)
        & (archive.model == "raw128")
    ]

    if (
        outer.gsm.duplicated().any()
        or len(outer) < 25
    ):
        raise RuntimeError(
            "Outer GSM roster invalid: " + cohort
        )

    ids = outer.gsm.tolist()
    y = outer.true_class.to_numpy(
        dtype=int
    )

    names = (
        "raw128",
        "raw512",
        "q_anchor",
        "q_residual"
    )

    models = {
        name: check_prob(
            subset_table(
                archive, cohort, name, ids
            )[list(P)].to_numpy(dtype=float),
            name
        )
        for name in names
    }

    for name in names[1:]:
        archived_labels = subset_table(
            archive, cohort, name, ids
        ).true_class.to_numpy(dtype=int)

        if not np.array_equal(
            archived_labels, y
        ):
            raise RuntimeError(
                "Archive labels differ: "
                + cohort + "/" + name
            )

    evidence = evidence4(
        models["raw128"],
        models["raw512"]
    )

    q = cohort_quantile(evidence)

    head = head_parameters(
        P14 / "checkpoints"
        / f"{cohort}_q_residual.npz"
    )

    reconstructed = residual_forward(
        q, head
    )

    replay = float(
        np.max(np.abs(
            reconstructed
            - models["q_residual"]
        ))
    )

    if replay > 1e-4:
        raise RuntimeError(
            f"Phase14 residual replay mismatch "
            f"{cohort}: {replay:.3e}"
        )

    reconstructed_anchor = residual_forward(
        q, head, residual=False
    )

    anchor_replay = float(
        np.max(np.abs(
            reconstructed_anchor
            - models["q_anchor"]
        ))
    )

    if anchor_replay > 1e-4:
        raise RuntimeError(
            f"Phase14 anchor replay mismatch "
            f"{cohort}: {anchor_replay:.3e}"
        )

    d_raw = np.linalg.norm(
        evidence[:, (1, 3)]
        - evidence[:, (0, 2)],
        axis=1
    )

    d_q = np.linalg.norm(
        q[:, (1, 3)]
        - q[:, (0, 2)],
        axis=1
    )

    summary = summarize_group(
        cohort,
        y,
        models["raw128"],
        models["raw512"],
        models["q_anchor"],
        models["q_residual"],
        d_raw,
        d_q
    )

    ablated = residual_forward(
        q,
        head,
        basis_off=(1, 2)
    )

    summary["conditional_basis_nll_benefit"] = float(
        loss(y, ablated).mean()
        - loss(
            y, models["q_residual"]
        ).mean()
    )

    summary["max_frozen_head_replay_error"] = replay

    source_metrics.append(summary)

    anchor_loss = loss(
        y, models["q_anchor"]
    )

    residual_loss = loss(
        y, models["q_residual"]
    )

    for i, gsm in enumerate(ids):
        source_sample.append({
            "cohort": cohort,
            "gsm": gsm,
            "true_class": int(y[i]),
            "disagreement_raw": float(
                d_raw[i]
            ),
            "disagreement_quantile": float(
                d_q[i]
            ),
            "lr128_pred": int(
                models["raw128"][i].argmax()
            ),
            "lr512_pred": int(
                models["raw512"][i].argmax()
            ),
            "residual_pred": int(
                models["q_residual"][i].argmax()
            ),
            "gain_vs_anchor_nll": float(
                anchor_loss[i]
                - residual_loss[i]
            )
        })


source_df = pd.DataFrame(
    source_metrics
)

samples_df = pd.DataFrame(
    source_sample
)

if (
    len(samples_df) != 843
    or samples_df[
        ["cohort", "gsm"]
    ].duplicated().any()
):
    raise RuntimeError(
        "Unexpected development sample count"
    )


frozen = pd.read_csv(
    P23 / "FROZEN_EXTERNAL_PREDICTIONS_UNSCORED.csv",
    dtype={"gsm": str}
)

scored = pd.read_csv(
    P24 / "SCORED_FROZEN_EXTERNAL_PREDICTIONS.csv",
    dtype={"gsm": str}
)

matched = pd.read_csv(
    P25 / "MATCHED_SAMPLE_PREDICTIONS.csv",
    dtype={"gsm": str}
)


for frame in (frozen, scored, matched):
    frame["gsm"] = (
        frame.gsm.str.strip().str.upper()
    )

    if frame.gsm.duplicated().any():
        raise RuntimeError(
            "Duplicate GSM in external files"
        )


if (
    len(frozen) != 121
    or len(scored) != 114
    or len(matched) != 114
    or set(matched.gsm) != set(scored.gsm)
):
    raise RuntimeError(
        "Unexpected external evaluation roster"
    )


matched = (
    matched.set_index("gsm")
    .loc[scored.gsm]
    .reset_index()
)

frozen = (
    frozen.set_index("gsm")
    .loc[scored.gsm]
    .reset_index()
)

y_ext = scored.true_class.to_numpy(
    dtype=int
)

if tuple(
    np.bincount(
        y_ext, minlength=3
    )
) != (17, 39, 58):
    raise RuntimeError(
        "External genotype class counts changed"
    )


external = {}

model_names = (
    "LR128",
    "LR512",
    "EQUAL_PROBABILITY_MEAN",
    "FROZEN_META_NO_RESIDUAL",
    "FROZEN_RESIDUAL"
)

for key in model_names:
    columns = [
        f"{key}_{p}"
        for p in P
    ]

    external[key] = check_prob(
        matched[
            columns
        ].to_numpy(dtype=float),
        key
    )


locked = check_prob(
    scored[list(P)].to_numpy(dtype=float),
    "locked scored"
)

if (
    np.max(np.abs(
        locked
        - external["FROZEN_RESIDUAL"]
    )) > 1e-12
    or np.max(np.abs(
        locked
        - frozen[list(P)].to_numpy(
            dtype=float
        )
    )) > 1e-12
):
    raise RuntimeError(
        "Residual external predictions "
        "do not match frozen lock"
    )


temps = pd.read_csv(
    P26 / "DEVELOPMENT_TEMPERATURES.csv"
)

tmp = temps.loc[
    temps.baseline == "EQUAL_MEAN",
    "temperature"
]

if (
    len(tmp) != 1
    or not np.isfinite(tmp.iloc[0])
    or tmp.iloc[0] <= 0
):
    raise RuntimeError(
        "Invalid Phase26 development-only temperature"
    )

T = float(tmp.iloc[0])

q_temp = np.log(
    np.clip(
        external["EQUAL_PROBABILITY_MEAN"],
        1e-15, 1
    )
) / T

external["TEMP_EQUAL_MEAN"] = np.exp(
    q_temp
    - logsumexp(
        q_temp,
        axis=1,
        keepdims=True
    )
)


if abs(
    loss(
        y_ext,
        external["TEMP_EQUAL_MEAN"]
    ).mean() - 0.768057
) > 5e-5:
    raise RuntimeError(
        "Phase26 calibrated ensemble not reproduced"
    )


if abs(
    loss(y_ext, locked).mean()
    - 0.796132
) > 5e-5:
    raise RuntimeError(
        "Phase24 frozen residual score not reproduced"
    )


evidence_ext = evidence4(
    external["LR128"],
    external["LR512"]
)

d_raw_ext = np.linalg.norm(
    evidence_ext[:, (1, 3)]
    - evidence_ext[:, (0, 2)],
    axis=1
)

external_metrics = summarize_group(
    "GSE66863",
    y_ext,
    external["LR128"],
    external["LR512"],
    external["FROZEN_META_NO_RESIDUAL"],
    locked,
    d_raw_ext
)

external_metrics[
    "nll_gain_over_calibrated_ensemble"
] = float(
    loss(
        y_ext,
        external["TEMP_EQUAL_MEAN"]
    ).mean()
    - loss(y_ext, locked).mean()
)


rescue_rows = []

for class_id, label in enumerate(
    ("EGFR", "KRAS", "DWT")
):
    mask = y_ext == class_id

    r_correct = (
        locked.argmax(axis=1) == y_ext
    )

    for base_name in (
        "LR512",
        "TEMP_EQUAL_MEAN"
    ):
        b_correct = (
            external[base_name].argmax(axis=1)
            == y_ext
        )

        rescue_rows.append({
            "class": label,
            "baseline": base_name,
            "n": int(mask.sum()),
            "rescued_by_residual": int(
                (
                    mask
                    & ~b_correct
                    & r_correct
                ).sum()
            ),
            "lost_by_residual": int(
                (
                    mask
                    & b_correct
                    & ~r_correct
                ).sum()
            ),
            "both_wrong": int(
                (
                    mask
                    & ~b_correct
                    & ~r_correct
                ).sum()
            )
        })

rescue_df = pd.DataFrame(
    rescue_rows
)


with np.load(
    DATA, allow_pickle=False
) as z:
    cohort_src = np.char.upper(
        np.char.strip(
            z["cohort"].astype(str)
        )
    )

    gsm_src = np.char.upper(
        np.char.strip(
            z["gsm"].astype(str)
        )
    )

    y_src = np.asarray(
        z["y"], dtype=int
    )

    genes_src = np.char.upper(
        np.char.strip(
            z["genes"].astype(str)
        )
    )

    with np.load(
        P19 / "FINAL_lr512.npz",
        allow_pickle=False
    ) as b512:
        indices512 = np.asarray(
            b512["gene_indices"],
            dtype=int
        )

    with np.load(
        P19 / "FINAL_lr128.npz",
        allow_pickle=False
    ) as b128:
        indices128 = np.asarray(
            b128["gene_indices"],
            dtype=int
        )

    selected = np.unique(
        np.concatenate((
            indices128,
            indices512
        ))
    )

    if (
        len(genes_src) != 13016
        or not np.array_equal(
            np.unique(selected), selected
        )
        or np.any(selected < 0)
        or np.any(selected >= 13016)
    ):
        raise RuntimeError(
            "Selected gene schema invalid"
        )

    X = np.asarray(
        z["X"], dtype=np.float32
    )


if (
    X.shape != (843, 13016)
    or not np.isfinite(X).all()
):
    raise RuntimeError(
        "Development expression matrix schema changed"
    )


rank_selected = np.empty(
    (843, len(selected)),
    dtype=np.float32
)

for start in range(0, len(X), 32):
    end = min(
        start + 32, len(X)
    )

    full_ranks = rankdata(
        X[start:end],
        axis=1,
        method="average"
    )

    rank_selected[start:end] = (
        (
            full_ranks[:, selected]
            .astype(np.float32) - 1
        ) / 13015
    )


d_lookup = samples_df.set_index(
    ["cohort", "gsm"]
)["disagreement_raw"]

keys = pd.MultiIndex.from_arrays((
    cohort_src,
    gsm_src
))

if (
    not keys.is_unique
    or not keys.isin(
        d_lookup.index
    ).all()
):
    raise RuntimeError(
        "Source expression rows cannot be matched "
        "to LOCO disagreement scores"
    )

d_src = d_lookup.loc[
    keys
].to_numpy(dtype=float)


label_lookup = samples_df.set_index(
    ["cohort", "gsm"]
)["true_class"]

if not np.array_equal(
    label_lookup.loc[
        keys
    ].to_numpy(dtype=int),
    y_src
):
    raise RuntimeError(
        "Archived LOCO labels differ from "
        "development expression labels"
    )


rho_rows = []

for cohort in COHORTS:
    loc = np.flatnonzero(
        cohort_src == cohort
    )

    r = rank_selected[
        loc
    ].astype(np.float64)

    d = d_src[loc].copy()

    for class_id in (0, 1, 2):
        mask = (
            y_src[loc] == class_id
        )

        if mask.sum() < 10:
            raise RuntimeError(
                "Insufficient genotype class "
                "for gene scan: " + cohort
            )

        r[mask] -= r[
            mask
        ].mean(axis=0)

        d[mask] -= d[
            mask
        ].mean()

    numer = d @ r

    denom = np.sqrt(
        (d @ d)
        * np.sum(
            r * r, axis=0
        )
    )

    rho = np.divide(
        numer,
        denom,
        out=np.zeros(
            len(selected)
        ),
        where=denom > 0
    )

    rho_rows.append(rho)


rhos = np.stack(
    rho_rows
)

coherent = (
    np.all(
        rhos > 0, axis=0
    )
    | np.all(
        rhos < 0, axis=0
    )
)

med_abs = np.median(
    np.abs(rhos),
    axis=0
)


gene_table = pd.DataFrame({
    "gene": genes_src[selected],
    "median_abs_within_genotype_r": med_abs,
    "same_sign_all_4_cohorts": coherent
})

for j, cohort in enumerate(COHORTS):
    gene_table[
        "r_" + cohort
    ] = rhos[j]


gene_table[
    "candidate_threshold_pass"
] = (
    coherent
    & (med_abs >= 0.15)
    & (
        np.min(
            np.abs(rhos),
            axis=0
        ) >= 0.05
    )
)

gene_table = gene_table.sort_values(
    "median_abs_within_genotype_r",
    ascending=False
)


OUT.mkdir(
    parents=True,
    exist_ok=True
)

RUN = OUT / (
    datetime.now(
        timezone.utc
    ).strftime("%Y%m%dT%H%M%SZ_")
    + uuid.uuid4().hex[:8]
)

RUN.mkdir(exist_ok=False)


source_df.to_csv(
    RUN / "DEVELOPMENT_LOCO_MECHANISM.csv",
    index=False
)

samples_df.to_csv(
    RUN / "DEVELOPMENT_SAMPLE_DISAGREEMENT.csv",
    index=False
)

pd.DataFrame([
    external_metrics
]).to_csv(
    RUN / "EXTERNAL_DISAGREEMENT.csv",
    index=False
)

rescue_df.to_csv(
    RUN / "EXTERNAL_RESCUE_HARM_BY_GENOTYPE.csv",
    index=False
)

gene_table.to_csv(
    RUN / "GENE_ASSOCIATIONS_EXPLORATORY.csv",
    index=False
)


write_json(
    RUN / "AUDIT_PROTOCOL.json",
    {
        "status": "EXPLORATORY_NO_RETRAINING",
        "frozen_external_sha256": LOCK,
        "external_outcomes_seen_before_this_analysis": True,
        "raw_disagreement": (
            "Euclidean LR512-minus-LR128 "
            "contrast-logit difference"
        ),
        "quantile_disagreement": (
            "Euclidean difference of "
            "cohort-ranked evidence coordinates"
        ),
        "disagreement_subgroup": (
            "Top quartile defined from predictions "
            "within each cohort; labels unused"
        ),
        "gene_screen": (
            "Within-cohort, within-genotype "
            "centered correlations of selected "
            "transcript ranks with raw disagreement"
        ),
        "gene_candidate_threshold": (
            "Same sign in all four cohorts; "
            "median absolute r >= 0.15; "
            "every cohort absolute r >= 0.05"
        ),
        "confidence_control": (
            "LR512 maximum-probability quartiles; "
            "within-bin high-versus-low disagreement"
        ),
        "conditional_basis_ablation": (
            "Zero anchor-conditioned and "
            "uncertainty-weighted residual terms "
            "without refitting"
        ),
        "no_model_fitting_or_external_threshold_tuning": True,
        "limitations": [
            "Development cohorts historically exposed during architecture design",
            "External cohort already used for benchmarking",
            "Gene associations are descriptive, not biological validation"
        ]
    }
)


write_json(
    RUN / "RUN_MANIFEST.json",
    {
        "created_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "input_sha256": {
            str(p): file_sha(p)
            for p in paths
        },
        "outputs_sha256": {
            p.name: file_sha(p)
            for p in RUN.iterdir()
            if p.name != "RUN_MANIFEST.json"
        }
    }
)


print(
    "PHASE27: DISAGREEMENT_INFORMATION_AUDIT_COMPLETE"
)

print("DEVELOPMENT_COHORTS:")

print(
    source_df[
        [
            "cohort",
            "residual_gain_vs_anchor_nll",
            "conditional_basis_nll_benefit",
            "gain_high_minus_other",
            "lr512_error_auc_from_raw_disagreement",
            "lr512_error_gap_within_confidence_quartiles",
            "rescue_vs_lr512",
            "harm_vs_lr512",
            "correct_when_both_bases_wrong"
        ]
    ].round(5).to_string(index=False)
)

print("EXTERNAL_MECHANISM:")

print(
    pd.DataFrame([
        external_metrics
    ])[
        [
            "residual_gain_vs_anchor_nll",
            "gain_high_minus_other",
            "lr512_error_auc_from_raw_disagreement",
            "lr512_error_gap_within_confidence_quartiles",
            "rescue_vs_lr512",
            "harm_vs_lr512",
            "correct_when_both_bases_wrong",
            "nll_gain_over_calibrated_ensemble"
        ]
    ].round(5).to_string(index=False)
)

print("EXTERNAL_RESCUE_HARM_BY_CLASS:")

print(
    rescue_df.to_string(index=False)
)

print("CROSS_COHORT_GENE_CANDIDATES:")

print(
    gene_table.loc[
        gene_table.candidate_threshold_pass
    ].head(12).round(3).to_string(index=False)
)

print(
    "CONSISTENT_EXPRESSION_CANDIDATES:",
    int(
        gene_table.candidate_threshold_pass.sum()
    ),
    "/",
    len(selected)
)

print(
    "STATUS: EXPLORATORY_MECHANISM_EVIDENCE_NOT_PROOF_OF_NOVELTY"
)

print("RUN:", RUN)

PHASE27: DISAGREEMENT_INFORMATION_AUDIT_COMPLETE
DEVELOPMENT_COHORTS:
  cohort  residual_gain_vs_anchor_nll  conditional_basis_nll_benefit  gain_high_minus_other  lr512_error_auc_from_raw_disagreement  lr512_error_gap_within_confidence_quartiles  rescue_vs_lr512  harm_vs_lr512  correct_when_both_bases_wrong
GSE13213                      0.01565                        0.00646                0.04544                                0.36712                                      0.00997               46             19                             33
GSE31210                      0.01620                        0.00588               -0.13044                                0.43504                                     -0.00265               52             21                             25
GSE32863                      0.00276                       -0.00162               -0.01587                                0.29510                                     -0.27648                9              1      

In [37]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import uuid

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy.optimize import minimize_scalar
from scipy.special import expit, logsumexp
from scipy.stats import rankdata
from sklearn.metrics import f1_score, roc_auc_score

ROOT = Path(r"D:\LUAD_DATAFIRST_METHOD_2026")
P14 = ROOT / "14_quantile_residual_transfer" / "20261023T160742Z_f9435239"
OUT = ROOT / "28_matched_mechanism_comparison"

COHORTS = (
    "GSE13213", "GSE31210",
    "GSE32863", "GSE72094"
)

COUNTS = {
    "GSE13213": (45, 15, 57),
    "GSE31210": (127, 20, 79),
    "GSE32863": (17, 22, 19),
    "GSE72094": (47, 154, 241)
}

PCOLS = ["p_egfr", "p_kras", "p_dwt"]

SEED = 20261023
EPOCHS = 400
LR = 0.045
L2 = 0.02
CAP = 1.5
NBOOT = 3000


def checksum(path):
    sha = hashlib.sha256()

    with path.open("rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            sha.update(block)

    return sha.hexdigest()


def valid(p, name):
    p = np.asarray(p, dtype=np.float64)

    if (
        p.ndim != 2
        or p.shape[1] != 3
        or not np.isfinite(p).all()
        or np.any(p < 0)
        or np.any(p > 1)
        or not np.allclose(
            p.sum(1), 1, atol=1e-7
        )
    ):
        raise RuntimeError(
            "Invalid probabilities: " + name
        )

    return p


def evidence(p128, p512):
    def e(p):
        pe = np.clip(
            p[:, 0], 1e-12, 1 - 1e-12
        )

        return np.column_stack((
            np.log(pe / (1 - pe)),
            np.log(
                np.clip(p[:, 1], 1e-12, 1)
                / np.clip(p[:, 2], 1e-12, 1)
            )
        ))

    a = e(p128)
    b = e(p512)

    return np.column_stack((
        a[:, 0], b[:, 0],
        a[:, 1], b[:, 1]
    ))


def from_evidence(v):
    pe = expit(v[:, 0])
    pk = expit(v[:, 1])

    return valid(
        np.column_stack((
            pe,
            (1 - pe) * pk,
            (1 - pe) * (1 - pk)
        )),
        "from evidence"
    )


def rank_evidence(raw, source):
    raw = np.asarray(raw, float)
    source = np.asarray(source).astype(str)

    q = np.empty_like(raw)

    for c in np.unique(source):
        ix = np.flatnonzero(source == c)

        if len(ix) < 25:
            raise RuntimeError(
                "Too few samples: " + c
            )

        for j in range(4):
            u = np.clip(
                (
                    rankdata(
                        raw[ix, j],
                        method="average"
                    ) - 0.5
                ) / len(ix),
                1e-5,
                1 - 1e-5
            )

            q[ix, j] = (
                np.log(u / (1 - u)) / 2.5
            )

    return q


def phi(q, kind):
    a = q[:, (0, 2)]
    d = q[:, (1, 3)] - a

    if kind == "CORDIB":
        return np.stack((
            d,
            d * np.tanh(1.5 * a),
            np.tanh(2 * d)
            * (1 - np.tanh(a) ** 2)
        ), axis=-1)

    if kind == "D_ONLY_6":
        return np.stack((
            d,
            np.tanh(2 * d),
            d * np.tanh(d) ** 2
        ), axis=-1)

    if kind == "GENERIC_CONDITIONAL_6":
        return np.stack((
            d,
            d * a,
            d * a * a
        ), axis=-1)

    if kind == "AFFINE_STACK_6":
        return np.stack((
            np.ones_like(a),
            a,
            q[:, (1, 3)]
        ), axis=-1)

    if kind == "ANCHOR_NONLINEAR_6":
        return np.stack((
            a,
            np.tanh(a),
            a * np.tanh(a)
        ), axis=-1)

    if kind == "STATIC":
        return d[:, :, None]

    raise ValueError(kind)


def anchor_score(q, bias, raw_scale):
    return (
        bias
        + np.logaddexp(0, raw_scale)
        * q[:, (0, 2)]
    )


def prob_from_scores(s):
    pe = expit(s[:, 0])
    pk = expit(s[:, 1])

    return valid(
        np.column_stack((
            pe,
            (1 - pe) * pk,
            (1 - pe) * (1 - pk)
        )),
        "logit head"
    )


def checkpoint(fp, k):
    with np.load(
        fp, allow_pickle=False
    ) as z:
        needed = (
            "bias",
            "raw_scale",
            "weights"
        )

        if not set(needed).issubset(z.files):
            raise RuntimeError(
                "Missing checkpoint keys: "
                + str(fp)
            )

        b, s, w = (
            np.asarray(
                z[n], dtype=np.float64
            )
            for n in needed
        )

        cap = float(
            np.asarray(
                z["correction_bound"]
            ).item()
        )

        g = float(
            np.asarray(
                z["guard_lambda"]
            ).item()
        )

    if (
        b.shape != (2,)
        or s.shape != (2,)
        or w.shape != (2, k)
        or abs(cap - CAP) > 1e-12
        or not 0 <= g <= 1
    ):
        raise RuntimeError(
            "Checkpoint mismatch: "
            + str(fp)
        )

    return b, s, w, g


def predict_head(
    q, b, s,
    w=None, g=1.0, kind=None
):
    logits = anchor_score(q, b, s)

    if w is not None:
        basis = phi(q, kind)

        logits += (
            g * CAP
            * np.tanh(
                np.sum(
                    basis * w[None],
                    axis=-1
                ) / CAP
            )
        )

    return prob_from_scores(logits)


def fit_frozen_anchor_residual(
    q, y, source, b, s, kind
):
    x = torch.as_tensor(
        phi(q, kind),
        dtype=torch.float64
    )

    base = torch.as_tensor(
        anchor_score(q, b, s),
        dtype=torch.float64
    )

    yy = torch.as_tensor(
        y, dtype=torch.long
    )

    groups = [
        torch.as_tensor(source == c)
        for c in np.unique(source)
    ]

    w = torch.nn.Parameter(
        torch.zeros(
            (2, 3),
            dtype=torch.float64
        )
    )

    optimizer = torch.optim.Adam(
        [w], lr=LR
    )

    for _ in range(EPOCHS):
        optimizer.zero_grad(
            set_to_none=True
        )

        z = (
            base
            + CAP * torch.tanh(
                (x * w).sum(-1) / CAP
            )
        )

        logp = torch.stack((
            F.logsigmoid(z[:, 0]),
            F.logsigmoid(-z[:, 0])
            + F.logsigmoid(z[:, 1]),
            F.logsigmoid(-z[:, 0])
            + F.logsigmoid(-z[:, 1])
        ), dim=1)

        rowloss = -logp[
            torch.arange(len(yy)), yy
        ]

        objective = (
            torch.stack([
                rowloss[m].mean()
                for m in groups
            ]).mean()
            + L2 * w.square().mean()
        )

        if not torch.isfinite(objective):
            raise RuntimeError(
                "Nonfinite optimization: "
                + kind
            )

        objective.backward()
        optimizer.step()

    return (
        w.detach()
        .numpy()
        .copy()
    )


def temperature_fit(p, y, source):
    source = np.asarray(
        source
    ).astype(str)

    lp = np.log(
        np.clip(p, 1e-15, 1)
    )

    groupix = [
        np.flatnonzero(source == c)
        for c in np.unique(source)
    ]

    def objective(logt):
        logits = lp / np.exp(logt)

        losses = (
            logsumexp(logits, axis=1)
            - logits[
                np.arange(len(y)), y
            ]
        )

        return np.mean([
            losses[ix].mean()
            for ix in groupix
        ])

    fit = minimize_scalar(
        objective,
        bounds=(
            np.log(0.25),
            np.log(20)
        ),
        method="bounded",
        options={"xatol": 1e-9}
    )

    if not fit.success:
        raise RuntimeError(
            "Temperature fitting failed"
        )

    return float(np.exp(fit.x))


def apply_temperature(p, t):
    lp = np.log(
        np.clip(p, 1e-15, 1)
    ) / t

    return valid(
        np.exp(
            lp - logsumexp(
                lp,
                axis=1,
                keepdims=True
            )
        ),
        "temperature"
    )


def sample_loss(y, p):
    return -np.log(
        np.clip(
            p[np.arange(len(y)), y],
            1e-15, 1
        )
    )


def report_metrics(y, p):
    pred = p.argmax(axis=1)

    return {
        "nll": float(
            sample_loss(y, p).mean()
        ),
        "macro_f1": float(
            f1_score(
                y, pred,
                labels=[0, 1, 2],
                average="macro",
                zero_division=0
            )
        ),
        "macro_auc": float(
            np.mean([
                roc_auc_score(
                    y == c,
                    p[:, c]
                )
                for c in range(3)
            ])
        ),
        "brier": float(
            np.mean(
                np.sum(
                    (
                        p - np.eye(3)[y]
                    ) ** 2,
                    axis=1
                )
            )
        )
    }


outer_fp = (
    P14 / "LOCO_PREDICTIONS.csv"
)

if not outer_fp.is_file():
    raise FileNotFoundError(outer_fp)

outer = pd.read_csv(
    outer_fp,
    dtype={
        "held_cohort": str,
        "gsm": str,
        "model": str
    }
)

required = {
    "held_cohort",
    "gsm",
    "model",
    "true_class",
    *PCOLS
}

if not required.issubset(
    outer.columns
):
    raise RuntimeError(
        "Phase14 prediction columns missing"
    )

if outer.duplicated([
    "held_cohort",
    "gsm",
    "model"
]).any():
    raise RuntimeError(
        "Duplicate archived predictions"
    )


models = (
    "q_anchor",
    "q_static",
    "q_residual",
    "CORDIB_INTERACTIONS_OFF",
    "D_ONLY_6",
    "GENERIC_CONDITIONAL_6",
    "AFFINE_STACK_6",
    "ANCHOR_NONLINEAR_6",
    "TEMP_EQUAL_MEAN"
)

comparators = tuple(
    m for m in models
    if m != "q_residual"
)

results = []
by_cohort = {}
row_dump = []
fitted_params = []
inputs = [outer_fp]
replay_max = 0.0


for held in COHORTS:
    train_fp = (
        P14 / "inner_oof"
        / f"{held}.npz"
    )

    head_dir = P14 / "checkpoints"

    head_paths = {
        m: (
            head_dir
            / f"{held}_{m}.npz"
        )
        for m in (
            "q_anchor",
            "q_static",
            "q_residual"
        )
    }

    for fp in (
        train_fp,
        *head_paths.values()
    ):
        if not fp.is_file():
            raise FileNotFoundError(fp)

        inputs.append(fp)

    with np.load(
        train_fp, allow_pickle=False
    ) as z:
        needed = {
            "rank_evidence",
            "oof_scores",
            "label",
            "source"
        }

        if not needed.issubset(z.files):
            raise RuntimeError(
                "Inner OOF schema mismatch: "
                + held
            )

        train_q = np.asarray(
            z["rank_evidence"],
            dtype=np.float64
        )

        raw = np.asarray(
            z["oof_scores"],
            dtype=np.float64
        )

        train_y = np.asarray(
            z["label"], dtype=int
        )

        source = np.asarray(
            z["source"]
        ).astype(str)

    if (
        train_q.shape != raw.shape
        or raw.ndim != 2
        or raw.shape[1] != 4
        or len(train_y) != len(source)
        or not np.isfinite(
            train_q
        ).all()
        or not np.array_equal(
            np.unique(train_y),
            [0, 1, 2]
        )
        or set(source) != (
            set(COHORTS) - {held}
        )
    ):
        raise RuntimeError(
            "Invalid inner OOF data: "
            + held
        )

    if not np.allclose(
        train_q,
        rank_evidence(raw, source),
        atol=1e-10,
        rtol=0
    ):
        raise RuntimeError(
            "Quantile transform mismatch: "
            + held
        )

    for c in set(COHORTS) - {held}:
        actual = tuple(
            int(
                (
                    (source == c)
                    & (train_y == i)
                ).sum()
            )
            for i in range(3)
        )

        if actual != COUNTS[c]:
            raise RuntimeError(
                "Training roster mismatch: "
                + held + "/" + c
            )

    def get_model(name):
        f = outer.loc[
            (outer.held_cohort == held)
            & (outer.model == name)
        ].sort_values("gsm")

        if len(f) != sum(COUNTS[held]):
            raise RuntimeError(
                "Incomplete outer predictions: "
                + held + "/" + name
            )

        return f

    anchor_df = get_model(
        "q_anchor"
    )

    gsms = anchor_df.gsm.to_numpy(
        dtype=str
    )

    y = anchor_df.true_class.to_numpy(
        dtype=int
    )

    actual_counts = tuple(
        int((y == i).sum())
        for i in range(3)
    )

    if (
        len(set(gsms)) != len(gsms)
        or actual_counts != COUNTS[held]
    ):
        raise RuntimeError(
            "Outer cohort roster mismatch: "
            + held
        )

    outer_p = {}

    for key in (
        "raw128",
        "raw512",
        "q_anchor",
        "q_static",
        "q_residual"
    ):
        f = get_model(key)

        if (
            not np.array_equal(
                f.gsm.to_numpy(str), gsms
            )
            or not np.array_equal(
                f.true_class.to_numpy(int), y
            )
        ):
            raise RuntimeError(
                "Prediction alignment mismatch: "
                + held + "/" + key
            )

        outer_p[key] = valid(
            f[PCOLS].to_numpy(float),
            key
        )

    eq = evidence(
        outer_p["raw128"],
        outer_p["raw512"]
    )

    test_q = rank_evidence(
        eq,
        np.repeat(
            held, len(y)
        )
    )

    heads = {
        m: checkpoint(
            path,
            0 if m == "q_anchor"
            else (
                1 if m == "q_static"
                else 3
            )
        )
        for m, path in head_paths.items()
    }

    ab, asc, aw, ag = (
        heads["q_anchor"]
    )

    for kind in (
        "q_anchor",
        "q_static",
        "q_residual"
    ):
        b, s, w, g = heads[kind]

        if (
            not np.allclose(
                b, ab, atol=1e-10
            )
            or not np.allclose(
                s, asc, atol=1e-10
            )
        ):
            raise RuntimeError(
                "Frozen anchor mismatch: "
                + held
            )

        replay = predict_head(
            test_q,
            b, s,
            None if kind == "q_anchor"
            else w,
            g,
            "STATIC"
            if kind == "q_static"
            else "CORDIB"
        )

        err = float(
            np.max(
                np.abs(
                    replay - outer_p[kind]
                )
            )
        )

        replay_max = max(
            replay_max, err
        )

        if err > 2e-6:
            raise RuntimeError(
                f"Phase14 replay failed: "
                f"{held}/{kind}, "
                f"error={err:.3g}"
            )

    result_p = {
        m: outer_p[m]
        for m in (
            "q_anchor",
            "q_static",
            "q_residual"
        )
    }

    b, s, w, g = (
        heads["q_residual"]
    )

    w_off = w.copy()
    w_off[:, 1:] = 0.0

    result_p[
        "CORDIB_INTERACTIONS_OFF"
    ] = predict_head(
        test_q,
        b, s,
        w_off, g,
        "CORDIB"
    )

    for kind in (
        "D_ONLY_6",
        "GENERIC_CONDITIONAL_6",
        "AFFINE_STACK_6",
        "ANCHOR_NONLINEAR_6"
    ):
        w = fit_frozen_anchor_residual(
            train_q,
            train_y,
            source,
            ab, asc,
            kind
        )

        result_p[kind] = predict_head(
            test_q,
            ab, asc,
            w, 1.0,
            kind
        )

        fitted_params.append({
            "held_cohort": held,
            "model": kind,
            "weights": w.tolist(),
            "train_source_count": int(
                len(np.unique(source))
            ),
            "train_n": len(train_y)
        })

    inner128 = from_evidence(
        raw[:, (0, 2)]
    )

    inner512 = from_evidence(
        raw[:, (1, 3)]
    )

    inner_mean = valid(
        (inner128 + inner512) / 2,
        "inner mean"
    )

    temp = temperature_fit(
        inner_mean,
        train_y,
        source
    )

    test_mean = valid(
        (
            outer_p["raw128"]
            + outer_p["raw512"]
        ) / 2,
        "outer mean"
    )

    result_p[
        "TEMP_EQUAL_MEAN"
    ] = apply_temperature(
        test_mean, temp
    )

    fitted_params.append({
        "held_cohort": held,
        "model": "TEMP_EQUAL_MEAN",
        "temperature": temp,
        "train_n": len(train_y)
    })

    by_cohort[held] = {
        "y": y,
        "p": result_p,
        "gsm": gsms
    }

    for m in models:
        results.append({
            "cohort": held,
            "model": m,
            "n": len(y),
            **report_metrics(
                y, result_p[m]
            )
        })

        for i, gsm in enumerate(gsms):
            row_dump.append({
                "cohort": held,
                "gsm": gsm,
                "true_class": int(y[i]),
                "model": m,
                **{
                    col: float(
                        result_p[m][i, j]
                    )
                    for j, col
                    in enumerate(PCOLS)
                }
            })


if sum(
    len(by_cohort[c]["y"])
    for c in COHORTS
) != 843:
    raise RuntimeError(
        "Development sample count mismatch"
    )


met = pd.DataFrame(results)

summary = (
    met.groupby("model", sort=False)[
        [
            "nll",
            "macro_f1",
            "macro_auc",
            "brier"
        ]
    ]
    .mean()
    .reindex(models)
)

per_source = []
comparisons = []

rng = np.random.default_rng(
    SEED
)

for comp in comparators:
    gains = []

    bootstrap = np.empty(
        (len(COHORTS), NBOOT)
    )

    for ci, cohort in enumerate(
        COHORTS
    ):
        r = by_cohort[cohort]
        yc = r["y"]

        diff = (
            sample_loss(
                yc,
                r["p"][comp]
            )
            - sample_loss(
                yc,
                r["p"]["q_residual"]
            )
        )

        gain = float(diff.mean())

        gains.append(gain)

        per_source.append({
            "cohort": cohort,
            "comparator": comp,
            "n": len(yc),
            "nll_gain_cordib": gain,
            "macro_f1_gain_cordib": float(
                report_metrics(
                    yc,
                    r["p"]["q_residual"]
                )["macro_f1"]
                - report_metrics(
                    yc,
                    r["p"][comp]
                )["macro_f1"]
            )
        })

        groups = [
            np.flatnonzero(yc == k)
            for k in range(3)
        ]

        simulated = np.zeros(
            NBOOT
        )

        for group in groups:
            draws = rng.choice(
                group,
                size=(
                    NBOOT,
                    len(group)
                ),
                replace=True
            )

            simulated += (
                diff[draws].sum(axis=1)
            )

        bootstrap[ci] = (
            simulated / len(yc)
        )

    mean = float(
        np.mean(gains)
    )

    ci_lo, ci_hi = np.quantile(
        bootstrap.mean(axis=0),
        [0.025, 0.975]
    )

    jackknife = [
        float(
            np.mean(
                np.delete(gains, i)
            )
        )
        for i in range(4)
    ]

    comparisons.append({
        "comparator": comp,
        "mean_equal_cohort_nll_gain": mean,
        "bootstrap_ci_low": float(ci_lo),
        "bootstrap_ci_high": float(ci_hi),
        "cohorts_gain_positive": int(
            np.sum(
                np.asarray(gains) > 0
            )
        ),
        "jackknife_min": min(jackknife),
        "jackknife_max": max(jackknife)
    })


OUT.mkdir(
    parents=True,
    exist_ok=True
)

RUN = OUT / (
    datetime.now(
        timezone.utc
    ).strftime("%Y%m%dT%H%M%SZ_")
    + uuid.uuid4().hex[:8]
)

RUN.mkdir(exist_ok=False)

summary.reset_index().to_csv(
    RUN / "EQUAL_COHORT_MODEL_METRICS.csv",
    index=False
)

met.to_csv(
    RUN / "PER_COHORT_MODEL_METRICS.csv",
    index=False
)

pd.DataFrame(
    comparisons
).to_csv(
    RUN / "PAIRED_NLL_COMPARISONS.csv",
    index=False
)

pd.DataFrame(
    per_source
).to_csv(
    RUN / "PER_COHORT_PAIRED_COMPARISONS.csv",
    index=False
)

pd.DataFrame(
    row_dump
).to_csv(
    RUN / "MODEL_PREDICTIONS.csv",
    index=False
)

(
    RUN / "FITTED_CONTROL_PARAMETERS.json"
).write_text(
    json.dumps(
        fitted_params, indent=2
    ),
    encoding="utf-8"
)

protocol = {
    "phase": 28,
    "analysis": (
        "Exploratory development-only "
        "capacity-matched comparison"
    ),
    "cohort_count": 4,
    "n_total": 843,
    "external_test_used": False,
    "original_cordib_model": (
        "Archived Phase14 q_residual, unchanged"
    ),
    "frozen_ablation": (
        "Conditional interaction terms zeroed "
        "without refitting"
    ),
    "controls": (
        "Same frozen anchor; six learned "
        "correction coefficients; bounded tanh; "
        "same optimizer, epochs and L2 penalty. "
        "Basis scaling differs by construction."
    ),
    "additional_control": (
        "Equal probability ensemble with "
        "development-only temperature fitting"
    ),
    "primary_metric": (
        "Equal-cohort mean paired multiclass NLL"
    ),
    "bootstrap": (
        "3000 within-cohort, class-stratified "
        "paired resamples; conditional on "
        "historically exposed development cohorts"
    ),
    "inference": (
        "Residual models use target-cohort "
        "unlabeled quantile transformation; "
        "calibrated mean uses raw probabilities"
    ),
    "interpretation": (
        "Exploratory comparisons only; "
        "not fresh confirmatory validation"
    ),
    "input_sha256": {
        str(p): checksum(p)
        for p in inputs
    },
    "checkpoint_replay_max_error": replay_max
}

(
    RUN / "AUDIT_PROTOCOL.json"
).write_text(
    json.dumps(
        protocol, indent=2
    ),
    encoding="utf-8"
)

(
    RUN / "RUN_MANIFEST.json"
).write_text(
    json.dumps(
        {
            p.name: checksum(p)
            for p in RUN.iterdir()
            if p.is_file()
        },
        indent=2
    ),
    encoding="utf-8"
)

print(
    "PHASE28: CONTROLLED_MECHANISM_COMPARISON_COMPLETE"
)

print(
    "ARCHIVED_CHECKPOINT_REPLAY_MAX_ERROR:",
    f"{replay_max:.3e}"
)

print("EQUAL_SOURCE_MODEL_METRICS:")

print(
    summary.round(5).to_string()
)

print(
    "PAIRED_NLL_GAIN_VS_CORDIB: "
    "positive favors CoRDiB"
)

print(
    pd.DataFrame(
        comparisons
    ).round(5).to_string(index=False)
)

print(
    "PER_COHORT_PAIRED_NLL_GAIN:"
)

print(
    pd.DataFrame(
        per_source
    ).pivot(
        index="cohort",
        columns="comparator",
        values="nll_gain_cordib"
    ).round(5).to_string()
)

print(
    "STATUS: DEVELOPMENT_EXPLORATORY_ONLY; "
    "NO_EXTERNAL_TEST_CONSUMED"
)

print("RUN:", RUN)

PHASE28: CONTROLLED_MECHANISM_COMPARISON_COMPLETE
ARCHIVED_CHECKPOINT_REPLAY_MAX_ERROR: 2.776e-16
EQUAL_SOURCE_MODEL_METRICS:
                             nll  macro_f1  macro_auc    brier
model                                                         
q_anchor                 0.92097   0.52502    0.80660  0.56848
q_static                 0.91314   0.52567    0.81001  0.56231
q_residual               0.90927   0.54525    0.81064  0.55901
CORDIB_INTERACTIONS_OFF  0.91346   0.51998    0.81037  0.56246
D_ONLY_6                 0.91505   0.53563    0.80987  0.56285
GENERIC_CONDITIONAL_6    0.91304   0.54065    0.80895  0.56102
AFFINE_STACK_6           0.91097   0.54448    0.81057  0.55954
ANCHOR_NONLINEAR_6       0.91797   0.52403    0.80548  0.56716
TEMP_EQUAL_MEAN          1.06010   0.45685    0.81883  0.63641
PAIRED_NLL_GAIN_VS_CORDIB: positive favors CoRDiB
             comparator  mean_equal_cohort_nll_gain  bootstrap_ci_low  bootstrap_ci_high  cohorts_gain_positive  jackknife_min  jac